# KVO_06 v1.1.1 — U-Net-guided SAM3 Musanga crown extraction on Baego

**Notebook version: `KVO_06 v1.1.1`**

**Primary observational basis:** six exhaustively digitized crown-validation areas in `ROI_Crowns_Kvo.shp`, with **all visible crowns** digitized in `Crown_Kristof.shp`.

The notebook now separates two tasks:

1. **Observational validation:** within the six crown ROIs, use *all* U-Net candidates passing the fixed probability/coherence criteria, run SAM3, and compare automatic objects against the exhaustive visible-crown reference.
2. **Baego production pilot:** outside this validation logic, the existing ranked top-500 workflow remains available as an optional production-scale pilot.

This makes the six ROIs the observational basis for evaluating the fully automatic **U-Net detection → SAM3 crown delineation** workflow. KVO_03 remains the independent species-classification validation and KVO_05 the prompt-conditioned SAM3 boundary benchmark.


# Table of contents

Colab's **Table of contents** sidebar also lists these Markdown headers automatically. Select any heading there to jump to its block.

- [0R — Resume a new Colab session](#0r-resume-a-new-colab-session)
- [0S — Restore runner without running original SAM3 inference](#0s-restore-runner-without-running-original-sam3-inference)
- [0 — Host environment and configuration](#0-host-environment-and-configuration)
- [1 — Extract coherent U-Net candidates](#1-extract-coherent-u-net-candidates)
- [2 — Isolated official SAM3 environment](#2-isolated-official-sam3-environment)
- [3 — Hugging Face authentication](#3-hugging-face-authentication)
- [4 — SAM3 delineation of U-Net candidates](#4-sam3-delineation-of-u-net-candidates)
- [5 — Convert SAM3 masks to georeferenced candidate crown polygons](#5-convert-sam3-masks-to-georeferenced-candidate-crown-polygons)
- [6 — Add object-level U-Net probability summaries](#6-add-object-level-u-net-probability-summaries)
- [7 — Primary observational validation against Crown_Kristof](#7-primary-observational-validation-against-crown_kristof)
- [8 — Diagnostics and production summary](#8-diagnostics-and-production-summary)
- [9 — Interpretation / next step](#9-interpretation-next-step)
- [10 — Experiment A: adaptive U-Net-guided SAM3 prompts](#10-experiment-a-adaptive-u-net-guided-sam3-prompts)
- [11 — Adaptive crown agreement and reproducibility](#11-adaptive-crown-agreement-and-reproducibility)
- [12 — Experiment B: species-independent RGB crown candidates](#12-experiment-b-species-independent-rgb-crown-candidates)
- [13 — Generic RGB-guided SAM3 and U-Net attribution](#13-generic-rgb-guided-sam3-and-u-net-attribution)
- [14 — Revised all-crown reference and Baego DSM audit](#14-revised-all-crown-reference-and-baego-dsm-audit)
- [15 — Multi-scale DSM candidate detection and ranked RGB fusion](#15-multi-scale-dsm-candidate-detection-and-ranked-rgb-fusion)
- [16 — Sensitivity and crown-size-specific coverage](#16-sensitivity-and-crown-size-specific-coverage)
- [17–19 — New RGB/DSM and SAM3 crown-discovery experiments](#1719-new-rgbdsm-and-sam3-crown-discovery-experiments)
- [20 — Multiscale SAM3 pilot (40 m RGB tiles)](#20-multiscale-sam3-pilot-40-m-rgb-tiles)
- [20B — Actual multiscale RGB/SAM3 visual QA](#20b-actual-multiscale-rgbsam3-visual-qa)
- [21 — DSM-assisted SAM3 crown quality assessment](#21-dsm-assisted-sam3-crown-quality-assessment)
- [22 — Full ROI RGB-only 8 m SAM3 and full manual-outline validation](#22-full-roi-rgb-only-8-m-sam3-and-full-manual-outline-validation)
- [22V — Statistical validation (no GPU)](#22v-statistical-validation-no-gpu)
- [23 — Statistical proposal-selection benchmark (CPU only)](#23-statistical-proposal-selection-benchmark-cpu-only)
- [24 — GIS topology and crown geometry optimization (CPU only)](#24-gis-topology-and-crown-geometry-optimization-cpu-only)
- [25 — Learn to rank SAM3 crown proposals (CPU only)](#25-learn-to-rank-sam3-crown-proposals-cpu-only)
- [26 — Proposal-budget curves and overlap-aware selection (CPU only)](#26-proposal-budget-curves-and-overlap-aware-selection-cpu-only)
- [27 — Stricter one-to-one training labels and model comparison (CPU only)](#27-stricter-one-to-one-training-labels-and-model-comparison-cpu-only)
- [28 — Multiregion SAM3 crown discovery (same zero-shot RGB configuration)](#28-multiregion-sam3-crown-discovery-same-zero-shot-rgb-configuration)
- [29 — Six-ROI crown proposal classifiers: RGB versus RGB + DSM (CPU only)](#29-six-roi-crown-proposal-classifiers-rgb-versus-rgb--dsm-cpu-only)
- [30 — SAM3 crown fine-tuning pilot: export masks and verify official training setup](#30-sam3-crown-fine-tuning-pilot-export-masks-and-verify-official-training-setup)
- [30B — SAM3 fine-tuning smoke test](#30b-sam3-fine-tuning-smoke-test-single-cell)

- [31 — SAM3 two-epoch crown fine-tuning](#31-sam3-two-epoch-crown-fine-tuning-training-only)

- [32 — Paired SAM3 evaluation on ROI 4](#32-paired-sam3-evaluation-on-roi-4)

- [33 — ROI 4 classifier transfer and polygon examples](#33-roi-4-classifier-transfer-and-crown-polygon-examples)

- [34 — Pilot alternative SAM3 geometric prompts](#34-pilot-grid-versus-centre-proxy-versus-adaptive-box-prompts)


## 0R — Resume a new Colab session

Run **0R**, then **Block 2 only if the SAM3 environment is missing**, then **0S**, then **Blocks 10–13**. Skip Blocks 1 and 3–9. Saved Drive masks are preserved.


In [ ]:
# BLOCK 0R — Fresh runtime resume (no original inference)
import os,sys,subprocess,shutil,warnings
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import Point,shape
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive,userdata
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
if not ROOT.exists():drive.mount("/content/drive")
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3";RAW=OUT/"sam3_raw"
RASTER=GEO/"Baego_georef.tif";PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
CROWN_REF=AD/"Crown_Kristof.shp";CROWN_ROI=AD/"ROI_Crowns_Kvo.shp"
CHIP_SIZE_M=24.;MIN_PRED_AREA_M2=2.;MAX_PRED_AREA_M2=500.
manifest=pd.read_csv(OUT/"candidate_manifest.csv")
roi=gpd.read_file(CROWN_ROI)
if roi.crs is None:raise ValueError("ROI missing CRS")
with rasterio.open(RASTER) as src:
    if roi.crs!=src.crs:roi=roi.to_crs(src.crs)
roi_union=roi.geometry.union_all()
SAMPY="/content/sam3env/bin/python";RUNNER=Path("/content/run_sam3_kvo06.py")
env=os.environ.copy();token=userdata.get("HF_TOKEN")
if token:env["HF_TOKEN"]=token
print("Saved candidates:",len(manifest),"existing SAM3 masks:",len(list(RAW.glob("cand_*.npz"))))
print("SAM3 env:",Path(SAMPY).exists())


## 0S — Restore runner without running original SAM3 inference

Recreates only the Python runner script that was previously embedded in Block 4.


In [ ]:
# BLOCK 0S — Runner definition only
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')

print('SAM3 runner restored:',RUNNER)


## 0 — Host environment and configuration

Use a **GPU runtime**. KVO_06 reuses the same isolated official SAM3 setup as KVO_05.


In [ ]:
import sys, subprocess, os, json, random, warnings, gc, shutil, math
from pathlib import Path
print("Host Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)
!pip -q install -U "numpy>=2,<2.6" "huggingface_hub<2" geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm pillow


In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape, Point
from tqdm.auto import tqdm
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT))

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3"
CHIPS=OUT/"chips"; RAW=OUT/"sam3_raw"; VIS=OUT/"examples"
for p in [OUT,CHIPS,RAW,VIS]: p.mkdir(parents=True,exist_ok=True)

RASTER=GEO/"Baego_georef.tif"
PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
CROWN_REF=AD/"Crown_Kristof.shp"
CROWN_ROI=AD/"ROI_Crowns_Kvo.shp"
assert RASTER.exists(),RASTER
assert PROB.exists(),PROB
assert CROWN_REF.exists(),CROWN_REF
assert CROWN_ROI.exists(),CROWN_ROI

# Candidate extraction. This is intentionally conservative and computationally cheap.
PROB_THRESHOLD=0.90
SCAN_WINDOW_PX=2048
PEAK_MIN_DISTANCE_M=10.0
CONTEXT_RADIUS_M=4.0       # score support around each peak
MIN_CONTEXT_MEAN=0.60      # reject isolated hot pixels
VALIDATION_MODE=True        # primary KVO_06 observational experiment
MAX_CANDIDATES=500         # used only if VALIDATION_MODE=False
CHIP_SIZE_M=24.0          # local RGB context around each U-Net candidate
BOX_SIZE_M=10.0           # SAM3 geometric prompt centered on probability peak
MIN_PRED_AREA_M2=2.0
MAX_PRED_AREA_M2=500.0
FORCE_CHIPS=False
FORCE_SAM=False
print("OUT:",OUT)


## 1 — Extract coherent U-Net candidates

Candidate detection: probability ≥0.90, local-context support, then 10 m spatial NMS.

**Validation mode (default):** retain every qualifying candidate whose point lies within the six polygons of `ROI_Crowns_Kvo.shp`. There is no top-N confidence cap inside the observational areas. RGB chips and SAM3 are generated only for those candidates.

**v0.8:** the complete Block 1 is resume-safe: candidate tuples are converted to the downstream dataframe/manifest format within the same block, and the ROI shapefile is read with SHX restoration enabled for robust Drive-mounted shapefiles.


In [ ]:
# v0.8 — coherent candidate extraction + ROI-filtered chip/manifest preparation.
# Pass 1: large-window scan; retain the strongest pixel per 10-m metric grid cell.
with rasterio.open(PROB) as ps:
    pcrs=ps.crs; ptr=ps.transform
    gsd=float((abs(ptr.a)+abs(ptr.e))/2)
    grid_m=float(PEAK_MIN_DISTANCE_M)
    nx=math.ceil(ps.width/SCAN_WINDOW_PX); ny=math.ceil(ps.height/SCAN_WINDOW_PX)
    nwin=nx*ny; best={}
    thr=int(round(PROB_THRESHOLD*10000))
    for r0 in tqdm(range(0,ps.height,SCAN_WINDOW_PX),total=ny,desc="U-NET SCAN"):
        for c0 in range(0,ps.width,SCAN_WINDOW_PX):
            hh=min(SCAN_WINDOW_PX,ps.height-r0); ww=min(SCAN_WINDOW_PX,ps.width-c0)
            a=ps.read(1,window=Window(c0,r0,ww,hh))
            rr,cc=np.where((a!=65535) & (a>=thr))
            if len(rr)==0: continue
            vals=a[rr,cc].astype(np.float32)/10000.0
            gr=rr+r0; gc=cc+c0
            xs,ys=rasterio.transform.xy(ptr,gr,gc,offset="center")
            xs=np.asarray(xs); ys=np.asarray(ys)
            gx=np.floor(xs/grid_m).astype(np.int64); gy=np.floor(ys/grid_m).astype(np.int64)
            order=np.argsort(vals)[::-1]; seen=set()
            for j in order:
                key=(int(gx[j]),int(gy[j]))
                if key in seen: continue
                seen.add(key)
                rec=(float(vals[j]),int(gr[j]),int(gc[j]),float(xs[j]),float(ys[j]))
                old=best.get(key)
                if old is None or rec[0]>old[0]: best[key]=rec

    print("Large raster windows scanned:",nwin)
    print("Strong coarse candidate cells:",len(best))

    # Pass 2: measure local support around each strong peak directly from probability map.
    rad=max(1,int(round(CONTEXT_RADIUS_M/gsd)))
    coherent=[]
    for peak,row,col,x,y in tqdm(best.values(),desc="CONTEXT SCORE"):
        w=Window(col-rad,row-rad,2*rad+1,2*rad+1)
        a=ps.read(1,window=w,boundless=True,fill_value=65535)
        v=a[a!=65535].astype(np.float32)/10000.0
        context_mean=float(v.mean()) if len(v) else 0.0
        context_high=float((v>=PROB_THRESHOLD).mean()) if len(v) else 0.0
        if context_mean < MIN_CONTEXT_MEAN: continue
        # Ranking rewards both a strong peak and broad local support.
        score=0.40*peak + 0.45*context_mean + 0.15*context_high
        coherent.append((score,peak,context_mean,context_high,row,col,x,y))

print("Coherent candidates before spatial NMS:",len(coherent))

# Pass 3: exact 10-m NMS using a spatial hash, highest coherent score first.
coherent.sort(key=lambda z:z[0],reverse=True)
accepted=[]; occupied={}
for rec in tqdm(coherent,desc="CANDIDATE NMS"):
    score,peak,cmean,chigh,row,col,x,y=rec
    kx=int(math.floor(x/grid_m)); ky=int(math.floor(y/grid_m)); ok=True
    for dx in (-1,0,1):
        for dy in (-1,0,1):
            for qx,qy in occupied.get((kx+dx,ky+dy),[]):
                if (x-qx)**2+(y-qy)**2 < PEAK_MIN_DISTANCE_M**2:
                    ok=False; break
            if not ok: break
        if not ok: break
    if ok:
        accepted.append(rec)
        occupied.setdefault((kx,ky),[]).append((x,y))

cand=pd.DataFrame(accepted,columns=["candidate_score","peak_prob","context_mean","context_high_frac",
                                    "row","col","x","y"])
cand=cand.sort_values("candidate_score",ascending=False).reset_index(drop=True)
n_before=len(cand)
cg=gpd.GeoDataFrame(cand,geometry=gpd.points_from_xy(cand.x,cand.y),crs=pcrs)

if VALIDATION_MODE:
    # Robust to a temporarily missing/stale .shx view on mounted Google Drive.
    with rasterio.Env(SHAPE_RESTORE_SHX='YES'):
        roi=gpd.read_file(CROWN_ROI)
    if roi.crs is None: raise ValueError("ROI_Crowns_Kvo.shp has no CRS.")
    if roi.crs!=pcrs: roi=roi.to_crs(pcrs)
    roi=roi[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
    roi["geometry"]=roi.geometry.make_valid()
    roi_union=roi.geometry.union_all()
    cg=cg[cg.geometry.intersects(roi_union)].copy()
    print("Validation ROIs:",len(roi))
    print("Qualifying U-Net candidates inside validation ROIs:",len(cg))
else:
    if MAX_CANDIDATES is not None: cg=cg.head(MAX_CANDIDATES).copy()

cg=cg.reset_index(drop=True)
cg["candidate_id"]=np.arange(len(cg),dtype=int)
cg.to_file(OUT/"Baego_UNet_candidates.gpkg",layer="candidates",driver="GPKG")
print("Peak threshold:",PROB_THRESHOLD)
print("Minimum context mean:",MIN_CONTEXT_MEAN)
print("Minimum separation m:",PEAK_MIN_DISTANCE_M)
print("Candidates after NMS:",n_before)
print("Candidates retained for SAM3:",len(cg),"(validation mode)" if VALIDATION_MODE else "(production pilot)")
display(cg[["candidate_id","candidate_score","peak_prob","context_mean","context_high_frac"]].head(10))

# Prepare one compact RGB chip per retained candidate only.
manifest=[]
with rasterio.open(RASTER) as src:
    if src.crs!=pcrs: raise ValueError("RGB and probability raster CRS differ")
    px=float((abs(src.transform.a)+abs(src.transform.e))/2)
    chip_px=max(128,int(round(CHIP_SIZE_M/px)))
    if chip_px%2: chip_px+=1
    for r in tqdm(cg.itertuples(),total=len(cg),desc="RGB CHIPS"):
        row,col=src.index(r.x,r.y)
        w=Window(col-chip_px//2,row-chip_px//2,chip_px,chip_px)
        op=CHIPS/f"cand_{r.candidate_id:05d}.jpg"
        if FORCE_CHIPS or not op.exists():
            rgb=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1).astype(np.uint8)
            Image.fromarray(rgb).save(op,quality=95)
        manifest.append(dict(candidate_id=r.candidate_id,path=str(op),x=r.x,y=r.y,
                             candidate_score=r.candidate_score,peak_prob=r.peak_prob,
                             context_mean=r.context_mean,context_high_frac=r.context_high_frac,
                             row_off=int(w.row_off),col_off=int(w.col_off),
                             height=chip_px,width=chip_px,gsd=px))
manifest=pd.DataFrame(manifest)
manifest.to_csv(OUT/"candidate_manifest.csv",index=False)
print("RGB chips:",len(manifest)," chip pixels:",chip_px)


## 2 — Isolated official SAM3 environment

In [ ]:
# SAM3 needs numpy<2, while current Colab geospatial packages need numpy>=2.
# Use uv to create a real isolated Python 3.12 environment; do NOT modify host Python.
UV=Path("/content/uv")
SAMENV=Path("/content/sam3env")
SAMREPO=Path("/content/sam3")

if not UV.exists():
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/content sh"],check=True)
    # uv installer may name the binary directly under /content.
    candidates=[Path("/content/uv"),Path("/content/uv/uv"),Path.home()/".local/bin/uv"]
    UV=next((p for p in candidates if p.exists() and p.is_file()),None)
    assert UV is not None,"uv installation failed"
else:
    UV=Path("/content/uv")
print("uv:",subprocess.check_output([str(UV),"--version"],text=True).strip())

# Recreate only the isolated SAM environment if a previous failed venv is present.
if SAMENV.exists() and not (SAMENV/"bin/python").exists():
    shutil.rmtree(SAMENV)
if not (SAMENV/"bin/python").exists():
    subprocess.run([str(UV),"venv","--python","3.12",str(SAMENV)],check=True)

SAMPY=str(SAMENV/"bin/python")
# Reuse Colab's CUDA-enabled torch wheel via explicit install in the isolated env.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "torch==2.10.0","torchvision",
                "--index-url","https://download.pytorch.org/whl/cu128"],check=True)

if not SAMREPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/facebookresearch/sam3.git",str(SAMREPO)],check=True)
else:
    subprocess.run(["git","-C",str(SAMREPO),"pull","--ff-only"],check=True)

# Official SAM3 currently requires numpy>=1.26,<2. Keep that constraint inside this env only.
subprocess.run([str(UV),"pip","install","--python",SAMPY,"setuptools<82"],check=True)
# The image builder imports modules that use packages listed by Meta under
# notebook/dev extras (not only the minimal core dependencies). Install the
# official extras rather than discovering them one-by-one at runtime.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "-e",str(SAMREPO)+"[notebooks,dev]"],check=True)

subprocess.run([SAMPY,"-c",
    "import sys,numpy,torch,sam3; print('SAM Python',sys.version.split()[0],'numpy',numpy.__version__,'torch',torch.__version__,'CUDA',torch.cuda.is_available())"],
    check=True)
print("HOST unchanged: Python",sys.version.split()[0],"numpy",np.__version__,"rasterio",rasterio.__version__)


## 3 — Hugging Face authentication

In [ ]:
from google.colab import userdata
try: HF_TOKEN=userdata.get("HF_TOKEN")
except Exception: HF_TOKEN=None
assert HF_TOKEN,"Add your Hugging Face read token to Colab Secrets as HF_TOKEN."
print("HF_TOKEN found (value not displayed).")


## 4 — SAM3 delineation of U-Net candidates

SAM3 is now used as a **local delineator**, not a wall-to-wall detector. Each U-Net candidate supplies a small approximate box centred on the probability peak. SAM3 receives the local RGB chip plus that geometric prompt and returns a crown mask.

This is the computational advantage of the revised workflow: SAM3 is run only where the U-Net already provides evidence for Musanga.


In [ ]:
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')
env=os.environ.copy(); env["HF_TOKEN"]=HF_TOKEN
cmd=[SAMPY,"-u",str(RUNNER),str(OUT/"candidate_manifest.csv"),str(RAW),str(BOX_SIZE_M),"1" if FORCE_SAM else "0"]
print("Starting targeted SAM3 delineation with live output...")
p=subprocess.run(cmd,env=env)
print("Return code:",p.returncode)
if p.returncode!=0: raise RuntimeError("SAM3 delineation subprocess failed; error printed above.")


## 5 — Convert SAM3 masks to georeferenced candidate crown polygons

For each candidate we retain the SAM3 mask component containing the U-Net peak (the chip centre), convert it to map coordinates, and attach the original U-Net peak probability. Implausibly small/large objects are discarded.


In [ ]:
pred=[]
with rasterio.open(RASTER) as src:
    for r in tqdm(manifest.itertuples(),total=len(manifest),desc="POLYGONIZE"):
        z=np.load(RAW/f"cand_{int(r.candidate_id):05d}.npz")
        masks=z["masks"]; scores=z["scores"]
        if len(masks)==0: continue
        # Prefer highest-score mask containing chip centre.
        order=np.argsort(scores)[::-1]; chosen=None
        for j in order:
            m=masks[j].astype(np.uint8)
            if m.shape!=(r.height,r.width):
                m=np.array(Image.fromarray(m).resize((r.width,r.height),resample=Image.Resampling.NEAREST))
            if m[r.height//2,r.width//2]>0: chosen=(m,float(scores[j])); break
        if chosen is None: continue
        m,score=chosen
        wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
        geoms=[]
        for gj,val in shapes(m,mask=m.astype(bool),transform=wt):
            if val==1: geoms.append(shape(gj))
        if not geoms: continue
        pt=Point(r.x,r.y)
        geom=max((q for q in geoms if q.contains(pt) or q.touches(pt)),key=lambda q:q.area,default=None)
        if geom is None: continue
        area=geom.area
        if not (MIN_PRED_AREA_M2<=area<=MAX_PRED_AREA_M2): continue
        pred.append(dict(candidate_id=int(r.candidate_id),peak_prob=float(r.peak_prob),
                         sam_score=score,crown_area_m2=area,geometry=geom))
crowns=gpd.GeoDataFrame(pred,crs=src.crs)
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print("Candidates:",len(manifest))
print("Valid SAM3 crown objects:",len(crowns))
display(crowns.drop(columns="geometry").head())


## 6 — Add object-level U-Net probability summaries

The peak probability detects the candidate; here we also summarize the KVO_04 probability raster inside each SAM3 crown. These are useful object-level attributes for later filtering and satellite aggregation.


In [ ]:
from rasterio.mask import mask as rio_mask
rows=[]
with rasterio.open(PROB) as ps:
    for r in tqdm(crowns.itertuples(),total=len(crowns),desc="OBJECT PROBABILITY"):
        arr,_=rio_mask(ps,[r.geometry],crop=True,filled=False,indexes=1)
        vals=np.asarray(arr.compressed(),dtype=np.float32)
        vals=vals[vals!=65535]/10000.0
        rows.append(dict(candidate_id=r.candidate_id,
                         mean_prob=float(np.mean(vals)) if len(vals) else np.nan,
                         median_prob=float(np.median(vals)) if len(vals) else np.nan,
                         p90_prob=float(np.quantile(vals,.9)) if len(vals) else np.nan))
stats=pd.DataFrame(rows)
crowns=crowns.merge(stats,on="candidate_id",how="left")
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print(crowns[["peak_prob","mean_prob","median_prob","crown_area_m2"]].describe())


## 7 — Primary observational validation against Crown_Kristof

`Crown_Kristof.shp` contains **all visible crowns digitized within the six areas of `ROI_Crowns_Kvo.shp`**. This is therefore the primary observational basis for KVO_06.

The analysis is restricted to the six ROIs and reports both directions:

- **Reference recovery:** for every digitized visible crown, the best overlapping automatic U-Net→SAM3 object and its IoU.
- **Automatic-object matching:** for every automatic crown, the best overlapping digitized crown and its IoU.

This allows unmatched automatic objects *inside these exhaustively digitized visible-crown areas* to be identified explicitly, while keeping the geometric match statistics separate from the U-Net species-probability evidence.


In [ ]:
from shapely.validation import make_valid

def safe_iou(a,b):
    try:
        a=make_valid(a); b=make_valid(b)
        inter=a.intersection(b).area
        if inter<=0: return 0.0
        union=a.union(b).area
        return float(inter/union) if union>0 else 0.0
    except Exception:
        return 0.0

ref=gpd.read_file(CROWN_REF)
roi=gpd.read_file(CROWN_ROI)
if ref.crs is None:
    print("Assigning EPSG:32635 to Crown_Kristof (Baego digitizing CRS)")
    ref=ref.set_crs("EPSG:32635")
if roi.crs is None: raise ValueError("ROI_Crowns_Kvo missing CRS")
if ref.crs!=crowns.crs: ref=ref.to_crs(crowns.crs)
if roi.crs!=crowns.crs: roi=roi.to_crs(crowns.crs)
ref=ref[ref.geometry.notna() & ~ref.geometry.is_empty].copy()
roi=roi[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
ref["geometry"]=ref.geometry.apply(make_valid)
roi["geometry"]=roi.geometry.apply(make_valid)
roi_union=roi.geometry.union_all()

# Enforce the observational footprint for both datasets.
ref=ref[ref.geometry.intersects(roi_union)].copy().reset_index(drop=True)
auto=crowns[crowns.geometry.intersects(roi_union)].copy().reset_index(drop=True)
print("Validation ROIs:",len(roi))
print("Digitized visible reference crowns:",len(ref))
print("Automatic SAM3 crowns in ROIs:",len(auto))

# Automatic object -> best reference crown.
ref_sidx=ref.sindex
auto_rows=[]
for r in tqdm(auto.itertuples(),total=len(auto),desc="AUTO -> REFERENCE"):
    ids=list(ref_sidx.query(r.geometry,predicate="intersects"))
    pairs=[(int(i),safe_iou(r.geometry,ref.iloc[i].geometry)) for i in ids]
    bi,bv=max(pairs,key=lambda z:z[1],default=(-1,0.0))
    auto_rows.append(dict(candidate_id=int(r.candidate_id),best_ref_index=bi,best_ref_iou=bv))
auto_qa=pd.DataFrame(auto_rows)
auto_qa.to_csv(OUT/"validation_auto_to_reference.csv",index=False)

# Reference crown -> best automatic object.
auto_sidx=auto.sindex
ref_rows=[]
for i,r in tqdm(ref.iterrows(),total=len(ref),desc="REFERENCE -> AUTO"):
    ids=list(auto_sidx.query(r.geometry,predicate="intersects"))
    pairs=[(int(j),safe_iou(r.geometry,auto.iloc[j].geometry)) for j in ids]
    bj,bv=max(pairs,key=lambda z:z[1],default=(-1,0.0))
    cid=int(auto.iloc[bj].candidate_id) if bj>=0 else -1
    ref_rows.append(dict(ref_index=int(i),best_candidate_id=cid,best_auto_iou=bv))
ref_qa=pd.DataFrame(ref_rows)
ref_qa.to_csv(OUT/"validation_reference_to_auto.csv",index=False)

nref=len(ref); nauto=len(auto)
print("\nREFERENCE RECOVERY")
print("Any overlap:",int((ref_qa.best_auto_iou>0).sum()),"/",nref)
print("IoU >= 0.5:",int((ref_qa.best_auto_iou>=0.5).sum()),"/",nref)
print("IoU >= 0.7:",int((ref_qa.best_auto_iou>=0.7).sum()),"/",nref)
if (ref_qa.best_auto_iou>0).any():
    print("Median IoU among overlapping reference crowns:",
          float(ref_qa.loc[ref_qa.best_auto_iou>0,"best_auto_iou"].median()))

print("\nAUTOMATIC OBJECT MATCHING")
print("Any reference overlap:",int((auto_qa.best_ref_iou>0).sum()),"/",nauto)
print("IoU >= 0.5:",int((auto_qa.best_ref_iou>=0.5).sum()),"/",nauto)
print("IoU >= 0.7:",int((auto_qa.best_ref_iou>=0.7).sum()),"/",nauto)
if (auto_qa.best_ref_iou>0).any():
    print("Median IoU among overlapping automatic crowns:",
          float(auto_qa.loc[auto_qa.best_ref_iou>0,"best_ref_iou"].median()))


## 8 — Diagnostics and production summary

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
if len(crowns):
    ax.scatter(crowns.peak_prob,crowns.mean_prob,s=12,alpha=.6)
ax.set_xlabel("U-Net candidate peak probability"); ax.set_ylabel("Mean U-Net probability inside SAM3 crown")
ax.set_title("Baego: detector → delineator consistency")
plt.show()

summary={
    "probability_threshold":PROB_THRESHOLD,
    "minimum_candidate_separation_m":PEAK_MIN_DISTANCE_M,
    "n_candidates":int(len(manifest)),
    "n_valid_sam3_crowns":int(len(crowns)),
    "median_crown_area_m2":float(crowns.crown_area_m2.median()) if len(crowns) else None,
    "median_object_mean_probability":float(crowns.mean_prob.median()) if len(crowns) else None,
}
(OUT/"summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


## 9 — Interpretation / next step

KVO_06 now uses the six exhaustively digitized crown areas as its **observational basis**.

- **KVO_03:** independent leave-one-site-out evidence for U-Net species classification.
- **KVO_04:** final continuous Musanga probability maps.
- **KVO_05:** prompt-conditioned SAM3 crown-boundary benchmark.
- **KVO_06:** fully automatic U-Net → SAM3 workflow evaluated inside six areas where all visible crowns were digitized.

The production-scale Baego inventory should only be expanded after examining recovery, matching IoU, and unmatched automatic objects in these six validation areas.


## 10 — Experiment A: adaptive U-Net-guided SAM3 prompts

Uses local 0.60 probability components for approximate size and four prompt bins (10/14/18/22m). The original 0.90 candidate threshold is retained for seeds. This is a first sensitivity experiment, not final calibration.


In [ ]:

# BLOCK 10 — Adaptive U-Net box prompts; reuses existing chips and SAM3 runner.
from scipy.ndimage import label as ndi_label
from shapely.validation import make_valid
EXP=OUT/"experiments_v09"; EXP.mkdir(exist_ok=True,parents=True)
MAN=pd.read_csv(OUT/"candidate_manifest.csv")
BOX_VALUES=[10.,14.,18.,22.]
def box_from_probability(r,threshold=.60,margin=1.25):
    with rasterio.open(PROB) as ds:
        rr,cc=ds.index(float(r.x),float(r.y))
        rad=int(round(15/abs(ds.transform.a)))
        a=ds.read(1,window=Window(cc-rad,rr-rad,2*rad+1,2*rad+1),boundless=True,fill_value=65535)
        good=(a!=65535)&(a>=int(threshold*10000))
        labs,num=ndi_label(good)
        k=labs[rad,rad]
        if k==0:return 14.
        ys,xs=np.where(labs==k)
        extent=max((xs.max()-xs.min()+1)*abs(ds.transform.a),(ys.max()-ys.min()+1)*abs(ds.transform.e))
        return float(np.clip(extent*margin,8,22))
MAN["adaptive_box_m"]=[box_from_probability(r) for r in MAN.itertuples()]
MAN.to_csv(EXP/"adaptive_boxes.csv",index=False)
print(MAN.adaptive_box_m.describe())
# Existing runner accepts one box size per call; group adaptive boxes into 4 bins,
# retaining original candidate_id and output paths for reproducibility.
MAN["box_group"]=MAN.adaptive_box_m.apply(lambda x:min(BOX_VALUES,key=lambda b:abs(b-x)))
adaptive_raw=EXP/"adaptive_raw";adaptive_raw.mkdir(exist_ok=True)
for b,g in MAN.groupby("box_group"):
    mf=EXP/f"adaptive_manifest_{int(b)}m.csv";g.to_csv(mf,index=False)
    p=subprocess.run([SAMPY,"-u",str(RUNNER),str(mf),str(adaptive_raw),str(b),"0"],env=env)
    if p.returncode:raise RuntimeError(f"SAM3 failed for {b} m")
print("Adaptive masks ready:",len(list(adaptive_raw.glob("cand_*.npz"))))


## 11 — Adaptive crown agreement and reproducibility

Compare against manual interpreted boundaries; no species-level validation claim.


In [ ]:

# BLOCK 11 — Shared polygonization and manual–automatic agreement.
from rasterio.features import shapes
from shapely.geometry import Point,shape
from PIL import Image
def polygons_from_masks(manifest_df,raw_dir):
    records=[]
    with rasterio.open(RASTER) as src:
        for r in tqdm(manifest_df.itertuples(),total=len(manifest_df),desc="POLYGONIZE"):
            p=raw_dir/f"cand_{int(r.candidate_id):05d}.npz"
            if not p.exists():continue
            z=np.load(p);masks=z["masks"];scores=z["scores"]
            if not len(masks):continue
            for j in np.argsort(scores)[::-1]:
                m=masks[j].astype("uint8")
                if m.ndim==3:m=m[0]
                if m.shape!=(r.height,r.width):
                    m=np.array(Image.fromarray(m).resize((r.width,r.height),Image.Resampling.NEAREST))
                if not m[r.height//2,r.width//2]:continue
                wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
                pt=Point(r.x,r.y)
                geoms=[shape(g) for g,v in shapes(m,mask=m.astype(bool),transform=wt) if v==1]
                geom=max((g for g in geoms if g.intersects(pt)),key=lambda g:g.area,default=None)
                if geom is not None and MIN_PRED_AREA_M2<=geom.area<=MAX_PRED_AREA_M2:
                    records.append(dict(candidate_id=int(r.candidate_id),area_m2=geom.area,geometry=make_valid(geom)))
                break
    return gpd.GeoDataFrame(records,geometry="geometry",crs=rasterio.open(RASTER).crs)
def agreement(auto,reference):
    if auto.crs!=reference.crs:auto=auto.to_crs(reference.crs)
    idx=reference.sindex;rows=[]
    for a in auto.itertuples():
        best=(-1,0.,0.,np.nan)
        for j in idx.query(a.geometry,predicate="intersects"):
            g=reference.iloc[j].geometry
            inter=a.geometry.intersection(g).area
            union=a.geometry.union(g).area
            iou=inter/union if union else 0.
            if iou>best[1]:best=(int(j),iou,inter,g.area)
        j,iou,inter,refarea=best
        rows.append(dict(candidate_id=a.candidate_id,manual_index=j,iou=iou,area_ratio=a.geometry.area/refarea if j>=0 else np.nan,manual_coverage=inter/refarea if j>=0 else np.nan,sam_inside_manual=inter/a.geometry.area if a.geometry.area else np.nan))
    return pd.DataFrame(rows)
reference=gpd.read_file(CROWN_REF)
if reference.crs is None:reference=reference.set_crs("EPSG:32635")
reference=reference[reference.geometry.intersects(roi_union)].copy()
adaptive_crowns=polygons_from_masks(MAN,adaptive_raw)
adaptive_qa=agreement(adaptive_crowns,reference)
adaptive_qa.to_csv(EXP/"adaptive_agreement.csv",index=False)
adaptive_crowns.to_file(EXP/"adaptive_crowns.gpkg",driver="GPKG")
print("Adaptive valid crowns:",len(adaptive_crowns),"overlap:",(adaptive_qa.iou>0).sum(),"median matched IoU:",adaptive_qa.loc[adaptive_qa.iou>0,"iou"].median())
# Reproducibility: compare the original 10m mask files against the sensitivity-run 10m files.
original_ids={int(p.stem.split("_")[-1]) for p in RAW.glob("cand_*.npz")}
print("Original 10m raw masks:",len(original_ids),"adaptive masks:",len(list(adaptive_raw.glob("cand_*.npz"))))


## 12 — Experiment B: species-independent RGB crown candidates

**Pilot approximation:** RGB local-extrema proposals followed by SAM3 geometric prompts. This is not native SAM3 automatic discovery and may miss crowns. The 846 manual crowns are observational comparisons only.


In [ ]:

# BLOCK 12 — Species-independent crown proposals within the six ROIs.
# Generic RGB candidate generation via local canopy brightness/texture extrema.
# This is a pilot heuristic, NOT native automatic SAM3 discovery; compare empirically.
from scipy.ndimage import gaussian_filter,maximum_filter
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
generic=[]
with rasterio.open(RASTER) as src:
    for k,region in enumerate(roi.geometry):
        arr,tr=rio_mask(src,[mapping(region)],crop=True,indexes=[1,2,3],filled=True,nodata=0)
        rgb=arr.astype("float32").mean(axis=0)
        # Local RGB texture peaks, spaced ~5m; exclude masked background.
        px=abs(src.transform.a); sep=max(3,int(round(5/px)))
        smooth=gaussian_filter(rgb,sigma=max(2,sep/5))
        peak=(smooth==maximum_filter(smooth,size=2*sep+1))&(rgb>20)
        ys,xs=np.where(peak)
        for y,x in zip(ys,xs):
            xx,yy=rasterio.transform.xy(tr,int(y),int(x))
            if region.covers(Point(xx,yy)):generic.append((k,xx,yy))
print("Generic RGB candidates:",len(generic))
# Build independent chips, same manifest contract as Block 4.
grows=[]
with rasterio.open(RASTER) as src:
    px=abs(src.transform.a);size=max(128,int(round(CHIP_SIZE_M/px)));size+=size%2
    for i,(region,x,y) in enumerate(tqdm(generic,desc="GENERIC RGB CHIPS")):
        row,col=src.index(x,y);w=Window(col-size//2,row-size//2,size,size)
        out=src.read([1,2,3],window=w,boundless=True,fill_value=0).transpose(1,2,0)
        path=EXP/f"generic_{i:05d}.png"
        if not path.exists():Image.fromarray(out.astype("uint8")).save(path)
        grows.append(dict(candidate_id=i,region=region,x=x,y=y,row_off=int(w.row_off),col_off=int(w.col_off),width=size,height=size,gsd=px,path=str(path)))
generic_manifest=pd.DataFrame(grows)
generic_manifest.to_csv(EXP/"generic_manifest.csv",index=False)
print("Generic chips:",len(generic_manifest))


## 13 — Generic RGB-guided SAM3 and U-Net attribution


In [ ]:

# BLOCK 13 — Species-independent SAM3 segmentation + agreement.
# Uses 14m generic prompts; independent of U-Net seed threshold.
generic_raw=EXP/"generic_raw";generic_raw.mkdir(exist_ok=True)
p=subprocess.run([SAMPY,"-u",str(RUNNER),str(EXP/"generic_manifest.csv"),str(generic_raw),"14","0"],env=env)
if p.returncode:raise RuntimeError("Generic SAM3 failed")
generic_crowns=polygons_from_masks(generic_manifest,generic_raw)
generic_qa=agreement(generic_crowns,reference)
generic_qa.to_csv(EXP/"generic_agreement.csv",index=False)
generic_crowns.to_file(EXP/"generic_crowns.gpkg",driver="GPKG")
print("Generic valid:",len(generic_crowns),"overlapping manual:",int((generic_qa.iou>0).sum()),"median matched IoU:",generic_qa.loc[generic_qa.iou>0,"iou"].median())
# Overlay species probability only AFTER delineation.
from rasterio.mask import mask as rio_mask
with rasterio.open(PROB) as ps:
    vals=[]
    for g in generic_crowns.geometry:
        a,_=rio_mask(ps,[mapping(g)],crop=True,indexes=1,filled=False)
        v=np.asarray(a.compressed(),dtype="float32");v=v[v!=65535]/10000
        vals.append(float(v.mean()) if len(v) else np.nan)
generic_crowns["musanga_mean_prob"]=vals
generic_crowns.to_file(EXP/"generic_crowns_with_probability.gpkg",driver="GPKG")
display(pd.DataFrame([{"method":"adaptive","valid":len(adaptive_crowns),"matched":int((adaptive_qa.iou>0).sum()),"median_iou":adaptive_qa.loc[adaptive_qa.iou>0,"iou"].median()},{"method":"generic_rgb","valid":len(generic_crowns),"matched":int((generic_qa.iou>0).sum()),"median_iou":generic_qa.loc[generic_qa.iou>0,"iou"].median()}]))


## 14 — Revised all-crown reference and Baego DSM audit

Run **0R** first in a new session; **Blocks 14–16 do not need SAM3, GPU, or Blocks 1–13**. This reads the revised Crown_Kristof shapefile afresh and checks DSM/RGB alignment.


In [ ]:
# BLOCK 14 — Audit revised all-crown reference and DSM (no SAM3/GPU)
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
from scipy.ndimage import gaussian_filter,maximum_filter,minimum_filter
from shapely.validation import make_valid
DSM=GEO/"Baego_DSM.tif"
assert DSM.exists(),f"DSM not found: {DSM}"
reference_v2=gpd.read_file(CROWN_REF)
if reference_v2.crs is None:
    print("Crown_Kristof CRS absent: assigning EPSG:32635 (verify digitizing CRS)")
    reference_v2=reference_v2.set_crs("EPSG:32635")
with rasterio.open(RASTER) as rgb,rasterio.open(DSM) as dsm:
    reference_v2=reference_v2.to_crs(rgb.crs)
    roi_v2=gpd.read_file(CROWN_ROI).to_crs(rgb.crs)
    print("RGB:",rgb.crs,rgb.res,"DSM:",dsm.crs,dsm.res)
    dx=(dsm.transform.c-rgb.transform.c)/rgb.res[0]
    dy=(dsm.transform.f-rgb.transform.f)/rgb.res[1]
    print("DSM pixel-grid compatible:",rgb.crs==dsm.crs and np.allclose(rgb.res,dsm.res,atol=1e-7,rtol=0) and np.allclose([dx,dy],np.round([dx,dy]),atol=1e-5),"offset pixels:",(dx,dy))
reference_v2=reference_v2[reference_v2.geometry.notna()&(~reference_v2.geometry.is_empty)].copy()
reference_v2["geometry"]=reference_v2.geometry.map(make_valid)
ref_roi_union=roi_v2.geometry.union_all()
reference_v2=reference_v2[reference_v2.geometry.intersects(ref_roi_union)].copy()
print("Revised crowns:",len(reference_v2),"ROIs:",len(roi_v2),"median area m2:",round(reference_v2.geometry.area.median(),1))
print("Invalid after repair:",(~reference_v2.is_valid).sum())
print("Per-ROI counts:",[int(reference_v2.intersects(g).sum()) for g in roi_v2.geometry])


## 15 — Multi-scale DSM candidate detection and ranked RGB fusion

Uses aligned **Baego_DSM.tif**, elevations in metres. Detects canopy maxima at 3 smoothing scales, ranks by local relief, tests 1.5–4m separation, preserves RGB candidates. CPU-only; no SAM3.


In [ ]:
# BLOCK 15 v2 — Multi-scale DSM maxima + ranked fusion; CPU only
# Reads aligned metre-valued DSM, preserving original RGB candidates.
from scipy.ndimage import gaussian_filter, maximum_filter, minimum_filter, label
from scipy.spatial import cKDTree
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping, Point
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from pathlib import Path

DSM = GEO/"Baego_DSM.tif"
if not DSM.exists():
    raise FileNotFoundError(f"DSM not found: {DSM}")
EXP15=OUT/"experiments_v15_dsm";EXP15.mkdir(parents=True,exist_ok=True)
if "generic_manifest" not in globals():
    options=list(OUT.rglob("*generic*manifest*.csv"))
    if not options: raise FileNotFoundError("Run Block 12 or provide the saved generic_manifest CSV.")
    generic_manifest=pd.read_csv(options[0])
    print("Restored RGB candidates from:",options[0])
rgb_points=generic_manifest[["x","y"]].dropna().drop_duplicates().copy()
rgb_points["source"]="RGB"
rgb_points["local_relief_m"]=np.nan
print("RGB candidates:",len(rgb_points))

# Use ROI crops, not full-resolution DSM. Check actual grid match and overlap.
peak_rows=[]
with rasterio.open(RASTER) as rgb, rasterio.open(DSM) as dsm:
    dx=(dsm.transform.c-rgb.transform.c)/rgb.res[0]
    dy=(dsm.transform.f-rgb.transform.f)/rgb.res[1]
    same=(rgb.crs==dsm.crs and np.allclose(rgb.res,dsm.res,atol=1e-7,rtol=0) and np.allclose([dx,dy],np.round([dx,dy]),atol=1e-5))
    print("DSM/RGB pixel-grid compatible:",same,"offset:",(dx,dy),"DSM units: metres (user confirmed)")
    if not same: raise ValueError("DSM pixel grids differ; check Snap Raster and cell size in ArcGIS.")
    if not (dsm.bounds.left<=rgb.bounds.right and dsm.bounds.right>=rgb.bounds.left and dsm.bounds.bottom<=rgb.bounds.top and dsm.bounds.top>=rgb.bounds.bottom):
        raise ValueError("DSM and RGB do not overlap.")
    gsd=abs(rgb.transform.a)
    for roi_id,geom in enumerate(roi_v2.geometry):
        masked,tr=rio_mask(dsm,[mapping(geom)],crop=True,indexes=1,filled=False)
        valid=(~np.ma.getmaskarray(masked)) & np.isfinite(masked.data)
        if not valid.any(): continue
        arr=masked.astype("float32").filled(np.nan)
        baseline=float(np.nanmedian(arr[valid]))
        arr=np.where(valid,arr,baseline)
        # Multi-scale smoothing; local relief removes slow topographic gradients.
        for smooth_m in (0.35,0.70,1.20):
            sigma=max(0.5,smooth_m/gsd)
            sm=gaussian_filter(arr,sigma=sigma)
            base=minimum_filter(sm,size=2*max(1,round(10/gsd))+1)
            relief=sm-base
            # 1.5m neighborhood to generate candidates; separation tuned downstream.
            r=max(1,round(1.5/gsd))
            peaks=(sm>=maximum_filter(sm,size=2*r+1)-1e-5)&valid&(relief>=0.35)
            # Collapse plateaus into one location rather than hundreds of equal-value pixels.
            labs,nlab=label(peaks)
            for labid in range(1,nlab+1):
                yy,xx=np.where(labs==labid)
                k=np.argmax(relief[yy,xx]); y=int(yy[k]);x=int(xx[k])
                east,north=rasterio.transform.xy(tr,y,x)
                if geom.covers(Point(east,north)):
                    peak_rows.append((roi_id,east,north,float(relief[y,x]),float(sm[y,x]),smooth_m))
        print(f"ROI {roi_id+1}: raw multiscale peaks so far {len(peak_rows)}")
peaks_df=pd.DataFrame(peak_rows,columns=["roi","x","y","local_relief_m","surface_elevation_m","smoothing_m"])
peaks_df.to_csv(EXP15/"dsm_multiscale_raw_peaks.csv",index=False)
print("Multi-scale peaks:",len(peaks_df))

# Rank strong local-relief peaks first; avoid duplicate maxima from different scales.
def greedy_spatial_nms(frame,sep_m,initial=None):
    if frame.empty:return frame.copy()
    f=frame.sort_values(["local_relief_m","smoothing_m"],ascending=[False,False]).reset_index(drop=True)
    coords=f[["x","y"]].to_numpy()
    tree=cKDTree(coords)
    blocked=np.zeros(len(f),bool)
    if initial is not None and len(initial):
        near=cKDTree(initial[["x","y"]].to_numpy()).query(coords)[0]
        blocked |= near<sep_m
    chosen=[]
    for i in range(len(f)):
        if blocked[i]:continue
        chosen.append(i)
        blocked[tree.query_ball_point(coords[i],r=sep_m)]=True
    return f.iloc[chosen].copy().reset_index(drop=True)

# Compare alternative minimum distances; preserve all RGB candidates.
fusions={}
for spacing in (1.5,2.0,2.5,3.0,4.0):
    dsm_keep=greedy_spatial_nms(peaks_df,spacing,initial=rgb_points)
    fusion=pd.concat([rgb_points,dsm_keep.assign(source="DSM")],ignore_index=True)
    fusion["candidate_id"]=np.arange(len(fusion))
    fusions[spacing]=fusion
    print(f"Spacing {spacing:.1f}m: DSM added {len(dsm_keep)}, fused {len(fusion)}")
    fusion.to_csv(EXP15/f"fused_spacing_{str(spacing).replace('.','p')}m.csv",index=False)
# Default 2m, chosen provisionally; Block 16 compares all variants.
accepted=fusions[2.0]
dsm_points=peaks_df
print("Default fused candidates:",len(accepted))


## 16 — Sensitivity and crown-size-specific coverage

Compares all candidate-spacing variants against the revised full-coverage crown annotations. Saves CSV reports; no SAM3.


In [ ]:
# BLOCK 16 v2 — Evaluate multiscale DSM fusion against revised all-crown reference
import geopandas as gpd
from scipy.spatial import cKDTree
def coverage_v2(points,ref):
    pts=gpd.GeoDataFrame(points.copy(),geometry=gpd.points_from_xy(points.x,points.y),crs=ref.crs)
    hits=gpd.sjoin(pts,ref[["geometry"]].reset_index(drop=True),how="left",predicate="within")
    idx=hits.index_right.dropna().astype(int)
    inside=hits.loc[hits.index_right.notna()].index.nunique()
    return dict(candidates=len(pts),crowns=len(ref),crowns_covered=idx.nunique(),coverage_pct=round(100*idx.nunique()/len(ref),1),candidates_inside=int(inside),candidates_outside=int(len(pts)-inside)),set(idx.tolist())
results=[];covered_sets={}
for name,pts in [("RGB",rgb_points)]+[(f"RGB+DSM {spacing:g}m",pts) for spacing,pts in fusions.items()]:
    stats,covered=coverage_v2(pts,reference_v2)
    results.append(dict(method=name,**stats))
    covered_sets[name]=covered
table15=pd.DataFrame(results)
display(table15)
table15.to_csv(EXP15/"candidate_coverage_sensitivity.csv",index=False)
# Crown-size stratification for RGB and default DSM fusion.
areas=reference_v2.reset_index(drop=True).geometry.area
bands=pd.cut(areas,bins=[0,20,40,80,float("inf")],labels=["<20","20-40","40-80",">80"],include_lowest=True)
size_rows=[]
for method in ("RGB","RGB+DSM 2m"):
    covered=covered_sets[method]
    for band in bands.cat.categories:
        ids=set(np.flatnonzero((bands==band).to_numpy()))
        size_rows.append(dict(method=method,crown_area_m2=str(band),crowns=len(ids),covered=len(ids&covered),coverage_pct=round(100*len(ids&covered)/len(ids),1) if ids else np.nan))
display(pd.DataFrame(size_rows))
print("Choose spacing based on coverage versus number of candidates; do not run SAM3 yet.")


## 17–19 — New RGB/DSM and SAM3 crown-discovery experiments

**Current recommended workflow: Block 19** runs SAM3 directly on a dense geometric-prompt grid over one complete ROI, with no treetop-candidate bottleneck. It uses 24 m overlapping RGB tiles, 6 m prompt spacing and 10 m boxes. It caches every SAM3 prediction on Drive, conservatively removes duplicates, and evaluates against the revised manually digitized crowns.

**Run in a fresh Colab runtime:** **0R → 2 (only if /content/sam3env is missing) → 14 → 19**. Block 0S and Blocks 15–18 are **not required** for Block 19. Use a GPU runtime for Block 19. Set `MAX_TILES=4` for a smoke test or `None` for the full ROI. Smoke-test and full-run outputs are isolated.

Earlier Blocks 17/18/18B were developed interactively; their saved results remain on Google Drive, and the full-ROI successor is consolidated below.


In [ ]:
# BLOCK 19 — Full ROI, candidate-free SAM3 crown discovery (resumable)
# Run after 0R, 2 (if environment absent), 14. No need for 0S or 15–18.
import os, sys, subprocess, json, math
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.enums import Resampling
from shapely.geometry import box, Point, shape
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt

ROI_INDEX = 0
TILE_M = 24.0
OVERLAP_M = 6.0
GRID_M = 6.0
BOX_M = 10.0
MAX_TILES = None       # None = complete ROI; 4 = technical smoke test
MIN_AREA_M2, MAX_AREA_M2 = 2.0, 500.0
EDGE_MARGIN_M = 0.4
DUP_IOU = 0.65
NESTED_CONTAINMENT = 0.95
SIMILAR_AREA_RATIO = 0.65

assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14"
assert Path(SAMPY).exists(), "Run Block 2 to install isolated SAM3 environment"
B19 = OUT / f"experiments_v19_full_roi_{ROI_INDEX}"
TILES, RAW19 = B19/"tiles", B19/"raw"
for d in (B19,TILES,RAW19): d.mkdir(parents=True,exist_ok=True)

with rasterio.open(RASTER) as src:
    crs, gsd = src.crs, abs(src.transform.a)
    roi19 = roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    n = int(round(TILE_M/gsd))
    stride = max(1,int(round((TILE_M-OVERLAP_M)/gsd)))
    minx,miny,maxx,maxy = roi19.bounds
    r0,c0 = src.index(minx,maxy)
    r1,c1 = src.index(maxx,miny)
    def starts(lo,hi):
        if hi-lo <= n: return [int(round((lo+hi-n)/2))]
        seq=list(range(int(lo),int(hi)-n+1,stride))
        last=int(hi)-n
        if not seq or seq[-1]!=last: seq.append(last)
        return sorted(set(seq))
    tiles=[]
    for rr in starts(r0,r1+1):
        for cc in starts(c0,c1+1):
            win=Window(cc,rr,n,n)
            b=rasterio.windows.bounds(win,src.transform)
            if not box(*b).intersects(roi19): continue
            tid=len(tiles); path=TILES/f"tile_{tid:04d}.png"
            tiles.append(dict(tile_id=tid,row_off=rr,col_off=cc,width=n,height=n,
                              xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],path=str(path)))
    if MAX_TILES is not None: tiles=tiles[:MAX_TILES]
    assert tiles,"No ROI tiles"
    for t in tiles:
        p=Path(t["path"])
        if p.exists(): continue
        win=Window(t["col_off"],t["row_off"],n,n)
        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        Image.fromarray(rgb.astype(np.uint8)).save(p)
tm=pd.DataFrame(tiles)
# Scope outputs by the selected tile IDs so smoke tests never masquerade as full ROI.
run_tag="full" if MAX_TILES is None else f"first_{MAX_TILES}_tiles"
work=B19/run_tag; raw=work/"raw"; raw.mkdir(parents=True,exist_ok=True)
rows=[]
for t in tm.itertuples():
    for dy in np.arange(GRID_M/2,TILE_M,GRID_M):
        for dx in np.arange(GRID_M/2,TILE_M,GRID_M):
            x,y=t.xmin+dx,t.ymax-dy
            if roi19.covers(Point(x,y)):
                rows.append(dict(tile_id=int(t.tile_id),path=t.path,cx=dx/TILE_M,cy=dy/TILE_M,
                                 box_m=BOX_M,width=n,height=n,gsd=gsd,
                                 row_off=int(t.row_off),col_off=int(t.col_off)))
pm=pd.DataFrame(rows)
assert len(pm),"No grid prompts inside ROI"
pm.insert(0,"prompt_id",np.arange(len(pm)))
pm.to_csv(work/"prompt_manifest.csv",index=False)
print(f"ROI {ROI_INDEX}: {roi19.area/10000:.3f} ha | tiles={len(tm)} | prompts={len(pm)} | mode={run_tag}")
print("Existing masks:",sum((raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id))

runner=Path("/content/run_sam3_kvo19.py")
runner.write_text(r'''
import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]); outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("CUDA GPU required")
torch.backends.cuda.matmul.allow_tf32=True
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path=bpe)
    processor=Sam3Processor(model)
    for tid,g in df.groupby("tile_id",sort=True):
        pending=[r for r in g.itertuples() if not (outdir/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not pending:continue
        img=Image.open(pending[0].path).convert("RGB")
        for r in tqdm(pending,desc=f"Tile {tid}",leave=False):
            # Fresh prompt state per box, to avoid accumulated prompts.
            state=processor.set_image(img)
            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))
            out=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(outdir/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"Tile {tid} complete",flush=True)
print("Inference complete",flush=True)
''')
missing=sum(not (raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id)
if missing:
    run_env=os.environ.copy()
    if "env" in globals(): run_env.update(env)
    print("Running SAM3 for",missing,"unsaved prompts")
    subprocess.run([SAMPY,"-u",str(runner),str(work/"prompt_manifest.csv"),str(raw)],
                   env=run_env,check=True)
else: print("All masks cached; skipping SAM3")

# Convert every returned instance, not just the highest-scoring mask.
features=[]
with rasterio.open(RASTER) as src:
    for r in pm.itertuples():
        p=raw/f"prompt_{int(r.prompt_id):06d}.npz"
        if not p.exists():continue
        with np.load(p) as z:
            masks=z["masks"]; scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        transform=src.window_transform(win)
        for k,m in enumerate(masks):
            m=np.asarray(m)>0
            if m.shape!=(n,n) or not m.any():continue
            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))
            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():
                continue
            geoms=[shape(g) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not (MIN_AREA_M2<=geom.area<=MAX_AREA_M2):continue
            if not roi19.covers(geom.representative_point()):continue
            features.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),
                                 instance=int(k),score=float(scores[min(k,len(scores)-1)]),
                                 area_m2=float(geom.area),geometry=geom))
print("Valid raw proposals:",len(features))
assert features,"No polygons survived filtering; inspect tile-edge masks"
allpred=gpd.GeoDataFrame(features,geometry="geometry",crs=crs)
allpred.to_file(work/"all_proposals.gpkg",driver="GPKG")

# Spatial-index-assisted conservative duplicate removal.
ordered=allpred.sort_values("score",ascending=False).reset_index(drop=True)
keep=[]; removed=[]
for i,r in ordered.iterrows():
    g=r.geometry; duplicate=False
    for j in keep:
        h=ordered.geometry.iloc[j]
        if not g.intersects(h):continue
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        containment=inter/min(g.area,h.area)
        ratio=min(g.area,h.area)/max(g.area,h.area)
        if iou>=DUP_IOU or (containment>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):
            duplicate=True;removed.append(dict(prompt_id=int(r.prompt_id),against=int(j),iou=iou,
                                                containment=containment,area_ratio=ratio))
            break
    if not duplicate:keep.append(i)
clean=ordered.iloc[keep].copy().reset_index(drop=True)
clean["crown_id"]=np.arange(1,len(clean)+1)
clean.to_file(work/"clean_crowns.gpkg",driver="GPKG")
pd.DataFrame(removed).to_csv(work/"removed_duplicates.csv",index=False)
print(f"Clean predictions: {len(clean)} | removed duplicates: {len(removed)}")

# Evaluate only reference crowns fully contained in the processed tile union
# (and ROI). This prevents partial smoke-test coverage being labelled full-ROI.
footprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]
from shapely.ops import unary_union
covered=unary_union(footprints).intersection(roi19)
reference=reference_v2.to_crs(crs).copy()
reference=reference[reference.geometry.apply(covered.covers)].copy().reset_index(drop=True)
mat=np.zeros((len(reference),len(clean)),dtype=np.float32)
idx=clean.sindex
for i,g in enumerate(reference.geometry):
    for j in idx.query(g,predicate="intersects"):
        h=clean.geometry.iloc[j]; inter=g.intersection(h).area
        union=g.area+h.area-inter
        if union>0:mat[i,j]=inter/union
matches=[]
if mat.size:
    utility=(mat>=.5).astype(float)*1000+mat
    ii,jj=linear_sum_assignment(utility,maximize=True)
    matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=.5]
matched_ref={i for i,j,v in matches}
reference["area_m2"]=reference.geometry.area
reference["matched"]=[i in matched_ref for i in range(len(reference))]
reference["best_iou"]=mat.max(axis=1) if len(clean) else 0
reference["size_class"]=pd.cut(reference.area_m2,[0,20,40,80,np.inf],
                               labels=["<20","20–40","40–80",">80"])
size=reference.groupby("size_class",observed=True).agg(
    n=("matched","size"),matched=("matched","sum"),recall=("matched","mean"),
    median_best_iou=("best_iou","median"))
size.to_csv(work/"recall_by_size.csv")
display(size)
recall=len(matches)/len(reference) if len(reference) else float("nan")
precision=len(matches)/len(clean) if len(clean) else float("nan")
summary=dict(roi_index=ROI_INDEX,mode=run_tag,roi_area_ha=roi19.area/10000,
             tiles=len(tm),prompts=len(pm),manual_crowns=len(reference),
             raw_proposals=len(allpred),clean_predictions=len(clean),
             matched_iou_05=len(matches),recall_05=recall,precision_05=precision,
             median_matched_iou=float(np.median([v for _,_,v in matches])) if matches else None)
(work/"summary.json").write_text(json.dumps(summary,indent=2))
print("RESULTS:",json.dumps(summary,indent=2))

# Full ROI visualization, downsampled to avoid loading the giant raster.
with rasterio.open(RASTER) as src:
    from rasterio.windows import from_bounds
    win=from_bounds(*roi19.bounds,transform=src.transform).round_offsets().round_lengths()
    scale=max(1,int(math.ceil(max(win.width,win.height)/1600)))
    hh=max(1,int(win.height/scale));ww=max(1,int(win.width/scale))
    img=src.read([1,2,3],window=win,out_shape=(3,hh,ww),
                 resampling=Resampling.average).transpose(1,2,0)
    bb=rasterio.windows.bounds(win,src.transform)
fig,ax=plt.subplots(figsize=(12,12))
ax.imshow(img,extent=(bb[0],bb[2],bb[1],bb[3]),origin="upper")
gpd.GeoSeries([roi19],crs=crs).boundary.plot(ax=ax,color="white",linewidth=2)
if len(reference):reference.boundary.plot(ax=ax,color="yellow",linewidth=1)
clean.boundary.plot(ax=ax,color="cyan",linewidth=.8)
ax.set_title(f"ROI {ROI_INDEX} | {run_tag} | manual {len(reference)} | SAM3 {len(clean)} | recall@0.5 {recall:.1%}")
ax.set_aspect("equal");plt.tight_layout()
fig.savefig(work/"visual_QA.png",dpi=180,bbox_inches="tight")
plt.show()
print("BLOCK 19 COMPLETE:",work)


## 20 — Multiscale SAM3 pilot (40 m RGB tiles)

**Run:** 0R → Block 2 only if SAM3 environment missing → Block 14 → **Block 20**. Requires the saved **Block 19B** diagnostic CSV and Block 19 clean predictions. No need to rerun Blocks 19/19B. GPU for inference; cached masks allow resuming. Four representative pilot areas (selected using diagnostic reference results, not used as SAM3 prompt shapes), fixed 6 m grid, box sizes 8/12/16/20 m. Compare each scale and combined scales with the Block 19 baseline on the **same fully contained manual crowns**. Outputs saved to `experiments_v20_multiscale_roi_0/`.

**Scientific note:** pilot areas are deliberately selected, so results are diagnostic rather than unbiased site-wide accuracy estimates.

In [ ]:
# BLOCK 20 — Multiscale SAM3 pilot: 40m RGB context, 8/12/16/20m boxes
# Run 0R -> 2 (if needed) -> 14 -> 20. Does not rerun Block 19.
import os, sys, json, subprocess, math
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.enums import Resampling
from shapely.geometry import Point, box, shape
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt

ROI_INDEX=0
TILE_M=40.0
GRID_M=6.0
BOX_SIZES_M=(8.,12.,16.,20.)
N_AREAS=4
EDGE_MARGIN_M=0.4
MIN_AREA_M2,MAX_AREA_M2=2.,700.
DUP_IOU=.65
NESTED_CONTAINMENT=.95
SIMILAR_AREA_RATIO=.65
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14"
assert Path(SAMPY).exists(), "Run Block 2 to install SAM3"
B19=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
DIAG=B19/"diagnostics_19B"/"crown_diagnostics_clean.csv"
assert DIAG.exists(), "Run Block 19B first; diagnostic CSV missing"
assert (B19/"clean_crowns.gpkg").exists(), "Block 19 clean predictions missing"
B20=OUT/f"experiments_v20_multiscale_roi_{ROI_INDEX}"
TILES=B20/"tiles";RAW20=B20/"raw"
for p in (B20,TILES,RAW20):p.mkdir(parents=True,exist_ok=True)

with rasterio.open(RASTER) as src:
    crs=src.crs;gsd=abs(src.transform.a)
    roi20=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    ref=reference_v2.to_crs(crs)
    ref=ref[ref.geometry.apply(roi20.covers)].copy().reset_index(drop=True)
    d=pd.read_csv(DIAG)
    assert len(d)==len(ref),"Block 19B reference order/count differs; rerun 19B after Block 14"
    # Choose two large poorly recovered crowns, one medium low-IoU, one well-matched control.
    d=d.copy()
    d["group"]=np.select([
        (d.crown_area_m2>80)&(d.best_iou<.5),
        (d.crown_area_m2.between(20,80))&(d.best_iou<.5),
        d.best_iou>=.5],
        ["large_missed","medium_low_iou","matched_control"],default="other")
    targets=[]
    for group,quota in [("large_missed",2),("medium_low_iou",1),("matched_control",1)]:
        subset=d[d.group==group].sort_values(
            ["combined_coverage","crown_area_m2"],ascending=[True,False])
        for rr in subset.itertuples():
            p=ref.geometry.iloc[int(rr.crown_id)].representative_point()
            if all(p.distance(q["point"])>=TILE_M*.7 for q in targets):
                targets.append(dict(point=p,group=group,ref_id=int(rr.crown_id)))
                if sum(q["group"]==group for q in targets)>=quota:break
    if len(targets)<N_AREAS:
        for rr in d.sort_values("crown_area_m2",ascending=False).itertuples():
            p=ref.geometry.iloc[int(rr.crown_id)].representative_point()
            if all(p.distance(q["point"])>=TILE_M*.5 for q in targets):
                targets.append(dict(point=p,group="additional",ref_id=int(rr.crown_id)))
                if len(targets)>=N_AREAS:break
    targets=targets[:N_AREAS]
    assert targets,"No pilot target areas found"
    n=int(round(TILE_M/gsd))
    tiles=[]
    for i,t in enumerate(targets):
        rr,cc=src.index(t["point"].x,t["point"].y)
        win=Window(int(cc-n//2),int(rr-n//2),n,n)
        b=rasterio.windows.bounds(win,src.transform)
        png=TILES/f"area_{i:02d}.png"
        if not png.exists():
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            Image.fromarray(rgb.astype(np.uint8)).save(png)
        tiles.append(dict(area_id=i,group=t["group"],reference_id=t["ref_id"],
                          path=str(png),row_off=int(win.row_off),col_off=int(win.col_off),
                          xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],n=n))
tm=pd.DataFrame(tiles);tm.to_csv(B20/"pilot_areas.csv",index=False)
print("Pilot areas:");display(tm[["area_id","group","reference_id"]])
# Same centres for all box sizes. 40m tiles: 6m grid positions 3,9,...,39m.
# Keep centres >= BOX_MAX/2 from tile edges so large boxes remain in the image.
offsets=np.arange(GRID_M/2,TILE_M,GRID_M)
offsets=[v for v in offsets if max(BOX_SIZES_M)/2<=v<=TILE_M-max(BOX_SIZES_M)/2]
rows=[]
for t in tm.itertuples():
    for dy in offsets:
        for dx in offsets:
            x,y=t.xmin+dx,t.ymax-dy
            if not roi20.covers(Point(x,y)):continue
            for sz in BOX_SIZES_M:
                rows.append(dict(area_id=int(t.area_id),path=t.path,row_off=int(t.row_off),
                                 col_off=int(t.col_off),n=n,gsd=gsd,
                                 cx=dx/TILE_M,cy=dy/TILE_M,box_m=sz))
pm=pd.DataFrame(rows)
assert len(pm),"No prompt centres inside selected areas"
pm.insert(0,"prompt_id",np.arange(len(pm)))
pm.to_csv(B20/"prompt_manifest.csv",index=False)
print(f"Prompts: {len(pm)} across {len(tm)} areas, {len(BOX_SIZES_M)} scales")
print("Existing masks:",sum((RAW20/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id))

runner=Path("/content/run_sam3_kvo20.py")
runner.write_text(r'''
import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]);dest=Path(sys.argv[2]);dest.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(),"SAM3 requires GPU"
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
    processor=Sam3Processor(model)
    for area,g in df.groupby("area_id",sort=True):
        todo=[r for r in g.itertuples() if not (dest/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not todo:continue
        img=Image.open(todo[0].path).convert("RGB")
        for r in tqdm(todo,desc=f"Area {area}",leave=False):
            state=processor.set_image(img)
            frac=float(r.box_m)/(float(r.n)*float(r.gsd))
            out=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(dest/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"Area {area} complete",flush=True)
print("All done",flush=True)
''')
missing=sum(not (RAW20/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id)
if missing:
    run_env=os.environ.copy()
    if "env" in globals():run_env.update(env)
    subprocess.run([SAMPY,"-u",str(runner),str(B20/"prompt_manifest.csv"),str(RAW20)],
                   env=run_env,check=True)
else:print("All inference cached")

# Polygonize masks for every scale; keep all returned instances.
records=[]
with rasterio.open(RASTER) as src:
    for r in pm.itertuples():
        p=RAW20/f"prompt_{int(r.prompt_id):06d}.npz"
        if not p.exists():continue
        with np.load(p) as z:
            masks=z["masks"];scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        transform=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.n),int(r.n)))
        for k,m in enumerate(masks):
            m=np.asarray(m)>0
            if m.shape!=(int(r.n),int(r.n)) or not m.any():continue
            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))
            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():continue
            geoms=[shape(g) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not MIN_AREA_M2<=geom.area<=MAX_AREA_M2:continue
            if not roi20.covers(geom.representative_point()):continue
            records.append(dict(prompt_id=int(r.prompt_id),area_id=int(r.area_id),
                                box_m=float(r.box_m),instance=int(k),
                                score=float(scores[min(k,len(scores)-1)]),geometry=geom))
assert records,"No valid masks; inspect raw outputs"
pred=gpd.GeoDataFrame(records,geometry="geometry",crs=crs)
pred.to_file(B20/"all_multiscale_proposals.gpkg",driver="GPKG")

def dedup(gdf):
    ordered=gdf.sort_values("score",ascending=False).reset_index(drop=True)
    keep=[]
    for i,r in ordered.iterrows():
        g=r.geometry;dup=False
        for j in keep:
            h=ordered.geometry.iloc[j]
            if not g.intersects(h):continue
            inter=g.intersection(h).area
            if inter<=0:continue
            iou=inter/(g.area+h.area-inter)
            contain=inter/min(g.area,h.area)
            ratio=min(g.area,h.area)/max(g.area,h.area)
            if iou>=DUP_IOU or (contain>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):
                dup=True;break
        if not dup:keep.append(i)
    return ordered.iloc[keep].copy().reset_index(drop=True)

# Evaluation population: crowns fully contained in the ROI AND in at least
# one of the four 40m windows. Identical population for all scales + baseline.
pilot_footprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]
refpilot=ref[ref.geometry.apply(lambda g:any(w.covers(g) for w in pilot_footprints))].copy().reset_index(drop=True)
print("Manual crowns fully inside pilot tiles:",len(refpilot))

def match_metrics(reference,predictions):
    nref,npred=len(reference),len(predictions)
    mat=np.zeros((nref,npred),dtype=np.float32)
    if nref and npred:
        idx=predictions.sindex
        for i,g in enumerate(reference.geometry):
            for j in idx.query(g,predicate="intersects"):
                h=predictions.geometry.iloc[j];a=g.intersection(h).area
                if a>0:mat[i,j]=a/(g.area+h.area-a)
        utility=(mat>=.5).astype(float)*1000+mat
        ii,jj=linear_sum_assignment(utility,maximize=True)
        matches=[float(mat[i,j]) for i,j in zip(ii,jj) if mat[i,j]>=.5]
    else:matches=[]
    return dict(manual=nref,predictions=npred,matched=len(matches),
                recall=len(matches)/nref if nref else np.nan,
                precision=len(matches)/npred if npred else np.nan,
                median_matched_iou=float(np.median(matches)) if matches else np.nan,
                large_recall=float((mat[refpilot.geometry.area.to_numpy()>80].max(axis=1)>=.5).mean())
                  if npred and (refpilot.geometry.area>80).any() else np.nan)

# Keep only proposals whose representative points fall inside pilot windows.
# Same geographic scope for baseline and new multiscale proposals.
def in_pilot(g):
    return any(w.covers(g.representative_point()) for w in pilot_footprints)

baseline=gpd.read_file(B19/"clean_crowns.gpkg").to_crs(crs)
baseline=baseline[baseline.geometry.apply(in_pilot)].copy().reset_index(drop=True)
results=[dict(scale="Block19 baseline 10m/24m",**match_metrics(refpilot,baseline))]
for sz in BOX_SIZES_M:
    subset=pred[np.isclose(pred.box_m,sz)].copy()
    clean=dedup(subset)
    clean.to_file(B20/f"clean_{int(sz)}m.gpkg",driver="GPKG")
    results.append(dict(scale=f"{int(sz)}m",**match_metrics(refpilot,clean)))
combined=dedup(pred)
combined.to_file(B20/"clean_multiscale_combined.gpkg",driver="GPKG")
results.append(dict(scale="all scales combined",**match_metrics(refpilot,combined)))
result=pd.DataFrame(results)
result.to_csv(B20/"scale_comparison.csv",index=False)
display(result.round(3))
fig,ax=plt.subplots(figsize=(10,5))
ax.bar(result.scale,result.recall)
ax.set_ylabel("Recall at IoU ≥ 0.5");ax.set_ylim(0,1)
ax.tick_params(axis="x",rotation=35);plt.tight_layout()
fig.savefig(B20/"scale_recall.png",dpi=160,bbox_inches="tight")
plt.show()
print("BLOCK 20 COMPLETE:",B20)


## 20B — Actual multiscale RGB/SAM3 visual QA

**Run 0R → 14 → 20B.** CPU only. Plots actual orthomosaic pixels, manual crown boundaries and the saved Block 19/20 predictions at identical extents. Produces four comparison figures in `experiments_v20_multiscale_roi_0/visual_QA_20B/`. No model inference.


In [ ]:
# BLOCK 20B — TRUE RGB / SAM3 multiscale visual QA (CPU only)
# Run 0R -> 14 -> 20B. Does NOT run SAM3.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from shapely.geometry import box
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROI_INDEX=0
B19=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
B20=OUT/f"experiments_v20_multiscale_roi_{ROI_INDEX}"
VIS=B20/"visual_QA_20B";VIS.mkdir(parents=True,exist_ok=True)
areas=pd.read_csv(B20/"pilot_areas.csv")
assert len(areas),"No pilot areas"
paths=[
    ("Block 19 (10 m, 24 m tiles)",B19/"clean_crowns.gpkg"),
    ("8 m",B20/"clean_8m.gpkg"),
    ("12 m",B20/"clean_12m.gpkg"),
    ("16 m",B20/"clean_16m.gpkg"),
    ("20 m",B20/"clean_20m.gpkg"),
    ("Combined",B20/"clean_multiscale_combined.gpkg"),
]
for label,path in paths:
    assert path.exists(),f"Missing {label}: {path}. Run Block 20."
with rasterio.open(RASTER) as src:
    crs=src.crs
    preds=[(label,gpd.read_file(path).to_crs(crs)) for label,path in paths]
    reference=reference_v2.to_crs(crs)
    roi20=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    reference=reference[reference.geometry.intersects(roi20)].copy()
    records=[]
    for area in areas.itertuples():
        footprint=box(float(area.xmin),float(area.ymin),float(area.xmax),float(area.ymax))
        # Same exact raster window for every column in a row.
        win=Window(int(area.col_off),int(area.row_off),int(area.n),int(area.n))
        rgb=src.read([1,2,3],window=win,out_shape=(3,800,800),
                     resampling=Resampling.average,boundless=True,fill_value=0)
        rgb=np.moveaxis(rgb,0,-1).astype(np.uint8)
        extent=(area.xmin,area.xmax,area.ymin,area.ymax)
        ref_area=reference[reference.geometry.intersects(footprint)]
        fig,axes=plt.subplots(1,6,figsize=(25,4.8),sharex=True,sharey=True)
        for ax,(label,gdf) in zip(axes,preds):
            subset=gdf[gdf.geometry.intersects(footprint)]
            ax.imshow(rgb,extent=extent,origin="upper")
            if len(ref_area):
                ref_area.boundary.plot(ax=ax,color="yellow",linewidth=1.0,zorder=3)
            if len(subset):
                subset.boundary.plot(ax=ax,color="cyan",linewidth=.8,zorder=4)
            ax.set_xlim(area.xmin,area.xmax);ax.set_ylim(area.ymin,area.ymax)
            ax.set_aspect("equal");ax.set_title(f"{label}\n{len(subset)} proposals",fontsize=10)
            ax.set_xticks([]);ax.set_yticks([])
            records.append(dict(area_id=int(area.area_id),group=area.group,
                                scale=label,visible_predictions=len(subset),
                                visible_reference=len(ref_area)))
        fig.suptitle(f"Baego ROI {ROI_INDEX} | Pilot area {area.area_id}: {area.group} "
                     f"| reference ID {area.reference_id}",fontsize=13)
        fig.legend(handles=[
            Line2D([0],[0],color="yellow",lw=2,label="Manual crown boundaries"),
            Line2D([0],[0],color="cyan",lw=2,label="SAM3 crown proposals")
        ],loc="lower center",ncol=2,bbox_to_anchor=(.5,-.02))
        fig.tight_layout(rect=(0,.06,1,.94))
        p=VIS/f"area_{int(area.area_id):02d}_real_RGB_comparison.png"
        fig.savefig(p,dpi=190,bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("Saved:",p)
pd.DataFrame(records).to_csv(VIS/"visible_polygon_counts.csv",index=False)
print("\nBLOCK 20B COMPLETE — real RGB and actual SAM3 polygons")
print("Figures:",VIS)


## 21 — DSM-assisted SAM3 crown quality assessment

**Run 0R → 14 → 21.** CPU only. Uses saved Block 19/20 polygons and DSM, aligns DSM to RGB with WarpedVRT, calculates geometry, DSM local contrast, nesting and overlap, and evaluates cautious quality filters against fully contained manual reference crowns in the same pilot areas. No SAM3 rerun. Outputs in `experiments_v21_DSM_quality_roi_0/`. DSM is surface elevation, not a terrain-normalized canopy height model. Flags are exploratory review priorities, not validated quality probabilities.


In [ ]:
# BLOCK 21 — DSM-assisted crown quality assessment (CPU only)
# Run 0R -> 14 -> 21. No SAM3 rerun. No modifications to source polygons.
from pathlib import Path
import math, json
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask
from rasterio.enums import Resampling
from shapely.geometry import box
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROI_INDEX=0
B19=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
B20=OUT/f"experiments_v20_multiscale_roi_{ROI_INDEX}"
B21=OUT/f"experiments_v21_DSM_quality_roi_{ROI_INDEX}"
B21.mkdir(parents=True,exist_ok=True)
DSM=GEO/"Baego_DSM.tif"
assert DSM.exists(),"Baego DSM missing"
assert "roi_v2" in globals() and "reference_v2" in globals(),"Run Block 14"
sources={
    "baseline_19":B19/"clean_crowns.gpkg",
    "scale_8m":B20/"clean_8m.gpkg",
    "multiscale":B20/"clean_multiscale_combined.gpkg",
}
for key,path in sources.items():assert path.exists(),f"Missing {key}: {path}"
areas=pd.read_csv(B20/"pilot_areas.csv")
assert len(areas),"Block 20 pilot areas missing"

# Quality metrics calculated from polygon geometry and aligned DSM.
# DSM is SURFACE ELEVATION, not canopy height; local contrasts are proxies only.
# We sample a local 2m exterior ring for relative elevation.
def dsm_metrics(g, vrt, transform, pixel_size, buffer_m=2.0):
    minx,miny,maxx,maxy=g.buffer(buffer_m).bounds
    win=from_bounds(minx,miny,maxx,maxy,transform=transform)
    win=win.round_offsets().round_lengths()
    if win.width<2 or win.height<2:return np.nan,np.nan,np.nan,np.nan,0
    # Prevent pathological reads for very large polygons.
    if win.width*win.height>2_000_000:return np.nan,np.nan,np.nan,np.nan,0
    z=vrt.read(1,window=win,masked=True)
    wt=rasterio.windows.transform(win,transform)
    inside=geometry_mask([g],out_shape=z.shape,transform=wt,invert=True)
    outer=geometry_mask([g.buffer(buffer_m)],out_shape=z.shape,transform=wt,invert=True)
    arr=np.asarray(z,dtype=float)
    valid=np.isfinite(arr)&(~np.ma.getmaskarray(z))&(arr>0)
    a=arr[inside&valid];b=arr[outer&~inside&valid]
    if len(a)<5:return np.nan,np.nan,np.nan,np.nan,len(a)
    med=float(np.median(a))
    spread=float(np.percentile(a,90)-np.percentile(a,10))
    contrast=med-float(np.median(b)) if len(b)>=5 else np.nan
    rough=float(np.std(a))
    return med,spread,contrast,rough,len(a)

def quality_dataset(gdf, vrt, transform, pixel_size):
    gdf=gdf.copy().reset_index(drop=True)
    gdf["area_m2"]=gdf.geometry.area
    gdf["perimeter_m"]=gdf.geometry.length
    gdf["compactness"]=4*np.pi*gdf.area_m2/(gdf.perimeter_m**2).clip(lower=1e-6)
    gdf["equiv_diameter_m"]=2*np.sqrt(gdf.area_m2/np.pi)
    metrics=[dsm_metrics(g,vrt,transform,pixel_size) for g in gdf.geometry]
    gdf[["dsm_median","dsm_p90_p10","dsm_local_contrast",
         "dsm_std","dsm_valid_pixels"]]=pd.DataFrame(metrics,index=gdf.index)
    # Pairwise spatial overlap and containment (not just IoU).
    si=gdf.sindex
    overlap=np.zeros(len(gdf),dtype=int)
    nested=np.zeros(len(gdf),dtype=int)
    max_iou=np.zeros(len(gdf),dtype=float)
    for i,g in enumerate(gdf.geometry):
        for j in si.query(g,predicate="intersects"):
            j=int(j)
            if j<=i:continue
            h=gdf.geometry.iloc[j]
            inter=g.intersection(h).area
            if inter<=0:continue
            union=g.area+h.area-inter
            iou=inter/union
            overlap[i]+=1;overlap[j]+=1
            max_iou[i]=max(max_iou[i],iou)
            max_iou[j]=max(max_iou[j],iou)
            if inter/min(g.area,h.area)>=.95:
                nested[i]+=1;nested[j]+=1
    gdf["overlap_count"]=overlap
    gdf["nested_count"]=nested
    gdf["max_pair_iou"]=max_iou
    # Interpretable, deliberately cautious flags; no reference labels used.
    # No absolute elevation/height threshold: terrain may vary.
    area_flag=(gdf.area_m2<5)
    shape_flag=(gdf.compactness<.12)
    redundancy_flag=(gdf.nested_count>=2)|(gdf.max_pair_iou>=.55)
    low_contrast=(gdf.dsm_local_contrast.notna())&(gdf.dsm_local_contrast<-.75)
    gdf["flag_tiny"]=area_flag.astype(int)
    gdf["flag_irregular"]=shape_flag.astype(int)
    gdf["flag_redundant"]=redundancy_flag.astype(int)
    gdf["flag_low_relative_DSM"]=low_contrast.astype(int)
    gdf["flag_count"]=gdf[["flag_tiny","flag_irregular",
                           "flag_redundant","flag_low_relative_DSM"]].sum(axis=1)
    gdf["quality_class"]=np.select(
        [gdf.flag_count>=2,gdf.flag_count==1],
        ["likely_artifact_review","uncertain"],default="higher_priority")
    # Note: these are review priorities, not classifier probabilities.
    return gdf

def evaluate(ref,pred):
    mat=np.zeros((len(ref),len(pred)),dtype=np.float32)
    if len(ref) and len(pred):
        ix=pred.sindex
        for i,g in enumerate(ref.geometry):
            for j in ix.query(g,predicate="intersects"):
                h=pred.geometry.iloc[j];a=g.intersection(h).area
                if a>0:mat[i,j]=a/(g.area+h.area-a)
        utility=(mat>=.5).astype(float)*1000+mat
        ii,jj=linear_sum_assignment(utility,maximize=True)
        matched=sum(mat[i,j]>=.5 for i,j in zip(ii,jj))
    else:matched=0
    return dict(manual=len(ref),predictions=len(pred),matched=int(matched),
                recall=matched/len(ref) if len(ref) else np.nan,
                precision=matched/len(pred) if len(pred) else np.nan)

with rasterio.open(RASTER) as rgb, rasterio.open(DSM) as dsm:
    crs=rgb.crs;roi21=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    ref=reference_v2.to_crs(crs)
    # Evaluation on the exact same 40m pilot tiles as Block 20.
    footprints=[box(float(t.xmin),float(t.ymin),float(t.xmax),float(t.ymax))
                for t in areas.itertuples()]
    ref=ref[ref.geometry.apply(
        lambda g:roi21.covers(g) and any(w.covers(g) for w in footprints)
    )].copy().reset_index(drop=True)
    print("Pilot reference crowns:",len(ref))
    # Align DSM to RGB pixels using a VRT (accounts for one-pixel shift).
    with WarpedVRT(dsm,crs=crs,transform=rgb.transform,
                   width=rgb.width,height=rgb.height,
                   resampling=Resampling.nearest) as vrt:
        datasets={}
        results=[]
        for name,path in sources.items():
            original=gpd.read_file(path).to_crs(crs)
            # Same pilot geography for all three sources.
            subset=original[original.geometry.apply(
                lambda g:any(w.covers(g.representative_point()) for w in footprints)
            )].copy().reset_index(drop=True)
            scored=quality_dataset(subset,vrt,rgb.transform,abs(rgb.transform.a))
            scored.to_file(B21/f"{name}_quality.gpkg",driver="GPKG")
            scored.drop(columns="geometry").to_csv(B21/f"{name}_quality.csv",index=False)
            datasets[name]=scored
            for filter_name,mask in [
                ("unfiltered",np.ones(len(scored),dtype=bool)),
                ("exclude_2plus_flags",scored.flag_count<2),
                ("higher_priority_only",scored.flag_count==0)]:
                selected=scored.loc[mask].copy().reset_index(drop=True)
                results.append(dict(source=name,filter=filter_name,
                                    **evaluate(ref,selected)))
        comparison=pd.DataFrame(results)
        comparison.to_csv(B21/"filter_sensitivity.csv",index=False)
        display(comparison.round(3))
        for area in areas.itertuples():
            win=rasterio.windows.Window(int(area.col_off),int(area.row_off),
                                        int(area.n),int(area.n))
            chip=rgb.read([1,2,3],window=win,out_shape=(3,850,850),
                          resampling=Resampling.average,boundless=True,fill_value=0)
            chip=np.moveaxis(chip,0,-1).astype(np.uint8)
            ext=(area.xmin,area.xmax,area.ymin,area.ymax)
            foot=box(area.xmin,area.ymin,area.xmax,area.ymax)
            fig,axes=plt.subplots(1,3,figsize=(16,5.6),sharex=True,sharey=True)
            for ax,(name,df) in zip(axes,datasets.items()):
                ax.imshow(chip,extent=ext,origin="upper")
                rr=ref[ref.geometry.intersects(foot)]
                if len(rr):rr.boundary.plot(ax=ax,color="yellow",linewidth=1.1,zorder=4)
                sub=df[df.geometry.intersects(foot)]
                for cls,color in [("higher_priority","lime"),
                                  ("uncertain","cyan"),
                                  ("likely_artifact_review","red")]:
                    layer=sub[sub.quality_class==cls]
                    if len(layer):layer.boundary.plot(ax=ax,color=color,linewidth=.9,zorder=5)
                ax.set_title(f"{name}\n{len(sub)} visible polygons",fontsize=11)
                ax.set_xlim(area.xmin,area.xmax);ax.set_ylim(area.ymin,area.ymax)
                ax.set_xticks([]);ax.set_yticks([]);ax.set_aspect("equal")
            fig.legend(handles=[
                Line2D([0],[0],color="yellow",lw=2,label="Manual reference"),
                Line2D([0],[0],color="lime",lw=2,label="Higher priority"),
                Line2D([0],[0],color="cyan",lw=2,label="Uncertain"),
                Line2D([0],[0],color="red",lw=2,label="Likely artifact — review")
            ],loc="lower center",ncol=4)
            fig.suptitle(f"Block 21 — area {area.area_id} ({area.group})")
            fig.tight_layout(rect=(0,.07,1,.94))
            fig.savefig(B21/f"area_{int(area.area_id):02d}_quality.png",
                        dpi=170,bbox_inches="tight")
            plt.show();plt.close(fig)
print("BLOCK 21 COMPLETE:",B21)
print("Review filter_sensitivity.csv and quality figures before discarding any polygons.")


## 22 — Full ROI RGB-only 8 m SAM3 and full manual-outline validation

**Inference:** GPU Colab, run `0R → 2 (if needed) → 14 → 22`. Resumable, 40 m RGB tiles, 8 m box prompts, 6 m grid, 10 m overlap. Saves masks and polygon outputs separately from Block 19. **Validation:** CPU, run `0R → 14 → 22V` after full inference. Manual `Crown_Kristof.shp` outlines are used only for evaluation, never for prompts. ROI from `ROI_Crowns_Kvo.shp`. Both methods evaluated against the identical fully-contained manual crowns; no DSM. Compare recall at IoU 0.3/0.5/0.7, matched crown area bias, fragmentation and size classes. Unmatched predictions are not automatically false positives.


In [ ]:
# BLOCK 22 — Full ROI RGB SAM3, 8m boxes / 40m tiles (resumable)
# Run after 0R, 2 (if environment absent), 14. No need for 0S or 15–18.
import os, sys, subprocess, json, math
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.enums import Resampling
from shapely.geometry import box, Point, shape
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt

ROI_INDEX = 0
TILE_M = 40.0
OVERLAP_M = 10.0
GRID_M = 6.0
BOX_M = 8.0
MAX_TILES = None       # None = complete ROI; 4 = technical smoke test
MIN_AREA_M2, MAX_AREA_M2 = 2.0, 700.0
EDGE_MARGIN_M = 0.4
DUP_IOU = 0.65
NESTED_CONTAINMENT = 0.95
SIMILAR_AREA_RATIO = 0.65

assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14"
assert Path(SAMPY).exists(), "Run Block 2 to install isolated SAM3 environment"
B19 = OUT / f"experiments_v22_full_roi_8m_{ROI_INDEX}"
TILES, RAW19 = B19/"tiles", B19/"raw"
for d in (B19,TILES,RAW19): d.mkdir(parents=True,exist_ok=True)

with rasterio.open(RASTER) as src:
    crs, gsd = src.crs, abs(src.transform.a)
    roi19 = roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    n = int(round(TILE_M/gsd))
    stride = max(1,int(round((TILE_M-OVERLAP_M)/gsd)))
    minx,miny,maxx,maxy = roi19.bounds
    r0,c0 = src.index(minx,maxy)
    r1,c1 = src.index(maxx,miny)
    def starts(lo,hi):
        if hi-lo <= n: return [int(round((lo+hi-n)/2))]
        seq=list(range(int(lo),int(hi)-n+1,stride))
        last=int(hi)-n
        if not seq or seq[-1]!=last: seq.append(last)
        return sorted(set(seq))
    tiles=[]
    for rr in starts(r0,r1+1):
        for cc in starts(c0,c1+1):
            win=Window(cc,rr,n,n)
            b=rasterio.windows.bounds(win,src.transform)
            if not box(*b).intersects(roi19): continue
            tid=len(tiles); path=TILES/f"tile_{tid:04d}.png"
            tiles.append(dict(tile_id=tid,row_off=rr,col_off=cc,width=n,height=n,
                              xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],path=str(path)))
    if MAX_TILES is not None: tiles=tiles[:MAX_TILES]
    assert tiles,"No ROI tiles"
    for t in tiles:
        p=Path(t["path"])
        if p.exists(): continue
        win=Window(t["col_off"],t["row_off"],n,n)
        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        Image.fromarray(rgb.astype(np.uint8)).save(p)
tm=pd.DataFrame(tiles)
# Scope outputs by the selected tile IDs so smoke tests never masquerade as full ROI.
run_tag="full" if MAX_TILES is None else f"first_{MAX_TILES}_tiles"
work=B19/run_tag; raw=work/"raw"; raw.mkdir(parents=True,exist_ok=True)
rows=[]
for t in tm.itertuples():
    for dy in np.arange(BOX_M/2,TILE_M-BOX_M/2+1e-6,GRID_M):
        for dx in np.arange(GRID_M/2,TILE_M,GRID_M):
            x,y=t.xmin+dx,t.ymax-dy
            if roi19.covers(Point(x,y)):
                rows.append(dict(tile_id=int(t.tile_id),path=t.path,cx=dx/TILE_M,cy=dy/TILE_M,
                                 box_m=BOX_M,width=n,height=n,gsd=gsd,
                                 row_off=int(t.row_off),col_off=int(t.col_off)))
pm=pd.DataFrame(rows)
assert len(pm),"No grid prompts inside ROI"
pm.insert(0,"prompt_id",np.arange(len(pm)))
pm.to_csv(work/"prompt_manifest.csv",index=False)
print(f"ROI {ROI_INDEX}: {roi19.area/10000:.3f} ha | tiles={len(tm)} | prompts={len(pm)} | mode={run_tag}")
print("Existing masks:",sum((raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id))

runner=Path("/content/run_sam3_kvo22.py")
runner.write_text(r'''
import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]); outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("CUDA GPU required")
torch.backends.cuda.matmul.allow_tf32=True
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path=bpe)
    processor=Sam3Processor(model)
    for tid,g in df.groupby("tile_id",sort=True):
        pending=[r for r in g.itertuples() if not (outdir/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not pending:continue
        img=Image.open(pending[0].path).convert("RGB")
        for r in tqdm(pending,desc=f"Tile {tid}",leave=False):
            # Fresh prompt state per box, to avoid accumulated prompts.
            state=processor.set_image(img)
            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))
            out=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(outdir/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"Tile {tid} complete",flush=True)
print("Inference complete",flush=True)
''')
missing=sum(not (raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id)
if missing:
    run_env=os.environ.copy()
    if "env" in globals(): run_env.update(env)
    print("Running SAM3 for",missing,"unsaved prompts")
    subprocess.run([SAMPY,"-u",str(runner),str(work/"prompt_manifest.csv"),str(raw)],
                   env=run_env,check=True)
else: print("All masks cached; skipping SAM3")

# Convert every returned instance, not just the highest-scoring mask.
features=[]
with rasterio.open(RASTER) as src:
    for r in pm.itertuples():
        p=raw/f"prompt_{int(r.prompt_id):06d}.npz"
        if not p.exists():continue
        with np.load(p) as z:
            masks=z["masks"]; scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        transform=src.window_transform(win)
        for k,m in enumerate(masks):
            m=np.asarray(m)>0
            if m.shape!=(n,n) or not m.any():continue
            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))
            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():
                continue
            geoms=[shape(g) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not (MIN_AREA_M2<=geom.area<=MAX_AREA_M2):continue
            if not roi19.covers(geom.representative_point()):continue
            features.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),
                                 instance=int(k),score=float(scores[min(k,len(scores)-1)]),
                                 area_m2=float(geom.area),geometry=geom))
print("Valid raw proposals:",len(features))
assert features,"No polygons survived filtering; inspect tile-edge masks"
allpred=gpd.GeoDataFrame(features,geometry="geometry",crs=crs)
allpred.to_file(work/"all_proposals.gpkg",driver="GPKG")

# Spatial-index-assisted conservative duplicate removal.
ordered=allpred.sort_values("score",ascending=False).reset_index(drop=True)
keep=[]; removed=[]
for i,r in ordered.iterrows():
    g=r.geometry; duplicate=False
    for j in keep:
        h=ordered.geometry.iloc[j]
        if not g.intersects(h):continue
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        containment=inter/min(g.area,h.area)
        ratio=min(g.area,h.area)/max(g.area,h.area)
        if iou>=DUP_IOU or (containment>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):
            duplicate=True;removed.append(dict(prompt_id=int(r.prompt_id),against=int(j),iou=iou,
                                                containment=containment,area_ratio=ratio))
            break
    if not duplicate:keep.append(i)
clean=ordered.iloc[keep].copy().reset_index(drop=True)
clean["crown_id"]=np.arange(1,len(clean)+1)
clean.to_file(work/"clean_crowns.gpkg",driver="GPKG")
pd.DataFrame(removed).to_csv(work/"removed_duplicates.csv",index=False)
print(f"Clean predictions: {len(clean)} | removed duplicates: {len(removed)}")

# Evaluate only reference crowns fully contained in the processed tile union
# (and ROI). This prevents partial smoke-test coverage being labelled full-ROI.
footprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]
from shapely.ops import unary_union
covered=unary_union(footprints).intersection(roi19)
reference=reference_v2.to_crs(crs).copy()
reference=reference[reference.geometry.apply(covered.covers)].copy().reset_index(drop=True)
mat=np.zeros((len(reference),len(clean)),dtype=np.float32)
idx=clean.sindex
for i,g in enumerate(reference.geometry):
    for j in idx.query(g,predicate="intersects"):
        h=clean.geometry.iloc[j]; inter=g.intersection(h).area
        union=g.area+h.area-inter
        if union>0:mat[i,j]=inter/union
matches=[]
if mat.size:
    utility=(mat>=.5).astype(float)*1000+mat
    ii,jj=linear_sum_assignment(utility,maximize=True)
    matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=.5]
matched_ref={i for i,j,v in matches}
reference["area_m2"]=reference.geometry.area
reference["matched"]=[i in matched_ref for i in range(len(reference))]
reference["best_iou"]=mat.max(axis=1) if len(clean) else 0
reference["size_class"]=pd.cut(reference.area_m2,[0,20,40,80,np.inf],
                               labels=["<20","20–40","40–80",">80"])
size=reference.groupby("size_class",observed=True).agg(
    n=("matched","size"),matched=("matched","sum"),recall=("matched","mean"),
    median_best_iou=("best_iou","median"))
size.to_csv(work/"recall_by_size.csv")
display(size)
recall=len(matches)/len(reference) if len(reference) else float("nan")
precision=len(matches)/len(clean) if len(clean) else float("nan")
summary=dict(roi_index=ROI_INDEX,mode=run_tag,roi_area_ha=roi19.area/10000,
             tiles=len(tm),prompts=len(pm),manual_crowns=len(reference),
             raw_proposals=len(allpred),clean_predictions=len(clean),
             matched_iou_05=len(matches),recall_05=recall,precision_05=precision,
             median_matched_iou=float(np.median([v for _,_,v in matches])) if matches else None)
(work/"summary.json").write_text(json.dumps(summary,indent=2))
print("RESULTS:",json.dumps(summary,indent=2))

print("BLOCK 22 COMPLETE. Run Block 22V for full statistical validation:",work)


## 22V — Statistical validation (no GPU)

Run after Block 22 completes. No SAM3 rerun; exports CSVs in `experiments_v22_full_roi_8m_0/full/validation/`.


In [ ]:
# BLOCK 22V — Full ROI statistical validation: manual crowns vs SAM3
# CPU only; run 0R -> 14 -> 22V after Block 22. No SAM3 inference.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from scipy.optimize import linear_sum_assignment
from shapely.ops import unary_union
from shapely.geometry import box
import json
ROI_INDEX=0
NEW=OUT/f"experiments_v22_full_roi_8m_{ROI_INDEX}"/"full"
OLD=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
V=NEW/"validation";V.mkdir(parents=True,exist_ok=True)
assert (NEW/"clean_crowns.gpkg").exists(),"Run Block 22 full inference first"
assert (OLD/"clean_crowns.gpkg").exists(),"Block 19 baseline missing"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14"
with rasterio.open(RASTER) as src:crs=src.crs
roi=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
ref=reference_v2.to_crs(crs)
# Only fully-contained reference crowns, common to both full-ROI runs.
ref=ref[ref.geometry.apply(roi.covers)].copy().reset_index(drop=True)
ref["manual_id"]=np.arange(len(ref))
ref["manual_area_m2"]=ref.geometry.area
ref["size_class"]=pd.cut(ref.manual_area_m2,[0,20,40,80,np.inf],
                           labels=["<20","20-40","40-80",">80"])
thresholds=[.3,.5,.7]
records=[];size_rows=[];manual_rows=[];match_rows=[];pred_rows=[]
for name,source in [("Block19_10m_24m",OLD),("Block22_8m_40m",NEW)]:
    p=gpd.read_file(source/"clean_crowns.gpkg").to_crs(crs).reset_index(drop=True)
    # Filter prediction representative points to ROI consistently.
    p=p[p.geometry.apply(lambda g:roi.covers(g.representative_point()))].reset_index(drop=True)
    ix=p.sindex
    mat=np.zeros((len(ref),len(p)),dtype=np.float32)
    overlap_counts=np.zeros(len(ref),dtype=int)
    covered=np.zeros(len(ref),dtype=float)
    best_pred=np.full(len(ref),-1,dtype=int)
    # Coverage union is calculated per manual crown, not by summing overlaps.
    for i,g in enumerate(ref.geometry):
        js=list(ix.query(g,predicate="intersects"))
        fragments=[]
        for j in js:
            h=p.geometry.iloc[j];inter=g.intersection(h)
            a=inter.area
            if a<=0:continue
            mat[i,j]=a/(g.area+h.area-a)
            if a/g.area>=.05:overlap_counts[i]+=1
            fragments.append(inter)
        if fragments:covered[i]=unary_union(fragments).area/g.area
        if len(p):best_pred[i]=int(np.argmax(mat[i]))
    best=mat.max(axis=1) if len(p) else np.zeros(len(ref))
    for thr in thresholds:
        if mat.size:
            utility=(mat>=thr).astype(float)*1000+mat
            ii,jj=linear_sum_assignment(utility,maximize=True)
            matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=thr]
        else:matches=[]
        records.append(dict(model=name,threshold=thr,manual=len(ref),predictions=len(p),
                            matched=len(matches),recall=len(matches)/len(ref) if len(ref) else np.nan,
                            reference_matched_prediction_fraction=len(matches)/len(p) if len(p) else np.nan,
                            median_matched_iou=np.median([v for _,_,v in matches]) if matches else np.nan))
        for cls,group in ref.groupby("size_class",observed=True):
            ids=set(group.index)
            m=sum(i in ids for i,j,v in matches)
            size_rows.append(dict(model=name,threshold=thr,size_class=str(cls),
                                  manual=len(group),matched=m,recall=m/len(group)))
        if thr==.5:
            for i,j,iou in matches:
                a=ref.geometry.iloc[i].area;b=p.geometry.iloc[j].area
                match_rows.append(dict(model=name,manual_id=i,pred_id=j,iou=iou,
                                       manual_area_m2=a,pred_area_m2=b,
                                       area_ratio=b/a,relative_area_error=(b-a)/a))
    for i in range(len(ref)):
        manual_rows.append(dict(model=name,manual_id=i,size_class=str(ref.size_class.iloc[i]),
                                manual_area_m2=float(ref.manual_area_m2.iloc[i]),
                                best_iou=float(best[i]),combined_coverage=float(covered[i]),
                                significant_overlap_count=int(overlap_counts[i]),
                                best_prediction_id=int(best_pred[i])))
    # Flag predictions overlapping no manual crown; not necessarily false positives
    for j,g in enumerate(p.geometry):
        pred_rows.append(dict(model=name,pred_id=j,area_m2=g.area,
                              best_reference_iou=float(mat[:,j].max()) if len(ref) else 0,
                              intersects_reference=bool((mat[:,j]>0).any()) if len(ref) else False))
metrics=pd.DataFrame(records);by_size=pd.DataFrame(size_rows)
diagnostics=pd.DataFrame(manual_rows);matches=pd.DataFrame(match_rows)
pred_diagnostics=pd.DataFrame(pred_rows)
metrics.to_csv(V/"iou_threshold_comparison.csv",index=False)
by_size.to_csv(V/"recall_by_size.csv",index=False)
diagnostics.to_csv(V/"manual_crown_fragmentation.csv",index=False)
matches.to_csv(V/"one_to_one_matches_iou05.csv",index=False)
pred_diagnostics.to_csv(V/"prediction_reference_overlap.csv",index=False)
print("Full-ROI manual crowns:",len(ref))
display(metrics.round(3))
display(by_size[by_size.threshold==.5].round(3))
if len(matches):
    print("Matched crown area bias at IoU 0.5:")
    display(matches.groupby("model").agg(n=("iou","size"),
        median_area_ratio=("area_ratio","median"),
        median_relative_area_error=("relative_area_error","median")).round(3))
print("Saved validation tables:",V)
print("Note: unmatched SAM3 polygons may be genuine unannotated crowns.")


## 23 — Statistical proposal-selection benchmark (CPU only)

Run `0R → 14 → 23`. Uses the existing full-ROI Block 22 SAM3 polygons. No DSM, no new inference. Tunes confidence and overlap/nesting suppression on a western spatial reference subset, locks the parameters, and evaluates them on the eastern subset with a 6 m split buffer. Produces `spatial_holdout_comparison.csv`, `training_parameter_sweep.csv`, `selected_parameters.csv`, and `selected_full_roi.gpkg`. Manual crowns are never used to choose individual polygons. This is a within-ROI holdout, not a separate-site validation.


In [ ]:
# BLOCK 23 — CPU-only proposal selection benchmark, full ROI 0
# Run 0R -> 14 -> 23. No DSM, no SAM3 rerun, no modification of original polygons.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from shapely.geometry import box
ROI_INDEX=0
SOURCE=OUT/f"experiments_v22_full_roi_8m_{ROI_INDEX}"/"full"/"clean_crowns.gpkg"
DEST=OUT/f"experiments_v23_proposal_selection_roi_{ROI_INDEX}"
DEST.mkdir(parents=True,exist_ok=True)
assert SOURCE.exists(),"Run Block 22 first"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14"
with rasterio.open(RASTER) as src:crs=src.crs
roi=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
pred=gpd.read_file(SOURCE).to_crs(crs).reset_index(drop=True)
ref=reference_v2.to_crs(crs)
ref=ref[ref.geometry.apply(roi.covers)].copy().reset_index(drop=True)
assert len(pred) and len(ref),"Empty input"
pred["selection_id"]=np.arange(len(pred))
pred["area_m2"]=pred.geometry.area
pred["compactness"]=4*np.pi*pred.area_m2/(pred.geometry.length**2).clip(lower=1e-8)
if "score" not in pred.columns:
    raise ValueError("SAM3 score column missing; cannot benchmark confidence filters")
pred["score"]=pd.to_numeric(pred["score"],errors="coerce")
print("SAM3 score distribution:",pred.score.describe().round(3).to_dict())
# Split by reference crown spatial location. Spatial blocks reduce train/test leakage.
# Thresholds are selected only from the west side; east is untouched until final evaluation.
split=float(ref.geometry.centroid.x.median())
train_idx=np.flatnonzero(ref.geometry.centroid.x.to_numpy()<split)
test_idx=np.flatnonzero(ref.geometry.centroid.x.to_numpy()>=split)
assert len(train_idx)>20 and len(test_idx)>20
print(f"Manual reference: train {len(train_idx)}, test {len(test_idx)}; east/west spatial split x={split:.1f}")
# Precompute IoU for ALL predictions. Reference outlines never guide filtering.
mat=np.zeros((len(ref),len(pred)),dtype=np.float32)
pidx=pred.sindex
for i,g in enumerate(ref.geometry):
    for j in pidx.query(g,predicate="intersects"):
        h=pred.geometry.iloc[j];inter=g.intersection(h).area
        if inter>0:mat[i,j]=inter/(g.area+h.area-inter)
# A candidate selected anywhere in ROI may be matched to references in its evaluation subset.
def metric(ref_ids,selected,thr=.5):
    m=mat[np.ix_(ref_ids,selected)]
    if not m.size:return dict(n_ref=len(ref_ids),n_predictions=len(selected),matched=0,recall=0.,matched_fraction=0.)
    utility=(m>=thr).astype(float)*1000+m
    ii,jj=linear_sum_assignment(utility,maximize=True)
    good=np.array([m[i,j]>=thr for i,j in zip(ii,jj)],dtype=bool)
    count=int(good.sum())
    return dict(n_ref=len(ref_ids),n_predictions=len(selected),matched=count,
                recall=count/len(ref_ids),matched_fraction=count/len(selected) if len(selected) else 0.)
# Spatial filtering: only candidate centres within a half-ROI count toward that half's
# candidate count. We still use full prediction polygons for IoU.
px=pred.geometry.centroid.x.to_numpy()
# Use a 6m exclusion buffer at the split for parameter selection and evaluation,
# avoiding nearby training/test reference crowns.
buffer_m=6.
train_idx=np.array([i for i in train_idx if ref.geometry.iloc[i].centroid.x<split-buffer_m])
test_idx=np.array([i for i in test_idx if ref.geometry.iloc[i].centroid.x>split+buffer_m])
assert len(train_idx)>15 and len(test_idx)>15,"Not enough crowns after spatial buffer"
print("After split buffer: train",len(train_idx),"test",len(test_idx))
train_candidates=np.flatnonzero(px<split-buffer_m)
test_candidates=np.flatnonzero(px>split+buffer_m)
# Cache polygon-pair overlap for NMS; no manual reference geometry used.
pair_iou={};pair_contain={}
idx=pred.sindex
for i,g in enumerate(pred.geometry):
    for j in idx.query(g,predicate="intersects"):
        j=int(j)
        if j<=i:continue
        h=pred.geometry.iloc[j];a=g.intersection(h).area
        if a<=0:continue
        pair_iou[i,j]=a/(g.area+h.area-a)
        pair_contain[i,j]=a/min(g.area,h.area)
def select(candidate_ids,score_cut,nms_iou,containment,rank_mode):
    ids=[int(i) for i in candidate_ids if np.isfinite(pred.score.iloc[i]) and pred.score.iloc[i]>=score_cut]
    if rank_mode=="score":
        ids.sort(key=lambda i:(float(pred.score.iloc[i]),float(pred.area_m2.iloc[i])),reverse=True)
    else:
        # Soft preference for plausible-sized, compact proposals, NOT fitted to labels.
        ids.sort(key=lambda i:(float(pred.score.iloc[i])+.08*float(pred.compactness.iloc[i])
                 -.03*abs(np.log(max(float(pred.area_m2.iloc[i]),1)/40))),reverse=True)
    kept=[]
    for i in ids:
        reject=False
        for j in kept:
            k=(min(i,j),max(i,j))
            if pair_iou.get(k,0)>=nms_iou or pair_contain.get(k,0)>=containment:
                reject=True;break
        if not reject:kept.append(i)
    return np.array(kept,dtype=int)
# Score quantiles, not absolute score assumptions.
q=np.unique(np.quantile(pred.score.dropna(),[0,.1,.25,.4,.55,.7]))
settings=[]
for cutoff in q:
    for iou,contain in [(1.01,1.01),(.7,1.01),(.5,1.01),(.5,.95),(.35,.9)]:
        for rank in ["score","score_shape"]:
            settings.append(dict(cutoff=float(cutoff),nms_iou=iou,containment=contain,rank=rank))
rows=[]
for k,cfg in enumerate(settings):
    selected=select(train_candidates,cfg["cutoff"],cfg["nms_iou"],cfg["containment"],cfg["rank"])
    m=metric(train_idx,selected)
    rows.append(dict(setting_id=k,**cfg,**m))
tuning=pd.DataFrame(rows)
tuning.to_csv(DEST/"training_parameter_sweep.csv",index=False)
baseline=metric(train_idx,train_candidates)
# Conservative optimization: maximize recall first, prefer fewer proposals only
# among configurations retaining at least 95% of unfiltered training matches.
target=.95*baseline["matched"]
eligible=tuning[tuning.matched>=target]
if len(eligible):
    chosen=eligible.sort_values(["n_predictions","recall"],ascending=[True,False]).iloc[0]
else:
    chosen=tuning.sort_values(["matched","n_predictions"],ascending=[False,True]).iloc[0]
cfg=dict(cutoff=float(chosen.cutoff),nms_iou=float(chosen.nms_iou),
         containment=float(chosen.containment),rank=str(chosen["rank"]))
print("\nSelected on training region only:",cfg)
print("Unfiltered training:",baseline)
print("Filtered training:",metric(train_idx,select(train_candidates,**dict(
    score_cut=cfg["cutoff"],nms_iou=cfg["nms_iou"],containment=cfg["containment"],
    rank_mode=cfg["rank"]))))
# Locked settings applied to independent east test region.
test_sel=select(test_candidates,score_cut=cfg["cutoff"],nms_iou=cfg["nms_iou"],
                containment=cfg["containment"],rank_mode=cfg["rank"])
train_sel=select(train_candidates,score_cut=cfg["cutoff"],nms_iou=cfg["nms_iou"],
                 containment=cfg["containment"],rank_mode=cfg["rank"])
comparisons=[]
for region,refs,candidates,selected in [
    ("training_west",train_idx,train_candidates,train_sel),
    ("heldout_east",test_idx,test_candidates,test_sel)]:
    for method,ids in [("unfiltered",candidates),("selected",selected)]:
        for thr in [.3,.5,.7]:
            comparisons.append(dict(region=region,method=method,iou_threshold=thr,
                                    **metric(refs,ids,thr)))
comparison=pd.DataFrame(comparisons)
comparison.to_csv(DEST/"spatial_holdout_comparison.csv",index=False)
display(comparison.round(3))
# Save full-ROI application of locked parameters, no use of test labels.
full_ids=select(np.arange(len(pred)),score_cut=cfg["cutoff"],
                nms_iou=cfg["nms_iou"],containment=cfg["containment"],rank_mode=cfg["rank"])
selected_gdf=pred.iloc[full_ids].copy().reset_index(drop=True)
selected_gdf.to_file(DEST/"selected_full_roi.gpkg",driver="GPKG")
pd.DataFrame([cfg]).to_csv(DEST/"selected_parameters.csv",index=False)
print("\nFull ROI:",len(pred),"input proposals ->",len(full_ids),"selected proposals")
print("Outputs:",DEST)
print("Caution: training/test are spatial halves of one ROI, not independent forest sites.")
print("Unmatched polygons are not necessarily false detections.")


## 24 — GIS topology and crown geometry optimization (CPU only)

Run **0R → 14 → 24**. Tests multipart/sliver cleanup, small-candidate area thresholds, morphological opening and nested-polygon removal (retain larger versus smaller). Selects parameters on western manual crowns and reports performance on spatially held-out eastern crowns. **No SAM3 inference, DSM or GPU.** Original Block 22 polygons are unchanged.


In [ ]:
# BLOCK 24 — GIS topology / crown geometry benchmark (CPU only)
# Run 0R -> 14 -> 24. Uses Block 22 ORIGINAL 879 polygons, not Block 23 output.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from shapely.geometry import Polygon, MultiPolygon
from scipy.optimize import linear_sum_assignment
from shapely.validation import make_valid
from itertools import product
SRC=OUT/"experiments_v22_full_roi_8m_0"/"full"/"clean_crowns.gpkg"
DST=OUT/"experiments_v24_gis_cleanup_roi_0";DST.mkdir(parents=True,exist_ok=True)
assert SRC.exists(),"Run Block 22 first"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14"
with rasterio.open(RASTER) as ds:crs=ds.crs
roi=roi_v2.to_crs(crs).geometry.iloc[0]
P=gpd.read_file(SRC).to_crs(crs).reset_index(drop=True)
R=reference_v2.to_crs(crs)
R=R[R.geometry.apply(roi.covers)].reset_index(drop=True)
P["pid"]=np.arange(len(P))
score_col=next((c for c in ["score","confidence","predicted_iou"] if c in P),None)
scores=pd.to_numeric(P[score_col],errors="coerce").fillna(0).to_numpy() if score_col else np.zeros(len(P))
def polys(g):
    if g is None or g.is_empty:return []
    if g.geom_type=="Polygon":return [g]
    if hasattr(g,"geoms"):return [p for x in g.geoms for p in polys(x)]
    return []
def repair(g):
    try:return make_valid(g) if not g.is_valid else g
    except Exception:return g.buffer(0)
orig=[repair(g) for g in P.geometry]
def clean(g,component_min=0,opening=0):
    parts=[p for p in polys(g) if p.area>=component_min]
    if not parts:return None
    # Remove disconnected fragments; never combine separated parts into a single crown.
    g=max(parts,key=lambda x:x.area)
    if opening:
        try:
            t=g.buffer(-opening,join_style=1)
            if t.is_empty:return None
            t=t.buffer(opening,join_style=1)
            parts=polys(repair(t))
            if not parts:return None
            g=max(parts,key=lambda x:x.area)
        except Exception:return None
    return repair(g)
# Precompute a small, explicit grid of geometry-only cleanup choices.
geoms={}
for comp,opening in product([0,.5,1.0],[0,.15,.3]):
    geoms[(comp,opening)]=[clean(g,comp,opening) for g in orig]
# Fixed spatial holdout. Reference outlines never influence which polygons get selected.
mid=float(np.median(R.geometry.centroid.x))
rx=R.geometry.centroid.x.to_numpy()
px=P.geometry.centroid.x.to_numpy()
buffer_m=6
west=np.flatnonzero(rx<mid-buffer_m);east=np.flatnonzero(rx>mid+buffer_m)
west_p=np.flatnonzero(px<mid-buffer_m);east_p=np.flatnonzero(px>mid+buffer_m)
assert min(len(west),len(east))>15
def select(gs,ids,min_area=0,contain=.99,policy="larger"):
    valid=[int(i) for i in ids if gs[i] is not None and gs[i].area>=min_area]
    if contain>1:return valid
    # Keep larger or higher-confidence polygon in highly nested pairs.
    if policy=="larger":order=sorted(valid,key=lambda i:(gs[i].area,scores[i]),reverse=True)
    elif policy=="smaller":order=sorted(valid,key=lambda i:(-gs[i].area,scores[i]),reverse=True)
    else:order=sorted(valid,key=lambda i:(scores[i],gs[i].area),reverse=True)
    kept=[];removed=set()
    for i in order:
        if i in removed:continue
        kept.append(i)
        a=gs[i]
        for j in order:
            if j==i or j in removed or j in kept:continue
            b=gs[j]
            if not a.intersects(b):continue
            overlap=a.intersection(b).area/min(a.area,b.area)
            if overlap>=contain:removed.add(j)
    return kept
def evaluate(gs,ref_ids,ids,thr=.5):
    ids=list(ids);ref_ids=list(ref_ids)
    if not ids:return dict(matched=0,recall=0.,median_iou=np.nan,n_predictions=0)
    sg=gpd.GeoSeries([gs[j] for j in ids],crs=crs)
    si=sg.sindex
    M=np.zeros((len(ref_ids),len(ids)),dtype=np.float32)
    for k,rid in enumerate(ref_ids):
        g=R.geometry.iloc[rid]
        for j in si.query(g,predicate="intersects"):
            h=sg.iloc[j];inter=g.intersection(h).area
            if inter>0:M[k,j]=inter/(g.area+h.area-inter)
    utility=(M>=thr).astype(np.float32)*1000+M
    ii,jj=linear_sum_assignment(utility,maximize=True)
    values=np.array([M[i,j] for i,j in zip(ii,jj)])
    hits=values[values>=thr]
    return dict(matched=len(hits),recall=len(hits)/len(ref_ids),
                median_iou=float(np.median(hits)) if len(hits) else np.nan,n_predictions=len(ids))
# Tune on west: preserve >=95% of baseline matches, minimize number of proposals.
baseline=evaluate(orig,west,west_p)
rows=[]
for (comp,opening),gs in geoms.items():
    for min_area,contain,policy in product([0,1,3],[1.01,.99,.95],["larger","smaller"]):
        ids=select(gs,west_p,min_area,contain,policy)
        m=evaluate(gs,west,ids)
        rows.append(dict(component_min_m2=comp,opening_m=opening,min_area_m2=min_area,
                         containment=contain,policy=policy,**m))
tune=pd.DataFrame(rows)
tune.to_csv(DST/"training_parameter_sweep.csv",index=False)
eligible=tune[tune.matched>=.95*baseline["matched"]]
if eligible.empty:
    best=tune.sort_values(["matched","n_predictions"],ascending=[False,True]).iloc[0]
else:
    best=eligible.sort_values(["n_predictions","matched"],ascending=[True,False]).iloc[0]
cfg={k:best[k] for k in ["component_min_m2","opening_m","min_area_m2","containment","policy"]}
gs=geoms[(float(cfg["component_min_m2"]),float(cfg["opening_m"]))]
def ids_for(region):return select(gs,region,float(cfg["min_area_m2"]),float(cfg["containment"]),str(cfg["policy"]))
rows=[]
for region,refs,candidates in [("training_west",west,west_p),("heldout_east",east,east_p)]:
    for method,gg,ids in [("original",orig,list(candidates)),("GIS_cleaned",gs,ids_for(candidates))]:
        for thr in [.3,.5,.7]:
            rows.append(dict(region=region,method=method,iou_threshold=thr,
                             n_ref=len(refs),**evaluate(gg,refs,ids,thr)))
results=pd.DataFrame(rows)
results.to_csv(DST/"spatial_holdout_comparison.csv",index=False)
pd.DataFrame([cfg]).to_csv(DST/"selected_parameters.csv",index=False)
# Apply fixed settings to full ROI, without consulting east/test labels.
full_ids=ids_for(np.arange(len(P)))
cleaned=P.iloc[full_ids].copy().reset_index(drop=True)
cleaned.geometry=gpd.GeoSeries([gs[i] for i in full_ids],index=cleaned.index,crs=crs)
cleaned=cleaned[cleaned.geometry.notna() & ~cleaned.geometry.is_empty]
cleaned.to_file(DST/"selected_full_roi.gpkg",driver="GPKG")
print("Block 24: input polygons",len(P),"manual crowns",len(R))
print("Train/test crowns",len(west),len(east),"6m spatial buffer")
print("Training baseline IoU>=.5",baseline)
print("Selected settings:",cfg)
display(results.round(3))
print("Full ROI:",len(P),"->",len(cleaned),"polygons")
print("Saved:",DST)
print("Important: candidate selection uses geometry only; unmatched polygons may be real trees.")


## 25 — Learn to rank SAM3 crown proposals (CPU only)

**Fresh Colab:** run **Block 0R → Block 14 → Block 25**. Requires the saved Block 22 `clean_crowns.gpkg`; **do not rerun Block 22 or install SAM3**. Uses geometry, confidence, overlap and RGB polygon statistics. Labels high-IoU candidates as positive and only clearly conflicting proposals as negative; all others remain uncertain. Train on western proposals and report crown recall on held-out eastern crowns. Outputs scored GeoPackages, feature importances and held-out evaluation. Note: candidate discrimination on the labeled subset does not establish precision over all proposals.


In [ ]:
# BLOCK 25 — Crown proposal classifier; CPU only; run 0R -> 14 -> 25
# Uses ORIGINAL Block 22 proposals; never runs SAM3. Test labels are never used for training.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds, Window
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib
from shapely.validation import make_valid
import matplotlib.pyplot as plt
SRC=OUT/"experiments_v22_full_roi_8m_0"/"full"/"clean_crowns.gpkg"
DEST=OUT/"experiments_v25_proposal_classifier_roi_0";DEST.mkdir(parents=True,exist_ok=True)
assert SRC.exists(),"Block 22 saved proposals missing"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14 first"
with rasterio.open(RASTER) as src: crs=src.crs
P=gpd.read_file(SRC).to_crs(crs).reset_index(drop=True)
R=reference_v2.to_crs(crs)
roi=roi_v2.to_crs(crs).geometry.iloc[0]
R=R[R.geometry.apply(roi.covers)].reset_index(drop=True)
P["proposal_id"]=np.arange(len(P))
assert len(P)>0 and len(R)>0
if "score" not in P.columns:
    print("WARNING: score missing; using constant confidence")
    P["score"]=0.5
P["score"]=pd.to_numeric(P["score"],errors="coerce").fillna(.5)
# 1. Unsupervised proposal geometry and overlap features
areas=P.geometry.area.to_numpy()
perim=P.geometry.length.to_numpy()
features=pd.DataFrame({"proposal_id":P.proposal_id,
    "score":P.score.to_numpy(),"area_m2":areas,"log_area":np.log1p(areas),
    "perimeter_m":perim,"compactness":4*np.pi*areas/np.maximum(perim**2,1e-8),
    "bbox_fill":areas/np.maximum(np.array([g.envelope.area for g in P.geometry]),1e-8)})
n=len(P);max_iou=np.zeros(n);max_contain=np.zeros(n);n_overlap=np.zeros(n,dtype=int)
n_nested=np.zeros(n,dtype=int);higher_score_overlap=np.zeros(n,dtype=int)
si=P.sindex
for i,g in enumerate(P.geometry):
    for j in si.query(g,predicate="intersects"):
        j=int(j)
        if j<=i:continue
        h=P.geometry.iloc[j];inter=g.intersection(h).area
        if inter<=0:continue
        ov=inter/(areas[i]+areas[j]-inter)
        cont=inter/max(min(areas[i],areas[j]),1e-8)
        max_iou[i]=max(max_iou[i],ov);max_iou[j]=max(max_iou[j],ov)
        max_contain[i]=max(max_contain[i],cont);max_contain[j]=max(max_contain[j],cont)
        n_overlap[i]+=1;n_overlap[j]+=1
        if cont>=.9:n_nested[i]+=1;n_nested[j]+=1
        if P.score.iloc[j]>P.score.iloc[i]:higher_score_overlap[i]+=1
        if P.score.iloc[i]>P.score.iloc[j]:higher_score_overlap[j]+=1
features["max_proposal_iou"]=max_iou;features["max_containment"]=max_contain
features["n_overlaps"]=n_overlap;features["n_nested"]=n_nested
features["n_higher_score_overlaps"]=higher_score_overlap
# 2. RGB polygon statistics; raster window per polygon, no DSM.
rgb_mean=np.full((n,3),np.nan);rgb_sd=np.full((n,3),np.nan)
with rasterio.open(RASTER) as src:
    assert src.count>=3,"Expected RGB raster"
    for i,g in enumerate(P.geometry):
        try:
            win=from_bounds(*g.bounds,transform=src.transform).round_offsets().round_lengths()
            win=win.intersection(Window(0,0,src.width,src.height))
            if win.width<1 or win.height<1:continue
            # Downsample large windows to limit CPU and RAM
            step=max(1,int(np.ceil(max(win.width,win.height)/256)))
            arr=src.read([1,2,3],window=win)[:,::step,::step]
            transform=src.window_transform(win)*rasterio.Affine.scale(step,step)
            mask=geometry_mask([g],out_shape=arr.shape[1:],transform=transform,invert=True)
            if mask.sum()<3:continue
            vals=arr[:,mask].astype(np.float32)
            rgb_mean[i]=vals.mean(axis=1);rgb_sd[i]=vals.std(axis=1)
        except Exception as exc:
            if i<3:print("RGB sample warning:",exc)
for k,c in enumerate("rgb"):
    features[f"{c}_mean"]=rgb_mean[:,k]
    features[f"{c}_std"]=rgb_sd[:,k]
# 3. Reference matching for labels ONLY. Conservative negatives are competing
# proposals that substantially occupy a documented reference crown but fit poorly.
# Unmatched polygons elsewhere are UNLABELED, not assumed to be false crowns.
ref_index=R.sindex
best_iou=np.zeros(n);best_ref=np.full(n,-1,dtype=int)
pred_cover=np.zeros(n)
for i,g in enumerate(P.geometry):
    for j in ref_index.query(g,predicate="intersects"):
        j=int(j);h=R.geometry.iloc[j]
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        pred_cover[i]=max(pred_cover[i],inter/max(g.area,1e-8))
        if iou>best_iou[i]:best_iou[i]=iou;best_ref[i]=j
# Strict positive: best IoU>=.5; ambiguous .25-.5 omitted;
# negative: IoU<.25 AND >=70% candidate area overlaps an annotated crown.
labels=np.full(n,-1,dtype=int)
labels[best_iou>=.5]=1
labels[(best_iou<.25)&(pred_cover>=.70)]=0
# Split spatially, with a 6m no-training/test buffer
split=float(R.geometry.centroid.x.median());buf=6.
x=P.geometry.centroid.x.to_numpy()
train=(x<split-buf);test=(x>split+buf)
known=(labels>=0)
train_idx=np.flatnonzero(train&known);test_idx=np.flatnonzero(test&known)
print("Candidate labels: positive",int((labels==1).sum()),"negative",int((labels==0).sum()),
      "uncertain/unlabeled",int((labels<0).sum()))
print("Spatial train/test labeled:",len(train_idx),len(test_idx))
assert len(np.unique(labels[train_idx]))==2 and len(np.unique(labels[test_idx]))==2,(
    "Not enough reliable positive/negative examples on both sides; inspect label counts.")
# 4. Fit model on training only, no target-based tuning on held-out region.
cols=[c for c in features if c!="proposal_id"]
X=features[cols].replace([np.inf,-np.inf],np.nan)
fill=X.iloc[train_idx].median().fillna(0)
X=X.fillna(fill)
clf=RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
                            class_weight="balanced_subsample",random_state=42,n_jobs=-1)
clf.fit(X.iloc[train_idx],labels[train_idx])
proba=clf.predict_proba(X)[:,list(clf.classes_).index(1)]
features["crown_probability"]=proba;features["reference_label"]=labels
features["best_reference_iou_diagnostic_only"]=best_iou
features["spatial_partition"]=np.where(train,"train_west",np.where(test,"test_east","buffer"))
# 5. Test classifier discrimination only on reliable labeled held-out candidates.
print("Held-out labeled candidate ROC AUC:",round(roc_auc_score(labels[test_idx],proba[test_idx]),3))
print("Held-out labeled candidate average precision:",
      round(average_precision_score(labels[test_idx],proba[test_idx]),3))
# 6. Proposal selection: train-only threshold, calibrated to preserve 95% of
# positive training candidates. This is NOT full crown recall.
positive_train=proba[train_idx[labels[train_idx]==1]]
cutoff=float(np.quantile(positive_train,.05))
print("Training-derived classifier cutoff:",round(cutoff,3))
# Apply to entire ROI and retain all uncertain candidates in a separate audit layer.
selected=proba>=cutoff
# Compare one-to-one reference crown recall on spatial held-out side.
def recall(ref_ids,candidate_ids,threshold):
    if len(candidate_ids)==0:return 0
    rr=R.iloc[ref_ids];pp=P.iloc[candidate_ids]
    M=np.zeros((len(rr),len(pp)),dtype=np.float32)
    local_idx=pp.sindex
    for i,g in enumerate(rr.geometry):
        for j in local_idx.query(g,predicate="intersects"):
            h=pp.geometry.iloc[j];inter=g.intersection(h).area
            if inter>0:M[i,j]=inter/(g.area+h.area-inter)
    row,col=linear_sum_assignment((M>=threshold).astype(float)*1000+M,maximize=True)
    return int(sum(M[i,j]>=threshold for i,j in zip(row,col)))
rx=R.geometry.centroid.x.to_numpy()
held_ref=np.flatnonzero(rx>split+buf)
held_candidates=np.flatnonzero(test)
held_selected=np.flatnonzero(test&selected)
rows=[]
confidence_order=held_candidates[np.argsort(-P.score.to_numpy()[held_candidates],kind="stable")]
confidence_selected=confidence_order[:len(held_selected)]
for method,ids in [("original",held_candidates),("classifier_selected",held_selected),("confidence_same_count",confidence_selected)]:
    for t in [.3,.5,.7]:
        matched=recall(held_ref,ids,t)
        rows.append(dict(method=method,iou_threshold=t,manual=len(held_ref),
                         predictions=len(ids),matched=matched,recall=matched/len(held_ref)))
report=pd.DataFrame(rows)
display(report.round(3))
report.to_csv(DEST/"heldout_crown_recall.csv",index=False)
features.to_csv(DEST/"proposal_features_and_scores.csv",index=False)
P["crown_probability"]=proba
P["spatial_partition"]=features.spatial_partition.to_numpy()
P["selection_25"]=selected.astype(int)
P.to_file(DEST/"all_scored_proposals.gpkg",driver="GPKG")
P[selected].to_file(DEST/"classifier_selected_full_roi.gpkg",driver="GPKG")
pd.DataFrame({"feature":cols,"importance":clf.feature_importances_}).sort_values(
    "importance",ascending=False).to_csv(DEST/"feature_importance.csv",index=False)
print("Full ROI:",len(P),"->",int(selected.sum()),"selected")
print("Outputs:",DEST)
print("CAVEAT: negatives are partial/competing proposals, NOT all false crowns.")
print("CAVEAT: test split is within one ROI; next validate on an independent ROI.")

# Supplemental evaluation and reusable training artifacts.
report["reference_matched_prediction_fraction"]=report["matched"]/report["predictions"].clip(lower=1)
report["proposal_reduction_pct"]=100*(1-report["predictions"]/max(len(held_candidates),1))
report.to_csv(DEST/"heldout_crown_recall.csv",index=False)
size_rows=[]
for method,ids in [("original",held_candidates),("classifier_selected",held_selected),("confidence_same_count",confidence_selected)]:
    for lo,hi,label in [(0,20,"under_20"),(20,40,"20_to_40"),(40,80,"40_to_80"),(80,float("inf"),"over_80")]:
        group=np.array([i for i in held_ref if lo<=R.geometry.iloc[i].area<hi],dtype=int)
        if not len(group):continue
        for t in (.3,.5,.7):
            matched=recall(group,ids,t)
            size_rows.append(dict(method=method,size_class=label,iou_threshold=t,manual=len(group),matched=matched,recall=matched/len(group)))
pd.DataFrame(size_rows).to_csv(DEST/"heldout_recall_by_crown_size.csv",index=False)
features["label_confidence"]=np.where(labels==1,"reference_iou_ge_05",np.where(labels==0,"reference_conflict","unlabeled"))
features.to_csv(DEST/"proposal_features_and_scores.csv",index=False)
features[["proposal_id","reference_label","label_confidence","best_reference_iou_diagnostic_only","spatial_partition"]].to_csv(DEST/"training_labels_and_confidence.csv",index=False)
joblib.dump(dict(model=clf,feature_columns=cols,fill_values=fill,cutoff=cutoff,split_x=split,buffer_m=buf,random_state=42),DEST/"crown_candidate_rf.joblib")
print("Saved model, label categories, same-count confidence benchmark and size-stratified recall.")
print("Caution: threshold uses in-sample training scores, so may be optimistic.")


## 26 — Proposal-budget curves and overlap-aware selection (CPU only)

**Fresh runtime:** run **0R → 14 → 26**. Reads cached Block 25 scored polygons (`all_scored_proposals.gpkg`) and the original Block 22 candidates. Does **not** rerun SAM3, retrain the classifier, or require Block 2.

Tests fixed retention budgets (100, 90, 80, 70, 60, 50, 40, 30%) with three selection methods: classifier ranking, confidence ranking, and greedy overlap-aware classifier ranking. Overlap-aware ranking suppresses candidates only above a specified pairwise IoU and cannot promise tree-level uniqueness. Evaluation is exclusively on **76-style eastern held-out manual crowns** defined from Block 25's spatial split; report counts are generated from the actual cached geometries, not assumed.

**Scientific guardrails:** fixed method settings are exploratory and not chosen by held-out performance; curves should not be mined for an optimal threshold on this test split. Each reference is matched to at most one candidate through one-to-one bipartite matching; reference-matched prediction fraction is **not** true precision. Training-side candidates are excluded from held-out prediction evaluation. Outputs are new Block 26 CSVs and geopackages, leaving Blocks 22 and 25 unchanged.


In [ ]:
# BLOCK 26 — CPU-only ranked proposal-budget curves; run 0R -> 14 -> 26
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

SRC25 = OUT / "experiments_v25_proposal_classifier_roi_0" / "all_scored_proposals.gpkg"
SRC22 = OUT / "experiments_v22_full_roi_8m_0" / "full" / "clean_crowns.gpkg"
DEST26 = OUT / "experiments_v26_ranked_selection_roi_0"
DEST26.mkdir(parents=True, exist_ok=True)
assert SRC25.exists(), f"Missing {SRC25}. Run Block 25 once to cache scores."
assert SRC22.exists(), f"Missing {SRC22}"
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14 first"
with rasterio.open(RASTER) as src:
    crs26 = src.crs
P26 = gpd.read_file(SRC25).to_crs(crs26).reset_index(drop=True)
B26 = gpd.read_file(SRC22).to_crs(crs26).reset_index(drop=True)
assert len(P26) == len(B26), "Block 25 output and Block 22 proposal counts differ."
assert "crown_probability" in P26 and "score" in P26, "Cached model score columns missing."
# Verify stable input ordering before comparing to original proposal geometry.
assert all(a.equals(b) for a,b in zip(P26.geometry, B26.geometry)), (
    "Proposals were reordered or modified. Cannot safely compare with Block 22."
)
roi26 = roi_v2.to_crs(crs26).geometry.iloc[0]
R26 = reference_v2.to_crs(crs26)
R26 = R26[R26.geometry.apply(roi26.covers)].reset_index(drop=True)
# Reconstruct exactly the Block 25 spatial split.
split26 = float(R26.geometry.centroid.x.median())
buffer26 = 6.0
px26 = P26.geometry.centroid.x.to_numpy()
rx26 = R26.geometry.centroid.x.to_numpy()
test26 = np.flatnonzero(px26 > split26 + buffer26)
ref26 = np.flatnonzero(rx26 > split26 + buffer26)
assert len(test26)>0 and len(ref26)>0
# Candidate scores use no held-out reference geometry.
scores26 = pd.to_numeric(P26.crown_probability, errors="coerce").fillna(-1).to_numpy()
conf26 = pd.to_numeric(P26.score, errors="coerce").fillna(-1).to_numpy()
# Make reference-to-prediction IoU matrix once for efficient repeated budget evaluation.
Rtest26 = R26.iloc[ref26].reset_index(drop=True)
Ptest26 = P26.iloc[test26].reset_index(drop=True)
M26 = np.zeros((len(Rtest26), len(Ptest26)), dtype=np.float32)
rindex26 = Rtest26.sindex
for j,g in enumerate(Ptest26.geometry):
    for i in rindex26.query(g, predicate="intersects"):
        i=int(i)
        h=Rtest26.geometry.iloc[i]
        inter=g.intersection(h).area
        if inter>0:
            M26[i,j]=inter/max(g.area+h.area-inter,1e-9)
# Fixed-budget ranking without any reference-derived optimization.
def ranked26(index, values):
    return np.array(sorted(index, key=lambda i:(-float(values[i]),int(i))), dtype=int)

# Greedy NMS: only suppress when pairwise candidate IoU >= 0.5.
# Fill remaining budget with best-ranked previously suppressed candidates.
# This makes budgets comparable while maintaining an overlap preference.
def overlap_rank26(index, values, nms_iou=0.5):
    ordered=ranked26(index,values)
    pool=set(int(i) for i in index)
    si=P26.sindex
    kept=[]; deferred=[]
    for idx in ordered:
        g=P26.geometry.iloc[idx]
        rejected=False
        for j in si.query(g,predicate="intersects"):
            j=int(j)
            if j not in pool or j==idx or j not in kept:
                continue
            h=P26.geometry.iloc[j]
            intersection=g.intersection(h).area
            if intersection>0 and intersection/max(g.area+h.area-intersection,1e-9)>=nms_iou:
                rejected=True
                break
        (deferred if rejected else kept).append(int(idx))
    return np.array(kept+deferred,dtype=int)

rankings26 = {
    "classifier":ranked26(test26,scores26),
    "sam3_confidence":ranked26(test26,conf26),
    "overlap_aware_classifier":overlap_rank26(test26,scores26,nms_iou=0.5),
}
fractions26=[1.0,.9,.8,.7,.6,.5,.4,.3]
thresholds26=[.3,.5,.7]
pos26={int(idx):j for j,idx in enumerate(test26)}
def matched26(ids, threshold, ref_subset=None):
    rows=np.arange(len(Rtest26),dtype=int) if ref_subset is None else np.asarray(ref_subset,dtype=int)
    cols=np.array([pos26[int(i)] for i in ids],dtype=int)
    if not len(rows) or not len(cols):return 0
    sub=M26[np.ix_(rows,cols)]
    # One-to-one matching, prioritizing count above threshold over fine IoU.
    valid=(sub>=threshold).astype(np.float64)
    ii,jj=linear_sum_assignment(valid*2+sub.astype(np.float64),maximize=True)
    return int(np.count_nonzero(sub[ii,jj]>=threshold))

rows26=[]; size_rows26=[]
size_bins26=[(0,20,"under_20"),(20,40,"20_to_40"),(40,80,"40_to_80"),(80,np.inf,"over_80")]
for name,ranking in rankings26.items():
    for fraction in fractions26:
        k=max(1,int(np.ceil(len(test26)*fraction)))
        chosen=ranking[:k]
        for threshold in thresholds26:
            hits=matched26(chosen,threshold)
            rows26.append(dict(method=name,retained_fraction_target=fraction,
                predictions=k,original_predictions=len(test26),reduction_pct=100*(1-k/len(test26)),
                iou_threshold=threshold,reference_crowns=len(ref26),
                matched=hits,recall=hits/len(ref26),
                reference_matched_prediction_fraction=hits/k))
            for lo,hi,label in size_bins26:
                subset=np.flatnonzero((Rtest26.geometry.area.to_numpy()>=lo)&(Rtest26.geometry.area.to_numpy()<hi))
                if len(subset):
                    matched=matched26(chosen,threshold,subset)
                    size_rows26.append(dict(method=name,retained_fraction_target=fraction,
                        size_class=label,iou_threshold=threshold,reference_crowns=len(subset),
                        matched=matched,recall=matched/len(subset)))
curves26=pd.DataFrame(rows26)
sizes26=pd.DataFrame(size_rows26)
curves26.to_csv(DEST26/"heldout_budget_recall_curves.csv",index=False)
sizes26.to_csv(DEST26/"heldout_budget_recall_by_size.csv",index=False)
# Explicit fixed 70%-retention example, not a tuned or recommended selection.
selection26=rankings26["overlap_aware_classifier"][:int(np.ceil(.7*len(test26)))]
P26.iloc[selection26].to_file(DEST26/"example_heldout_overlap_aware_70pct.gpkg",driver="GPKG")
# Save full-heldout rankings as CSV for audit and re-use.
pd.DataFrame({name:pd.Series(rank) for name,rank in rankings26.items()}).to_csv(
    DEST26/"heldout_candidate_rankings.csv",index=False)
fig,ax=plt.subplots(figsize=(8,5))
for name in rankings26:
    sub=curves26[(curves26.method==name)&(curves26.iou_threshold==.5)].sort_values("reduction_pct")
    ax.plot(sub.reduction_pct,sub.recall,marker="o",label=name)
ax.set(xlabel="Held-out proposal reduction (%)",ylabel="One-to-one crown recall (IoU ≥ 0.5)",
       title="Block 26 — held-out proposal-budget trade-off")
ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(DEST26/"heldout_recall_vs_reduction.png",dpi=150)
plt.show()
display(curves26[curves26.iou_threshold==.5].round(3))
print("Saved Block 26 outputs:",DEST26)
print("CAUTION: exploratory fixed-budget comparisons, not threshold selection on held-out references.")
print("CAUTION: baseline reference-matched fraction is not true precision.")


## 27 — Stricter one-to-one training labels and model comparison (CPU only)

**Fresh Colab:** run **Block 0R → Block 14 → Block 27**. Requires cached Block 25 `proposal_features_and_scores.csv` and `all_scored_proposals.gpkg`; no SAM3 environment, GPU, or rerun of Blocks 22–26.

**Scope:** ROI 0 only (not all six annotated ROIs). Train on western, spatially buffered subset; evaluate only eastern references/proposals. Enforce **one-to-one positive labels** on training-side manual crowns via maximum-cardinality IoU matching at 0.5; define conservative negatives as poor-IoU polygons almost entirely covering an annotated training crown; leave all other candidates *unlabeled*. Reference IoU and reference-derived label diagnostics are **not** model features. Compare a compact logistic regression and Random Forest with the previous Block 25 classifier and SAM3 confidence at fixed candidate budgets. Fixed hyperparameters and budgets: do not tune based on eastern results.

**Cautions:** a candidate that is unmatched to a manual crown is not necessarily a false tree; even high-overlap alternative polygons may be plausible trees. The held-out region has already been inspected in previous blocks and is an exploratory benchmark rather than a pristine final test. For a definitive generalization estimate, produce candidate inventories for other ROIs and reserve an independent ROI.


In [ ]:
# BLOCK 27 — Controlled one-to-one proposal-label experiment; CPU only
# Fresh runtime: 0R -> 14 -> 27. Cached Block 25 features required.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, joblib
from scipy.optimize import linear_sum_assignment
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

B25 = OUT/"experiments_v25_proposal_classifier_roi_0"
DEST27 = OUT/"experiments_v27_one_to_one_training_roi_0"
DEST27.mkdir(parents=True,exist_ok=True)
assert (B25/"all_scored_proposals.gpkg").exists(), "Run Block 25 once to save proposal scores."
assert (B25/"proposal_features_and_scores.csv").exists(), "Block 25 feature CSV missing."
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14."
with rasterio.open(RASTER) as src: crs27=src.crs
P27=gpd.read_file(B25/"all_scored_proposals.gpkg").to_crs(crs27).reset_index(drop=True)
F27=pd.read_csv(B25/"proposal_features_and_scores.csv")
assert len(P27)==len(F27) and "proposal_id" in F27
assert np.array_equal(F27.proposal_id.to_numpy(),np.arange(len(P27))), "Proposal feature order changed."
R27=reference_v2.to_crs(crs27)
roi27=roi_v2.to_crs(crs27).geometry.iloc[0]
R27=R27[R27.geometry.apply(roi27.covers)].reset_index(drop=True)
split27=float(R27.geometry.centroid.x.median())
buf27=6.0
train_p27=np.flatnonzero(P27.geometry.centroid.x.to_numpy()<split27-buf27)
test_p27=np.flatnonzero(P27.geometry.centroid.x.to_numpy()>split27+buf27)
train_r27=np.flatnonzero(R27.geometry.centroid.x.to_numpy()<split27-buf27)
test_r27=np.flatnonzero(R27.geometry.centroid.x.to_numpy()>split27+buf27)
assert min(map(len,[train_p27,test_p27,train_r27,test_r27]))>0

# For training labels, compare ONLY western proposals with western manual crowns.
def iou_matrix27(ref_ids,prop_ids):
    rr=R27.iloc[ref_ids].reset_index(drop=True)
    pp=P27.iloc[prop_ids].reset_index(drop=True)
    mm=np.zeros((len(rr),len(pp)),dtype=np.float32)
    coverage=np.zeros(len(pp),dtype=np.float32)
    si=rr.sindex
    for j,g in enumerate(pp.geometry):
        for i in si.query(g,predicate="intersects"):
            i=int(i);h=rr.geometry.iloc[i]
            area=g.intersection(h).area
            if area<=0:continue
            mm[i,j]=area/max(g.area+h.area-area,1e-9)
            coverage[j]=max(coverage[j],area/max(g.area,1e-9))
    return mm,coverage
TR27,COVER27=iou_matrix27(train_r27,train_p27)
labels27=np.full(len(P27),-1,dtype=np.int8)
# Maximum-cardinality one-to-one positive matching, then prioritize IoU.
ii,jj=linear_sum_assignment((TR27>=.5).astype(float)*2+TR27.astype(float),maximize=True)
positive_local=np.array([j for i,j in zip(ii,jj) if TR27[i,j]>=.5],dtype=int)
labels27[train_p27[positive_local]]=1
# Negatives must be mostly *inside* a reference crown yet poorly shaped to match it.
# Avoid treating unmatched crown candidates elsewhere as negatives.
best_train_iou=TR27.max(axis=0) if len(train_r27) else np.zeros(len(train_p27))
negative_local=np.flatnonzero((best_train_iou<.25)&(COVER27>=.70))
labels27[train_p27[negative_local]]=0
# Exclude ambiguous candidate polygons from fitting.
labeled_train=train_p27[labels27[train_p27]>=0]
n_positive=int((labels27[labeled_train]==1).sum())
n_negative=int((labels27[labeled_train]==0).sum())
print("ROI 0: proposals",len(P27),"reference crowns",len(R27))
print("West train crowns",len(train_r27),"East held-out crowns",len(test_r27))
print("One-to-one training labels: positive",n_positive,"negative",n_negative,
      "unlabeled western",int((labels27[train_p27]<0).sum()))
if min(n_positive,n_negative)<5:
    raise RuntimeError("Insufficient reliable labels for robust classifier; inspect west-side counts.")

# Only predictors that can be measured without knowing reference crowns.
# Explicit allowlist prevents reference-derived columns leaking into training.
feature_columns27=[
    "score","area_m2","log_area","perimeter_m","compactness","bbox_fill",
    "max_proposal_iou","max_containment","n_overlaps","n_nested",
    "n_higher_score_overlaps","r_mean","g_mean","b_mean",
    "r_std","g_std","b_std"
]
missing27=[c for c in feature_columns27 if c not in F27.columns]
assert not missing27,f"Missing cached Block 25 predictors: {missing27}"
X27=F27[feature_columns27].replace([np.inf,-np.inf],np.nan)
y27=labels27[labeled_train]
models27={
    "logistic_one_to_one":make_pipeline(
        SimpleImputer(strategy="median",add_indicator=False),StandardScaler(),
        LogisticRegression(C=1.,class_weight="balanced",max_iter=1000,random_state=42)),
    "rf_one_to_one":make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
                               class_weight="balanced_subsample",n_jobs=-1,random_state=42))
}
score_columns27={}
for key,model in models27.items():
    model.fit(X27.iloc[labeled_train],y27)
    score_columns27[key]=model.predict_proba(X27)[:,list(model.classes_).index(1)]
    joblib.dump({"model":model,"predictors":feature_columns27,"train_indices":labeled_train,
                 "split_x":split27,"buffer_m":buf27,"positive_iou":.5},
                DEST27/(key+".joblib"))
score_columns27["block25_classifier"]=pd.to_numeric(P27.crown_probability,errors="coerce").fillna(-1).to_numpy()
score_columns27["sam3_confidence"]=pd.to_numeric(P27.score,errors="coerce").fillna(-1).to_numpy()

# Strict held-out one-to-one evaluation. Held-out references NEVER influence selection.
TE27,_=iou_matrix27(test_r27,test_p27)
test_position27={int(v):j for j,v in enumerate(test_p27)}
def evaluate27(global_candidate_ids,t):
    jj=np.array([test_position27[int(v)] for v in global_candidate_ids],dtype=int)
    if len(jj)==0:return 0
    sub=TE27[:,jj].astype(float)
    rr,cc=linear_sum_assignment((sub>=t).astype(float)*2+sub,maximize=True)
    return int(np.count_nonzero(sub[rr,cc]>=t))
budgets27=[1.,.9,.8,.7,.6,.5,.4,.3]
thresholds27=[.3,.5,.7]
rows27=[]
for method,scores in score_columns27.items():
    ranking=np.array(sorted(test_p27,key=lambda i:(-float(scores[i]),int(i))),dtype=int)
    for budget in budgets27:
        k=max(1,int(np.ceil(budget*len(test_p27))))
        selected=ranking[:k]
        for t in thresholds27:
            matches=evaluate27(selected,t)
            rows27.append(dict(method=method,retained_fraction_target=budget,
                predictions=k,original_predictions=len(test_p27),
                proposal_reduction_pct=100*(1-k/len(test_p27)),
                iou_threshold=t,reference_crowns=len(test_r27),
                matched=matches,recall=matches/len(test_r27),
                reference_matched_prediction_fraction=matches/k))
    # Save rankings, not only a cherry-picked test threshold.
    pd.DataFrame({"proposal_id":ranking,
        "score":scores[ranking],"rank":np.arange(1,len(ranking)+1)}
        ).to_csv(DEST27/(method+"_heldout_ranking.csv"),index=False)
    P27[method+"_score"]=scores
evaluation27=pd.DataFrame(rows27)
evaluation27.to_csv(DEST27/"heldout_budget_comparison.csv",index=False)
labels_table27=pd.DataFrame({"proposal_id":np.arange(len(P27)),"training_label":labels27,
    "partition":np.where(np.isin(np.arange(len(P27)),train_p27),"train_west",
                  np.where(np.isin(np.arange(len(P27)),test_p27),"test_east","buffer")),
    "label_category":np.where(labels27==1,"one_to_one_reference_match",
                   np.where(labels27==0,"poor_fit_reference_overlap","unlabeled"))})
labels_table27.to_csv(DEST27/"training_labels.csv",index=False)
P27.to_file(DEST27/"all_scored_proposals.gpkg",driver="GPKG")
display(evaluation27[(evaluation27.iou_threshold==.5)&
                     (evaluation27.retained_fraction_target.isin([1.,.7,.6,.5]))].round(3))
print("Saved:",DEST27)
print("CAVEAT: ROI 0 only; reference inventory incomplete; eastern split previously inspected.")
print("CAVEAT: models are fixed beforehand; use independent ROIs for definitive model selection.")


## 28 — Multiregion SAM3 crown discovery (same zero-shot RGB configuration)

**Purpose:** extend the *unchanged Block 22 geometric-prompt SAM3 experiment* from ROI 0 to remaining ROIs (indices 1–5). This is GPU inference for ROIs lacking cached masks; unlike Blocks 25–27, **run Block 2** in a new runtime.

**Restart sequence:** **0R → 2 → 14 → 28**. Block 28 embeds the exact Block 22 implementation so it has no dependency on rerunning Block 22 or fetching notebook code. It leaves ROI 0 intact, checks existing completed ROIs, and resumes missing cached SAM3 prompt masks. Each region writes under `experiments_v22_full_roi_8m_<ROI_INDEX>/full/`, with `all_proposals.gpkg`, `clean_crowns.gpkg`, `summary.json`, and per-size recall. If GPU time is limited, edit `ROI_IDS_28` to a subset (e.g. `[1]`); repeated runs skip complete ROIs and resume incomplete ones.

**Fixed settings:** RGB only; 40 m tiles, 10 m overlap, 6 m prompt grid, 8 m boxes; no DSM, CHM, treetop seeds or manual labels in inference. **DSM is reserved for the next supervised classifier comparison:** use DSM-derived height, local relief and crown-to-surroundings height contrast as additional predictors after all ROI proposals have been generated. This avoids conflating RGB-versus-DSM classifier benefits with a change in SAM3 proposal generation.

**Methodological caveat:** Each ROI's recall is evaluated only against manual crowns covered by the tile union. Reference-matched prediction fraction is not true precision. The original Block 22 code's JSON key `precision_05` refers to *reference-matched proposal fraction* and should not be interpreted as detection precision.


In [ ]:
# BLOCK 28 — ROI 1–5 multiregion expansion; exact Block 22 engine, GPU and resumable
# Fresh runtime: 0R -> 2 -> 14 -> 28. Select smaller groups to manage Colab GPU time.
import json
from pathlib import Path
import pandas as pd
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14 first"
assert "SAMPY" in globals() and Path(SAMPY).exists(), "Run Block 2 for the SAM3 GPU environment"
ROI_IDS_28 = list(range(1, len(roi_v2)))  # Default: all regions other than ROI 0
assert len(roi_v2)>=2, "Expected multiple evaluation ROIs"
# Captured verbatim from committed Block 22; only the ROI_INDEX assignment is changed.
BLOCK22_SOURCE_28 = "# BLOCK 22 — Full ROI RGB SAM3, 8m boxes / 40m tiles (resumable)\n# Run after 0R, 2 (if environment absent), 14. No need for 0S or 15–18.\nimport os, sys, subprocess, json, math\nfrom pathlib import Path\nimport numpy as np, pandas as pd, geopandas as gpd, rasterio\nfrom rasterio.windows import Window\nfrom rasterio.features import shapes\nfrom rasterio.enums import Resampling\nfrom shapely.geometry import box, Point, shape\nfrom scipy.optimize import linear_sum_assignment\nfrom PIL import Image\nimport matplotlib.pyplot as plt\n\nROI_INDEX = 0\nTILE_M = 40.0\nOVERLAP_M = 10.0\nGRID_M = 6.0\nBOX_M = 8.0\nMAX_TILES = None       # None = complete ROI; 4 = technical smoke test\nMIN_AREA_M2, MAX_AREA_M2 = 2.0, 700.0\nEDGE_MARGIN_M = 0.4\nDUP_IOU = 0.65\nNESTED_CONTAINMENT = 0.95\nSIMILAR_AREA_RATIO = 0.65\n\nassert \"roi_v2\" in globals() and \"reference_v2\" in globals(), \"Run Block 14\"\nassert Path(SAMPY).exists(), \"Run Block 2 to install isolated SAM3 environment\"\nB19 = OUT / f\"experiments_v22_full_roi_8m_{ROI_INDEX}\"\nTILES, RAW19 = B19/\"tiles\", B19/\"raw\"\nfor d in (B19,TILES,RAW19): d.mkdir(parents=True,exist_ok=True)\n\nwith rasterio.open(RASTER) as src:\n    crs, gsd = src.crs, abs(src.transform.a)\n    roi19 = roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]\n    n = int(round(TILE_M/gsd))\n    stride = max(1,int(round((TILE_M-OVERLAP_M)/gsd)))\n    minx,miny,maxx,maxy = roi19.bounds\n    r0,c0 = src.index(minx,maxy)\n    r1,c1 = src.index(maxx,miny)\n    def starts(lo,hi):\n        if hi-lo <= n: return [int(round((lo+hi-n)/2))]\n        seq=list(range(int(lo),int(hi)-n+1,stride))\n        last=int(hi)-n\n        if not seq or seq[-1]!=last: seq.append(last)\n        return sorted(set(seq))\n    tiles=[]\n    for rr in starts(r0,r1+1):\n        for cc in starts(c0,c1+1):\n            win=Window(cc,rr,n,n)\n            b=rasterio.windows.bounds(win,src.transform)\n            if not box(*b).intersects(roi19): continue\n            tid=len(tiles); path=TILES/f\"tile_{tid:04d}.png\"\n            tiles.append(dict(tile_id=tid,row_off=rr,col_off=cc,width=n,height=n,\n                              xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],path=str(path)))\n    if MAX_TILES is not None: tiles=tiles[:MAX_TILES]\n    assert tiles,\"No ROI tiles\"\n    for t in tiles:\n        p=Path(t[\"path\"])\n        if p.exists(): continue\n        win=Window(t[\"col_off\"],t[\"row_off\"],n,n)\n        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)\n        Image.fromarray(rgb.astype(np.uint8)).save(p)\ntm=pd.DataFrame(tiles)\n# Scope outputs by the selected tile IDs so smoke tests never masquerade as full ROI.\nrun_tag=\"full\" if MAX_TILES is None else f\"first_{MAX_TILES}_tiles\"\nwork=B19/run_tag; raw=work/\"raw\"; raw.mkdir(parents=True,exist_ok=True)\nrows=[]\nfor t in tm.itertuples():\n    for dy in np.arange(BOX_M/2,TILE_M-BOX_M/2+1e-6,GRID_M):\n        for dx in np.arange(GRID_M/2,TILE_M,GRID_M):\n            x,y=t.xmin+dx,t.ymax-dy\n            if roi19.covers(Point(x,y)):\n                rows.append(dict(tile_id=int(t.tile_id),path=t.path,cx=dx/TILE_M,cy=dy/TILE_M,\n                                 box_m=BOX_M,width=n,height=n,gsd=gsd,\n                                 row_off=int(t.row_off),col_off=int(t.col_off)))\npm=pd.DataFrame(rows)\nassert len(pm),\"No grid prompts inside ROI\"\npm.insert(0,\"prompt_id\",np.arange(len(pm)))\npm.to_csv(work/\"prompt_manifest.csv\",index=False)\nprint(f\"ROI {ROI_INDEX}: {roi19.area/10000:.3f} ha | tiles={len(tm)} | prompts={len(pm)} | mode={run_tag}\")\nprint(\"Existing masks:\",sum((raw/f\"prompt_{i:06d}.npz\").exists() for i in pm.prompt_id))\n\nrunner=Path(\"/content/run_sam3_kvo22.py\")\nrunner.write_text(r'''\nimport sys\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\nfrom PIL import Image\nfrom tqdm import tqdm\nfrom sam3.model_builder import build_sam3_image_model\nfrom sam3.model.sam3_image_processor import Sam3Processor\ndf=pd.read_csv(sys.argv[1]); outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)\nif not torch.cuda.is_available(): raise RuntimeError(\"CUDA GPU required\")\ntorch.backends.cuda.matmul.allow_tf32=True\nbpe=\"/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz\"\nwith torch.inference_mode(),torch.autocast(\"cuda\",dtype=torch.bfloat16):\n    model=build_sam3_image_model(bpe_path=bpe)\n    processor=Sam3Processor(model)\n    for tid,g in df.groupby(\"tile_id\",sort=True):\n        pending=[r for r in g.itertuples() if not (outdir/f\"prompt_{int(r.prompt_id):06d}.npz\").exists()]\n        if not pending:continue\n        img=Image.open(pending[0].path).convert(\"RGB\")\n        for r in tqdm(pending,desc=f\"Tile {tid}\",leave=False):\n            # Fresh prompt state per box, to avoid accumulated prompts.\n            state=processor.set_image(img)\n            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))\n            out=processor.add_geometric_prompt(\n                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),\n                label=np.array([1],dtype=np.int32),state=state)\n            masks=out[\"masks\"].detach().cpu().numpy()\n            scores=out[\"scores\"].detach().float().cpu().numpy()\n            if masks.ndim==4:masks=masks[:,0]\n            if masks.ndim==2:masks=masks[None]\n            np.savez_compressed(outdir/f\"prompt_{int(r.prompt_id):06d}.npz\",\n                                masks=(masks>0.5).astype(np.uint8),scores=scores)\n        print(f\"Tile {tid} complete\",flush=True)\nprint(\"Inference complete\",flush=True)\n''')\nmissing=sum(not (raw/f\"prompt_{i:06d}.npz\").exists() for i in pm.prompt_id)\nif missing:\n    run_env=os.environ.copy()\n    if \"env\" in globals(): run_env.update(env)\n    print(\"Running SAM3 for\",missing,\"unsaved prompts\")\n    subprocess.run([SAMPY,\"-u\",str(runner),str(work/\"prompt_manifest.csv\"),str(raw)],\n                   env=run_env,check=True)\nelse: print(\"All masks cached; skipping SAM3\")\n\n# Convert every returned instance, not just the highest-scoring mask.\nfeatures=[]\nwith rasterio.open(RASTER) as src:\n    for r in pm.itertuples():\n        p=raw/f\"prompt_{int(r.prompt_id):06d}.npz\"\n        if not p.exists():continue\n        with np.load(p) as z:\n            masks=z[\"masks\"]; scores=np.asarray(z[\"scores\"]).reshape(-1)\n        if masks.ndim==4:masks=masks[:,0]\n        if masks.ndim==2:masks=masks[None]\n        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))\n        transform=src.window_transform(win)\n        for k,m in enumerate(masks):\n            m=np.asarray(m)>0\n            if m.shape!=(n,n) or not m.any():continue\n            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))\n            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():\n                continue\n            geoms=[shape(g) for g,v in shapes(m.astype(\"uint8\"),mask=m,transform=transform) if v==1]\n            if not geoms:continue\n            geom=max(geoms,key=lambda g:g.area)\n            if not (MIN_AREA_M2<=geom.area<=MAX_AREA_M2):continue\n            if not roi19.covers(geom.representative_point()):continue\n            features.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),\n                                 instance=int(k),score=float(scores[min(k,len(scores)-1)]),\n                                 area_m2=float(geom.area),geometry=geom))\nprint(\"Valid raw proposals:\",len(features))\nassert features,\"No polygons survived filtering; inspect tile-edge masks\"\nallpred=gpd.GeoDataFrame(features,geometry=\"geometry\",crs=crs)\nallpred.to_file(work/\"all_proposals.gpkg\",driver=\"GPKG\")\n\n# Spatial-index-assisted conservative duplicate removal.\nordered=allpred.sort_values(\"score\",ascending=False).reset_index(drop=True)\nkeep=[]; removed=[]\nfor i,r in ordered.iterrows():\n    g=r.geometry; duplicate=False\n    for j in keep:\n        h=ordered.geometry.iloc[j]\n        if not g.intersects(h):continue\n        inter=g.intersection(h).area\n        if inter<=0:continue\n        iou=inter/(g.area+h.area-inter)\n        containment=inter/min(g.area,h.area)\n        ratio=min(g.area,h.area)/max(g.area,h.area)\n        if iou>=DUP_IOU or (containment>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):\n            duplicate=True;removed.append(dict(prompt_id=int(r.prompt_id),against=int(j),iou=iou,\n                                                containment=containment,area_ratio=ratio))\n            break\n    if not duplicate:keep.append(i)\nclean=ordered.iloc[keep].copy().reset_index(drop=True)\nclean[\"crown_id\"]=np.arange(1,len(clean)+1)\nclean.to_file(work/\"clean_crowns.gpkg\",driver=\"GPKG\")\npd.DataFrame(removed).to_csv(work/\"removed_duplicates.csv\",index=False)\nprint(f\"Clean predictions: {len(clean)} | removed duplicates: {len(removed)}\")\n\n# Evaluate only reference crowns fully contained in the processed tile union\n# (and ROI). This prevents partial smoke-test coverage being labelled full-ROI.\nfootprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]\nfrom shapely.ops import unary_union\ncovered=unary_union(footprints).intersection(roi19)\nreference=reference_v2.to_crs(crs).copy()\nreference=reference[reference.geometry.apply(covered.covers)].copy().reset_index(drop=True)\nmat=np.zeros((len(reference),len(clean)),dtype=np.float32)\nidx=clean.sindex\nfor i,g in enumerate(reference.geometry):\n    for j in idx.query(g,predicate=\"intersects\"):\n        h=clean.geometry.iloc[j]; inter=g.intersection(h).area\n        union=g.area+h.area-inter\n        if union>0:mat[i,j]=inter/union\nmatches=[]\nif mat.size:\n    utility=(mat>=.5).astype(float)*1000+mat\n    ii,jj=linear_sum_assignment(utility,maximize=True)\n    matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=.5]\nmatched_ref={i for i,j,v in matches}\nreference[\"area_m2\"]=reference.geometry.area\nreference[\"matched\"]=[i in matched_ref for i in range(len(reference))]\nreference[\"best_iou\"]=mat.max(axis=1) if len(clean) else 0\nreference[\"size_class\"]=pd.cut(reference.area_m2,[0,20,40,80,np.inf],\n                               labels=[\"<20\",\"20–40\",\"40–80\",\">80\"])\nsize=reference.groupby(\"size_class\",observed=True).agg(\n    n=(\"matched\",\"size\"),matched=(\"matched\",\"sum\"),recall=(\"matched\",\"mean\"),\n    median_best_iou=(\"best_iou\",\"median\"))\nsize.to_csv(work/\"recall_by_size.csv\")\ndisplay(size)\nrecall=len(matches)/len(reference) if len(reference) else float(\"nan\")\nprecision=len(matches)/len(clean) if len(clean) else float(\"nan\")\nsummary=dict(roi_index=ROI_INDEX,mode=run_tag,roi_area_ha=roi19.area/10000,\n             tiles=len(tm),prompts=len(pm),manual_crowns=len(reference),\n             raw_proposals=len(allpred),clean_predictions=len(clean),\n             matched_iou_05=len(matches),recall_05=recall,precision_05=precision,\n             median_matched_iou=float(np.median([v for _,_,v in matches])) if matches else None)\n(work/\"summary.json\").write_text(json.dumps(summary,indent=2))\nprint(\"RESULTS:\",json.dumps(summary,indent=2))\n\nprint(\"BLOCK 22 COMPLETE. Run Block 22V for full statistical validation:\",work)\n"
assert BLOCK22_SOURCE_28.count("ROI_INDEX = 0")==1
results28=[]
for target_roi28 in ROI_IDS_28:
    assert 0<=target_roi28<len(roi_v2), f"Unknown ROI index {target_roi28}"
    region_dir28=OUT/f"experiments_v22_full_roi_8m_{target_roi28}"/"full"
    summary_path28=region_dir28/"summary.json"
    cleaned_path28=region_dir28/"clean_crowns.gpkg"
    proposals_path28=region_dir28/"all_proposals.gpkg"
    if summary_path28.exists() and cleaned_path28.exists() and proposals_path28.exists():
        summary28=json.loads(summary_path28.read_text())
        if summary28.get("mode")=="full" and summary28.get("roi_index")==target_roi28:
            print(f"ROI {target_roi28}: full outputs already available, skipping GPU")
            results28.append(summary28)
            continue
    print(f"\\n=== Running or resuming ROI {target_roi28} ===",flush=True)
    engine28=BLOCK22_SOURCE_28.replace("ROI_INDEX = 0",f"ROI_INDEX = {target_roi28}",1)
    # Shared notebook namespace is intentional: mirrors running Block 22 for each ROI.
    exec(compile(engine28,f"<Block22_ROI_{target_roi28}>","exec"),globals())
    assert summary_path28.exists() and cleaned_path28.exists(), f"Missing completed outputs for ROI {target_roi28}"
    results28.append(json.loads(summary_path28.read_text()))
# Aggregate every completed ROI, including the original ROI 0 baseline.
all28=[]
for idx28 in range(len(roi_v2)):
    d28=OUT/f"experiments_v22_full_roi_8m_{idx28}"/"full"
    s28=d28/"summary.json"
    if s28.exists() and (d28/"clean_crowns.gpkg").exists():
        v28=json.loads(s28.read_text())
        if v28.get("mode")=="full" and v28.get("roi_index")==idx28:
            v28["reference_matched_prediction_fraction_05"]=v28.get("matched_iou_05",0)/max(v28.get("clean_predictions",0),1)
            all28.append(v28)
summary_table28=pd.DataFrame(all28).sort_values("roi_index") if all28 else pd.DataFrame()
if len(summary_table28):
    aggregate_dir28=OUT/"experiments_v28_multiregion_summary"
    aggregate_dir28.mkdir(parents=True,exist_ok=True)
    summary_table28.to_csv(aggregate_dir28/"roi_summary.csv",index=False)
    display(summary_table28[[c for c in ["roi_index","roi_area_ha","tiles","prompts","manual_crowns","raw_proposals","clean_predictions","matched_iou_05","recall_05","reference_matched_prediction_fraction_05"] if c in summary_table28.columns]].round(3))
    print("Complete ROI inventories:",len(summary_table28),"/",len(roi_v2))
    print("Saved aggregate summary:",aggregate_dir28/"roi_summary.csv")
else:
    print("No complete full-ROI inventories found.")


## 29 — Six-ROI crown proposal classifiers: RGB versus RGB + DSM (CPU only)

**Fresh Colab:** **Block 0R → Block 14 → Block 29**. Uses all six cached Block 28 `clean_crowns.gpkg` inventories, RGB orthomosaic and `Baego_DSM.tif`. No SAM3 GPU inference, no Block 2. Existing predictions and previous experiments are never overwritten.

**Experiment:** same Random Forest and identical splits/parameters for (A) confidence + polygon geometry + spatial overlap + RGB and (B) all A features + DSM statistics (within-crown height variability, local crown-to-surroundings contrast, local relative height). DSM elevations alone should not be interpreted as tree height without a terrain model (DTM); features emphasize **local differences**. Extracted from geographically aligned raster windows with `WarpedVRT` when needed.

**Validation:** leave-one-ROI-out for each of the six ROIs. On the other five ROIs, create one-to-one reference-matched positive labels (IoU ≥ 0.5), conservative poor-fit canopy negatives, and high-ground-fraction negatives; keep ambiguous candidates unlabeled. The held-out ROI's manual crowns are used **only for evaluation**, never for training labels, feature engineering, thresholds or model choice. Pre-specified fixed retention budgets: 100%, 90%, 80%, 70%, 60%, 50%, 40%, 30%; IoU thresholds 0.3/0.5/0.7. Include a SAM3-confidence baseline; compare recall **at equal proposal counts**, including crown-size classes and ROI 4. All reported reference-matched prediction fractions are distinct from genuine detection precision.

**Caveats:** exhaustive visible-crown coverage allows more informative negative examples, but no claim of perfect annotations; mis-segmented canopy objects and unlabeled proposals should not be blindly treated as background. The six ROIs may be spatially close and not fully independent forest sites. This experiment evaluates **selection**, not whether DSM improves SAM3 segmentation boundaries or generates additional crowns. Reference-validation association remains based on the original complete manual polygons.


In [ ]:
# BLOCK 29 — All six ROIs; RGB vs RGB+DSM proposal ranking; CPU ONLY
# Run 0R -> 14 -> 29. No SAM3 rerun. Writes only experiments_v29_*.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds, Window
from shapely.geometry import mapping
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import joblib

DEST29=OUT/"experiments_v29_rgb_vs_dsm_six_roi"
DEST29.mkdir(parents=True,exist_ok=True)
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14 first"
assert len(roi_v2)==6,"Expected exactly 6 ROIs"
DSM29=GEO/"Baego_DSM.tif"
assert DSM29.exists(),f"DSM missing: {DSM29}"
with rasterio.open(RASTER) as rgb:
    crs29=rgb.crs
roi29=roi_v2.to_crs(crs29).reset_index(drop=True)
references29=reference_v2.to_crs(crs29)
candidates29=[];ref_by_roi29={}
for roi_id in range(6):
    path=OUT/f"experiments_v22_full_roi_8m_{roi_id}"/"full"/"clean_crowns.gpkg"
    assert path.exists(),f"Missing Block 28 clean proposals: {path}"
    p=gpd.read_file(path).to_crs(crs29)
    p=p[p.geometry.notna()&~p.geometry.is_empty].copy().reset_index(drop=True)
    p["roi_id"]=roi_id
    p["local_proposal_id"]=np.arange(len(p))
    candidates29.append(p)
    roi_shape=roi29.geometry.iloc[roi_id]
    ref=references29[references29.geometry.apply(roi_shape.covers)].copy().reset_index(drop=True)
    assert len(ref),f"No fully contained reference crowns ROI {roi_id}"
    ref_by_roi29[roi_id]=ref
P29=gpd.GeoDataFrame(pd.concat(candidates29,ignore_index=True),geometry="geometry",crs=crs29)
P29["proposal_id"]=np.arange(len(P29))
assert len(P29)>=5000,f"Unexpected number of SAM3 proposals: {len(P29)}"
print("Loaded",len(P29),"candidates and",sum(map(len,ref_by_roi29.values())),"references")

# Geometry and overlap features computed strictly within each ROI (no reference labels).
area29=P29.geometry.area.to_numpy()
per29=P29.geometry.length.to_numpy()
base29=pd.DataFrame(dict(proposal_id=P29.proposal_id.to_numpy(),
    roi_id=P29.roi_id.to_numpy(),
    sam3_score=pd.to_numeric(P29["score"],errors="coerce").to_numpy(),
    area_m2=area29,log_area=np.log1p(area29),perimeter_m=per29,
    compactness=4*np.pi*area29/np.maximum(per29**2,1e-9),
    bbox_fill=area29/np.maximum(np.array([g.envelope.area for g in P29.geometry]),1e-9)))
for key in ("max_overlap_iou","overlap_count","containment_count","higher_score_overlap"):
    base29[key]=0.0
for roi_id in range(6):
    ids=np.flatnonzero(P29.roi_id.to_numpy()==roi_id)
    grp=P29.iloc[ids].reset_index(drop=True); idx=grp.sindex
    local_area=grp.geometry.area.to_numpy()
    scores=pd.to_numeric(grp.score,errors="coerce").fillna(-1).to_numpy()
    for i,g in enumerate(grp.geometry):
        for j in idx.query(g,predicate="intersects"):
            j=int(j)
            if j<=i:continue
            h=grp.geometry.iloc[j];inter=g.intersection(h).area
            if inter<=0:continue
            overlap=inter/max(local_area[i]+local_area[j]-inter,1e-9)
            containment=inter/max(min(local_area[i],local_area[j]),1e-9)
            a,b=int(ids[i]),int(ids[j])
            base29.loc[[a,b],"max_overlap_iou"]=np.maximum(base29.loc[[a,b],"max_overlap_iou"],overlap)
            base29.loc[[a,b],"overlap_count"]+=1
            if containment>=.9:base29.loc[[a,b],"containment_count"]+=1
            if scores[i]<scores[j]:base29.loc[a,"higher_score_overlap"]+=1
            elif scores[j]<scores[i]:base29.loc[b,"higher_score_overlap"]+=1

# Raster features use polygon interior and nearby ring in a bounded window.
# DSM is treated as a SURFACE (not a CHM); local contrasts are physically interpretable.
rgbcols29=["r_mean","g_mean","b_mean","r_std","g_std","b_std",
           "rgb_ring_contrast"]
dsmcols29=["dsm_median","dsm_std","dsm_p90_p10","dsm_ring_contrast",
           "dsm_local_relief"]
raster_columns29=rgbcols29+dsmcols29
raster_values29=np.full((len(P29),len(raster_columns29)),np.nan,dtype=np.float32)
with rasterio.open(RASTER) as rgb, rasterio.open(DSM29) as raw_dsm:
    aligned_dsm=WarpedVRT(raw_dsm,crs=rgb.crs,transform=rgb.transform,
                           width=rgb.width,height=rgb.height,
                           resampling=Resampling.bilinear)
    try:
        for i,g in enumerate(P29.geometry):
            try:
                outer=g.buffer(2.0)
                win=from_bounds(*outer.bounds,transform=rgb.transform).round_offsets().round_lengths()
                win=win.intersection(Window(0,0,rgb.width,rgb.height))
                if win.width<2 or win.height<2:continue
                step=max(1,int(np.ceil(max(win.width,win.height)/192)))
                out_w=max(1,int(np.ceil(win.width/step)));out_h=max(1,int(np.ceil(win.height/step)))
                out_shape=(out_h,out_w)
                rgb_array=rgb.read([1,2,3],window=win,out_shape=(3,out_h,out_w),
                                   resampling=Resampling.average,masked=True)
                dsm_array=aligned_dsm.read(1,window=win,out_shape=out_shape,
                                           resampling=Resampling.average,masked=True)
                tr=rgb.window_transform(win)*rasterio.Affine.scale(win.width/out_w,win.height/out_h)
                inside=geometry_mask([mapping(g)],out_shape=out_shape,transform=tr,invert=True)
                ring=geometry_mask([mapping(outer)],out_shape=out_shape,transform=tr,invert=True)&~inside
                if inside.sum()<3:continue
                c=np.asarray(rgb_array.filled(np.nan),dtype=float)
                z=np.asarray(dsm_array.filled(np.nan),dtype=float)
                means=[];stds=[]
                for channel in range(3):
                    v=c[channel,inside];v=v[np.isfinite(v)]
                    means.append(float(np.mean(v)) if len(v) else np.nan)
                    stds.append(float(np.std(v)) if len(v) else np.nan)
                gray=c.mean(axis=0)
                inside_gray=gray[inside];ring_gray=gray[ring]
                iv=inside_gray[np.isfinite(inside_gray)]
                rv=ring_gray[np.isfinite(ring_gray)]
                rgb_diff=float(np.nanmedian(iv)-np.nanmedian(rv)) if len(iv) and len(rv) else np.nan
                iz=z[inside];iz=iz[np.isfinite(iz)]
                rz=z[ring];rz=rz[np.isfinite(rz)]
                if len(iz)>=3:
                    z_median=float(np.median(iz));z_std=float(np.std(iz))
                    z_range=float(np.percentile(iz,90)-np.percentile(iz,10))
                    z_contrast=float(z_median-np.median(rz)) if len(rz)>=3 else np.nan
                    relief=float(np.percentile(iz,90)-np.percentile(rz,10)) if len(rz)>=3 else np.nan
                else:z_median=z_std=z_range=z_contrast=relief=np.nan
                raster_values29[i,:]=means+stds+[rgb_diff,z_median,z_std,z_range,z_contrast,relief]
            except Exception as error:
                if i<4:print("Raster feature warning",i,str(error))
            if (i+1)%1000==0:print(f"Raster features {i+1}/{len(P29)}")
    finally:aligned_dsm.close()
for k,col in enumerate(raster_columns29):base29[col]=raster_values29[:,k]
# Do not include absolute DSM elevation; no DTM or elevation normalization assumed.
rgb_features29=[c for c in base29.columns if c not in
                (["proposal_id","roi_id"]+dsmcols29)]
dsm_features29=rgb_features29+["dsm_std","dsm_p90_p10","dsm_ring_contrast","dsm_local_relief"]
base29.to_csv(DEST29/"candidate_features_all_rois.csv",index=False)

# Build labels separately for each ROI. Only TRAINING ROI labels can enter fitting.
# One positive proposal per reference crown; others remain unlabeled unless
# confidently identifiable as wrong crown geometry or mostly ground.
labels29=np.full(len(P29),-1,dtype=np.int8)
reference_matrices29={}
for roi_id in range(6):
    ids=np.flatnonzero(P29.roi_id.to_numpy()==roi_id)
    pp=P29.iloc[ids].reset_index(drop=True)
    rr=ref_by_roi29[roi_id]
    ref_index=rr.sindex
    iou=np.zeros((len(rr),len(pp)),dtype=np.float32)
    candidate_covered=np.zeros(len(pp),dtype=np.float32)
    for j,g in enumerate(pp.geometry):
        intersections=[]
        for k in ref_index.query(g,predicate="intersects"):
            k=int(k);h=rr.geometry.iloc[k]
            overlap=g.intersection(h).area
            if overlap<=0:continue
            iou[k,j]=overlap/max(g.area+h.area-overlap,1e-9)
            intersections.append(h.intersection(g))
        if intersections:
            candidate_covered[j]=min(1.,unary_union(intersections).area/max(g.area,1e-9))
    reference_matrices29[roi_id]=iou
    row,col=linear_sum_assignment((iou>=.5).astype(float)*2+iou,maximize=True)
    positive=np.array([j for i,j in zip(row,col) if iou[i,j]>=.5],dtype=int)
    local=np.full(len(pp),-1,dtype=np.int8)
    local[positive]=1
    best_iou=iou.max(axis=0)
    # Poor fit despite mostly annotated-canopy overlap -> conservative negative.
    # Mostly outside exhaustively annotated canopy -> background negative.
    negatives=((best_iou<.25)&(candidate_covered>=.80)) | (candidate_covered<=.20)
    local[negatives&(local!=1)]=0
    labels29[ids]=local
base29["supervision_label"]=labels29
base29.to_csv(DEST29/"candidate_features_and_training_labels.csv",index=False)
print("Label counts:",pd.Series(labels29).value_counts().to_dict())
print("DSM valid proportions:",base29[dsm_features29[-4:]].notna().mean().round(3).to_dict())

# Leave-one-ROI-out: same RF family / parameters, no held-out label fitting.
budgets29=[1.,.9,.8,.7,.6,.5,.4,.3]
iou_thresholds29=[.3,.5,.7]
methods29={"rgb_only":rgb_features29,"rgb_plus_dsm":dsm_features29}
comparison29=[];importance29=[];prediction_rows29=[];size_rows29=[]
sizes29=[(0,20,"under_20"),(20,40,"20_to_40"),(40,80,"40_to_80"),(80,np.inf,"over_80")]
for held_id in range(6):
    train_ids=np.flatnonzero((P29.roi_id.to_numpy()!=held_id)&(labels29>=0))
    test_ids=np.flatnonzero(P29.roi_id.to_numpy()==held_id)
    ytrain=labels29[train_ids]
    assert set(np.unique(ytrain))=={0,1},f"Training fold {held_id} has one label class"
    mat=reference_matrices29[held_id].astype(np.float64)
    nref=mat.shape[0];ntest=len(test_ids)
    test_score29=pd.to_numeric(P29.iloc[test_ids].score,errors="coerce").fillna(-1).to_numpy()
    rankings={"sam3_confidence":np.argsort(-test_score29,kind="stable")}
    for method,features in methods29.items():
        Xtrain=base29.iloc[train_ids][features].replace([np.inf,-np.inf],np.nan)
        Xtest=base29.iloc[test_ids][features].replace([np.inf,-np.inf],np.nan)
        model=make_pipeline(SimpleImputer(strategy="median"),
            RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
            class_weight="balanced_subsample",n_jobs=-1,random_state=42))
        model.fit(Xtrain,ytrain)
        score=model.predict_proba(Xtest)[:,list(model.classes_).index(1)]
        rankings[method]=np.argsort(-score,kind="stable")
        for feature,value in zip(features,model[-1].feature_importances_):
            importance29.append(dict(heldout_roi=held_id,model=method,
                                     feature=feature,importance=float(value)))
        joblib.dump({"model":model,"features":features,"heldout_roi":held_id,
                     "training_rois":[j for j in range(6) if j!=held_id]},
                    DEST29/f"model_{method}_holdout_roi_{held_id}.joblib")
        prediction_rows29.extend(dict(heldout_roi=held_id,proposal_id=int(test_ids[j]),
                                      method=method,score=float(score[j])) for j in range(ntest))
    for method,rank in rankings.items():
        for budget in budgets29:
            k=max(1,int(np.ceil(ntest*budget)));chosen=rank[:k]
            sub=mat[:,chosen]
            for threshold in iou_thresholds29:
                ri,cj=linear_sum_assignment((sub>=threshold).astype(float)*2+sub,maximize=True)
                count=int(np.count_nonzero(sub[ri,cj]>=threshold))
                comparison29.append(dict(heldout_roi=held_id,method=method,
                    retained_fraction=budget,proposals_retained=k,
                    original_proposals=ntest,reduction_pct=100*(1-k/ntest),
                    iou_threshold=threshold,reference_crowns=nref,matched=count,
                    recall=count/nref,reference_matched_prediction_fraction=count/k))
                ref_areas=ref_by_roi29[held_id].geometry.area.to_numpy()
                for lo,hi,label in sizes29:
                    selected_ref=np.flatnonzero((ref_areas>=lo)&(ref_areas<hi))
                    if not len(selected_ref):continue
                    part=sub[selected_ref,:]
                    si,sj=linear_sum_assignment((part>=threshold).astype(float)*2+part,maximize=True)
                    cnt=int(np.count_nonzero(part[si,sj]>=threshold))
                    size_rows29.append(dict(heldout_roi=held_id,method=method,
                         retained_fraction=budget,iou_threshold=threshold,size_class=label,
                         reference_crowns=len(selected_ref),matched=cnt,recall=cnt/len(selected_ref)))
    print(f"ROI {held_id}: train {len(train_ids)} labeled candidates; held-out {ntest} proposals, {nref} crowns")
results29=pd.DataFrame(comparison29)
sizes29_df=pd.DataFrame(size_rows29)
importance29_df=pd.DataFrame(importance29)
results29.to_csv(DEST29/"leave_one_roi_out_budget_recall.csv",index=False)
sizes29_df.to_csv(DEST29/"leave_one_roi_out_crown_size.csv",index=False)
importance29_df.to_csv(DEST29/"feature_importance_by_fold.csv",index=False)
pd.DataFrame(prediction_rows29).to_csv(DEST29/"heldout_proposal_scores.csv",index=False)
pooled29=results29.groupby(["method","retained_fraction","iou_threshold"],as_index=False).agg(
    reference_crowns=("reference_crowns","sum"),matched=("matched","sum"),
    proposals_retained=("proposals_retained","sum"),original_proposals=("original_proposals","sum"))
pooled29["pooled_recall"]=pooled29.matched/pooled29.reference_crowns
pooled29["reduction_pct"]=100*(1-pooled29.proposals_retained/pooled29.original_proposals)
pooled29.to_csv(DEST29/"pooled_leave_one_roi_out_recall.csv",index=False)
print("Pooled IoU >= 0.5 (fixed proposal budgets):")
display(pooled29[(pooled29.iou_threshold==.5)&pooled29.retained_fraction.isin([1.,.7,.6,.5])].round(3))
print("ROI 4 at 40% reduction:")
display(results29[(results29.heldout_roi==4)&(results29.iou_threshold==.5)&
                  (results29.retained_fraction==.6)].round(3))
print("Mean feature importance:")
display(importance29_df.groupby(["model","feature"],as_index=False).importance.mean()
        .sort_values(["model","importance"],ascending=[True,False]).groupby("model").head(10))
print("Saved outputs:",DEST29)
print("CAUTION: Evaluation uses exhaustively digitized visible crowns; unmatched proposals may still be partial or merged canopy objects.")
print("CAUTION: DSM is a surface model, not canopy height above ground without a DTM.")


## 30 — SAM3 crown fine-tuning pilot: export masks and verify official training setup

**Fresh Colab GPU runtime:** run **0R → 2 → 14 → 30**. Unlike the CPU classifier experiments, SAM3 training requires Block 2 and a GPU.

This block creates a **reproducible COCO instance-segmentation dataset** from the manually delineated `Crown_Kristof.shp` polygons and existing Block 22 tile manifests. It reserves whole ROIs **0–3 for training, 4 for validation, and 5 for untouched testing**, avoiding random-image leakage. Each tile is annotated with all intersecting manual crowns, including clipped border instances; pixels outside the annotated ROI are blacked out in exported training imagery to avoid treating unannotated surroundings as background. Dataset includes category `tree crown`, COCO RLE masks, a dataset manifest, and checks. Original imagery and previous outputs are not modified.

The official SAM3 repository supports custom-data training via `sam3/train/train.py` and `Sam3ImageDataset`; its example Roboflow training YAML **defaults to box-only loss**. This block creates a conservative segmentation-enabled pilot configuration by enabling mask targets and loss, and reports the training command; the launcher is **disabled by default** until you inspect the exported samples and configuration. The official training setup is primarily concept/detection oriented; successful training does **not** automatically imply that a geometric-prompt decoder has been fine-tuned. Later we must verify which weights are updated and separately evaluate automatic geometric-grid inference versus prompted masks.

**No DSM used in this first pilot:** we keep the original RGB SAM3 input interface. Fine-tuning outputs, once trained, will need evaluation with ROI 5 held out.


In [ ]:
# BLOCK 30 — Fine-tuning DATASET and official trainer pilot configuration
# Run 0R -> 2 -> 14 -> 30 on GPU runtime. Does NOT launch training by default.
from pathlib import Path
import json, subprocess, sys, shutil
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import rasterize, geometry_mask
from shapely.geometry import box, mapping
from PIL import Image
try:
    from pycocotools import mask as coco_mask
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install","pycocotools"],check=True)
    from pycocotools import mask as coco_mask
import yaml

DEST30=OUT/"experiments_v30_sam3_finetune"
DATA30=DEST30/"dataset"/"musanga"
SPLIT30={0:"train",1:"train",2:"train",3:"train",4:"valid",5:"test"}
DEST30.mkdir(parents=True,exist_ok=True)
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14."
assert Path("/content/sam3/sam3/train/train.py").exists(), (
    "Official SAM3 train script missing; check the cloned repository in Block 2.")
with rasterio.open(RASTER) as src:
    crs30=src.crs
ref30=reference_v2.to_crs(crs30)
roi30=roi_v2.to_crs(crs30)
records30=[];summaries30=[]
combined30={split:{'images':[],'annotations':[]} for split in ('train','valid','test')}
image_id30=0;ann_id30=0
for roi_id30,split30 in SPLIT30.items():
    manifest30=OUT/f"experiments_v22_full_roi_8m_{roi_id30}"/"full"/"prompt_manifest.csv"
    assert manifest30.exists(),f"Missing Block 22 tile manifest ROI {roi_id30}"
    tiles30=pd.read_csv(manifest30).drop_duplicates("tile_id").sort_values("tile_id")
    folder30=DATA30/split30
    folder30.mkdir(parents=True,exist_ok=True)
    polygon30=roi30.geometry.iloc[roi_id30]
    crown30=ref30[ref30.geometry.intersects(polygon30)].copy()
    images30=[];annotations30=[]
    with rasterio.open(RASTER) as src:
        for t in tiles30.itertuples():
            win=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
            transform=src.window_transform(win)
            footprint=box(*rasterio.windows.bounds(win,src.transform))
            if not footprint.intersects(polygon30):continue
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            h,w=rgb.shape[:2]
            roi_mask=geometry_mask([mapping(polygon30)],out_shape=(h,w),transform=transform,invert=True)
            rgb[~roi_mask]=0
            filename=f"roi{roi_id30}_tile{int(t.tile_id):04d}.png"
            Image.fromarray(np.clip(rgb,0,255).astype(np.uint8)).save(folder30/filename)
            image_id30+=1
            images30.append(dict(id=image_id30,file_name=filename,width=w,height=h,roi_id=roi_id30))
            crown_intersections=crown30[crown30.geometry.intersects(footprint)]
            for crown in crown_intersections.geometry:
                clipped=crown.intersection(footprint).intersection(polygon30)
                if clipped.is_empty or clipped.area<2:continue
                mask=rasterize([(mapping(clipped),1)],out_shape=(h,w),transform=transform,
                               fill=0,dtype="uint8")
                pixels=int(mask.sum())
                if pixels<16:continue
                yy,xx=np.where(mask>0)
                x0,y0=int(xx.min()),int(yy.min())
                x1,y1=int(xx.max())+1,int(yy.max())+1
                encoded=coco_mask.encode(np.asfortranarray(mask))
                encoded["counts"]=encoded["counts"].decode("ascii")
                ann_id30+=1
                annotations30.append(dict(id=ann_id30,image_id=image_id30,category_id=1,
                    bbox=[x0,y0,x1-x0,y1-y0],area=pixels,
                    segmentation=encoded,iscrowd=0,noun_phrase="tree crown"))
    coco30=dict(info={"description":"Musanga Baego exhaustive visible tree crowns; ROI-separated"},
                licenses=[],images=images30,annotations=annotations30,
                categories=[{"id":1,"name":"tree crown","supercategory":"canopy"}])
    # Accumulate all ROIs belonging to each split before writing annotation JSON.
    combined30[split30]["images"].extend(images30)
    combined30[split30]["annotations"].extend(annotations30)
    summaries30.append(dict(roi_id=roi_id30,split=split30,
                            images=len(images30),mask_instances=len(annotations30)))
    records30.extend(dict(roi_id=roi_id30,split=split30,
                          image=x["file_name"]) for x in images30)
# Write a single complete COCO JSON for each split, not once for each ROI.
for split_name30,items30 in combined30.items():
    image_list30=items30["images"]; ann_list30=items30["annotations"]
    image_ids30={im["id"] for im in image_list30}
    assert len(image_ids30)==len(image_list30), "Duplicate image IDs"
    assert len({a["id"] for a in ann_list30})==len(ann_list30), "Duplicate annotation IDs"
    assert all(a["image_id"] in image_ids30 for a in ann_list30)
    coco_split30=dict(info={"description":"Musanga Baego crowns, ROI-separated"},
                      licenses=[],images=image_list30,annotations=ann_list30,
                      categories=[{"id":1,"name":"tree crown","supercategory":"canopy"}])
    (DATA30/split_name30/"_annotations.coco.json").write_text(json.dumps(coco_split30))
    print("COMBINED",split_name30,len(image_list30),"tiles",len(ann_list30),"masks")
summary30=pd.DataFrame(summaries30)
summary30.to_csv(DEST30/"dataset_manifest.csv",index=False)
display(summary30)
assert summary30.groupby("split").images.sum().min()>0
assert summary30.groupby("split").mask_instances.sum().min()>0
print("COCO mask dataset ready:",DATA30)
# Use official example only as a starting point; specifically enable MASK training.
example30=Path("/content/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml")
if example30.exists():
    config30=yaml.safe_load(example30.read_text())
    config30["paths"]["roboflow_vl_100_root"]=str(DEST30/"dataset")
    config30["paths"]["experiment_log_dir"]=str(DEST30/"training_logs")
    config30["paths"]["bpe_path"]="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
    config30["roboflow_train"]["supercategory"]="musanga"
    config30["roboflow_train"]["num_images"]=None
    config30["scratch"]["enable_segmentation"]=True
    config30["scratch"]["num_train_workers"]=2
    config30["scratch"]["num_val_workers"]=0
    config30["scratch"]["max_data_epochs"]=3
    config30["scratch"]["target_epoch_size"]=100
    config30["trainer"]["max_epochs"]=3
    config30["trainer"]["skip_saving_ckpts"]=False
    config30["trainer"]["val_epoch_freq"]=1
    config30["launcher"]["gpus_per_node"]=1
    loss30=config30["roboflow_train"]["loss"]["loss_fns_find"]
    if not any(str(item.get("_target_","")).endswith(".Masks") for item in loss30):
        loss30.append({"_target_":"sam3.train.loss.loss_fns.Masks",
                       "focal_alpha":0.25,"focal_gamma":2.,
                       "weight_dict":{"loss_mask":200.,"loss_dice":10.},
                       "compute_aux":False})
    # Validation expects the example's /test annotation folder; use reserved ROI 4 for validation.
    config30["trainer"]["data"]["val"]["dataset"]["img_folder"]=str(DATA30/"valid")+"/"
    config30["trainer"]["data"]["val"]["dataset"]["ann_file"]=str(DATA30/"valid"/"_annotations.coco.json")
    config30["trainer"]["data"]["train"]["dataset"]["img_folder"]=str(DATA30/"train")+"/"
    config30["trainer"]["data"]["train"]["dataset"]["ann_file"]=str(DATA30/"train"/"_annotations.coco.json")
    # Default Roboflow "supercategory" interpolation no longer needed for local paths.
    config30["trainer"]["model"]["enable_segmentation"]=True
    config_file30=DEST30/"sam3_crown_segmentation_pilot.yaml"
    config_file30.write_text('# ' + '@' + 'package _global_\n'+yaml.safe_dump(config30,sort_keys=False))
    print("Pilot trainer config:",config_file30)
    print("Official trainer command (after checking configuration and sample images):")
    print(f"{SAMPY} /content/sam3/sam3/train/train.py -c '{config_file30}' --use-cluster 0 --num-gpus 1")
else:
    print("Official example configuration not found; dataset exported, trainer config not written.")
# Explicit launch gate. DO NOT run training automatically.
RUN_FINE_TUNING_30=False
if RUN_FINE_TUNING_30:
    assert example30.exists()
    print("Launching official SAM3 trainer with segmentation losses enabled.")
    subprocess.run([SAMPY,"/content/sam3/sam3/train/train.py","-c",str(config_file30),
                    "--use-cluster","0","--num-gpus","1"],check=True)
else:
    print("TRAINING NOT STARTED. Inspect data/config first; set RUN_FINE_TUNING_30=True to launch.")
print("IMPORTANT: ROI 5 is never used for training or configuration selection.")


## 30B — SAM3 fine-tuning smoke test (single cell)

Run **Block 30B once** after Block 30 has generated the dataset. The single code cell verifies COCO annotations, prepares the frozen-vision Hydra configuration, then launches the two-batch GPU smoke test. Validation is deliberately disabled only in this smoke test; ROI 5 remains held out. Logs and checkpoints are saved under `experiments_v30b_training_smoke`.


In [ ]:
# BLOCK 30B — One-cell COCO QA, config and two-batch GPU smoke
from pathlib import Path
import json, shutil, yaml, numpy as np
from PIL import Image
from IPython.display import display
from pycocotools import mask as coco_mask
D30B=OUT/"experiments_v30b_training_smoke"
D30B.mkdir(parents=True,exist_ok=True)
DATA30B=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"
source30b=OUT/"experiments_v30_sam3_finetune"/"sam3_crown_segmentation_pilot.yaml"
assert source30b.exists(), "Run Block 30 first."
for part in ("train","valid","test"):
    folder=DATA30B/part
    obj=json.loads((folder/"_annotations.coco.json").read_text())
    assert obj["images"] and obj["annotations"]
    expected30b={"train":(105,1883),"valid":(33,549),"test":(30,497)}
    nimg30b,nmask30b=expected30b[part]
    assert (len(obj["images"]),len(obj["annotations"]))==(nimg30b,nmask30b), (
        f"{part} got {len(obj['images'])} tiles and {len(obj['annotations'])} masks, "
        f"expected {nimg30b} tiles and {nmask30b} masks. Re-run Block 30."
    )
    assert len({im["id"] for im in obj["images"]})==len(obj["images"])
    assert all(a["image_id"] in {im["id"] for im in obj["images"]} for a in obj["annotations"])
    imgs={im["id"]:im for im in obj["images"]}
    for a in obj["annotations"][:20]:
        im=imgs[a["image_id"]]
        enc=dict(a["segmentation"]);enc["counts"]=enc["counts"].encode("ascii")
        mask=coco_mask.decode(enc)
        assert mask.shape==(im["height"],im["width"])
        assert mask.sum()==int(a["area"]) and mask.any()
    first=obj["images"][0]
    pic=np.array(Image.open(folder/first["file_name"]).convert("RGB"))
    mask_all=np.zeros(pic.shape[:2],dtype=bool)
    for a in obj["annotations"]:
        if a["image_id"]==first["id"]:
            enc=dict(a["segmentation"]);enc["counts"]=enc["counts"].encode("ascii")
            mask_all|=coco_mask.decode(enc).astype(bool)
    pic[mask_all]=(0.5*pic[mask_all]+0.5*np.array([255,0,255])).astype("uint8")
    print(part, len(obj["images"]),"tiles,",len(obj["annotations"]),"instances")
    display(Image.fromarray(pic).resize((450,450)))
cfg30b=yaml.safe_load(source30b.read_text())
assert cfg30b["scratch"]["enable_segmentation"] is True
assert any(str(x.get("_target_","")).endswith(".Masks")
           for x in cfg30b["roboflow_train"]["loss"]["loss_fns_find"])
cfg30b["paths"]["experiment_log_dir"]=str(D30B/"training_logs")
cfg30b["trainer"]["max_epochs"]=1
cfg30b["trainer"]["skip_first_val"]=True
cfg30b["trainer"]["val_epoch_freq"]=999
# No validation in the two-batch infrastructure smoke; final validation runs regardless of frequency.
cfg30b["trainer"]["data"].pop("val",None)
cfg30b["trainer"]["meters"].pop("val",None)
cfg30b["trainer"]["skip_saving_ckpts"]=False
cfg30b["scratch"]["target_epoch_size"]=2
cfg30b["scratch"]["max_data_epochs"]=1
cfg30b["scratch"]["num_train_workers"]=0
cfg30b["scratch"]["num_val_workers"]=0
# Retain official 1008 resolution: SAM3 ViT RoPE has fixed positional frequencies.
cfg30b["scratch"]["resolution"]=1008
cfg30b["roboflow_train"]["num_images"]=2
# Official trainer requires a top-level launcher; create it if the pilot omitted it.
cfg30b.setdefault("launcher",{})
cfg30b["launcher"].update({
    "num_nodes":1,
    "gpus_per_node":1,
    "experiment_log_dir":str(D30B/"training_logs"),
    "multiprocessing_context":"forkserver",
})
cfg30b.setdefault("submitit",{})
cfg30b["submitit"].setdefault("port_range",[10000,65000])
cfg30b["submitit"]["use_cluster"]=False
cfg30b["roboflow_train"]["supercategory"]="musanga"
cfg30b["submitit"].pop("job_array",None)
# Single-GPU smoke: DDP static_graph tolerates consistently unused outputs.
cfg30b["trainer"]["distributed"]["find_unused_parameters"]=False
cfg30b["trainer"]["distributed"]["static_graph"]=True
cfg30b["trainer"]["logging"]["log_freq"]=1
cfg30b["trainer"]["data"]["train"]["num_workers"]=0

# Minimal smoke: freeze the pretrained vision trunk before Hydra creates the optimizer.
# Local wrapper leaves upstream SAM3 code unchanged and avoids gradients through
# sam3.perflib.fused.addmm_act (which explicitly requires no-grad).
builder_path30b=Path("/content/sam3/sam3/train/musanga_frozen_builder.py")
builder_path30b.write_text("""
from sam3.model_builder import build_sam3_image_model

def build_frozen_vision_sam3(**kwargs):
    model = build_sam3_image_model(**kwargs)
    for param in model.backbone.vision_backbone.parameters():
        param.requires_grad_(False)
    # SAM3 fused ViT MLP requires torch.is_grad_enabled() == False.
    # Parameter freezing alone does NOT switch off grad mode during forward.
    import types
    import torch
    original_forward_image = model.backbone.forward_image
    def frozen_forward_image(self, samples):
        with torch.no_grad():
            return original_forward_image(samples)
    model.backbone.forward_image = types.MethodType(frozen_forward_image, model.backbone)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 0 < trainable < total, (trainable, total)
    assert any(p.requires_grad for p in model.segmentation_head.parameters()), "Segmentation head frozen!"
    print(f"MUSANGA FROZEN VISION (NO_GRAD): {trainable:,} / {total:,} trainable parameters", flush=True)
    return model
""")
cfg30b["trainer"]["model"]["_target_"]="sam3.train.musanga_frozen_builder.build_frozen_vision_sam3"
print("Frozen-vision model builder:",builder_path30b)
# Official Hydra configuration is registered as part of sam3.train.
official_config30b=Path("/content/sam3/sam3/train/configs/musanga_kvo30b_smoke.yaml")
assert official_config30b.parent.exists(),"SAM3 official config folder not found"
official_config30b.write_text('# ' + '@' + 'package _global_\n'+yaml.safe_dump(cfg30b,sort_keys=False))
# Verify the actual YAML on disk, then test Hydra composition with the SAM3
# interpreter. This catches missing keys BEFORE launching a GPU trainer.
written30b=yaml.safe_load(official_config30b.read_text())
assert set(("launcher","trainer","submitit","scratch")).issubset(written30b), list(written30b)
assert written30b["launcher"]["experiment_log_dir"]
import subprocess
preflight30b=subprocess.run([str(SAMPY),"-c",'''
from hydra import compose, initialize_config_module
from omegaconf import OmegaConf
with initialize_config_module("sam3.train",version_base="1.2"):
    cfg=compose(config_name="configs/musanga_kvo30b_smoke.yaml")
    assert cfg.launcher.gpus_per_node == 1
    assert cfg.launcher.num_nodes == 1
    assert cfg.scratch.resolution == 1008
    assert cfg.trainer.distributed.static_graph is True
    assert cfg.launcher.experiment_log_dir
    print("Hydra config PASS; sections:",list(cfg.keys()))
    print("Training output:",cfg.launcher.experiment_log_dir)
'''],capture_output=True,text=True,cwd="/content/sam3")
print(preflight30b.stdout)
if preflight30b.returncode:
    print(preflight30b.stderr[-3000:])
    raise RuntimeError("SAM3 Hydra config preflight failed; do not launch training.")
print("COCO QA PASS; registered Hydra config:",official_config30b)

# Launch official SAM3 two-batch smoke after successful config and data preflight
import os, subprocess, torch
# The inference-only SAM3 environment may lack trainer dependencies.
check_submitit30b=subprocess.run([str(SAMPY),"-c","import submitit"],capture_output=True)
if check_submitit30b.returncode:
    print("Installing missing official trainer dependency: submitit")
    subprocess.run([str(SAMPY),"-m","pip","install","submitit"],check=True)
assert torch.cuda.is_available(), "Select a Colab GPU runtime"
assert official_config30b.exists(), "Run Block 30B-1 first"
log30b=D30B/"trainer_stdout.log"
cmd30b=[str(SAMPY),"/content/sam3/sam3/train/train.py",
        "-c","configs/musanga_kvo30b_smoke.yaml","--use-cluster","0","--num-gpus","1"]
print("GPU:",torch.cuda.get_device_name(0))
print("Starting short official trainer smoke. Log:",log30b)
with log30b.open("w") as fd:
    proc30b=subprocess.run(cmd30b,cwd="/content/sam3",stdout=fd,stderr=subprocess.STDOUT,check=False)
lines30b=log30b.read_text(errors="replace").splitlines()
brief30b=[line for line in lines30b if ("Train Epoch:" in line or "MUSANGA FROZEN VISION" in line or "RuntimeError:" in line or "Error:" in line or "Checkpoint" in line)]
print("\\n".join(brief30b[-25:]) if brief30b else "\\n".join(lines30b[-25:]))
if proc30b.returncode: print("\\n".join(lines30b[-30:]))
checkpoints30b=list((D30B/"training_logs").rglob("*.pt"))+list((D30B/"training_logs").rglob("*.pth"))
print("Exit code:",proc30b.returncode)
print("Checkpoint files:",[str(x) for x in checkpoints30b])
assert proc30b.returncode==0, f"SMOKE FAILED: inspect {log30b} before launching longer training"
print("Smoke trainer exited normally. Inspect loss/optimizer steps and checkpoint output.")


## 31 — SAM3 two-epoch crown fine-tuning (training only)

Run this **single cell** after Block 30B. It trains with all 105 ROI 0–3 training tiles for two epochs, writes a separate checkpoint, and reports progress. **No validation is run here:** the official validation loader has a COCO segmentation resize issue. ROI 4 evaluation will be a separate step; ROI 5 remains sealed for independent testing.


In [ ]:
# BLOCK 31 — Two-epoch SAM3 crown fine-tuning (ROIs 0–3 only)
# Run after successful Block 30B in the same Colab runtime. No validation/test access.
from pathlib import Path
import copy, json, subprocess, yaml, torch, re, shutil, zipfile
assert torch.cuda.is_available(), "Use an L4 GPU runtime"
assert "cfg30b" in globals() and "D30B" in globals(), "Run Block 30B once first"
assert Path("/content/sam3/sam3/train/musanga_frozen_builder.py").exists(), "Run Block 30B first"
D31=OUT/"experiments_v31_sam3_two_epoch"
D31.mkdir(parents=True,exist_ok=True)
training31=D31/"training_logs"
# Save trainer output locally: Google Drive FUSE produced 6.46 GB truncated ZIP files.
local31=Path("/content/musanga31_training_local")
localckpt31=local31/"checkpoints"/"checkpoint.pt"
ckpt31=training31/"checkpoints"/"checkpoint.pt"
log31=D31/"trainer_stdout.log"
# Avoid accidentally overwriting a completed run; delete D31 manually to rerun.
# A file can exist on Drive despite an interrupted/corrupt PyTorch ZIP write.
# Only rerun automatically if the existing checkpoint is demonstrably unreadable.
if ckpt31.exists():
    try:
        with zipfile.ZipFile(ckpt31,"r") as archive31:
            if archive31.testzip() is not None: raise zipfile.BadZipFile("corrupt checkpoint member")
    except (zipfile.BadZipFile,OSError) as exc:
        broken31=ckpt31.with_name("checkpoint_corrupt_backup.pt")
        if broken31.exists(): broken31=ckpt31.with_name("checkpoint_corrupt_backup_2.pt")
        ckpt31.rename(broken31)
        print("Corrupt checkpoint preserved at:",broken31,"| rebuilding Block 31")
        SKIP31=False
    else:
        print(f"Valid Block 31 checkpoint already exists: {ckpt31}. No retraining necessary.")
        SKIP31=True
else:
    SKIP31=False
dataset31=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"
train31=json.loads((dataset31/"train"/"_annotations.coco.json").read_text())
assert len(train31["images"])==105 and len(train31["annotations"])==1883, "Unexpected training dataset"
cfg31=copy.deepcopy(cfg30b)
cfg31["trainer"]["max_epochs"]=2
cfg31["trainer"]["skip_saving_ckpts"]=False
cfg31["trainer"]["skip_first_val"]=True
cfg31["trainer"]["data"].pop("val",None)
cfg31["trainer"]["meters"].pop("val",None)
cfg31["trainer"]["val_epoch_freq"]=999
cfg31["trainer"]["distributed"]["find_unused_parameters"]=False
cfg31["trainer"]["distributed"]["static_graph"]=True
cfg31["scratch"]["resolution"]=1008
cfg31["scratch"]["target_epoch_size"]=105
cfg31["scratch"]["max_data_epochs"]=2
cfg31["roboflow_train"]["num_images"]=None
cfg31["trainer"]["data"]["train"]["dataset"]["limit_ids"]=None
cfg31["scratch"]["num_train_workers"]=0
cfg31["trainer"]["data"]["train"]["num_workers"]=0
cfg31["launcher"]["experiment_log_dir"]=str(local31)
cfg31["paths"]["experiment_log_dir"]=str(local31)
cfg31["trainer"]["logging"]["log_dir"]=str(local31)
cfg31["trainer"]["model"]["_target_"]="sam3.train.musanga_frozen_builder.build_frozen_vision_sam3"
config31=Path("/content/sam3/sam3/train/configs/musanga_kvo31_two_epoch.yaml")
config31.write_text("# " + "@" + "package _global_\n" + yaml.safe_dump(cfg31,sort_keys=False))
if SKIP31:
    print("Block 31 already complete; skip rerun.")
else:
    free31=shutil.disk_usage("/content").free
    assert free31>=13_000_000_000, (
        f"Only {free31/1e9:.1f} GB local space. Need at least 13 GB for safe local checkpoint. "
        "Free space before training."
    )
    if localckpt31.exists():
        print("Removing previous local checkpoint before retraining:",localckpt31)
        localckpt31.unlink()
    print("Training only: 105 tiles, 1883 overlapping-tile masks; 2 epochs; ROI 4 and ROI 5 excluded")
    print("GPU:",torch.cuda.get_device_name(0),"| LOCAL checkpoint:",localckpt31)
    cmd31=[str(SAMPY),"/content/sam3/sam3/train/train.py",
           "-c","configs/musanga_kvo31_two_epoch.yaml","--use-cluster","0","--num-gpus","1"]
    with log31.open("w") as fd:
        result31=subprocess.run(cmd31,cwd="/content/sam3",stdout=fd,stderr=subprocess.STDOUT,check=False)
    lines31=log31.read_text(errors="replace").splitlines()
    progress31=[line for line in lines31 if "Train Epoch:" in line]
    for line in progress31[-8:]: print(line.strip()[:350])
    print("Epoch progress lines:",len(progress31),"| exit code:",result31.returncode)
    if result31.returncode:
        print("\\n".join(lines31[-22:]))
        raise RuntimeError(f"Block 31 training failed; inspect {log31}")
    assert localckpt31.exists(),"Trainer exited normally but local checkpoint missing"
    with zipfile.ZipFile(localckpt31,"r") as archive31:
        assert archive31.testzip() is None,"LOCAL checkpoint CRC verification failed"
        assert len(archive31.namelist())>10,"LOCAL checkpoint missing weight members"
    print(f"LOCAL checkpoint verified ({localckpt31.stat().st_size/1e9:.2f} GB). Copying to Drive.")
    ckpt31.parent.mkdir(parents=True,exist_ok=True)
    tmp31=ckpt31.with_suffix(".uploading")
    if tmp31.exists():tmp31.unlink()
    shutil.copyfile(localckpt31,tmp31)
    assert tmp31.stat().st_size==localckpt31.stat().st_size,"Incomplete Drive transfer"
    with zipfile.ZipFile(tmp31,"r") as archive31:
        assert archive31.testzip() is None,"DRIVE checkpoint failed CRC verification"
    tmp31.replace(ckpt31)
    print("Verified Drive checkpoint:",ckpt31)
    print("BLOCK 31 TRAINING + CHECKPOINT PASS. ROI 4 evaluation has NOT run; ROI 5 remains held out.")


## 32 — Paired SAM3 evaluation on ROI 4

Runs the Block 31 checkpoint on **exactly the existing Block 28 ROI 4 geometric prompts**, applies the Block 22 polygon filters and duplicate removal, and compares with the existing zero-shot predictions against fully contained manual ROI 4 crowns. Reports one-to-one precision, recall, F1, and segmentation IoU at 0.3/0.5/0.7, saving CSV, GeoPackage and a side-by-side outline figure. **ROI 5 is not accessed.** Requires an existing completed ROI 4 zero-shot cache.


In [ ]:
# BLOCK 32 — Paired ROI 4 validation: zero-shot versus fine-tuned SAM3
# One cell. Requires Blocks 0R, 2, 14; Block 28 ROI 4 baseline; Block 31 checkpoint.
import os, json, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from shapely.geometry import shape
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from IPython.display import display
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14 first"
assert Path(SAMPY).exists(), "Run Block 2 first"
ROI32=4
BASE32=OUT/f"experiments_v22_full_roi_8m_{ROI32}"/"full"
PM32=BASE32/"prompt_manifest.csv"
RAWBASE32=BASE32/"raw"
CLEANBASE32=BASE32/"clean_crowns.gpkg"
assert all(p.exists() for p in [PM32,CLEANBASE32]), "Run Block 28 for full ROI 4 zero-shot baseline first"
CKPT32=OUT/"experiments_v31_sam3_two_epoch"/"training_logs"/"checkpoints"/"checkpoint.pt"
# Validate archive BEFORE loading the 840M-parameter model or running 1056 prompts.
# Drive can contain a truncated checkpoint even after the trainer reports success.
import zipfile
candidates32=[CKPT32]+sorted(CKPT32.parent.glob("checkpoint_*.pt"),reverse=True)
valid32=[]
for candidate in candidates32:
    if not candidate.exists(): continue
    size=candidate.stat().st_size
    try:
        with zipfile.ZipFile(candidate,"r") as z:
            if z.testzip() is not None: raise ValueError("Corrupt member in ZIP archive")
        valid32.append(candidate)
        print("Valid checkpoint archive:",candidate.name, f"({size/1e9:.2f} GB)")
    except (zipfile.BadZipFile,ValueError,OSError) as exc:
        print("INVALID checkpoint:",candidate.name,f"({size/1e9:.2f} GB):",str(exc))
assert valid32, (
    "No readable Block 31 checkpoint. The Drive file is likely truncated. "
    "Preserve the log; rerun Block 31 to regenerate the checkpoint, then retry Block 32."
)
CKPT32=valid32[0]
D32=OUT/"experiments_v32_roi4_finetune_validation"
RAW32=D32/"raw_finetuned";RAW32.mkdir(parents=True,exist_ok=True)
RUN32=Path("/content/musanga_32_finetuned_inference.py")
RUN32.write_text(r'''
import sys,numpy as np,pandas as pd,torch
from pathlib import Path
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
manifest=pd.read_csv(sys.argv[1])
output=Path(sys.argv[2]);output.mkdir(parents=True,exist_ok=True)
ckpt_path=sys.argv[3]
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(
        bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz",
        device="cuda",eval_mode=True)
    ckpt=torch.load(ckpt_path,map_location="cpu",weights_only=False)
    assert "model" in ckpt and isinstance(ckpt["model"],dict),"Expected official trainer checkpoint"
    missing,unexpected=model.load_state_dict(ckpt["model"],strict=False)
    assert not missing and not unexpected,(missing[:15],unexpected[:15])
    del ckpt
    print("Block 31 model weights loaded exactly",flush=True)
    processor=Sam3Processor(model)
    for tid,group in manifest.groupby("tile_id",sort=True):
        todo=[r for r in group.itertuples() if not (output/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not todo:continue
        img=Image.open(todo[0].path).convert("RGB")
        for r in todo:
            state=processor.set_image(img)
            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))
            res=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=res["masks"].detach().cpu().numpy()
            scores=res["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(output/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"ROI4 tile {tid}: {len(todo)} prompts",flush=True)
''')
pm32=pd.read_csv(PM32)
assert len(pm32)>0 and pm32.prompt_id.is_unique
assert all((RAWBASE32/f"prompt_{int(i):06d}.npz").exists() for i in pm32.prompt_id), "Zero-shot prompt cache incomplete; finish Block 28"
missing32=sum(not (RAW32/f"prompt_{int(i):06d}.npz").exists() for i in pm32.prompt_id)
if missing32:
    print(f"Fine-tuned inference for {missing32} ROI 4 prompts (resumable)...")
    log32=D32/"inference_stdout.log"
    with log32.open("w") as fd:
        run32=subprocess.run([str(SAMPY),"-u",str(RUN32),str(PM32),str(RAW32),str(CKPT32)],
                             cwd="/content/sam3",stdout=fd,stderr=subprocess.STDOUT,check=False)
    tail32=log32.read_text(errors="replace").splitlines()
    print("\n".join(tail32[-15:]))
    assert run32.returncode==0, f"Fine-tuned ROI4 inference failed: {log32}"
assert all((RAW32/f"prompt_{int(i):06d}.npz").exists() for i in pm32.prompt_id)
# The same Block 22 polygon cleaning rules, applied to fine-tuned predictions.
from rasterio.windows import Window
from rasterio.features import shapes
MINA32,MAXA32=2.,700.
DUP32=.65;CONTAIN32=.95;RATIO32=.65
with rasterio.open(RASTER) as src:
    crs32=src.crs;gsd32=abs(src.transform.a)
    roi_geom32=roi_v2.to_crs(crs32).geometry.iloc[ROI32]
    features32=[]
    for r in pm32.itertuples():
        with np.load(RAW32/f"prompt_{int(r.prompt_id):06d}.npz") as z:
            masks=z["masks"];scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        trans=src.window_transform(win)
        margin=max(1,int(round(.4/gsd32)))
        for k,ma in enumerate(masks):
            ma=np.asarray(ma)>0
            if ma.shape!=(int(r.height),int(r.width)) or not ma.any():continue
            if ma[:margin].any() or ma[-margin:].any() or ma[:,:margin].any() or ma[:,-margin:].any():continue
            geoms=[shape(g) for g,v in shapes(ma.astype("uint8"),mask=ma,transform=trans) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not MINA32<=geom.area<=MAXA32:continue
            if not roi_geom32.covers(geom.representative_point()):continue
            features32.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),
                                   instance=int(k),score=float(scores[min(k,len(scores)-1)]),
                                   geometry=geom))
assert features32,"No fine-tuned crown polygons survived the common filtering!"
ordered32=gpd.GeoDataFrame(features32,geometry="geometry",crs=crs32).sort_values(
    "score",ascending=False).reset_index(drop=True)
keep32=[]
for i,row in ordered32.iterrows():
    g=row.geometry;duplicate=False
    for j in keep32:
        h=ordered32.geometry.iloc[j]
        if not g.intersects(h):continue
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        ratio=min(g.area,h.area)/max(g.area,h.area)
        if iou>=DUP32 or (inter/min(g.area,h.area)>=CONTAIN32 and ratio>=RATIO32):
            duplicate=True;break
    if not duplicate:keep32.append(i)
fin32=ordered32.iloc[keep32].copy().reset_index(drop=True)
fin32["crown_id"]=np.arange(1,len(fin32)+1)
fin32.to_file(D32/"finetuned_clean_crowns.gpkg",driver="GPKG")
zero32=gpd.read_file(CLEANBASE32).to_crs(crs32)
reference32=reference_v2.to_crs(crs32)
reference32=reference32[reference32.geometry.apply(roi_geom32.covers)].reset_index(drop=True)
assert len(reference32)>0,"No fully contained ROI4 manual crowns"
records32=[]
for name,preds in [("zero_shot",zero32),("fine_tuned_2ep",fin32)]:
    preds=preds[preds.geometry.apply(lambda g:roi_geom32.covers(g.representative_point()))].reset_index(drop=True)
    mat=np.zeros((len(reference32),len(preds)),dtype=np.float32)
    idx=preds.sindex
    for i,g in enumerate(reference32.geometry):
        for j in idx.query(g,predicate="intersects"):
            h=preds.geometry.iloc[j];a=g.intersection(h).area
            if a>0:mat[i,j]=a/(g.area+h.area-a)
    for thr in (.3,.5,.7):
        matches=[]
        if mat.size:
            utility=(mat>=thr).astype(float)*1000+mat
            ii,jj=linear_sum_assignment(utility,maximize=True)
            matches=[float(mat[i,j]) for i,j in zip(ii,jj) if mat[i,j]>=thr]
        tp=len(matches);precision=tp/len(preds) if len(preds) else 0.
        recall=tp/len(reference32);f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        records32.append(dict(model=name,iou_threshold=thr,manual_crowns=len(reference32),
            predicted_crowns=len(preds),matched=tp,precision=precision,recall=recall,
            f1=f1,median_matched_iou=float(np.median(matches)) if matches else np.nan,
            mean_best_reference_iou=float(mat.max(axis=1).mean()) if len(preds) else 0.))
results32=pd.DataFrame(records32)
results32.to_csv(D32/"paired_roi4_metrics.csv",index=False)
print("ROI 4 comparison — identical prompts, crown cleaning and manual reference")
display(results32.round(3))
print("Saved",D32/"paired_roi4_metrics.csv")
# Context map: visually compare same region with manual, baseline and fine-tuned outlines.
fig,axes=plt.subplots(1,2,figsize=(13,6))
for ax,(label,preds) in zip(axes,[("Zero-shot",zero32),("Fine-tuned (2 epochs)",fin32)]):
    reference32.boundary.plot(ax=ax,color="black",linewidth=.9,label="Manual")
    preds.boundary.plot(ax=ax,color="tab:orange",linewidth=.7,alpha=.7,label="SAM3")
    ax.set_title(f"ROI 4 — {label}");ax.set_aspect("equal")
    ax.legend(handles=[
        plt.Line2D([0],[0],color="black",label="Manual"),
        plt.Line2D([0],[0],color="tab:orange",label="SAM3")],loc="upper right")
plt.tight_layout()
fig.savefig(D32/"roi4_outline_comparison.png",dpi=180)
plt.show()
print("ROI 5 was not accessed. Block 32 PASS.")


## 33 — ROI 4 classifier transfer and crown polygon examples

CPU-only single cell. Refit Block 29 RGB Random Forest on ROIs 0–3 only, score both zero-shot and fine-tuned ROI 4 polygons, compare fixed proposal-retention budgets and one-to-one IoU matching, and create three RGB-image example panels with manual crown outlines. **ROI 5 labels are never used.** Low strict precision is diagnosed separately from overlap with manual crowns.


In [ ]:
# BLOCK 33 — ROI4 classifier transfer, precision diagnostics and polygon examples (CPU)
# Training labels from ROI 0–3 only. No ROI4/ROI5 labels enter fitting or ranking.
from pathlib import Path
import json, numpy as np, pandas as pd, geopandas as gpd, rasterio, joblib
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds,Window
from rasterio.features import geometry_mask
from rasterio.enums import Resampling
from shapely.geometry import mapping,box
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display
B33=OUT/"experiments_v33_classifier_roi4"
B33.mkdir(parents=True,exist_ok=True)
S29=OUT/"experiments_v29_rgb_vs_dsm_six_roi"/"candidate_features_and_training_labels.csv"
S32=OUT/"experiments_v32_roi4_finetune_validation"/"finetuned_clean_crowns.gpkg"
S0=OUT/"experiments_v22_full_roi_8m_4"/"full"/"clean_crowns.gpkg"
assert all(p.exists() for p in (S29,S32,S0)), "Require Blocks 29, 32 and ROI4 zero-shot baseline"
assert "reference_v2" in globals() and "roi_v2" in globals() and "RASTER" in globals()
table33=pd.read_csv(S29)
assert {"roi_id","supervision_label","proposal_id"}.issubset(table33.columns)
features33=["sam3_score","area_m2","log_area","perimeter_m","compactness","bbox_fill",
            "max_overlap_iou","overlap_count","containment_count","higher_score_overlap",
            "r_mean","g_mean","b_mean","r_std","g_std","b_std","rgb_ring_contrast"]
assert set(features33).issubset(table33.columns)
labeled33=table33[(table33.roi_id.isin([0,1,2,3]))&(table33.supervision_label.isin([0,1]))].copy()
assert labeled33.supervision_label.nunique()==2,"Training lacks one class"
rf33=make_pipeline(SimpleImputer(strategy="median"),
     RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
     class_weight="balanced_subsample",n_jobs=-1,random_state=42))
rf33.fit(labeled33[features33].replace([np.inf,-np.inf],np.nan),labeled33.supervision_label)
joblib.dump({"model":rf33,"features":features33,"training_rois":[0,1,2,3]},B33/"rgb_rf_train_roi0to3.joblib")
print("Trained RGB classifier on",len(labeled33),"labeled ROI 0–3 proposals; ROI4/ROI5 labels excluded.")
with rasterio.open(RASTER) as ds33:
    crs33=ds33.crs
    roi33=roi_v2.to_crs(crs33).geometry.iloc[4]
    manual33=reference_v2.to_crs(crs33)
    manual33=manual33[manual33.geometry.apply(roi33.covers)].copy().reset_index(drop=True)
    preds33={"zero_shot":gpd.read_file(S0).to_crs(crs33),
             "fine_tuned":gpd.read_file(S32).to_crs(crs33)}
    features_by_model33={}
    for model_name33,pp33 in preds33.items():
        pp33=pp33[pp33.geometry.apply(lambda g:roi33.covers(g.representative_point()))].copy().reset_index(drop=True)
        preds33[model_name33]=pp33
        arr33=pp33.geometry.area.to_numpy()
        per33=pp33.geometry.length.to_numpy()
        scores33=pd.to_numeric(pp33["score"],errors="coerce").fillna(-1).to_numpy()
        vals33=pd.DataFrame(dict(sam3_score=scores33,area_m2=arr33,log_area=np.log1p(arr33),
                    perimeter_m=per33,compactness=4*np.pi*arr33/np.maximum(per33**2,1e-9),
                    bbox_fill=arr33/np.maximum(np.array([g.envelope.area for g in pp33.geometry]),1e-9)))
        for key in ["max_overlap_iou","overlap_count","containment_count","higher_score_overlap"]:
            vals33[key]=0.
        si33=pp33.sindex
        for i,g in enumerate(pp33.geometry):
            for j in si33.query(g,predicate="intersects"):
                j=int(j)
                if j<=i:continue
                h=pp33.geometry.iloc[j]; a=g.intersection(h).area
                if a<=0:continue
                iou=a/max(g.area+h.area-a,1e-9)
                contain=a/min(g.area,h.area)
                vals33.loc[[i,j],"max_overlap_iou"]=np.maximum(vals33.loc[[i,j],"max_overlap_iou"],iou)
                vals33.loc[[i,j],"overlap_count"]+=1
                if contain>=.9:vals33.loc[[i,j],"containment_count"]+=1
                if scores33[i]<scores33[j]:vals33.loc[i,"higher_score_overlap"]+=1
                elif scores33[j]<scores33[i]:vals33.loc[j,"higher_score_overlap"]+=1
        raster_vals33=[]
        for g in pp33.geometry:
            v=[np.nan]*7
            try:
                outer=g.buffer(2.)
                win=from_bounds(*outer.bounds,transform=ds33.transform).round_offsets().round_lengths()
                win=win.intersection(Window(0,0,ds33.width,ds33.height))
                if win.width>=2 and win.height>=2:
                    step=max(1,int(np.ceil(max(win.width,win.height)/192)))
                    w=max(1,int(np.ceil(win.width/step)));h=max(1,int(np.ceil(win.height/step)))
                    data=ds33.read([1,2,3],window=win,out_shape=(3,h,w),resampling=Resampling.average,masked=True)
                    # Pixel-aligned scaled window transform
                    tr=ds33.window_transform(win)*rasterio.Affine.scale(win.width/w,win.height/h)
                    inside=geometry_mask([mapping(g)],out_shape=(h,w),transform=tr,invert=True)
                    ring=geometry_mask([mapping(outer)],out_shape=(h,w),transform=tr,invert=True)&~inside
                    if inside.sum()>=3:
                        x=np.asarray(data.filled(np.nan),dtype=float)
                        means=[];stds=[]
                        for channel in range(3):
                            a=x[channel,inside];a=a[np.isfinite(a)]
                            means.append(float(np.mean(a)) if len(a) else np.nan)
                            stds.append(float(np.std(a)) if len(a) else np.nan)
                        gray=x.mean(axis=0)
                        ai=gray[inside];ai=ai[np.isfinite(ai)]
                        ar=gray[ring];ar=ar[np.isfinite(ar)]
                        contrast=float(np.median(ai)-np.median(ar)) if len(ai) and len(ar) else np.nan
                        v=means+stds+[contrast]
            except Exception:
                pass
            raster_vals33.append(v)
        for j,k in enumerate(["r_mean","g_mean","b_mean","r_std","g_std","b_std","rgb_ring_contrast"]):
            vals33[k]=np.array(raster_vals33)[:,j]
        assert len(vals33)==len(pp33)
        vals33["rf_score"]=rf33.predict_proba(vals33[features33].replace([np.inf,-np.inf],np.nan))[:,1]
        vals33["model"]=model_name33
        vals33.to_csv(B33/f"{model_name33}_scores.csv",index=False)
        features_by_model33[model_name33]=vals33
        print(model_name33,"proposals:",len(pp33),"| median RF score",round(float(vals33.rf_score.median()),3))
# Evaluate fixed budgets, without selecting a favorable threshold on ROI4.
rows33=[]
label_diag33=[]
selected33={}
for model_name33,p in preds33.items():
    f=features_by_model33[model_name33]
    mat=np.zeros((len(manual33),len(p)),dtype=np.float32)
    pindex=p.sindex
    for i,g in enumerate(manual33.geometry):
        for j in pindex.query(g,predicate="intersects"):
            h=p.geometry.iloc[j]; area=g.intersection(h).area
            if area>0:mat[i,j]=area/(g.area+h.area-area)
    best=mat.max(axis=0) if len(manual33) else np.zeros(len(p))
    label_diag33.extend(dict(model=model_name33,pred_id=j,
                     best_manual_iou=float(best[j]),rf_score=float(f.rf_score.iloc[j]))
                      for j in range(len(p)))
    for method in ["sam_score","rgb_rf"]:
        sc=(f.sam3_score if method=="sam_score" else f.rf_score).to_numpy()
        ranking=np.argsort(-sc,kind="stable")
        for fraction in [1.,.9,.8,.7,.6,.5,.4,.3]:
            selected=ranking[:max(1,int(np.ceil(fraction*len(p))))]
            sub=mat[:,selected]
            for threshold in [.3,.5,.7]:
                matched=[]
                if sub.size:
                    ii,jj=linear_sum_assignment((sub>=threshold).astype(float)*2+sub,maximize=True)
                    matched=[float(sub[i,j]) for i,j in zip(ii,jj) if sub[i,j]>=threshold]
                tp=len(matched);precision=tp/len(selected);recall=tp/len(manual33)
                rows33.append(dict(model=model_name33,method=method,fraction=fraction,
                    iou_threshold=threshold,manual=len(manual33),predictions=len(selected),
                    matched=tp,precision=precision,recall=recall,
                    f1=2*precision*recall/(precision+recall) if precision+recall else 0.,
                    median_matched_iou=float(np.median(matched)) if matched else np.nan))
            if method=="rgb_rf" and fraction==.6:selected33[model_name33]=selected
results33=pd.DataFrame(rows33)
results33.to_csv(B33/"ranking_retention_comparison.csv",index=False)
pd.DataFrame(label_diag33).to_csv(B33/"proposal_iou_diagnostics.csv",index=False)
display(results33[(results33.iou_threshold==.5)&results33.fraction.isin([1.,.7,.6,.5])].round(3))
# Precision diagnostic: among unmatched proposals, count near-misses and severe overlaps.
diagnostics33=pd.DataFrame(label_diag33)
diagnostics33["overlaps_some_crown"]=diagnostics33.best_manual_iou>0
diagnostics33["near_match_03"]=diagnostics33.best_manual_iou>=.3
print("Best manual-IoU distribution (NOT one-to-one classification):")
display(diagnostics33.groupby("model").agg(
    total=("pred_id","count"),intersects_manual=("overlaps_some_crown","sum"),
    best_iou_ge03=("near_match_03","sum")).round(2))
# 3 fixed local windows around manual crowns: raw RGB + manual, baseline, fine-tuned and RF subset.
with rasterio.open(RASTER) as ds33:
    nshow=min(3,len(manual33))
    # Deterministic reference positions across the ROI, not chosen by performance.
    picks=np.linspace(0,len(manual33)-1,nshow,dtype=int)
    fig,axes=plt.subplots(nshow,3,figsize=(15,5*nshow),squeeze=False)
    for row,ind in enumerate(picks):
        center=manual33.geometry.iloc[ind].centroid
        extent=box(center.x-14,center.y-14,center.x+14,center.y+14)
        win=from_bounds(*extent.bounds,transform=ds33.transform).round_offsets().round_lengths()
        data=ds33.read([1,2,3],window=win,boundless=True,fill_value=0)
        transform=ds33.window_transform(win)
        bb=rasterio.windows.bounds(win,ds33.transform)
        rgb=np.moveaxis(data,0,-1)
        if rgb.dtype!=np.uint8:
            rgb=np.uint8(np.clip(rgb,0,255))
        panels=[("Zero-shot",preds33["zero_shot"]),
                ("Fine-tuned",preds33["fine_tuned"]),
                ("Fine-tuned + RF 60%",preds33["fine_tuned"].iloc[selected33["fine_tuned"]])]
        for col,(title,pp) in enumerate(panels):
            ax=axes[row,col]
            ax.imshow(rgb,extent=bb,origin="upper")
            refs=manual33[manual33.intersects(extent)]
            if len(refs):refs.boundary.plot(ax=ax,color="yellow",linewidth=1.7)
            props=pp[pp.intersects(extent)]
            if len(props):props.boundary.plot(ax=ax,color="cyan",linewidth=1.0)
            ax.set_xlim(extent.bounds[0],extent.bounds[2])
            ax.set_ylim(extent.bounds[1],extent.bounds[3])
            ax.set_aspect("equal");ax.set_title(f"{title} | window {row+1}")
            ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle("ROI 4: manual crowns yellow, SAM3 polygons cyan (same RGB background)",fontsize=13)
    fig.tight_layout()
    fig.savefig(B33/"roi4_examples_manual_vs_sam3.png",dpi=160)
    plt.show()
print("Saved Block 33:",B33)
print("Interpretation: precision requires strict one-to-one IoU; unmatched masks may overlap annotated crowns or represent splits.")


## 34 — Pilot: grid versus centre-proxy versus adaptive box prompts

Single-cell exploratory test on three fixed ROI 4 tiles. Uses **only previously predicted zero-shot polygons** to locate candidate centres; manual crowns are used **only after inference** for evaluation. SAM3's image processor has no native point prompt, so the 1 m centre box is a proxy, not a true point. Results are not a matched-budget full-ROI ranking. ROI 5 is untouched.


In [ ]:
# BLOCK 34 — Pilot prompting experiment (ROI 4 only; no annotation-guided prompts)
# Compares grid boxes, baseline-proposal-centred 1m boxes, and adaptive baseline boxes.
import json, subprocess, zipfile
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.features import shapes
from rasterio.windows import Window
from shapely.geometry import Point,shape,box
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
assert "reference_v2" in globals() and "roi_v2" in globals() and "SAMPY" in globals(), "Run Blocks 0R, 2 and 14"
D34=OUT/"experiments_v34_prompt_pilot";D34.mkdir(parents=True,exist_ok=True)
BASE34=OUT/"experiments_v22_full_roi_8m_4"/"full"
PM34=BASE34/"prompt_manifest.csv"
Z34=BASE34/"clean_crowns.gpkg"
CK34=OUT/"experiments_v31_sam3_two_epoch"/"training_logs"/"checkpoints"/"checkpoint.pt"
assert PM34.exists() and Z34.exists() and CK34.exists(),"Requires ROI4 Block28 baseline and Block31 model"
# Google Drive may expose an incomplete checkpoint; check all known copies first.
checkpoint_candidates34=[
    CK34,
    OUT/"experiments_v30b_training_smoke"/"training_logs"/"checkpoints"/"checkpoint.pt",
]
# The smoke checkpoint is NOT the two-epoch model and must never be used as a substitute.
good34=[]
for candidate34 in checkpoint_candidates34[:1]:
    if not candidate34.exists():continue
    print("Block 31 checkpoint:",round(candidate34.stat().st_size/1e9,2),"GB",
          "| readable ZIP:",zipfile.is_zipfile(candidate34))
    if zipfile.is_zipfile(candidate34):good34.append(candidate34)
if not good34:
    raise RuntimeError(
        "The Block 31 checkpoint is incomplete or corrupted on Drive. "
        "Do not rerun training yet. In Colab, check whether Block 32's saved "
        "fine-tuned ROI4 masks are complete; those outputs are still usable. "
        "A full checkpoint recovery requires saving the trainer model weights "
        "to local /content first, then copying the verified archive to Drive."
    )
CK34=good34[0]
with rasterio.open(RASTER) as src34:
    crs34=src34.crs;gsd34=abs(src34.transform.a)
roi34=roi_v2.to_crs(crs34).geometry.iloc[4]
base34=gpd.read_file(Z34).to_crs(crs34)
pm34=pd.read_csv(PM34)
# Block 22 stores pixel offsets, not xmin/ymin/xmax/ymax in its prompt manifest.
required34={"tile_id","row_off","col_off","width","height","path","cx","cy","box_m","gsd"}
assert required34.issubset(pm34.columns), f"Missing: {sorted(required34-set(pm34.columns))}"
with rasterio.open(RASTER) as src34:
    bounds34=[rasterio.windows.bounds(
        Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)),src34.transform)
        for r in pm34.itertuples()]
pm34[["xmin","ymin","xmax","ymax"]]=np.asarray(bounds34)
print("Block 22 manifest reconciled using raster pixel offsets:",len(pm34),"prompts")
# Deterministic sample: three tiles with the most independent zero-shot candidate centres.
tile_rows34=[]
for tid34,g34 in pm34.groupby("tile_id"):
    rr34=g34.iloc[0]
    footprint34=box(rr34.xmin,rr34.ymin,rr34.xmax,rr34.ymax)
    candidates34=base34[base34.geometry.apply(lambda g:footprint34.covers(g.representative_point()))]
    tile_rows34.append((int(tid34),len(candidates34)))
selected_tiles34=[tid for tid,_ in sorted(tile_rows34,key=lambda x:(-x[1],x[0]))[:3]]
print("ROI4 pilot tiles:",selected_tiles34)
rows34=[];pid34=0
for tid34 in selected_tiles34:
    tile34=pm34[pm34.tile_id==tid34]
    origin34=tile34.iloc[0]
    footprint34=box(origin34.xmin,origin34.ymin,origin34.xmax,origin34.ymax)
    ngrid34=min(12,len(tile34))
    for r34 in tile34.iloc[:ngrid34].itertuples():
        rows34.append(dict(prompt_id=pid34,method="grid_8m",tile_id=tid34,path=r34.path,
            cx=float(r34.cx),cy=float(r34.cy),box_m=8.,width=r34.width,
            height=r34.height,gsd=r34.gsd,row_off=r34.row_off,col_off=r34.col_off,xmin=r34.xmin,ymin=r34.ymin,xmax=r34.xmax,ymax=r34.ymax))
        pid34+=1
    candidates34=base34[base34.geometry.apply(lambda g:footprint34.covers(g.representative_point()))].copy()
    candidates34=candidates34.sort_values("score",ascending=False).head(12)
    for rr in candidates34.itertuples():
        center=rr.geometry.representative_point()
        cx=(center.x-origin34.xmin)/(origin34.xmax-origin34.xmin)
        cy=(origin34.ymax-center.y)/(origin34.ymax-origin34.ymin)
        # 1m positive box = point-like spatial hint, NOT an actual SAM3 point prompt.
        bounds=rr.geometry.bounds
        adaptive=max(3.,min(16.,1.3*max(bounds[2]-bounds[0],bounds[3]-bounds[1])))
        for method,width_m in (("centre_proxy_1m",1.),("adaptive_box",adaptive)):
            rows34.append(dict(prompt_id=pid34,method=method,tile_id=tid34,path=origin34.path,
                cx=float(cx),cy=float(cy),box_m=width_m,width=origin34.width,
                height=origin34.height,gsd=origin34.gsd,row_off=origin34.row_off,col_off=origin34.col_off,xmin=origin34.xmin,
                ymin=origin34.ymin,xmax=origin34.xmax,ymax=origin34.ymax))
            pid34+=1
manifest34=pd.DataFrame(rows34)
manifest34.to_csv(D34/"pilot_prompts.csv",index=False)
print("Prompts by method:",manifest34.method.value_counts().to_dict())
# Resume without changing completed masks. SAM3 uses the same Block31 checkpoint for all three methods.
RAW34=D34/"masks";RAW34.mkdir(exist_ok=True)
SCRIPT34=Path("/content/musanga_34_prompt_pilot.py")
SCRIPT34.write_text(r'''
import sys,torch,numpy as np,pandas as pd
from PIL import Image
from pathlib import Path
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]);outdir=Path(sys.argv[2]);ckpt_path=sys.argv[3]
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
    ckpt=torch.load(ckpt_path,map_location="cpu",weights_only=False)
    missing,unexpected=model.load_state_dict(ckpt["model"],strict=False)
    assert not missing and not unexpected,(missing[:5],unexpected[:5])
    del ckpt
    processor=Sam3Processor(model)
    for tid,group in df.groupby("tile_id"):
        todo=[r for r in group.itertuples() if not (outdir/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not todo:continue
        img=Image.open(todo[0].path).convert("RGB")
        for r in todo:
            state=processor.set_image(img)
            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))
            outputs=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=outputs["masks"].detach().cpu().numpy()
            scores=outputs["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(outdir/f"prompt_{int(r.prompt_id):06d}.npz",
                    masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"tile {tid} done: {len(todo)} prompts",flush=True)
''')
if any(not (RAW34/f"prompt_{int(pid):06d}.npz").exists() for pid in manifest34.prompt_id):
    print("Running short GPU prompting pilot...")
    with (D34/"inference.log").open("w") as fd:
        run34=subprocess.run([str(SAMPY),"-u",str(SCRIPT34),str(D34/"pilot_prompts.csv"),
             str(RAW34),str(CK34)],cwd="/content/sam3",stdout=fd,stderr=subprocess.STDOUT)
    print("\n".join((D34/"inference.log").read_text().splitlines()[-12:]))
    assert run34.returncode==0,"Pilot inference failed; inspect inference.log"
# Convert masks to polygons; edge exclusion matches Block22.
predictions34=[]
with rasterio.open(RASTER) as ds:
    for r in manifest34.itertuples():
        with np.load(RAW34/f"prompt_{int(r.prompt_id):06d}.npz") as d:
            masks=d["masks"];scores=np.asarray(d["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)) if hasattr(r,"col_off") else None
        if win is None:
            row,col=ds.index(r.xmin,r.ymax)
            win=Window(col,row,int(r.width),int(r.height))
        trans=ds.window_transform(win)
        mrg=max(1,int(round(.4/gsd34)))
        for k,ma in enumerate(masks):
            mask=np.asarray(ma)>0
            if mask.shape!=(int(r.height),int(r.width)) or not mask.any():continue
            if mask[:mrg].any() or mask[-mrg:].any() or mask[:,:mrg].any() or mask[:,-mrg:].any():continue
            poly=[shape(g) for g,v in shapes(mask.astype("uint8"),mask=mask,transform=trans) if v==1]
            if not poly:continue
            geom=max(poly,key=lambda g:g.area)
            if not (2<=geom.area<=700) or not roi34.covers(geom.representative_point()):continue
            predictions34.append(dict(tile_id=int(r.tile_id),method=r.method,prompt_id=int(r.prompt_id),
                                      score=float(scores[min(k,len(scores)-1)]),geometry=geom))
assert predictions34,"No usable polygons"
P34=gpd.GeoDataFrame(predictions34,crs=crs34)
# Compare on identical footprints, matching to fully contained reference crowns within each tile.
refs34=reference_v2.to_crs(crs34)
summary34=[]
for tid in selected_tiles34:
    g=manifest34[manifest34.tile_id==tid].iloc[0]
    tilepoly=box(g.xmin,g.ymin,g.xmax,g.ymax)
    rr=refs34[refs34.geometry.apply(lambda x:tilepoly.covers(x) and roi34.covers(x))].reset_index(drop=True)
    for method in ["grid_8m","centre_proxy_1m","adaptive_box"]:
        pp=P34[(P34.tile_id==tid)&(P34.method==method)].reset_index(drop=True)
        mat=np.zeros((len(rr),len(pp)),dtype=float)
        for i,rg in enumerate(rr.geometry):
            for j,pg in enumerate(pp.geometry):
                if rg.intersects(pg):
                    a=rg.intersection(pg).area
                    mat[i,j]=a/max(rg.area+pg.area-a,1e-9)
        matches=0
        if mat.size:
            ii,jj=linear_sum_assignment((mat>=.5).astype(float)*2+mat,maximize=True)
            matches=int(sum(mat[i,j]>=.5 for i,j in zip(ii,jj)))
        summary34.append(dict(tile_id=tid,method=method,manual=len(rr),proposals=len(pp),
                              matched_iou05=matches,recall=matches/len(rr) if len(rr) else np.nan,
                              precision=matches/len(pp) if len(pp) else 0.))
df34=pd.DataFrame(summary34)
df34.to_csv(D34/"pilot_comparison.csv",index=False)
display(df34.round(3))
# Image overlays for each tile (first 3 methods on the same RGB background).
with rasterio.open(RASTER) as ds:
    fig,axs=plt.subplots(len(selected_tiles34),3,figsize=(15,5*len(selected_tiles34)),squeeze=False)
    for i,tid in enumerate(selected_tiles34):
        r=manifest34[manifest34.tile_id==tid].iloc[0]
        row,col=ds.index(r.xmin,r.ymax)
        win=Window(col,row,int(r.width),int(r.height))
        rgb=ds.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        if rgb.dtype!=np.uint8:rgb=np.uint8(np.clip(rgb,0,255))
        extent=(r.xmin,r.xmax,r.ymin,r.ymax)
        bounds=box(r.xmin,r.ymin,r.xmax,r.ymax)
        refclip=refs34[refs34.intersects(bounds)]
        for j,method in enumerate(["grid_8m","centre_proxy_1m","adaptive_box"]):
            ax=axs[i,j];ax.imshow(rgb,extent=extent,origin="upper")
            pp=P34[(P34.tile_id==tid)&(P34.method==method)]
            if len(refclip):refclip.boundary.plot(ax=ax,color="yellow",linewidth=1.3)
            if len(pp):pp.boundary.plot(ax=ax,color="cyan",linewidth=.9)
            ax.set_xlim(r.xmin,r.xmax);ax.set_ylim(r.ymin,r.ymax)
            ax.set_title(f"Tile {tid}: {method} ({len(pp)} masks)");ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle("Yellow: manual crowns (evaluation only). Cyan: fine-tuned SAM3. Prompt budgets differ.",fontsize=13)
    fig.tight_layout();fig.savefig(D34/"prompt_strategy_examples.png",dpi=160);plt.show()
print("BLOCK 34 PILOT COMPLETE:",D34)
print("IMPORTANT: small non-matched prompt budgets; exploratory only, not a fair full-ROI benchmark.")


## 35 — ROI 4 RF-guided crown instance suppression

CPU only, using the cached Block 32 proposals and Block 33 RGB classifier scores. Compares global RF ranking against local IoU non-maximum suppression and spatial overlap exclusion at fixed budgets and overlap thresholds. Saves numeric precision/recall/F1 comparisons and RGB crown-outline panels. ROI 5 is never used; don't select final hyperparameters on it.


In [ ]:
# BLOCK 35 — RF-guided instance suppression on cached ROI 4 polygons (CPU only)
# No SAM3 inference, no model fitting, no ROI 5. Prespecified overlap sweeps.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box
from rasterio.windows import from_bounds
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D35=OUT/"experiments_v35_roi4_spatial_selection";D35.mkdir(parents=True,exist_ok=True)
D33=OUT/"experiments_v33_classifier_roi4"
P35=OUT/"experiments_v32_roi4_finetune_validation"/"finetuned_clean_crowns.gpkg"
S35=D33/"fine_tuned_scores.csv"
assert P35.exists() and S35.exists(),"Run Blocks 32 and 33 first"
assert "reference_v2" in globals() and "roi_v2" in globals() and "RASTER" in globals()
with rasterio.open(RASTER) as src: crs35=src.crs
roi35=roi_v2.to_crs(crs35).geometry.iloc[4]
p35=gpd.read_file(P35).to_crs(crs35)
p35=p35[p35.geometry.apply(lambda x:roi35.covers(x.representative_point()))].reset_index(drop=True)
f35=pd.read_csv(S35)
assert len(p35)==len(f35) and "rf_score" in f35, "Block33 prediction-score order mismatch"
assert len(p35)==1075, f"Expected 1075 fine-tuned ROI4 candidates, found {len(p35)}"
scores35=f35.rf_score.to_numpy(dtype=float)
assert np.isfinite(scores35).all()
ranking35=np.argsort(-scores35,kind="stable")
areas35=p35.geometry.area.to_numpy()
# Cache ONLY intersecting pairs: no dense all-pairs geometry calculation.
neighbors35=[[] for _ in range(len(p35))]
spatial35=p35.sindex
npairs35=0
for i,g in enumerate(p35.geometry):
    for j in spatial35.query(g,predicate="intersects"):
        j=int(j)
        if j<=i:continue
        h=p35.geometry.iloc[j];inter=g.intersection(h).area
        if inter<=0:continue
        union=max(areas35[i]+areas35[j]-inter,1e-9)
        iou=inter/union
        cover=inter/min(areas35[i],areas35[j])
        neighbors35[i].append((j,iou,cover))
        neighbors35[j].append((i,iou,cover))
        npairs35+=1
print(f"ROI4: {len(p35)} predictions; {npairs35} overlapping pairs.")
def choose35(mode,limit=538,threshold=.5):
    picked=[];selected=set()
    for i in ranking35:
        i=int(i)
        if mode=="ranking":
            passed=True
        else:
            # iou_nms: suppress only near-duplicates with IoU > threshold
            # spatial: reject masks covering > threshold of smaller polygon
            passed=all((iou if mode=="iou_nms" else cover)<=threshold
                       for j,iou,cover in neighbors35[i] if j in selected)
        if passed:
            selected.add(i);picked.append(i)
            if len(picked)>=limit:break
    return np.array(picked,dtype=int)
r35=reference_v2.to_crs(crs35)
r35=r35[r35.geometry.apply(roi35.covers)].reset_index(drop=True)
assert len(r35)==198,f"Expected 198 manual ROI4 crowns, got {len(r35)}"
mat35=np.zeros((len(r35),len(p35)),dtype=np.float32)
for i,g in enumerate(r35.geometry):
    for j in spatial35.query(g,predicate="intersects"):
        h=p35.geometry.iloc[int(j)]
        a=g.intersection(h).area
        if a>0:mat35[i,int(j)]=a/(g.area+h.area-a)
def evaluate35(ids,t=.5):
    n=len(ids);sub=mat35[:,ids]
    if not n:return 0,0.,0.,0.
    ii,jj=linear_sum_assignment((sub>=t).astype(float)*2+sub,maximize=True)
    tp=int(np.count_nonzero(sub[ii,jj]>=t))
    precision=tp/n;recall=tp/len(r35)
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.
    return tp,precision,recall,f1
rows35=[]; selections35={}
for budget35 in [538,645,753,1075]:
    for mode35,thresholds35 in [
        ("ranking",[None]),("iou_nms",[.3,.5,.65]),
        ("spatial",[.3,.5,.7])]:
        for threshold35 in thresholds35:
            ids35=choose35(mode35,budget35,threshold35 or .5)
            key35=(mode35,budget35,threshold35)
            selections35[key35]=ids35
            for matching35 in [.3,.5,.7]:
                tp,pr,rc,f1=evaluate35(ids35,matching35)
                rows35.append(dict(method=mode35,overlap_threshold=threshold35,
                    budget_target=budget35,retained=len(ids35),
                    matching_iou=matching35,matched=tp,precision=pr,recall=rc,f1=f1))
result35=pd.DataFrame(rows35)
result35.to_csv(D35/"spatial_selection_sweep.csv",index=False)
print("IoU ≥ 0.5; target 538 predictions (methods may retain fewer):")
display(result35[(result35.matching_iou==.5)&(result35.budget_target==538)]
        .sort_values("f1",ascending=False).round(3))
# Diagnostic subset visualizations: fixed windows, not selected using manual IoU.
fig,axes=plt.subplots(3,4,figsize=(17,14),squeeze=False)
visual35=[("RF top 50%",selections35[("ranking",538,None)]),
          ("RF + IoU NMS 0.5",selections35[("iou_nms",538,.5)]),
          ("RF + spatial 0.5",selections35[("spatial",538,.5)]),
          ("RF + spatial 0.3",selections35[("spatial",538,.3)])]
with rasterio.open(RASTER) as src:
    for row,idx in enumerate(np.linspace(0,len(r35)-1,3,dtype=int)):
        center=r35.geometry.iloc[idx].centroid
        extent=box(center.x-14,center.y-14,center.x+14,center.y+14)
        win=from_bounds(*extent.bounds,transform=src.transform).round_offsets().round_lengths()
        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        if rgb.dtype!=np.uint8:rgb=np.uint8(np.clip(rgb,0,255))
        bb=rasterio.windows.bounds(win,src.transform)
        localrefs=r35[r35.intersects(extent)]
        for col,(title,ids) in enumerate(visual35):
            ax=axes[row,col]
            ax.imshow(rgb,extent=bb,origin="upper")
            if len(localrefs):localrefs.boundary.plot(ax=ax,color="yellow",linewidth=1.4)
            localpred=p35.iloc[ids]
            localpred=localpred[localpred.intersects(extent)]
            if len(localpred):localpred.boundary.plot(ax=ax,color="cyan",linewidth=.9)
            ax.set_xlim(extent.bounds[0],extent.bounds[2])
            ax.set_ylim(extent.bounds[1],extent.bounds[3])
            ax.set_title(f"{title} | {len(ids)} total")
            ax.set_xticks([]);ax.set_yticks([])
fig.suptitle("ROI4: manual crowns yellow; fine-tuned SAM3 candidates cyan",fontsize=13)
fig.tight_layout()
fig.savefig(D35/"spatial_selection_examples.png",dpi=160)
plt.show()
# Save candidate IDs for reproducible comparison, without choosing the best threshold automatically.
records35=[]
for (mode,budget,threshold),ids in selections35.items():
    records35.extend(dict(method=mode,budget_target=budget,overlap_threshold=threshold,
                          proposal_id=int(idx),rf_score=float(scores35[idx]))
                     for idx in ids)
pd.DataFrame(records35).to_csv(D35/"selected_ids.csv",index=False)
print("BLOCK 35 PASS — outputs:",D35)
print("Interpretation: the highest-scoring setting on ROI4 is exploratory; reserve ROI5 for final fixed-method test.")


## 36 — GIS crown reconstruction

CPU-only ROI 4 comparison of (i) the existing RF top-538 candidates, (ii) smoothed simplified crown polygons, and (iii) polygon-union rasterization followed by seeded distance-transform watershed. Seeds are derived from RF-ranked SAM3 candidates, never reference crowns. Outputs: metrics, three GeoPackages, and RGB comparison figure. ROI 5 remains untouched.


In [ ]:
# BLOCK 36 — GIS crown reconstruction from existing SAM3 polygons (CPU only, ROI 4)
# Compare RF baseline, polygon simplification, and union + seeded watershed.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box,shape
from shapely.ops import unary_union
from rasterio.features import rasterize,shapes
from rasterio.transform import from_origin
from rasterio.windows import from_bounds
from scipy.ndimage import distance_transform_edt, label as cc_label
from scipy.optimize import linear_sum_assignment
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from IPython.display import display
D36=OUT/"experiments_v36_gis_crown_reconstruction";D36.mkdir(parents=True,exist_ok=True)
P36=OUT/"experiments_v32_roi4_finetune_validation"/"finetuned_clean_crowns.gpkg"
S36=OUT/"experiments_v33_classifier_roi4"/"fine_tuned_scores.csv"
assert P36.exists() and S36.exists(),"Run Blocks 32 and 33 first"
with rasterio.open(RASTER) as src:crs36=src.crs;pixel36=abs(src.transform.a)
roi36=roi_v2.to_crs(crs36).geometry.iloc[4]
pred36=gpd.read_file(P36).to_crs(crs36)
pred36=pred36[pred36.geometry.apply(lambda x:roi36.covers(x.representative_point()))].reset_index(drop=True)
scores36=pd.read_csv(S36).rf_score.to_numpy()
assert len(pred36)==len(scores36)==1075,"Block 33 scores are not aligned to Block 32 polygons"
ids36=np.argsort(-scores36,kind="stable")[:538]
original36=pred36.iloc[ids36].copy().reset_index(drop=True)
reference36=reference_v2.to_crs(crs36)
reference36=reference36[reference36.geometry.apply(roi36.covers)].reset_index(drop=True)
assert len(reference36)==198
# Keep the polygon-smoothing baseline entirely independent of reference boundaries.
def cleaned36(geoms,buffer_m=.45,min_area=8.):
    out=[]
    for g in geoms:
        if g is None or g.is_empty:continue
        smooth=g.buffer(buffer_m).buffer(-buffer_m).simplify(.25,preserve_topology=True)
        if smooth.is_empty:continue
        if smooth.geom_type=="MultiPolygon":smooth=max(smooth.geoms,key=lambda x:x.area)
        if smooth.is_valid and smooth.area>=min_area:
            out.append(smooth)
    return gpd.GeoDataFrame(geometry=out,crs=crs36)
smooth36=cleaned36(original36.geometry)
# Rasters for union reconstruction: mask is derived only from selected SAM3 polygons.
resolution36=max(.35,min(.6,pixel36*2))
minx,miny,maxx,maxy=roi36.bounds
margin36=4.
minx-=margin36;miny-=margin36;maxx+=margin36;maxy+=margin36
width36=int(np.ceil((maxx-minx)/resolution36))
height36=int(np.ceil((maxy-miny)/resolution36))
assert width36*height36<20_000_000, "Raster unexpectedly large; increase resolution36"
tr36=from_origin(minx,maxy,resolution36,resolution36)
mask36=rasterize(((g,1) for g in original36.geometry),out_shape=(height36,width36),
                 transform=tr36,fill=0,dtype="uint8").astype(bool)
region36=rasterize([(roi36,1)],out_shape=mask36.shape,transform=tr36,fill=0,dtype="uint8").astype(bool)
mask36 &= region36
dist36=distance_transform_edt(mask36)*resolution36
# Seed polygons: choose RF-ranked interior representative points >=2m apart;
# add a distance peak to any disconnected canopy island missing a seed.
markers36=np.zeros(mask36.shape,dtype=np.int32)
seedpoints36=[]
for g in original36.geometry:
    pt=g.representative_point()
    row=int((maxy-pt.y)/resolution36);col=int((pt.x-minx)/resolution36)
    if not (0<=row<height36 and 0<=col<width36 and mask36[row,col]):continue
    if any((pt.x-x)**2+(pt.y-y)**2<2.0**2 for x,y in seedpoints36):continue
    seedpoints36.append((pt.x,pt.y))
    markers36[row,col]=len(seedpoints36)
islands36,n_islands36=cc_label(mask36)
seeded_islands36=set(np.unique(islands36[markers36>0]))-{0}
for island_id36 in range(1,n_islands36+1):
    if island_id36 in seeded_islands36:continue
    coords36=np.argwhere(islands36==island_id36)
    if not len(coords36):continue
    point36=coords36[np.argmax(dist36[coords36[:,0],coords36[:,1]])]
    markers36[tuple(point36)]=markers36.max()+1
# Watershed uses distance-to-edge to define compact, non-overlapping regions.
labels36=watershed(-dist36,markers36,mask=mask36)
polygons36=[]
for geom36,val36 in shapes(labels36.astype("int32"),mask=labels36>0,transform=tr36):
    g=shape(geom36)
    if not g.is_valid:g=g.buffer(0)
    if g.is_empty:continue
    if g.geom_type=="MultiPolygon":g=max(g.geoms,key=lambda x:x.area)
    if g.area<8:continue
    g=g.simplify(.25,preserve_topology=True).intersection(roi36)
    if not g.is_empty and g.area>=8:polygons36.append(g)
watershed36=gpd.GeoDataFrame(geometry=polygons36,crs=crs36)
assert len(watershed36),"Watershed produced no crown objects"
# Strict one-to-one polygon matching and independent canopy coverage diagnostics.
def metric36(pp):
    mat=np.zeros((len(reference36),len(pp)),dtype=np.float32)
    si=pp.sindex
    for i,g in enumerate(reference36.geometry):
        for j in si.query(g,predicate="intersects"):
            h=pp.geometry.iloc[int(j)]
            a=g.intersection(h).area
            if a>0:mat[i,int(j)]=a/(g.area+h.area-a)
    out=[]
    for t in [.3,.5,.7]:
        if mat.size:
            ii,jj=linear_sum_assignment((mat>=t).astype(float)*2+mat,maximize=True)
            tp=int(np.count_nonzero(mat[ii,jj]>=t))
        else:tp=0
        pr=tp/len(pp) if len(pp) else 0.;rc=tp/len(reference36)
        out.append(dict(iou_threshold=t,predictions=len(pp),matched=tp,
                        precision=pr,recall=rc,f1=2*pr*rc/(pr+rc) if pr+rc else 0.))
    return out
rows36=[]
for name36,pp36 in [("RF_top_538",original36),("GIS_smoothed",smooth36),("union_seeded_watershed",watershed36)]:
    rows36.extend(dict(method=name36,**r) for r in metric36(pp36))
    pp36.to_file(D36/f"{name36}.gpkg",driver="GPKG")
results36=pd.DataFrame(rows36);results36.to_csv(D36/"reconstruction_metrics.csv",index=False)
print("Geometry matching:");display(results36[results36.iou_threshold==.5].round(3))
print("Watershed seeds:",int(markers36.max()),"| islands:",n_islands36,"| resulting objects:",len(watershed36))
# Illustration windows chosen deterministically, not by performance.
fig,axes=plt.subplots(3,3,figsize=(14,14),squeeze=False)
variants36=[("RF top 538",original36),("Smoothed",smooth36),("Union + watershed",watershed36)]
with rasterio.open(RASTER) as src:
    for i,ridx in enumerate(np.linspace(0,len(reference36)-1,3,dtype=int)):
        pt=reference36.geometry.iloc[ridx].centroid
        bounds=box(pt.x-16,pt.y-16,pt.x+16,pt.y+16)
        win=from_bounds(*bounds.bounds,transform=src.transform).round_offsets().round_lengths()
        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        if rgb.dtype!=np.uint8:rgb=np.uint8(np.clip(rgb,0,255))
        actual_bounds=rasterio.windows.bounds(win,src.transform)
        refs=reference36[reference36.intersects(bounds)]
        for j,(title,pp) in enumerate(variants36):
            ax=axes[i,j];ax.imshow(rgb,extent=actual_bounds,origin="upper")
            if len(refs):refs.boundary.plot(ax=ax,color="yellow",linewidth=1.4)
            shown=pp[pp.intersects(bounds)]
            if len(shown):shown.boundary.plot(ax=ax,color="cyan",linewidth=.9)
            ax.set_xlim(bounds.bounds[0],bounds.bounds[2]);ax.set_ylim(bounds.bounds[1],bounds.bounds[3])
            ax.set_title(f"{title} | {len(pp)} objects");ax.set_xticks([]);ax.set_yticks([])
fig.suptitle("ROI4 crowns: manual yellow, SAM3-derived objects cyan",fontsize=13)
fig.tight_layout();fig.savefig(D36/"gis_reconstruction_examples.png",dpi=160);plt.show()
print("BLOCK 36 PASS —",D36)
print("ROI4 is development; ROI5 untouched. New polygons must improve metrics AND visual crown coherence.")


## 37 — Musanga probability + DSM crown analysis

Uses the existing Baego CNN probability raster, local DSM relief and Block 32 SAM3 masks. Compares fixed ranking ablations against the Block 33 RF baseline on ROI 4, plus probability overlays. **All-crown IoU is not Musanga-specific species accuracy.** No retraining or GPU inference; ROI 5 untouched.


In [ ]:
# BLOCK 37 — Musanga-probability and DSM-guided crown analysis (CPU, ROI 4)
# The probability raster is model output, NOT independent Musanga species ground truth.
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,matplotlib.pyplot as plt
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import from_bounds,Window
from rasterio.features import geometry_mask
from shapely.geometry import mapping,box
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D37=OUT/"experiments_v37_musanga_dsm_crowns";D37.mkdir(parents=True,exist_ok=True)
P37=OUT/"experiments_v32_roi4_finetune_validation"/"finetuned_clean_crowns.gpkg"
S37=OUT/"experiments_v33_classifier_roi4"/"fine_tuned_scores.csv"
DSM37=GEO/"Baego_DSM.tif"
assert all(p.exists() for p in [PROB,DSM37,P37,S37]),"Check Block 0R probability path and saved Block32/33 artifacts"
assert "reference_v2" in globals() and "roi_v2" in globals()
with rasterio.open(RASTER) as rgb37:
    crs37=rgb37.crs
roi37=roi_v2.to_crs(crs37).geometry.iloc[4]
pp37=gpd.read_file(P37).to_crs(crs37)
pp37=pp37[pp37.geometry.apply(lambda g:roi37.covers(g.representative_point()))].reset_index(drop=True)
sc37=pd.read_csv(S37)
assert len(pp37)==len(sc37)==1075,"Block33 scores must match Block32 polygons"
refs37=reference_v2.to_crs(crs37)
refs37=refs37[refs37.geometry.apply(roi37.covers)].reset_index(drop=True)
assert len(refs37)==198
# Detect actual numeric encoding of the probability map; never silently rescale unknown inputs.
with rasterio.open(PROB) as src:
    raw37=src.read(1,masked=True,out_shape=(min(src.height,2048),min(src.width,2048)),
                   resampling=Resampling.nearest)
    pmax37=float(np.nanmax(raw37.astype(np.float32).filled(np.nan)))
    pdtype37=src.dtypes[0]
    print("Probability:",PROB.name,"dtype:",pdtype37,"sample max:",pmax37,
          "CRS:",src.crs,"pixel:",src.res)
if np.issubdtype(np.dtype(pdtype37),np.integer):
    scale37=10000. if pmax37<=10000 else 65535. if pmax37<=65535 else None
else:
    scale37=1. if pmax37<=1.01 else 100. if pmax37<=100.01 else None
assert scale37 is not None,"Unknown probability encoding: inspect raster before interpreting"
print("Probability normalization:",scale37,"(expected uint16 0–10000).")
if np.issubdtype(np.dtype(pdtype37),np.integer) and scale37!=10000.:
    raise ValueError("Unexpected integer probability encoding; verify against Block 0R and Block 8 before proceeding.")
# Sample within each polygon, including local DSM relief, in the common RGB grid.
records37=[]
with rasterio.open(RASTER) as rgb, rasterio.open(PROB) as psrc, rasterio.open(DSM37) as dsrc:
    with WarpedVRT(psrc,crs=rgb.crs,transform=rgb.transform,width=rgb.width,height=rgb.height,
                   resampling=Resampling.bilinear) as prob, WarpedVRT(dsrc,crs=rgb.crs,transform=rgb.transform,width=rgb.width,height=rgb.height,
                   resampling=Resampling.bilinear) as dsm:
        for i,g in enumerate(pp37.geometry):
            vals=dict(proposal_id=i,rf_score=float(sc37.rf_score.iloc[i]),area_m2=float(g.area),
                      prob_mean=np.nan,prob_median=np.nan,prob_high_fraction=np.nan,
                      prob_context_difference=np.nan,dsm_relief=np.nan,dsm_std=np.nan)
            try:
                around=g.buffer(2.)
                win=from_bounds(*around.bounds,transform=rgb.transform).round_offsets().round_lengths()
                win=win.intersection(Window(0,0,rgb.width,rgb.height))
                if win.width>=3 and win.height>=3:
                    step=max(1,int(np.ceil(max(win.width,win.height)/160)))
                    h=max(1,int(np.ceil(win.height/step)));w=max(1,int(np.ceil(win.width/step)))
                    t=rgb.window_transform(win)*rasterio.Affine.scale(win.width/w,win.height/h)
                    inside=geometry_mask([mapping(g)],out_shape=(h,w),transform=t,invert=True)
                    surrounding=geometry_mask([mapping(around)],out_shape=(h,w),transform=t,invert=True)&~inside
                    if inside.sum()>=3:
                        pa=prob.read(1,window=win,out_shape=(h,w),resampling=Resampling.bilinear,masked=True)
                        da=dsm.read(1,window=win,out_shape=(h,w),resampling=Resampling.bilinear,masked=True)
                        p=np.asarray(pa.astype(np.float32).filled(np.nan),dtype=float)/scale37
                        z=np.asarray(da.astype(np.float32).filled(np.nan),dtype=float)
                        a=p[inside];a=a[np.isfinite(a)]
                        b=p[surrounding];b=b[np.isfinite(b)]
                        c=z[inside];c=c[np.isfinite(c)]
                        d=z[surrounding];d=d[np.isfinite(d)]
                        if len(a):
                            vals["prob_mean"]=float(np.mean(a))
                            vals["prob_median"]=float(np.median(a))
                            vals["prob_high_fraction"]=float(np.mean(a>=.7))
                            if len(b):vals["prob_context_difference"]=float(np.mean(a)-np.mean(b))
                        if len(c):
                            vals["dsm_std"]=float(np.std(c))
                            if len(d):vals["dsm_relief"]=float(np.percentile(c,90)-np.median(d))
            except Exception as e:
                if i<3:print("Sampling warning",i,repr(e))
            records37.append(vals)
            if (i+1)%250==0:print("Sampled",i+1,"/",len(pp37))
features37=pd.DataFrame(records37)
assert features37.prob_mean.notna().mean()>.80,"Too many missing probability features; check projection/nodata."
assert features37.prob_mean.dropna().between(0,1.001).all(),"Probability scaling incorrect"
features37.to_csv(D37/"roi4_crown_musanga_dsm_features.csv",index=False)
# DSM absolute elevation is deliberately excluded; relative metrics only.
def percentile37(x):
    x=pd.to_numeric(x,errors="coerce").to_numpy(dtype=float)
    valid=np.isfinite(x)
    if not valid.any():return np.zeros(len(x))
    lo,hi=np.nanpercentile(x[valid],[5,95])
    return np.clip((np.where(valid,x,lo)-lo)/max(hi-lo,1e-9),0,1)
features37["rank_rf"]=percentile37(features37.rf_score)
features37["rank_prob"]=percentile37(features37.prob_mean)
features37["rank_high"]=percentile37(features37.prob_high_fraction)
features37["rank_relief"]=percentile37(features37.dsm_relief)
# Prespecified heuristic ablations (not trained classifiers): compare to RF without species cue.
scores37={
    "rf_baseline":features37.rank_rf.to_numpy(),
    "rf_plus_musanga":(.65*features37.rank_rf+.35*features37.rank_prob).to_numpy(),
    "rf_plus_probability_structure":(.55*features37.rank_rf+
         .35*features37.rank_prob+.10*features37.rank_relief).to_numpy()}
iou37=np.zeros((len(refs37),len(pp37)),dtype=np.float32)
for i,g in enumerate(refs37.geometry):
    for j in pp37.sindex.query(g,predicate="intersects"):
        h=pp37.geometry.iloc[int(j)];a=g.intersection(h).area
        if a>0:iou37[i,int(j)]=a/(g.area+h.area-a)
# Evaluate on generic all-crown reference; species-specific evaluation requires independent Musanga crown labels.
evaluations37=[]; ranks37=[]; kept37={}
for name37,score37 in scores37.items():
    order37=np.argsort(-score37,kind="stable")
    for count37 in [538,645,753]:
        chosen37=order37[:count37]
        kept37[(name37,count37)]=chosen37
        sub=iou37[:,chosen37]
        for t37 in [.3,.5,.7]:
            rr,cc=linear_sum_assignment((sub>=t37).astype(float)*2+sub,maximize=True)
            matched=int(np.count_nonzero(sub[rr,cc]>=t37))
            prec=matched/count37;rec=matched/len(refs37)
            evaluations37.append(dict(method=name37,retained=count37,iou_threshold=t37,
                     matches=matched,precision=prec,recall=rec,
                     f1=2*prec*rec/(prec+rec) if prec+rec else 0))
    ranks37.extend(dict(method=name37,proposal_id=int(j),rank=k+1,score=float(score37[j]))
                   for k,j in enumerate(order37))
results37=pd.DataFrame(evaluations37)
results37.to_csv(D37/"roi4_generic_crown_accuracy.csv",index=False)
pd.DataFrame(ranks37).to_csv(D37/"all_proposal_rankings.csv",index=False)
print("ALL-CROWN evaluation (NOT Musanga-specific ground truth), IoU >= 0.5:")
display(results37[(results37.iou_threshold==.5)&(results37.retained==538)].round(3))
# Show probability and candidate polygons on exactly the same RGB background.
fig,axes=plt.subplots(3,3,figsize=(15,14),squeeze=False)
with rasterio.open(RASTER) as rgb,rasterio.open(PROB) as ps:
    with WarpedVRT(ps,crs=rgb.crs,transform=rgb.transform,width=rgb.width,height=rgb.height,
                   resampling=Resampling.bilinear) as pv:
        for row,idx in enumerate(np.linspace(0,len(refs37)-1,3,dtype=int)):
            center=refs37.geometry.iloc[idx].centroid
            extent=box(center.x-16,center.y-16,center.x+16,center.y+16)
            win=from_bounds(*extent.bounds,transform=rgb.transform).round_offsets().round_lengths()
            data=rgb.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            if data.dtype!=np.uint8:data=np.uint8(np.clip(data,0,255))
            pr=pv.read(1,window=win,masked=True).astype(np.float32).filled(0)/scale37
            b=rasterio.windows.bounds(win,rgb.transform)
            localref=refs37[refs37.intersects(extent)]
            for col,(title,mode) in enumerate([("RGB + RF","rf_baseline"),
                          ("Musanga probability",None),("RGB + probability-guided","rf_plus_probability_structure")]):
                ax=axes[row,col];ax.imshow(data,extent=b,origin="upper")
                if mode is None:
                    ax.imshow(np.ma.masked_where(pr<.1,pr),extent=b,origin="upper",
                              vmin=0,vmax=1,cmap="magma",alpha=.55)
                else:
                    local=pp37.iloc[kept37[(mode,538)]]
                    local=local[local.intersects(extent)]
                    if len(local):local.boundary.plot(ax=ax,color="cyan",linewidth=.9)
                if len(localref):localref.boundary.plot(ax=ax,color="yellow",linewidth=1.2)
                ax.set_xlim(extent.bounds[0],extent.bounds[2]);ax.set_ylim(extent.bounds[1],extent.bounds[3])
                ax.set_title(title);ax.set_xticks([]);ax.set_yticks([])
fig.suptitle("ROI4: Musanga probability and crown selection | yellow manual all-species boundaries",fontsize=12)
fig.tight_layout();fig.savefig(D37/"musanga_probability_crown_overlays.png",dpi=160);plt.show()
print("BLOCK 37 PASS:",D37)
print("CAUTION: ROI4 is validation. Probability may share training imagery; independent species-specific accuracy is NOT established.")
print("ROI5 untouched. Next: evaluate against manually labelled Musanga crowns and spatially held-out probability predictions.")


## 39 — RGB U-Net crown interiors and boundaries (pilot)

Train a small 2-output CNN on Block 30 COCO crown instances from ROIs 0–3 (interior + boundary). Stitch ROI 4 predictions to the UAV raster, reconstruct individual polygons via watershed, and compare with existing all-crown reference. No ROI 5 access. This is a fast RGB-only baseline; DSM fusion is deferred until validation.


In [ ]:
# BLOCK 39 — Small RGB U-Net crown interiors + boundaries; ROI0–3 train, ROI4 validate
# Uses existing Block30 COCO tiles. ROI5 is not read. Independent of SAM3 GPU environment.
from pathlib import Path
import json,random,copy
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from PIL import Image
from pycocotools import mask as cmask
from scipy.ndimage import binary_erosion,distance_transform_edt,maximum_filter,label as ndi_label
from scipy.optimize import linear_sum_assignment
from rasterio.features import shapes,geometry_mask
from rasterio.windows import Window
from shapely.geometry import shape,mapping
from skimage.segmentation import watershed
import matplotlib.pyplot as plt
from IPython.display import display
assert torch.cuda.is_available(),"Select GPU runtime"
torch.manual_seed(42);np.random.seed(42);random.seed(42)
D39=OUT/"experiments_v39_unet_crown_pilot";D39.mkdir(parents=True,exist_ok=True)
DATA39=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"
assert (DATA39/"train"/"_annotations.coco.json").exists(),"Run Block30 to prepare data"
assert (DATA39/"valid"/"_annotations.coco.json").exists()
assert "roi_v2" in globals() and "reference_v2" in globals()
# COCO masks are crown instances. Train interior and narrow boundary independently.
class CrownTiles39(Dataset):
    def __init__(self,split):
        self.folder=DATA39/split
        obj=json.loads((self.folder/"_annotations.coco.json").read_text())
        self.images=obj["images"]
        self.byid={a["id"]:[] for a in self.images}
        for ann in obj["annotations"]:self.byid[ann["image_id"]].append(ann)
        if split=="train":assert all(im["roi_id"] in [0,1,2,3] for im in self.images)
        if split=="valid":assert all(im["roi_id"]==4 for im in self.images)
    def __len__(self):return len(self.images)
    def __getitem__(self,idx):
        im=self.images[idx]
        rgb=np.asarray(Image.open(self.folder/im["file_name"]).convert("RGB"),dtype=np.float32)/255.
        h,w=rgb.shape[:2]
        semantic=np.zeros((h,w),dtype=bool);edges=np.zeros((h,w),dtype=bool)
        for a in self.byid[im["id"]]:
            rle=dict(a["segmentation"]);rle["counts"]=rle["counts"].encode("ascii")
            m=cmask.decode(rle).astype(bool)
            semantic |= m
            edges |= m & ~binary_erosion(m,iterations=3)
        x=torch.from_numpy(rgb.transpose(2,0,1).copy())
        y=torch.from_numpy(np.stack([semantic,edges]).astype(np.float32))
        return x,y,idx
trainset39=CrownTiles39("train");valset39=CrownTiles39("valid")
print("COCO tiles train/validation:",len(trainset39),len(valset39))
# Tile subsampling keeps 40m inputs tractable at UAV 8cm GSD (~500px).
class Block39(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.f=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(True),
                             nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(True))
    def forward(self,x):return self.f(x)
class UNet39(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=Block39(3,24);self.e2=Block39(24,48);self.e3=Block39(48,96)
        self.pool=nn.MaxPool2d(2);self.mid=Block39(96,144)
        self.d3=Block39(144+96,96);self.d2=Block39(96+48,48);self.d1=Block39(48+24,24)
        self.out=nn.Conv2d(24,2,1)
    def forward(self,x):
        import torch.nn.functional as F
        a=self.e1(x);b=self.e2(self.pool(a));c=self.e3(self.pool(b));d=self.mid(self.pool(c))
        u=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1))
        u=self.d2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1))
        u=self.d1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        return self.out(u)
device39="cuda";model39=UNet39().to(device39)
optimizer39=torch.optim.AdamW(model39.parameters(),lr=0.0007,weight_decay=0.0001)
loss39=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.,5.],device=device39).view(1,2,1,1))
loader39=DataLoader(trainset39,batch_size=2,shuffle=True,num_workers=0)
N_EPOCH39=12
ckpt39=D39/"unet_rgb_interior_boundary.pt"
if not ckpt39.exists():
    for ep in range(N_EPOCH39):
        model39.train();total=0
        for x,y,_ in loader39:
            x=x.to(device39);y=y.to(device39)
            # Downsample to 256px for this quick controlled pilot.
            x=nn.functional.interpolate(x,size=(256,256),mode="bilinear",align_corners=False)
            y=nn.functional.interpolate(y,size=(256,256),mode="nearest")
            optimizer39.zero_grad(set_to_none=True)
            logits=model39(x)
            error=loss39(logits,y);error.backward();optimizer39.step()
            total+=error.item()
        print(f"Epoch {ep+1:02d}/{N_EPOCH39} mean loss {total/len(loader39):.4f}")
    torch.save({"model":model39.cpu().state_dict(),"epochs":N_EPOCH39},ckpt39)
    model39.to(device39)
else:
    model39.load_state_dict(torch.load(ckpt39,map_location="cpu",weights_only=True)["model"])
    print("Existing pilot checkpoint loaded:",ckpt39)
# Stitch predictions from validation tiles into ONE ROI4 probability raster.
# This avoids counting duplicate tile predictions as separate crowns.
model39.eval()
with rasterio.open(RASTER) as src:
    polygon39=roi_v2.to_crs(src.crs).geometry.iloc[4]
    bb39=polygon39.bounds
    top,left=src.index(bb39[0],bb39[3])
    bottom,right=src.index(bb39[2],bb39[1])
    row0,col0=max(0,min(top,bottom)),max(0,min(left,right))
    row1,col1=min(src.height,max(top,bottom)+1),min(src.width,max(left,right)+1)
    hh,ww=row1-row0,col1-col0
    assert hh*ww<25_000_000,"ROI4 mosaic too large for this pilot"
    accum=np.zeros((2,hh,ww),dtype=np.float32);count=np.zeros((hh,ww),dtype=np.float32)
    print("ROI4 mosaic grid:",ww,"x",hh)
    for k in range(len(valset39)):
        x,_,_=valset39[k]
        x=x.unsqueeze(0).to(device39)
        oh,ow=x.shape[-2:]
        with torch.no_grad():
            input39=nn.functional.interpolate(x,size=(256,256),mode="bilinear",align_corners=False)
            prob39=torch.sigmoid(model39(input39))
            prob39=nn.functional.interpolate(prob39,size=(oh,ow),mode="bilinear",align_corners=False)
        pred=prob39[0].cpu().numpy()
        fname=valset39.images[k]["file_name"]
        # Recover actual offsets from the original Block22 ROI4 tile manifest.
        name_id=int(fname.split("_tile")[-1].split(".")[0])
        if k==0:
            m39=pd.read_csv(OUT/"experiments_v22_full_roi_8m_4"/"full"/"prompt_manifest.csv")
            tile39=m39.drop_duplicates("tile_id").set_index("tile_id")
        tm=tile39.loc[name_id]
        r=int(tm.row_off)-row0;c=int(tm.col_off)-col0
        ay=max(0,r);ax=max(0,c);by=min(hh,r+oh);bx=min(ww,c+ow)
        if ay>=by or ax>=bx:continue
        sy=ay-r;sx=ax-c
        accum[:,ay:by,ax:bx]+=pred[:,sy:sy+by-ay,sx:sx+bx-ax]
        count[ay:by,ax:bx]+=1
    valid=count>0
    probm39=np.divide(accum,count[None],out=np.zeros_like(accum),where=valid[None])
    tr39=src.window_transform(Window(col0,row0,ww,hh))
    roi_mask39=geometry_mask([mapping(polygon39)],out_shape=(hh,ww),transform=tr39,invert=True)
mask39=(probm39[0]>=.5)&(probm39[1]<.5)&roi_mask39&valid
# Marker-controlled watershed using only predicted crown interior and boundary.
d39=distance_transform_edt(mask39)
peaks39=(d39==maximum_filter(d39,size=31))&(d39>=12)&mask39
mark39,_=ndi_label(peaks39)
labels39=watershed(-d39,mark39,mask=mask39)
geoms39=[]
for geom,val in shapes(labels39.astype(np.int32),mask=labels39>0,transform=tr39):
    poly=shape(geom)
    if poly.area>=8 and polygon39.covers(poly.representative_point()):
        geoms39.append(poly)
result39=gpd.GeoDataFrame(geometry=geoms39,crs=reference_v2.crs)
result39.to_file(D39/"roi4_unet_watershed_crowns.gpkg",driver="GPKG")
ref39=reference_v2.to_crs(result39.crs)
ref39=ref39[ref39.geometry.apply(polygon39.covers)].reset_index(drop=True)
iou39=np.zeros((len(ref39),len(result39)),np.float32)
if len(result39):
    for i,g in enumerate(ref39.geometry):
        for j in result39.sindex.query(g,predicate="intersects"):
            h=result39.geometry.iloc[int(j)];a=g.intersection(h).area
            if a>0:iou39[i,int(j)]=a/(g.area+h.area-a)
report39=[]
for t in [.3,.5,.7]:
    if iou39.size:
        ii,jj=linear_sum_assignment((iou39>=t).astype(float)*2+iou39,maximize=True)
        tp=int(np.count_nonzero(iou39[ii,jj]>=t))
    else:tp=0
    precision=tp/len(result39) if len(result39) else 0.
    recall=tp/len(ref39)
    report39.append(dict(model="rgb_unet_boundary_watershed",iou_threshold=t,
        manual=len(ref39),predicted=len(result39),matched=tp,precision=precision,
        recall=recall,f1=2*precision*recall/(precision+recall) if precision+recall else 0))
rep39=pd.DataFrame(report39);rep39.to_csv(D39/"roi4_metrics.csv",index=False)
display(rep39.round(3))
with rasterio.open(RASTER) as src:
    fig,axs=plt.subplots(1,3,figsize=(16,5))
    mid=ref39.geometry.iloc[len(ref39)//2].centroid
    extent=(mid.x-20,mid.y-20,mid.x+20,mid.y+20)
    win=Window(*[0,0,1,1]) # overwritten immediately
    from rasterio.windows import from_bounds
    win=from_bounds(*extent,transform=src.transform).round_offsets().round_lengths()
    rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
    if rgb.dtype!=np.uint8:rgb=np.uint8(np.clip(rgb,0,255))
    bounds=rasterio.windows.bounds(win,src.transform)
    sub=from_bounds(*extent,transform=tr39).round_offsets().round_lengths()
    r0=int(sub.row_off);c0=int(sub.col_off);rh=int(sub.height);cw=int(sub.width)
    probpart=np.zeros((rh,cw),dtype=np.float32)
    ya=max(0,r0);yb=min(hh,r0+rh);xa=max(0,c0);xb=min(ww,c0+cw)
    if ya<yb and xa<xb:probpart[ya-r0:yb-r0,xa-c0:xb-c0]=probm39[1,ya:yb,xa:xb]
    for ax in axs:ax.imshow(rgb,extent=bounds,origin="upper")
    axs[1].imshow(probpart,extent=bounds,origin="upper",cmap="magma",alpha=.55,vmin=0,vmax=1)
    shown=result39[result39.intersects(shape({"type":"Polygon","coordinates":[[(extent[0],extent[1]),(extent[2],extent[1]),(extent[2],extent[3]),(extent[0],extent[3]),(extent[0],extent[1])]]}))]
    if len(shown):shown.boundary.plot(ax=axs[2],color="cyan",linewidth=1)
    rr=ref39[ref39.intersects(shape({"type":"Polygon","coordinates":[[(extent[0],extent[1]),(extent[2],extent[1]),(extent[2],extent[3]),(extent[0],extent[3]),(extent[0],extent[1])]]}))]
    if len(rr):rr.boundary.plot(ax=axs[2],color="yellow",linewidth=1.2)
    for ax,title in zip(axs,["UAV RGB","Predicted crown boundaries","Predicted cyan / manual yellow"]):
        ax.set_title(title);ax.set_xlim(extent[0],extent[2]);ax.set_ylim(extent[1],extent[3]);ax.set_xticks([]);ax.set_yticks([])
    fig.tight_layout();fig.savefig(D39/"roi4_unet_examples.png",dpi=160);plt.show()
print("BLOCK 39 COMPLETE:",D39)
print("PILOT: 256px training subsampling, 12 epochs, fixed thresholds; ROI5 untouched.")


## 40 — Oracle-box SAM3 diagnostic (zero-shot vs fine-tuned)

Use the **ROI 4 reference crown polygon only to construct a tight enclosing box with 1.5 m padding**. Prompt the same zero-shot and two-epoch fine-tuned SAM3 image models. Evaluate the highest-scoring returned mask against its prompted crown, never select by manual IoU. This is a privileged-prompt upper-bound *diagnostic*, not independent detection accuracy. ROI 5 stays untouched.


In [ ]:
# BLOCK 40 — Oracle crown-box diagnostic: zero-shot vs fine-tuned SAM3 (ROI 4)
# IMPORTANT: annotation-derived boxes are privileged prompts. NOT independent crown detection.
import json, subprocess, zipfile
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box,mapping
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
D40=OUT/"experiments_v40_oracle_boxes";D40.mkdir(parents=True,exist_ok=True)
assert "reference_v2" in globals() and "roi_v2" in globals() and Path(SAMPY).exists()
CK40=OUT/"experiments_v31_sam3_two_epoch"/"training_logs"/"checkpoints"/"checkpoint.pt"
assert CK40.exists() and zipfile.is_zipfile(CK40),"Need valid fine-tuned Block31 checkpoint"
PM40=OUT/"experiments_v22_full_roi_8m_4"/"full"/"prompt_manifest.csv"
assert PM40.exists(),"Block28 ROI4 tiles needed"
pm40=pd.read_csv(PM40)
tiles40=pm40.drop_duplicates("tile_id").set_index("tile_id")
with rasterio.open(RASTER) as src:
    crs40=src.crs
    ref40=reference_v2.to_crs(crs40)
    roi40=roi_v2.to_crs(crs40).geometry.iloc[4]
    ref40=ref40[ref40.geometry.apply(roi40.covers)].reset_index(drop=True)
    assert len(ref40)==198,f"Expected 198 ROI4 crowns, got {len(ref40)}"
    tileinfo40=[]
    for r in tiles40.itertuples():
        w=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        tileinfo40.append((int(r.Index),box(*rasterio.windows.bounds(w,src.transform)),r))
    # For every reference crown, choose a SINGLE tile containing its geometry plus padding.
    # No tile-border clipping; omit crowns that cannot be fully prompted.
    prompts40=[];excluded40=0
    for crown_id40,g in enumerate(ref40.geometry):
        gx0,gy0,gx1,gy1=g.bounds
        padding40=1.5
        desired40=box(gx0-padding40,gy0-padding40,gx1+padding40,gy1+padding40)
        eligible40=[(tid,rect,r) for tid,rect,r in tileinfo40 if rect.covers(desired40)]
        if not eligible40:excluded40+=1;continue
        # Deterministic best centered tile, maximize distance to tile boundaries.
        tid,rect,r=max(eligible40,key=lambda z:z[1].boundary.distance(g.centroid))
        xmin,ymin,xmax,ymax=rect.bounds
        cx=((gx0+gx1)/2-xmin)/(xmax-xmin)
        cy=(ymax-(gy0+gy1)/2)/(ymax-ymin)
        bw=(gx1-gx0+2*padding40)/(xmax-xmin)
        bh=(gy1-gy0+2*padding40)/(ymax-ymin)
        prompts40.append(dict(crown_id=crown_id40,tile_id=tid,path=r.path,
                              col_off=int(r.col_off),row_off=int(r.row_off),
                              width=int(r.width),height=int(r.height),
                              cx=float(cx),cy=float(cy),bw=float(bw),bh=float(bh)))
manifest40=pd.DataFrame(prompts40)
assert len(manifest40)>=120,f"Only {len(manifest40)} usable reference crowns; inspect tile coverage"
manifest40.to_csv(D40/"oracle_manifest.csv",index=False)
print("ROI4 reference crowns:",len(ref40),"| oracle boxes:",len(manifest40),
      "| excluded (tile edge):",excluded40)
SCRIPT40=Path("/content/musanga40_oracle_inference.py")
SCRIPT40.write_text(r'''
import sys,torch,numpy as np,pandas as pd
from pathlib import Path
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
manifest=pd.read_csv(sys.argv[1]);outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)
ckpath=sys.argv[3];mode=sys.argv[4]
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz",device="cuda",eval_mode=True)
    if mode=="finetuned":
        ck=torch.load(ckpath,map_location="cpu",weights_only=False)
        missing,unexpected=model.load_state_dict(ck["model"],strict=False)
        assert not missing and not unexpected,(missing[:10],unexpected[:10])
        del ck
    proc=Sam3Processor(model)
    for tid,g in manifest.groupby("tile_id",sort=True):
        todo=[r for r in g.itertuples() if not (outdir/f"crown_{int(r.crown_id):04d}.npz").exists()]
        if not todo:continue
        image=Image.open(todo[0].path).convert("RGB")
        for r in todo:
            state=proc.set_image(image)
            result=proc.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),float(r.bw),float(r.bh)]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=result["masks"].detach().cpu().numpy()
            scores=result["scores"].detach().float().cpu().numpy().reshape(-1)
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(outdir/f"crown_{int(r.crown_id):04d}.npz",
                                masks=(masks>.5).astype(np.uint8),scores=scores)
        print(mode,"tile",tid,"done",flush=True)
''')
for mode40 in ["zero_shot","finetuned"]:
    target40=D40/mode40;target40.mkdir(exist_ok=True)
    missing40=sum(not (target40/f"crown_{int(i):04d}.npz").exists() for i in manifest40.crown_id)
    if missing40:
        print(mode40,":",missing40,"pending oracle boxes")
        log40=D40/f"{mode40}_inference.log"
        with log40.open("w") as out:
            proc40=subprocess.run([str(SAMPY),"-u",str(SCRIPT40),str(D40/"oracle_manifest.csv"),
                    str(target40),str(CK40),mode40],cwd="/content/sam3",
                    stdout=out,stderr=subprocess.STDOUT,check=False)
        print("\n".join(log40.read_text(errors="replace").splitlines()[-8:]))
        assert proc40.returncode==0,f"{mode40} inference failed; see {log40}"
# Evaluate per-crown oracle box: choose highest-score returned mask, not best oracle IoU.
rows40=[]
with rasterio.open(RASTER) as src:
    for r in manifest40.itertuples():
        transform=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)))
        truth=rasterize([(mapping(ref40.geometry.iloc[int(r.crown_id)]),1)],
                    out_shape=(int(r.height),int(r.width)),transform=transform).astype(bool)
        for mode40 in ["zero_shot","finetuned"]:
            with np.load(D40/mode40/f"crown_{int(r.crown_id):04d}.npz") as d:
                masks=d["masks"];scores=d["scores"].reshape(-1)
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            if len(scores) and len(masks):
                idx=int(np.argmax(scores));pred=np.asarray(masks[idx],dtype=bool)
                intersection=np.count_nonzero(pred&truth)
                union=np.count_nonzero(pred|truth)
                iou=intersection/union if union else 0.
                area_ratio=np.count_nonzero(pred)/max(np.count_nonzero(truth),1)
                score=float(scores[idx])
            else:iou=0.;area_ratio=0.;score=np.nan
            rows40.append(dict(crown_id=int(r.crown_id),model=mode40,
                    oracle_iou=iou,prediction_area_ratio=area_ratio,
                    score=score,manual_area_m2=ref40.geometry.iloc[int(r.crown_id)].area))
scores_table40=pd.DataFrame(rows40)
scores_table40.to_csv(D40/"oracle_per_crown_iou.csv",index=False)
overview40=[]
for mode40,g in scores_table40.groupby("model"):
    overview40.append(dict(model=mode40,crowns=len(g),
         median_iou=g.oracle_iou.median(),mean_iou=g.oracle_iou.mean(),
         fraction_ge_03=(g.oracle_iou>=.3).mean(),
         fraction_ge_05=(g.oracle_iou>=.5).mean(),
         fraction_ge_07=(g.oracle_iou>=.7).mean(),
         median_area_ratio=g.prediction_area_ratio.median()))
table40=pd.DataFrame(overview40)
table40.to_csv(D40/"oracle_summary.csv",index=False)
display(table40.round(3))
# Side-by-side RGB, prompted crown in yellow, zero-shot/fine-tuned mask in cyan.
pick40=np.linspace(0,len(manifest40)-1,min(4,len(manifest40)),dtype=int)
fig,axes=plt.subplots(len(pick40),2,figsize=(11,5*len(pick40)),squeeze=False)
for row,idx in enumerate(pick40):
    r=manifest40.iloc[int(idx)]
    image=np.asarray(Image.open(r.path).convert("RGB"))
    truth_poly=ref40.geometry.iloc[int(r.crown_id)]
    with rasterio.open(RASTER) as src:
        transform=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)))
        b=rasterio.windows.bounds(Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)),src.transform)
    for col,mode40 in enumerate(["zero_shot","finetuned"]):
        with np.load(D40/mode40/f"crown_{int(r.crown_id):04d}.npz") as d:
            masks=d["masks"];scores=d["scores"].reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        ax=axes[row,col];ax.imshow(image,extent=b,origin="upper")
        if len(scores) and len(masks):
            mask=masks[int(np.argmax(scores))]
            ax.contour(np.linspace(b[0],b[2],mask.shape[1]),
                np.linspace(b[3],b[1],mask.shape[0]),mask,levels=[.5],
                colors=["cyan"],linewidths=1.2)
        gpd.GeoSeries([truth_poly],crs=crs40).boundary.plot(ax=ax,color="yellow",linewidth=1.4)
        cx=truth_poly.centroid.x;cy=truth_poly.centroid.y
        ax.set_xlim(cx-14,cx+14);ax.set_ylim(cy-14,cy+14)
        val=scores_table40[(scores_table40.crown_id==int(r.crown_id))&
                            (scores_table40.model==mode40)].oracle_iou.iloc[0]
        ax.set_title(f"{mode40} | crown {int(r.crown_id)} | IoU={val:.2f}")
        ax.set_xticks([]);ax.set_yticks([])
fig.suptitle("ORACLE BOXES (manual-derived prompts; NOT autonomous detection) | yellow reference, cyan SAM3",fontsize=12)
fig.tight_layout();fig.savefig(D40/"oracle_examples.png",dpi=150);plt.show()
print("BLOCK 40 PASS:",D40)
print("Oracle results are CONDITIONAL segmentation quality only. ROI5 untouched.")


## 41 — Oracle-box padding sensitivity and mask selection

Zero-shot SAM3 on the same ROI 4 manual crowns, varying box padding (0, 0.5, 1.0, 1.5 m). Use a paired subset for a controlled comparison; inspect top-confidence mask versus best-IoU candidate as a privileged diagnostic only. No model training; ROI 5 remains untouched.


In [ ]:
# BLOCK 41 — Oracle box padding and candidate-mask diagnostic (ROI 4 only)
# PRIVILEGED prompts derived from manual polygons: diagnostic, NOT autonomous detection.
import subprocess
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box,mapping
from scipy.stats import spearmanr
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
D41=OUT/"experiments_v41_oracle_padding";D41.mkdir(parents=True,exist_ok=True)
D40=OUT/"experiments_v40_oracle_boxes"
manifest40=pd.read_csv(D40/"oracle_manifest.csv")
assert len(manifest40)>=120 and "reference_v2" in globals() and "roi_v2" in globals()
with rasterio.open(RASTER) as src:
    crs41=src.crs
ref41=reference_v2.to_crs(crs41)
roi41=roi_v2.to_crs(crs41).geometry.iloc[4]
ref41=ref41[ref41.geometry.apply(roi41.covers)].reset_index(drop=True)
# Same 169 reference crowns as Block40, same image tile, 4 prespecified padding levels.
padding41=[0.,.5,1.,1.5]
rows41=[]
with rasterio.open(RASTER) as src:
    for r in manifest40.itertuples():
        g=ref41.geometry.iloc[int(r.crown_id)]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        xmin,ymin,xmax,ymax=rasterio.windows.bounds(win,src.transform)
        gx0,gy0,gx1,gy1=g.bounds
        for pad in padding41:
            rect=box(gx0-pad,gy0-pad,gx1+pad,gy1+pad)
            if not box(xmin,ymin,xmax,ymax).covers(rect):continue
            rows41.append(dict(prompt_id=len(rows41),crown_id=int(r.crown_id),
                pad_m=pad,tile_id=int(r.tile_id),path=r.path,
                col_off=int(r.col_off),row_off=int(r.row_off),width=int(r.width),height=int(r.height),
                cx=((gx0+gx1)/2-xmin)/(xmax-xmin),cy=(ymax-(gy0+gy1)/2)/(ymax-ymin),
                bw=(gx1-gx0+2*pad)/(xmax-xmin),bh=(gy1-gy0+2*pad)/(ymax-ymin)))
pm41=pd.DataFrame(rows41);pm41.to_csv(D41/"oracle_padding_manifest.csv",index=False)
print("Oracle prompts by padding:",pm41.groupby("pad_m").size().to_dict())
SCRIPT41=Path("/content/musanga41_zero_shot_padding.py")
SCRIPT41.write_text(r'''
import sys,torch,numpy as np,pandas as pd
from pathlib import Path
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]);dest=Path(sys.argv[2]);dest.mkdir(parents=True,exist_ok=True)
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz",
                                 device="cuda",eval_mode=True)
    proc=Sam3Processor(model)
    for tid,group in df.groupby("tile_id",sort=True):
        todo=[r for r in group.itertuples() if not (dest/f"prompt_{int(r.prompt_id):04d}.npz").exists()]
        if not todo:continue
        img=Image.open(todo[0].path).convert("RGB")
        for r in todo:
            state=proc.set_image(img)
            out=proc.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),float(r.bw),float(r.bh)]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy().reshape(-1)
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(dest/f"prompt_{int(r.prompt_id):04d}.npz",
                                masks=(masks>.5).astype(np.uint8),scores=scores)
        print("tile",tid,":",len(todo),"prompts done",flush=True)
''')
RAW41=D41/"raw";RAW41.mkdir(exist_ok=True)
todo41=sum(not (RAW41/f"prompt_{int(i):04d}.npz").exists() for i in pm41.prompt_id)
if todo41:
    print("Zero-shot oracle padding prompts pending:",todo41)
    with (D41/"inference.log").open("w") as fd:
        done41=subprocess.run([str(SAMPY),"-u",str(SCRIPT41),str(D41/"oracle_padding_manifest.csv"),
                              str(RAW41)],cwd="/content/sam3",stdout=fd,stderr=subprocess.STDOUT,check=False)
    print("\n".join((D41/"inference.log").read_text(errors="replace").splitlines()[-12:]))
    assert done41.returncode==0,"Block41 inference failed; inspect inference.log"
# Paired crown-level IoU: native top score versus oracle-best candidate, if SAM3 yields multiple masks.
evalrows41=[]
with rasterio.open(RASTER) as src:
    for r in pm41.itertuples():
        tr=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)))
        truth=rasterize([(mapping(ref41.geometry.iloc[int(r.crown_id)]),1)],
                         out_shape=(int(r.height),int(r.width)),transform=tr).astype(bool)
        with np.load(RAW41/f"prompt_{int(r.prompt_id):04d}.npz") as z:
            masks=z["masks"];scores=z["scores"].reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        ious=[]
        for m in masks:
            m=np.asarray(m,dtype=bool)
            union=np.count_nonzero(m|truth)
            ious.append(np.count_nonzero(m&truth)/union if union else 0.)
        if len(ious):
            best_score_index=int(np.argmax(scores))
            native=float(ious[best_score_index])
            best=float(np.max(ious))
            area_ratio=np.count_nonzero(masks[best_score_index])/max(np.count_nonzero(truth),1)
        else:native=best=area_ratio=0.
        evalrows41.append(dict(crown_id=int(r.crown_id),pad_m=float(r.pad_m),
                               candidates=len(ious),top_score_iou=native,oracle_best_iou=best,
                               top_score_area_ratio=area_ratio,
                               crown_area_m2=ref41.geometry.iloc[int(r.crown_id)].area))
details41=pd.DataFrame(evalrows41);details41.to_csv(D41/"per_crown_padding_iou.csv",index=False)
# Restrict comparisons to crowns present at ALL padding levels (paired test).
complete41=details41.groupby("crown_id").pad_m.nunique()
paired41=details41[details41.crown_id.isin(complete41[complete41==4].index)]
summary41=paired41.groupby("pad_m").agg(
    crowns=("crown_id","count"),
    median_iou=("top_score_iou","median"),mean_iou=("top_score_iou","mean"),
    fraction_ge05=("top_score_iou",lambda x:(x>=.5).mean()),
    median_area_ratio=("top_score_area_ratio","median"),
    median_oracle_best_iou=("oracle_best_iou","median"),
    avg_candidate_count=("candidates","mean")).reset_index()
summary41.to_csv(D41/"paired_padding_summary.csv",index=False)
print("PAIRED crowns present at all four padding levels:")
display(summary41.round(3))
print("Note: oracle_best_iou uses reference-selected masks and is ONLY a diagnostic upper bound.")
if len(paired41):
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    paired41.boxplot(column="top_score_iou",by="pad_m",ax=axes[0])
    axes[0].set_title("SAM3 top-score mask IoU");axes[0].set_xlabel("Oracle box padding (m)")
    summary41.plot(x="pad_m",y=["median_iou","median_oracle_best_iou"],marker="o",ax=axes[1])
    axes[1].set_ylabel("Median crown IoU");axes[1].set_title("Native selection vs oracle-best mask")
    plt.suptitle("");plt.tight_layout()
    fig.savefig(D41/"oracle_padding_sensitivity.png",dpi=160);plt.show()
print("BLOCK 41 PASS:",D41)
print("ROI 5 untouched. Do NOT interpret oracle-derived prompts as independent detection.")


## 42 — CNN-guided SAM3 prompts: annotation source audit

First identify the **separate ~4,000 partial Musanga crown polygons** (not the 198 complete-crown ROI4 evaluation masks). This preflight scans plausible annotation sources, reports counts and metadata, excludes ROI4/5 with a buffer, and prepares a verified partial-label training layer. The next model will predict crown centres and box sizes without treating the unannotated canopy as negative examples.


In [ ]:
# BLOCK 42 — Identify the ~4,000 partial Musanga crown annotations before CNN-box training
# This is an annotation audit / preflight, NOT a detector trained on the wrong labels.
# Run after 0R and 14. It does not access ROI 5 reference data or train a model.
from pathlib import Path
import pandas as pd,geopandas as gpd,rasterio
from shapely.geometry import box
from IPython.display import display
D42=OUT/"experiments_v42_cnn_box_prompts"
D42.mkdir(parents=True,exist_ok=True)
assert "ROOT" in globals() and "RASTER" in globals() and "roi_v2" in globals()
# Set this to the correct file if multiple candidates are listed.
MUSANGA_ANNOTATIONS42=None
extensions42={".shp",".gpkg",".geojson",".fgb"}
# Search annotation directories, not gigabytes of raster/model outputs.
folders42=[ROOT/"data"/"Geo_Ref"/"Annotations",ROOT/"data"/"Annotations",ROOT/"annotations",
           ROOT/"KVO_work"/"KVO_04_final_multisite"]
candidates42=[]
for folder in folders42:
    if not folder.exists():continue
    for p in folder.rglob("*"):
        if p.suffix.lower() not in extensions42:continue
        if any(x in p.name.lower() for x in ["crow","musang","annot","label","polygon"]):
            candidates42.append(p)
candidates42=sorted(set(candidates42))
print("Potential original crown-annotation files:",len(candidates42))
report42=[]
with rasterio.open(RASTER) as rgb:
    roi42=roi_v2.to_crs(rgb.crs)
    reserved42=roi42.geometry.iloc[4].union(roi42.geometry.iloc[5])
    for p in candidates42:
        record=dict(file=str(p),count=None,columns="",crs="",overlaps_reserved=None,error="")
        try:
            if p.stem.lower() in {"crown_kristof","roi_crowns_kvo"}:
                record["error"]="Existing complete-crown ROI/reference; not the partial Musanga training dataset"
            else:
                layer=gpd.read_file(p)
                record["count"]=len(layer);record["columns"]=", ".join(map(str,layer.columns[:12]))
                record["crs"]=str(layer.crs)
                if layer.crs is not None and len(layer):
                    layer=layer.to_crs(rgb.crs)
                    record["overlaps_reserved"]=int(layer.geometry.intersects(reserved42).sum())
        except Exception as e:record["error"]=str(e)[:120]
        report42.append(record)
audit42=pd.DataFrame(report42)
audit42.to_csv(D42/"annotation_file_audit.csv",index=False)
display(audit42)
if MUSANGA_ANNOTATIONS42 is None:
    print("ACTION: Identify the actual ~4,000 Musanga polygon source(s) in the table.")
    print("Set MUSANGA_ANNOTATIONS42 to the source path (or list of paths), then we can train a partially-labelled crown-centre detector.")
    print("No model training has been performed; ROI4 and ROI5 remain reserved.")
else:
    selected42=[Path(MUSANGA_ANNOTATIONS42)] if isinstance(MUSANGA_ANNOTATIONS42,(str,Path)) else [Path(x) for x in MUSANGA_ANNOTATIONS42]
    assert all(p.exists() for p in selected42)
    training42=gpd.GeoDataFrame(pd.concat([gpd.read_file(p) for p in selected42],ignore_index=True),
                                geometry="geometry",crs=gpd.read_file(selected42[0]).crs)
    assert training42.crs is not None
    with rasterio.open(RASTER) as rgb:
        training42=training42.to_crs(rgb.crs)
        training42=training42[training42.geometry.notna()&~training42.geometry.is_empty]
        training42=training42[~training42.geometry.intersects(roi42.geometry.iloc[4].buffer(20).union(roi42.geometry.iloc[5].buffer(20)))]
    print("Usable partial crowns after ROI4/5 buffer exclusion:",len(training42))
    assert len(training42)>100,"Too few correctly sourced training crowns"
    training42.to_file(D42/"training_crowns_excluding_roi4_roi5.gpkg",driver="GPKG")
    print("PARTIAL CROWN TRAINING DATA READY. Next: masked centre/size CNN → automatic boxes → zero-shot SAM3.")


## 43 — Nadine-only CNN crown-box detector (pilot)

Train a lightweight local-supervision centre/box regression CNN using the **314 precise `Crown_Nadine` polygons**, excluding ROI 4/5 with 20 m buffer. Since annotations are scattered, train around known positive crown centres and **do not designate other crowns as negative examples**. Generate automatic ROI4 boxes and evaluate box IoU against complete `Crown_Kristof` ROI4 crowns, saving an overlay. This first pilot deliberately stops before launching SAM3; inspect box quality before spending GPU time on segmentation. ROI5 untouched.


In [ ]:
# BLOCK 43 — Nadine-only crown-centre/box CNN pilot (partial-positive supervision)
# Train ONLY on precise Nadine polygons outside ROI4+ROI5 buffered by 20m.
# Background is NOT treated as a negative crown label. ROI4 is evaluation; ROI5 untouched.
from pathlib import Path
import random, json, numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
import torch.nn as nn
import torch.nn.functional as F
from rasterio.windows import Window
from shapely.geometry import Point,box
from torch.utils.data import Dataset,DataLoader
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from IPython.display import display
torch.manual_seed(43);np.random.seed(43);random.seed(43)
assert torch.cuda.is_available(),"Use Colab GPU"
assert "reference_v2" in globals() and "roi_v2" in globals()
D43=OUT/"experiments_v43_nadine_box_detector";D43.mkdir(parents=True,exist_ok=True)
nadpath43=GEO/"Annotations"/"Crown_Nadine.shp"
assert nadpath43.exists(),nadpath43
with rasterio.open(RASTER) as src:
    crs43=src.crs;gsd43=abs(src.transform.a)
    rois43=roi_v2.to_crs(crs43)
    nad43=gpd.read_file(nadpath43).to_crs(crs43)
    assert len(nad43)==314,f"Nadine layer changed: {len(nad43)}"
    excluded43=rois43.geometry.iloc[4].buffer(20).union(rois43.geometry.iloc[5].buffer(20))
    nad43=nad43[nad43.geometry.notna()&~nad43.geometry.is_empty]
    nad43=nad43[~nad43.geometry.intersects(excluded43)].reset_index(drop=True)
    nad43=nad43[nad43.geometry.area.between(4,700)].copy().reset_index(drop=True)
    print("Nadine crowns available for training:",len(nad43))
    assert len(nad43)>=80,"Insufficient independent Nadine training crowns"
    ref43=reference_v2.to_crs(crs43)
    ref43=ref43[ref43.geometry.apply(rois43.geometry.iloc[4].covers)].reset_index(drop=True)
    assert len(ref43)==198
# Patch centred on each annotated crown. Supervise centre and approximate box size only near annotations.
# No random negative training tiles, because other crowns are unannotated.
PATCH43=256;TRAIN_SIZE43=128;SIGMA43=5.;HALF43=PATCH43//2
class Nadine43(Dataset):
    def __init__(self):self.geoms=list(nad43.geometry)
    def __len__(self):return len(self.geoms)*3
    def __getitem__(self,index):
        geom=self.geoms[index%len(self.geoms)]
        rng=np.random.default_rng(index+int(torch.initial_seed()%1000000))
        centre=geom.centroid
        # deterministic-scale random translation, augment positioning without fabricating negative labels
        shift=rng.uniform(-4,4,2)/gsd43
        with rasterio.open(RASTER) as ds:
            rr,cc=ds.index(centre.x,centre.y)
            row=int(rr-HALF43+shift[1]);col=int(cc-HALF43+shift[0])
            win=Window(col,row,PATCH43,PATCH43)
            rgb=ds.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)/255.
            transform=ds.window_transform(win)
        x0,y0,x1,y1=geom.bounds
        crow,ccol=~transform*(centre.x,centre.y)
        # affine inverse returns col,row
        cx,cy=crow,ccol
        yy,xx=np.mgrid[:PATCH43,:PATCH43]
        sigma=SIGMA43
        heat=np.exp(-((xx-cx)**2+(yy-cy)**2)/(2*sigma*sigma)).astype(np.float32)
        supervision=(np.hypot(xx-cx,yy-cy)<=15).astype(np.float32)
        bw=(x1-x0)/gsd43;bh=(y1-y0)/gsd43
        targets=np.stack([heat,np.full_like(heat,bw/PATCH43),
                          np.full_like(heat,bh/PATCH43),supervision])
        rgb=F.interpolate(torch.from_numpy(rgb)[None],size=(TRAIN_SIZE43,TRAIN_SIZE43),
                          mode="bilinear",align_corners=False)[0]
        y=F.interpolate(torch.from_numpy(targets)[None],size=(TRAIN_SIZE43,TRAIN_SIZE43),
                        mode="bilinear",align_corners=False)[0]
        return rgb,y
class Conv43(nn.Module):
    def __init__(self,a,b):
        super().__init__();self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.ReLU(),
             nn.Conv2d(b,b,3,padding=1),nn.ReLU())
    def forward(self,x):return self.net(x)
class Detector43(nn.Module):
    def __init__(self):
        super().__init__();self.e1=Conv43(3,24);self.e2=Conv43(24,48)
        self.e3=Conv43(48,96);self.mid=Conv43(96,96)
        self.d2=Conv43(144,48);self.d1=Conv43(72,24);self.out=nn.Conv2d(24,3,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(F.avg_pool2d(a,2));c=self.e3(F.avg_pool2d(b,2))
        z=self.mid(c)
        u=self.d2(torch.cat([F.interpolate(z,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1))
        u=self.d1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        y=self.out(u)
        return torch.cat([torch.sigmoid(y[:,0:1]),F.softplus(y[:,1:])],dim=1)
model43=Detector43().cuda();checkpoint43=D43/"nadine_centre_box_cnn.pt"
if checkpoint43.exists():
    model43.load_state_dict(torch.load(checkpoint43,map_location="cpu",weights_only=True))
    print("Loaded saved Nadine model")
else:
    loader43=DataLoader(Nadine43(),batch_size=8,shuffle=True,num_workers=0)
    opt43=torch.optim.AdamW(model43.parameters(),lr=.001)
    for epoch in range(15):
        model43.train();losses=[]
        for x,y in loader43:
            x=x.cuda();y=y.cuda()
            p=model43(x);heat=y[:,0:1];sup=y[:,3:4]
            # Focal-like positive-local loss; unlabeled crown pixels outside local target are ignored.
            loss_h=(((p[:,0:1]-heat)**2)*sup).sum()/sup.sum().clamp_min(1)
            loss_box=(((p[:,1:]-y[:,1:3])**2)*heat).sum()/heat.sum().clamp_min(1)
            loss=loss_h+2*loss_box
            opt43.zero_grad(set_to_none=True);loss.backward();opt43.step()
            losses.append(float(loss.item()))
        print(f"Nadine epoch {epoch+1:02d}/15: {np.mean(losses):.4f}")
    torch.save(model43.cpu().state_dict(),checkpoint43);model43.cuda()
# Sliding-window ROI4 inference; nonoverlapping central region prevents duplicate patch-centre predictions.
model43.eval()
with rasterio.open(RASTER) as ds:
    roi=rois43.geometry.iloc[4];xmin,ymin,xmax,ymax=roi.bounds
    top,left=ds.index(xmin,ymax);bottom,right=ds.index(xmax,ymin)
    row0=max(0,min(top,bottom));col0=max(0,min(left,right))
    row1=min(ds.height,max(top,bottom)+1);col1=min(ds.width,max(left,right)+1)
    hh=row1-row0;ww=col1-col0
    heat43=np.zeros((hh,ww),dtype=np.float32)
    width43=np.zeros((hh,ww),dtype=np.float32)
    height43=np.zeros((hh,ww),dtype=np.float32)
    stride43=128
    for r0 in range(row0,row1,stride43):
        for c0 in range(col0,col1,stride43):
            # each 256px patch contributes only central 128px to global mosaic
            win=Window(c0-64,r0-64,PATCH43,PATCH43)
            img=ds.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)/255.
            x=F.interpolate(torch.from_numpy(img)[None].cuda(),size=(TRAIN_SIZE43,TRAIN_SIZE43),
                            mode="bilinear",align_corners=False)
            with torch.no_grad():
                y=model43(x)
                z=F.interpolate(y,size=(PATCH43,PATCH43),mode="bilinear",align_corners=False)[0].cpu().numpy()
            yoff=r0-row0;xoff=c0-col0
            sy=min(stride43,hh-yoff);sx=min(stride43,ww-xoff)
            heat43[yoff:yoff+sy,xoff:xoff+sx]=z[0,64:64+sy,64:64+sx]
            width43[yoff:yoff+sy,xoff:xoff+sx]=z[1,64:64+sy,64:64+sx]
            height43[yoff:yoff+sy,xoff:xoff+sx]=z[2,64:64+sy,64:64+sx]
    transform43=ds.window_transform(Window(col0,row0,ww,hh))
# Conservative local maxima, with no annotation-derived seed locations.
peaks43=(heat43==maximum_filter(heat43,size=40))&(heat43>=.35)
iy43,ix43=np.where(peaks43)
rows=[]
for y,x in zip(iy43,ix43):
    worldx,worldy=transform43*(float(x)+.5,float(y)+.5)
    if not rois43.geometry.iloc[4].covers(Point(worldx,worldy)):continue
    w=np.clip(float(width43[y,x])*PATCH43*gsd43,2.,24.)
    h=np.clip(float(height43[y,x])*PATCH43*gsd43,2.,24.)
    rows.append(dict(cx=worldx,cy=worldy,width_m=w,height_m=h,score=float(heat43[y,x])))
boxes43=pd.DataFrame(rows).sort_values("score",ascending=False).reset_index(drop=True) if rows else pd.DataFrame(columns=["cx","cy","width_m","height_m","score"])
boxes43.to_csv(D43/"roi4_cnn_predicted_boxes.csv",index=False)
print("Automatic boxes on ROI4:",len(boxes43))
# Diagnostic box coverage (without supplying annotations to the detector).
# Note: Nadine is partially annotated, so do not interpret missed references as species-specific false negatives.
truth_boxes=[box(*g.bounds) for g in ref43.geometry]
if len(boxes43):
    pboxes=[box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
            for r in boxes43.itertuples()]
    mat=np.zeros((len(ref43),len(pboxes)),np.float32)
    for i,a in enumerate(truth_boxes):
        for j,b in enumerate(pboxes):
            if a.intersects(b):
                area=a.intersection(b).area
                mat[i,j]=area/(a.area+b.area-area)
    rr,cc=linear_sum_assignment((mat>=.5).astype(float)*2+mat,maximize=True)
    tp=int(np.count_nonzero(mat[rr,cc]>=.5))
else:tp=0
result43=dict(train_crowns=len(nad43),roi4_reference_crowns=len(ref43),predicted_boxes=len(boxes43),
              boxes_matched_iou05=tp,box_precision=tp/max(len(boxes43),1),
              box_recall=tp/len(ref43))
(D43/"box_diagnostic.json").write_text(json.dumps(result43,indent=2))
print("Box IoU diagnostic (all tree species, not Musanga-specific):",result43)
# Quick RGB diagnostic; automatic boxes cyan, complete crowns yellow.
with rasterio.open(RASTER) as ds:
    centre=ref43.geometry.iloc[len(ref43)//2].centroid
    xx0,yy0,xx1,yy1=centre.x-24,centre.y-24,centre.x+24,centre.y+24
    win=Window(*[0,0,1,1])
    from rasterio.windows import from_bounds
    win=from_bounds(xx0,yy0,xx1,yy1,transform=ds.transform).round_offsets().round_lengths()
    rgb=ds.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
    b=rasterio.windows.bounds(win,ds.transform)
    fig,ax=plt.subplots(figsize=(8,8));ax.imshow(rgb,extent=b,origin="upper")
    for geom in ref43.geometry:
        if geom.intersects(box(xx0,yy0,xx1,yy1)):
            gx,gy=geom.exterior.xy;ax.plot(gx,gy,color="yellow",linewidth=1.2)
    for r in boxes43.itertuples():
        g=box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
        if g.intersects(box(xx0,yy0,xx1,yy1)):
            x,y=g.exterior.xy;ax.plot(x,y,color="cyan",linewidth=1)
    ax.set_xlim(xx0,xx1);ax.set_ylim(yy0,yy1)
    ax.set_title("Nadine-only CNN boxes cyan; complete ROI4 crowns yellow")
    fig.tight_layout();fig.savefig(D43/"roi4_cnn_boxes.png",dpi=160);plt.show()
print("BLOCK 43 PASS:",D43)
print("Pilot stops at detector box evaluation; do not run SAM3 on boxes until box accuracy is assessed.")


## 44 — Diagnose zero detections in Nadine-only CNN

Do **not** interpret training loss reduction as detection performance. This lightweight diagnostic reads the existing trained model and ROI4 heatmap, compares training-patch versus ROI4 peak scores, and sweeps thresholds without accepting low-confidence detections as a fix. No retraining or ROI5 access.


In [ ]:
# BLOCK 44 — Diagnose why Nadine centre CNN returned 0 boxes (NO retraining)
# Run immediately after Block 43 in the SAME runtime.
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter
from shapely.geometry import Point
assert all(x in globals() for x in ["heat43","width43","height43","rois43","transform43","ref43","model43","nad43"]), "Run Block 43 first"
values44=heat43[np.isfinite(heat43)]
print("ROI4 heatmap: min/max/mean:",float(values44.min()),float(values44.max()),float(values44.mean()))
print("ROI4 heatmap quantiles:",{q:round(float(np.quantile(values44,q)),5) for q in [.5,.9,.99,.999,1.]})
print("ROI4 predicted width (normalized) max/median:",float(np.nanmax(width43)),float(np.nanmedian(width43)))
# Score of the heatmap at 215 labelled crowns (training centres)
model43.eval()
import torch,torch.nn.functional as F,rasterio
train_scores44=[]
for geom in nad43.geometry[:min(60,len(nad43))]:
    with rasterio.open(RASTER) as ds:
        rr,cc=ds.index(geom.centroid.x,geom.centroid.y)
        rgb=ds.read([1,2,3],window=Window(cc-128,rr-128,256,256),boundless=True,fill_value=0).astype(np.float32)/255.
    t=F.interpolate(torch.from_numpy(rgb)[None].cuda(),size=(128,128),mode="bilinear",align_corners=False)
    with torch.no_grad():output=model43(t)[0,0].cpu().numpy()
    train_scores44.append((float(output.max()),float(output[64,64])))
print("Training-patch peak and centre confidence (median):",np.median(train_scores44,axis=0))
maximum44=maximum_filter(heat43,size=40)
scores44=[]
for threshold in [.02,.05,.1,.15,.2,.25,.3,.35]:
    mask=(heat43==maximum44)&(heat43>=threshold)
    ys,xs=np.where(mask)
    cnt=0
    for y,x in zip(ys,xs):
        wx,wy=transform43*(float(x)+.5,float(y)+.5)
        if rois43.geometry.iloc[4].covers(Point(wx,wy)):cnt+=1
    scores44.append({"threshold":threshold,"roi4_peaks":cnt})
display(pd.DataFrame(scores44))
fig,axes=plt.subplots(1,2,figsize=(12,5))
vmax=float(np.quantile(values44,.999)) or 1.
axes[0].imshow(heat43,origin="upper",vmin=0,vmax=vmax,cmap="magma")
axes[0].set_title("ROI4 CNN centre heatmap")
axes[1].hist(values44[::max(1,len(values44)//80000)],bins=60)
axes[1].set_title("Heatmap pixel-confidence distribution")
fig.tight_layout();fig.savefig(D43/"roi4_heatmap_diagnostic.png",dpi=150);plt.show()
print("If heat scores are near-constant, lowering the threshold is NOT a valid fix: detector has not localized centres.")


## 45 — Pretrained crown-box detector: Kristof + Nadine

Replace the unsuccessful positive-only centre heatmap with a **pretrained YOLOv8-nano bounding-box detector**. Use spatially complete `Crown_Kristof` ROIs 0–2 for training and ROI 3 for internal validation; improve *matched* bounding boxes with `Crown_Nadine` geometry where possible. Nadine's scattered annotations are not treated as exhaustive negative-labelled tiles. Evaluate box predictions on held-out ROI 4. ROI 5 remains untouched. Requires the image/COCO dataset from Block 30, but does not need SAM3 training or Block 43.


In [ ]:
# BLOCK 45 — Pretrained YOLO crown-box detector; complete Kristof ROIs + Nadine geometry
# Prerequisites: Blocks 0R,14,30 completed, Block30 COCO images persisted.
# Train ROI0,1,2; internal validate ROI3; independent diagnostics ROI4; never use ROI5.
import os,sys,subprocess,json,shutil,re,yaml
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from shapely.geometry import box,Point
from shapely.validation import make_valid
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D45=OUT/"experiments_v45_yolo_nadine_kristof"
D45.mkdir(parents=True,exist_ok=True)
DATA45=D45/"yolo_data"
SOURCE45=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"
assert (SOURCE45/"train"/"_annotations.coco.json").exists(), "Run Block 30 to prepare tiles first."
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14 first."
try:
    from ultralytics import YOLO
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install","ultralytics>=8.3,<9"],check=True)
    from ultralytics import YOLO
with rasterio.open(RASTER) as ds:
    roi45=roi_v2.to_crs(ds.crs); ref45=reference_v2.to_crs(ds.crs)
    nad45=gpd.read_file(GEO/"Annotations"/"Crown_Nadine.shp").to_crs(ds.crs)
    invalid45=int((~nad45.geometry.is_valid).sum())
    nad45["geometry"]=nad45.geometry.map(lambda g: make_valid(g) if g is not None and not g.is_valid else g)
    print("Repaired invalid Nadine geometries:",invalid45)
    restricted45=roi45.geometry.iloc[4].buffer(20).union(roi45.geometry.iloc[5].buffer(20))
    nad45=nad45[nad45.geometry.notna() & ~nad45.geometry.is_empty]
    nad45=nad45[~nad45.geometry.intersects(restricted45)].copy()
    nad45=nad45[nad45.geometry.area.between(4,700)].copy()
    # Only use Nadine to replace a matching full-crown Kristof box. Unmatched scattered crowns
    # are NOT added as fully labelled object-detection tiles.
    nad_geoms45=list(nad45.geometry)
    manifest45={}
    for roi_id in range(5):
        csv=OUT/f"experiments_v22_full_roi_8m_{roi_id}"/"full"/"prompt_manifest.csv"
        if not csv.exists():raise FileNotFoundError(f"Missing tile manifest {csv}; run Block 22")
        tiles=pd.read_csv(csv).drop_duplicates("tile_id")
        for t in tiles.itertuples():
            manifest45[(roi_id,int(t.tile_id))]=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
    train_coco45=json.loads((SOURCE45/"train"/"_annotations.coco.json").read_text())
    valid_coco45=json.loads((SOURCE45/"valid"/"_annotations.coco.json").read_text())
    anns45={}
    for split,obj in [("train",train_coco45),("valid",valid_coco45)]:
        for a in obj["annotations"]:
            anns45.setdefault((split,a["image_id"]),[]).append(a)
    summary45=[]; replacements45=0
    for source_split,obj in [("train",train_coco45),("valid",valid_coco45)]:
        for im in obj["images"]:
            rid=int(im["roi_id"])
            if rid==5:continue
            split="train" if rid in (0,1,2) else "val" if rid==3 else "roi4"
            fname=im["file_name"]
            img_path=SOURCE45/source_split/fname
            assert img_path.exists(),img_path
            match=re.search(r"tile(\d+)",fname)
            assert match,fname
            win=manifest45[(rid,int(match.group(1)))]
            tx=ds.window_transform(win)
            out_im=DATA45/"images"/split/fname
            out_lbl=DATA45/"labels"/split/Path(fname).with_suffix(".txt").name
            out_im.parent.mkdir(parents=True,exist_ok=True);out_lbl.parent.mkdir(parents=True,exist_ok=True)
            if not out_im.exists():shutil.copyfile(img_path,out_im)
            lines=[];num_replace=0
            # Prevent marking truncated crowns as complete examples near ROI/tile borders.
            for a in anns45.get((source_split,im["id"]),[]):
                x,y,w,h=map(float,a["bbox"]);H=int(im["height"]);W=int(im["width"])
                if min(x,y,W-x-w,H-y-h)<3:continue
                if min(w,h)<8:continue
                cx=(x+w/2);cy=(y+h/2)
                wx,wy=tx*(cx,cy)
                if not roi45.geometry.iloc[rid].contains(Point(wx,wy)):continue
                # Find a Nadine annotation for the same crown by world-space box overlap;
                # only replace if it is on this tile and in training ROI.
                target=(x,y,w,h)
                if rid in (0,1,2,3):
                    kg=box(*rasterio.windows.bounds(Window(x,y,w,h),tx))
                    best=.0;choice=None
                    for ng in nad_geoms45:
                        if not ng.intersects(kg):continue
                        inter=ng.intersection(kg).area
                        overlap=inter/max(ng.area+kg.area-inter,1e-9)
                        if overlap>best:best=overlap;choice=ng
                    if best>=.5:
                        nx0,ny0,nx1,ny1=choice.bounds
                        # rasterio inverse affine -> column,row
                        c0,r0=(~tx)*(nx0,ny1);c1,r1=(~tx)*(nx1,ny0)
                        if min(c0,r0,W-c1,H-r1)>=3 and c1>c0 and r1>r0:
                            target=(c0,r0,c1-c0,r1-r0);num_replace+=1
                xx,yy,ww,hh=target
                lines.append(f"0 {(xx+ww/2)/W:.7f} {(yy+hh/2)/H:.7f} {ww/W:.7f} {hh/H:.7f}")
            out_lbl.write_text("\n".join(lines)+("\n" if lines else ""))
            replacements45+=num_replace
            summary45.append(dict(roi=rid,split=split,image=fname,boxes=len(lines),nadine_refined=num_replace))
    # ROI3 used internally as a real spatial hold-out; ROI4 is NEVER used by fit/early stopping.
    yml=DATA45/"dataset.yaml"
    yml.write_text(yaml.safe_dump(dict(path=str(DATA45),train="images/train",val="images/val",names={0:"crown"})))
report45=pd.DataFrame(summary45)
report45.to_csv(D45/"dataset_audit.csv",index=False)
display(report45.groupby(["split","roi"],as_index=False)[["boxes","nadine_refined"]].agg(["count","sum"]))
print("Nadine boxes replacing matching Kristof crowns:",replacements45)
assert (report45.query("split=='train'").boxes.sum()>150)
assert (report45.query("split=='val'").boxes.sum()>20)
weights45=D45/"yolo_nadine_kristof_best.pt"
if weights45.exists():
    print("Using saved trained weights",weights45)
else:
    detector45=YOLO("yolov8n.pt")  # pretrained small detector
    run45=detector45.train(data=str(yml),epochs=45,imgsz=512,batch=8,device=0,
                            workers=2,seed=45,patience=12,project=str(D45),
                            name="train",exist_ok=True,plots=False,verbose=False,
                            hsv_h=.0,fliplr=.5,flipud=.5,mosaic=.0,close_mosaic=0)
    best45=D45/"train"/"weights"/"best.pt"
    assert best45.exists(),f"No checkpoint: {best45}"
    shutil.copyfile(best45,weights45)
detector45=YOLO(str(weights45))
# Predict ROI4 only; do not optimize training against these results.
# Infer on original complete ROI4 tiles, deduplicate crowns in world coordinates.
from shapely.strtree import STRtree
predrows45=[]
with rasterio.open(RASTER) as ds:
    for im in valid_coco45["images"]:
        if int(im["roi_id"])!=4:continue
        fname=im["file_name"]
        tile=int(re.search(r"tile(\d+)",fname).group(1))
        win=manifest45[(4,tile)];tx=ds.window_transform(win)
        img=SOURCE45/"valid"/fname
        res=detector45.predict(source=str(img),imgsz=512,conf=.12,iou=.6,
                               device=0,verbose=False)[0]
        if res.boxes is None:continue
        for b,conf in zip(res.boxes.xyxy.cpu().numpy(),res.boxes.conf.cpu().numpy()):
            c0,r0,c1,r1=map(float,b)
            cx,cy=tx*((c0+c1)/2,(r0+r1)/2)
            if not roi45.geometry.iloc[4].contains(Point(cx,cy)):continue
            x0,y1=tx*(c0,r0);x1,y0=tx*(c1,r1)
            predrows45.append(dict(tile=tile,cx=cx,cy=cy,width_m=abs(x1-x0),
                                   height_m=abs(y1-y0),score=float(conf)))
pred45=pd.DataFrame(predrows45)
if len(pred45):
    pred45=pred45.sort_values("score",ascending=False).reset_index(drop=True)
    accepted=[]
    for row in pred45.itertuples():
        geom=box(row.cx-row.width_m/2,row.cy-row.height_m/2,
                 row.cx+row.width_m/2,row.cy+row.height_m/2)
        if any(geom.intersection(g).area/max(geom.union(g).area,1e-9)>.5 for g in accepted):continue
        accepted.append(geom)
    keep45=[];accepted=[]
    for i,r in pred45.iterrows():
        geom=box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
        if any(geom.intersection(g).area/max(geom.union(g).area,1e-9)>.5 for g in accepted):continue
        accepted.append(geom);keep45.append(i)
    pred45=pred45.loc[keep45].reset_index(drop=True)
pred45.to_csv(D45/"roi4_automatic_crown_boxes.csv",index=False)
refs45=ref45[ref45.geometry.apply(roi45.geometry.iloc[4].covers)].copy()
truth45=[box(*g.bounds) for g in refs45.geometry]
if len(pred45):
    p45=[box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
         for r in pred45.itertuples()]
    overlap45=np.zeros((len(truth45),len(p45)),dtype=np.float32)
    for i,a in enumerate(truth45):
        for j,b in enumerate(p45):
            if a.intersects(b):
                ai=a.intersection(b).area
                overlap45[i,j]=ai/(a.area+b.area-ai)
    rr,cc=linear_sum_assignment(overlap45,maximize=True)
    pair45=overlap45[rr,cc]
else:pair45=np.array([])
metrics45=[]
for threshold in [.3,.5,.7]:
    matched=int(np.sum(pair45>=threshold))
    metrics45.append(dict(box_iou=threshold,predictions=len(pred45),reference=len(truth45),
                          matched=matched,precision=matched/max(1,len(pred45)),
                          recall=matched/max(1,len(truth45)),
                          f1=2*matched/max(1,len(pred45)+len(truth45))))
display(pd.DataFrame(metrics45))
pd.DataFrame(metrics45).to_csv(D45/"roi4_box_metrics.csv",index=False)
print("BLOCK 45 PASS:",D45)
print("ROI4 held out from training. ROI5 untouched. These are BOX metrics, not SAM3 segmentation results.")


## 46 — YOLO boxes → zero-shot SAM3 (RGB-only baseline)

Feed *automatically predicted* Block45 YOLO crown boxes into the **zero-shot SAM3** model. Only boxes fully contained in an existing ROI4 image tile are submitted; count excluded boxes separately. Select the native SAM-score mask, polygonize, apply confidence-ordered spatial NMS, and evaluate one-to-one against 198 ROI4 reference crowns. **No DSM and no Musanga probability** in this baseline, so their future incremental value can be measured. No annotation-derived prompting or ROI5 access. Requires Blocks 0R/2/14 and successful Block45.


In [ ]:
# BLOCK 46 — Autonomous YOLO boxes -> ZERO-SHOT SAM3 crowns on ROI4
# No oracle prompting, DSM or Musanga probability. ROI5 untouched.
import subprocess,json
from pathlib import Path
import numpy as np,pandas as pd,rasterio,geopandas as gpd
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import box,shape
from scipy.optimize import linear_sum_assignment
from shapely.validation import make_valid
from IPython.display import display
D46=OUT/"experiments_v46_yolo_to_zero_shot_sam3"; D46.mkdir(parents=True,exist_ok=True)
D45=OUT/"experiments_v45_yolo_nadine_kristof"
assert Path(SAMPY).exists() and Path("/content/sam3").exists(),"Run environment setup Blocks 0R/2"
source46=D45/"roi4_automatic_crown_boxes.csv"
assert source46.exists(),"Run Block45"
pred46=pd.read_csv(source46);assert len(pred46)>0
tiles46=pd.read_csv(OUT/"experiments_v22_full_roi_8m_4"/"full"/"prompt_manifest.csv").drop_duplicates("tile_id")
with rasterio.open(RASTER) as src:
    all_tiles46=[]
    for r in tiles46.itertuples():
        w=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        all_tiles46.append((r,box(*rasterio.windows.bounds(w,src.transform))))
    roi46=roi_v2.to_crs(src.crs).geometry.iloc[4]
    ref46=reference_v2.to_crs(src.crs)
    ref46=ref46[ref46.geometry.apply(roi46.covers)].reset_index(drop=True)
    assert len(ref46)==198
    assigned46=[];excluded46=0
    for i,r in pred46.iterrows():
        requested=box(r.cx-r.width_m/2,r.cy-r.height_m/2,
                      r.cx+r.width_m/2,r.cy+r.height_m/2)
        eligible=[(t,p) for t,p in all_tiles46 if p.covers(requested)]
        if not eligible:
            excluded46+=1;continue
        t,p=max(eligible,key=lambda z:z[1].boundary.distance(requested.centroid))
        xmin,ymin,xmax,ymax=p.bounds
        assigned46.append(dict(prompt_id=len(assigned46),yolo_id=i,tile_id=int(t.tile_id),path=t.path,
             col_off=int(t.col_off),row_off=int(t.row_off),width=int(t.width),height=int(t.height),
             cx=(r.cx-xmin)/(xmax-xmin),cy=(ymax-r.cy)/(ymax-ymin),
             bw=r.width_m/(xmax-xmin),bh=r.height_m/(ymax-ymin),detector_score=r.score))
manifest46=pd.DataFrame(assigned46)
assert len(manifest46)>0,"No YOLO boxes fully enclosed by existing SAM3 tiles."
manifest46.to_csv(D46/"autonomous_prompt_manifest.csv",index=False)
print("YOLO boxes:",len(pred46),"| eligible SAM3 boxes:",len(manifest46),"| tile-edge exclusions:",excluded46)
# Reuse zero-shot SAM3 prompt interface established in Block41.
script46=Path("/content/musanga46_yolo_sam3.py")
script46.write_text(r'''
import sys,torch,numpy as np,pandas as pd
from pathlib import Path
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]);dest=Path(sys.argv[2]);dest.mkdir(parents=True,exist_ok=True)
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz",
                                 device="cuda",eval_mode=True)
    proc=Sam3Processor(model)
    for tile,group in df.groupby("tile_id",sort=True):
        todo=[r for r in group.itertuples() if not (dest/f"prompt_{int(r.prompt_id):04d}.npz").exists()]
        if not todo:continue
        img=Image.open(todo[0].path).convert("RGB")
        for r in todo:
            state=proc.set_image(img)
            result=proc.add_geometric_prompt(
                box=np.array([[r.cx,r.cy,r.bw,r.bh]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=result["masks"].detach().cpu().numpy()
            scores=result["scores"].detach().float().cpu().numpy().reshape(-1)
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(dest/f"prompt_{int(r.prompt_id):04d}.npz",
                masks=(masks>.5).astype(np.uint8),scores=scores)
        print("tile",tile,":",len(todo),"SAM3 prompts done",flush=True)
''')
raw46=D46/"raw";raw46.mkdir(exist_ok=True)
pending46=sum(not (raw46/f"prompt_{int(i):04d}.npz").exists() for i in manifest46.prompt_id)
if pending46:
    with (D46/"sam3_inference.log").open("w") as log:
        result=subprocess.run([str(SAMPY),"-u",str(script46),str(D46/"autonomous_prompt_manifest.csv"),
                               str(raw46)],cwd="/content/sam3",stdout=log,stderr=subprocess.STDOUT,check=False)
    print("\n".join((D46/"sam3_inference.log").read_text(errors="replace").splitlines()[-12:]))
    assert result.returncode==0,"SAM3 failed. Inspect sam3_inference.log"
# Extract highest SAM-score mask, polygonize; no reference-based mask selection.
features46=[]
with rasterio.open(RASTER) as src:
    for r in manifest46.itertuples():
        with np.load(raw46/f"prompt_{int(r.prompt_id):04d}.npz") as z:
            masks=z["masks"];scores=z["scores"].reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        if not len(scores) or not len(masks):continue
        mask=(masks[int(np.argmax(scores))]>0).astype("uint8")
        transform=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)))
        pieces=[make_valid(shape(g)) for g,v in shapes(mask,mask=mask.astype(bool),transform=transform) if v==1]
        if not pieces:continue
        # Select main connected component closest to the *predicted* YOLO box centre.
        cx,cy=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height)))*(r.cx*r.width,r.cy*r.height)
        pieces=sorted(pieces,key=lambda g:g.area,reverse=True)
        g=pieces[0]
        if not g.is_valid:g=make_valid(g)
        if g.is_empty or not g.is_valid or g.area<2 or g.area>700:continue
        g=g.intersection(roi46)
        if g.is_empty or g.area<2:continue
        features46.append(dict(prompt_id=int(r.prompt_id),detector_score=float(r.detector_score),
                               sam_score=float(scores.max()),geometry=g))
crowns46=gpd.GeoDataFrame(features46,geometry="geometry",crs=src.crs) if features46 else gpd.GeoDataFrame(columns=["geometry"],geometry="geometry",crs=src.crs)
if len(crowns46):
    # Independent spatial NMS: confidence based, never reference IoU.
    crowns46=crowns46.sort_values(["detector_score","sam_score"],ascending=False).reset_index(drop=True)
    keep=[];accepted=[]
    for k,g in enumerate(crowns46.geometry):
        if any(g.intersection(prev).area/max(g.area+prev.area-g.intersection(prev).area,1e-9)>.65 for prev in accepted):continue
        keep.append(k);accepted.append(g)
    crowns46=crowns46.iloc[keep].copy().reset_index(drop=True)
    crowns46.to_file(D46/"roi4_yolo_zero_shot_sam3_crowns.gpkg",driver="GPKG")
print("SAM3 polygons after basic filtering and NMS:",len(crowns46))
# Same independent one-to-one Hungarian matching at polygon IoU.
overlap46=np.zeros((len(ref46),len(crowns46)),dtype=np.float32)
for i,a in enumerate(ref46.geometry):
    for j,b in enumerate(crowns46.geometry):
        if a.intersects(b):
            inter=a.intersection(b).area
            overlap46[i,j]=inter/max(a.area+b.area-inter,1e-9)
rr,cc=linear_sum_assignment(overlap46,maximize=True)
paired46=overlap46[rr,cc]
metrics46=[]
for threshold in [.3,.5,.7]:
    tp=int(np.sum(paired46>=threshold))
    metrics46.append(dict(iou=threshold,reference=len(ref46),predicted=len(crowns46),matched=tp,
                   precision=tp/max(1,len(crowns46)),recall=tp/len(ref46),
                   f1=2*tp/max(1,len(ref46)+len(crowns46))))
summary46=pd.DataFrame(metrics46)
summary46.to_csv(D46/"roi4_sam3_metrics.csv",index=False)
display(summary46.round(3))
print("For comparison: Block36 RF+GIS watershed F1=0.391 at IoU 0.5; differing prediction counts.")
print("BLOCK 46 PASS:",D46,"| ROI5 untouched; no DSM/probability used.")


## 47 — Diagnose YOLO box versus SAM3 crown accuracy

Use the **same automatically detected crown proposals** to assess whether zero-shot SAM3 improves geometric IoU over YOLO boxes. Run a low-confidence YOLO-only threshold sweep on ROI 4 (no SAM3 re-inference), and sample Musanga probability and DSM at crown centres for descriptive analysis. **Important:** all-species `Crown_Kristof` ground truth does not validate Musanga species identity; the DSM is a surface model, not a canopy-height model. ROI 5 remains untouched.


In [ ]:
# BLOCK 47 — YOLO→SAM3 paired accuracy + lower-confidence detection + DSM/probability audit
# READ-ONLY evaluation: no training, no additional SAM3 runs, no ROI5 access.
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from shapely.geometry import box,Point
from scipy.optimize import linear_sum_assignment
from IPython.display import display
import matplotlib.pyplot as plt
import re
from ultralytics import YOLO
D47=OUT/"experiments_v47_yolo_sam3_diagnostics";D47.mkdir(parents=True,exist_ok=True)
D45=OUT/"experiments_v45_yolo_nadine_kristof"
D46=OUT/"experiments_v46_yolo_to_zero_shot_sam3"
assert (D45/"yolo_nadine_kristof_best.pt").exists()
assert (D46/"roi4_yolo_zero_shot_sam3_crowns.gpkg").exists()
assert "reference_v2" in globals() and "roi_v2" in globals()
with rasterio.open(RASTER) as ds:
    crs47=ds.crs
    roi47=roi_v2.to_crs(crs47).geometry.iloc[4]
    ref47=reference_v2.to_crs(crs47)
    ref47=ref47[ref47.geometry.apply(roi47.covers)].reset_index(drop=True)
assert len(ref47)==198
truth47=list(ref47.geometry)
auto47=pd.read_csv(D45/"roi4_automatic_crown_boxes.csv")
manifest47=pd.read_csv(D46/"autonomous_prompt_manifest.csv")
polys47=gpd.read_file(D46/"roi4_yolo_zero_shot_sam3_crowns.gpkg").to_crs(crs47)
assert "prompt_id" in polys47.columns
polys47=polys47.merge(manifest47[["prompt_id","yolo_id"]],on="prompt_id",validate="one_to_one")
# Paired best-reference IoUs per SAME autonomous prompt, and one-to-one Hungarian F1.
def iou47(a,b):
    try:
        if not a.intersects(b):return 0.
        inter=a.intersection(b).area
        return float(inter/max(a.area+b.area-inter,1e-12))
    except Exception:return 0.
paired47=[]
for row in polys47.itertuples():
    y=auto47.iloc[int(row.yolo_id)]
    bbox=box(y.cx-y.width_m/2,y.cy-y.height_m/2,y.cx+y.width_m/2,y.cy+y.height_m/2)
    bi=np.array([iou47(bbox,box(*g.bounds)) for g in truth47])
    mi=np.array([iou47(row.geometry,g) for g in truth47])
    k=int(mi.argmax())
    paired47.append(dict(prompt_id=int(row.prompt_id),yolo_id=int(row.yolo_id),
        box_best_iou=float(bi.max()),sam_best_iou=float(mi.max()),
        same_ref_box_iou=float(bi[k]),same_ref_sam_iou=float(mi[k]),
        sam_improvement=float(mi[k]-bi[k]),detector_score=float(y.score)))
paired47=pd.DataFrame(paired47)
paired47.to_csv(D47/"paired_box_vs_sam.csv",index=False)
print("Paired prompts:",len(paired47))
print("Median best box IoU:",round(paired47.box_best_iou.median(),3),
      "| median best SAM crown IoU:",round(paired47.sam_best_iou.median(),3))
print("Median SAM-minus-box IoU to SAM-best reference:",
      round(paired47.sam_improvement.median(),3))
print("Fraction SAM improves IoU to SAME reference:",
      round((paired47.sam_improvement>0).mean(),3))
# Additional unpaired box count is explicitly tracked (YOLO 132 -> 129 eligible)
print("Autonomous YOLO proposals:",len(auto47),"| SAM masks retained:",len(polys47))
# Confidence threshold sweep: YOLO inference only, independent of reference; compare BOX metrics.
det47=YOLO(str(D45/"yolo_nadine_kristof_best.pt"))
tile_csv47=OUT/"experiments_v22_full_roi_8m_4"/"full"/"prompt_manifest.csv"
tiles47=pd.read_csv(tile_csv47).drop_duplicates("tile_id")
srcimgs47=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"/"valid"
candidates47=[]
with rasterio.open(RASTER) as ds:
    for t in tiles47.itertuples():
        fname=f"roi4_tile{int(t.tile_id):04d}.png"
        path=srcimgs47/fname
        if not path.exists():continue
        tx=ds.window_transform(rasterio.windows.Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height)))
        result=det47.predict(str(path),imgsz=512,conf=.025,iou=.6,device=0,verbose=False)[0]
        if result.boxes is None:continue
        for coords,conf in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.conf.cpu().numpy()):
            c0,r0,c1,r1=map(float,coords)
            cx,cy=tx*((c0+c1)/2,(r0+r1)/2)
            if not roi47.contains(Point(cx,cy)):continue
            x0,y1=tx*(c0,r0);x1,y0=tx*(c1,r1)
            candidates47.append(dict(cx=cx,cy=cy,width_m=abs(x1-x0),
                height_m=abs(y1-y0),score=float(conf),tile=int(t.tile_id)))
candidate47=pd.DataFrame(candidates47)
candidate47.to_csv(D47/"roi4_low_conf_yolo_raw.csv",index=False)
def nms47(rows,overlap=.5):
    accepted=[];keep=[]
    for r in rows.sort_values("score",ascending=False).itertuples():
        p=box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
        if any(iou47(p,prev)>overlap for prev in accepted):continue
        keep.append(dict(cx=r.cx,cy=r.cy,width_m=r.width_m,height_m=r.height_m,score=r.score))
        accepted.append(p)
    return pd.DataFrame(keep)
ref_boxes47=[box(*g.bounds) for g in truth47]
sweep47=[]
for cutoff in [.025,.05,.08,.12,.18,.25]:
    subset=candidate47[candidate47.score>=cutoff] if len(candidate47) else candidate47
    dedup=nms47(subset)
    boxes=[box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
           for r in dedup.itertuples()] if len(dedup) else []
    mat=np.array([[iou47(a,b) for b in boxes] for a in ref_boxes47],dtype=np.float32) if boxes else np.zeros((len(ref_boxes47),0))
    matches=mat[linear_sum_assignment(mat,maximize=True)] if len(boxes) else np.array([])
    for threshold in [.3,.5]:
        tp=int((matches>=threshold).sum())
        sweep47.append(dict(confidence=cutoff,box_iou=threshold,predictions=len(boxes),matched=tp,
           precision=tp/max(1,len(boxes)),recall=tp/len(truth47),
           f1=2*tp/max(1,len(boxes)+len(truth47))))
sweep47=pd.DataFrame(sweep47)
sweep47.to_csv(D47/"yolo_confidence_box_sweep.csv",index=False)
display(sweep47.round(3))
print("Threshold sweep uses bounding boxes ONLY; no SAM3 run for new candidates.")
# Raster predictors: audit sampled values at existing SAM crown centroids; NO species ground-truth claim.
predictors47=[]
raster_paths47={"musanga_prob":PROB,"dsm_surface":GEO/"Baego_DSM.tif"}
for name,path in raster_paths47.items():
    try:
        if not Path(path).exists():continue
        with rasterio.open(path) as ds:
            centres=polys47.to_crs(ds.crs).geometry.centroid
            values=[]
            for a in ds.sample([(p.x,p.y) for p in centres],masked=True):
                v=a[0]
                values.append(np.nan if np.ma.is_masked(v) or not np.isfinite(v) or (ds.nodata is not None and v==ds.nodata) else float(v))
            if name=="musanga_prob":values=[v/10000 if np.isfinite(v) else np.nan for v in values]
            predictors47.append((name,values))
    except Exception as exc:
        print(name,"sample unavailable:",str(exc)[:180])
scores47=paired47.copy()
for name,values in predictors47:scores47[name]=values
scores47.to_csv(D47/"paired_with_predictor_samples.csv",index=False)
print("Raster values are descriptive only; DSM is a surface model, NOT canopy height.")
fig,ax=plt.subplots(figsize=(6,5))
ax.scatter(paired47.same_ref_box_iou,paired47.same_ref_sam_iou,s=20,alpha=.65)
ax.plot([0,1],[0,1],"--",linewidth=1)
ax.set(xlabel="YOLO box IoU (same SAM-best reference)",ylabel="SAM3 polygon IoU",
       title="Paired auto-prompt boundary improvement")
fig.tight_layout();fig.savefig(D47/"paired_box_sam_iou.png",dpi=160);plt.show()
print("BLOCK 47 PASS:",D47,"| ROI5 untouched.")


## 48 — Rank fixed YOLO candidates using DSM and Musanga probability

Starting from the existing low-confidence YOLO outputs (Block 47), hold the candidate geometry fixed at confidence ≥ 0.05. Compare **RGB-only confidence**, RGB + local DSM relative relief, RGB + mean Musanga CNN probability, and all three, using prespecified weights and identical candidate counts. Use ROI 4 for **exploratory all-species box detection only**. DSM is surface elevation, not canopy height; Musanga probability is model-derived and not an independent species label. ROI 5 untouched; no model training or SAM3 inference.


In [ ]:
# BLOCK 48 — Fixed YOLO candidate set: RGB vs DSM-relief vs Musanga-probability ranking (ROI4)
# No model fitting, no oracle prompting, no ROI5 use; all-species IoU only.
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import from_bounds,Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from shapely.geometry import box,Point
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import gaussian_filter
from IPython.display import display
import matplotlib.pyplot as plt
D48=OUT/"experiments_v48_dsm_probability_ranking";D48.mkdir(parents=True,exist_ok=True)
D47=OUT/"experiments_v47_yolo_sam3_diagnostics"
raw48=pd.read_csv(D47/"roi4_low_conf_yolo_raw.csv")
assert len(raw48)>0 and "reference_v2" in globals() and "roi_v2" in globals()
# Reproduce Block47 detector NMS, then restrict to the fixed confidence 0.05 candidates.
def box48(r):
    return box(r.cx-r.width_m/2,r.cy-r.height_m/2,r.cx+r.width_m/2,r.cy+r.height_m/2)
def iou48(a,b):
    if not a.intersects(b):return 0.
    v=a.intersection(b).area
    return float(v/max(a.area+b.area-v,1e-12))
keep48=[];geom48=[]
for row in raw48[raw48.score>=.05].sort_values("score",ascending=False).itertuples():
    g=box48(row)
    if any(iou48(g,p)>.5 for p in geom48):continue
    keep48.append(dict(cx=row.cx,cy=row.cy,width_m=row.width_m,height_m=row.height_m,
                       yolo_score=row.score,tile=row.tile))
    geom48.append(g)
cand48=pd.DataFrame(keep48)
print("Fixed 0.05-confidence candidate boxes:",len(cand48))
assert len(cand48)>0
# Windowed sampling: read only small aligned patches, not whole orthomosaic/DSM.
# DSM measures SURFACE elevation, not tree height: remove broad-scale trends locally.
def raster_features48(path,rect,is_probability=False):
    with rasterio.open(RASTER) as rgb, rasterio.open(path) as raw:
        with WarpedVRT(raw,crs=rgb.crs,transform=rgb.transform,
                       width=rgb.width,height=rgb.height,
                       resampling=Resampling.bilinear,
                       src_nodata=raw.nodata,nodata=np.nan) as ds:
            xmin,ymin,xmax,ymax=rect.bounds
            pad=5.
            win=from_bounds(xmin-pad,ymin-pad,xmax+pad,ymax+pad,transform=rgb.transform)
            win=win.round_offsets().round_lengths()
            # Clip to source bounds (WarpedVRT boundless reads unsupported).
            win=win.intersection(Window(0,0,rgb.width,rgb.height))
            if win.width<3 or win.height<3:return (np.nan,np.nan,np.nan)
            arr=ds.read(1,window=win,masked=True).astype(np.float32).filled(np.nan)
            tx=rgb.window_transform(win)
            h,w=arr.shape
            yy,xx=np.mgrid[:h,:w]
            xs=tx.c+(xx+.5)*tx.a+(yy+.5)*tx.b
            ys=tx.f+(xx+.5)*tx.d+(yy+.5)*tx.e
            inside=(xs>=xmin)&(xs<=xmax)&(ys>=ymin)&(ys<=ymax)&np.isfinite(arr)
            if not inside.any():return (np.nan,np.nan,np.nan)
            z=arr[inside]
            if is_probability:
                values=z/10000.
                return (float(np.nanmean(values)),float(np.nanquantile(values,.75)),float(np.mean(values>=.5)))
            # Local relief relative to a 5m ring/patch baseline; not CHM and not a crown apex metric.
            baseline=float(np.nanmedian(arr))
            relief=z-baseline
            return (float(np.nanquantile(relief,.9)),float(np.nanstd(z)),float(np.nanmedian(relief)))
rows48=[]
for i,row in cand48.iterrows():
    g=box48(row)
    pmean,p75,pcover=raster_features48(PROB,g,True)
    d90,dstd,dmed=raster_features48(GEO/"Baego_DSM.tif",g,False)
    rows48.append(dict(prob_mean=pmean,prob_p75=p75,prob_fraction_ge05=pcover,
                       dsm_relief_p90=d90,dsm_std=dstd,dsm_relief_median=dmed))
feat48=pd.concat([cand48,pd.DataFrame(rows48)],axis=1)
feat48.to_csv(D48/"roi4_260_candidates_features.csv",index=False)
display(feat48[["yolo_score","prob_mean","dsm_relief_p90","dsm_std"]].describe().round(3))
# Prespecified, unsupervised rank fusion. Scores are percentile ranks within SAME candidates.
# No ROI4 label-derived feature weights; thus this is a diagnostic ranking, not a calibrated classifier.
def rank48(col):
    x=feat48[col].replace([np.inf,-np.inf],np.nan)
    return x.rank(pct=True).fillna(.5)
feat48["rank_rgb"]=rank48("yolo_score")
feat48["rank_rgb_dsm"]=.8*feat48.rank_rgb+.2*rank48("dsm_relief_p90")
feat48["rank_rgb_prob"]=.8*feat48.rank_rgb+.2*rank48("prob_mean")
feat48["rank_rgb_dsm_prob"]=.7*feat48.rank_rgb+.15*rank48("dsm_relief_p90")+.15*rank48("prob_mean")
with rasterio.open(RASTER) as src:
    ref=reference_v2.to_crs(src.crs);roi=roi_v2.to_crs(src.crs).geometry.iloc[4]
    ref=ref[ref.geometry.apply(roi.covers)].reset_index(drop=True)
assert len(ref)==198
targets=[box(*g.bounds) for g in ref.geometry]
mat=np.zeros((len(targets),len(feat48)),np.float32)
for i,a in enumerate(targets):
    for j,r in enumerate(feat48.itertuples()):
        mat[i,j]=iou48(a,box48(r))
records=[]
for rank_name in ["rank_rgb","rank_rgb_dsm","rank_rgb_prob","rank_rgb_dsm_prob"]:
    for top_n in [75,100,132,185,260]:
        top=feat48.nlargest(min(top_n,len(feat48)),rank_name).index.to_numpy()
        sub=mat[:,top]
        rr,cc=linear_sum_assignment(sub,maximize=True)
        vals=sub[rr,cc]
        for threshold in [.3,.5]:
            tp=int(np.sum(vals>=threshold));n=len(top)
            records.append(dict(ranking=rank_name,top_n=top_n,iou=threshold,predicted=n,
                matched=tp,precision=tp/max(1,n),recall=tp/len(targets),
                f1=2*tp/max(1,n+len(targets))))
summary48=pd.DataFrame(records)
summary48.to_csv(D48/"ranking_comparison.csv",index=False)
print("All-species CROWN BOX comparison, IoU>=0.5:")
display(summary48[summary48.iou==.5].pivot(index="top_n",columns="ranking",values="f1").round(3))
# Preserve complete candidate diagnostics for future species-specific and paired analyses.
feat48.to_csv(D48/"ranked_candidates_with_features.csv",index=False)
print("Musanga probability is an existing CNN OUTPUT, not independent species ground truth.")
print("DSM metrics are relative SURFACE elevations, not height-normalized crown estimates.")
print("Block48 rank weights are fixed heuristics; ROI4 results are exploratory and cannot calibrate an independent test.")
print("BLOCK 48 PASS:",D48,"| ROI5 untouched. No extra training or SAM3 inference.")


## 49 — Diagnose YOLO box geometry and controlled size correction

Use the fixed 260 candidate boxes from Block 47 (confidence ≥ 0.05). Match boxes one-to-one to ROI4 crown references, quantify crown-centre displacement (metres), width/height bias and crown-size dependence. Test **prespecified, reference-free** width/height multipliers 0.8–1.2 on identical candidate centres, reporting box F1; these are sensitivity tests, not learned corrections or SAM3 outcomes. ROI4 is exploratory development; ROI5 remains untouched.


In [ ]:
# BLOCK 49 — Crown-box geometry diagnosis and reference-free box-size sensitivity
# ROI4 development diagnostics only. NO fitting against ROI4 labels, no SAM3 inference, ROI5 untouched.
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from shapely.geometry import box
from scipy.optimize import linear_sum_assignment
from IPython.display import display
import matplotlib.pyplot as plt
D49=OUT/"experiments_v49_box_geometry";D49.mkdir(parents=True,exist_ok=True)
D47=OUT/"experiments_v47_yolo_sam3_diagnostics"
assert (D47/"roi4_low_conf_yolo_raw.csv").exists()
assert "reference_v2" in globals() and "roi_v2" in globals()
raw49=pd.read_csv(D47/"roi4_low_conf_yolo_raw.csv")
def geom49(x):
    return box(float(x.cx)-float(x.width_m)/2,float(x.cy)-float(x.height_m)/2,
               float(x.cx)+float(x.width_m)/2,float(x.cy)+float(x.height_m)/2)
def iou49(a,b):
    if not a.intersects(b):return 0.
    v=a.intersection(b).area
    return float(v/max(a.area+b.area-v,1e-9))
# Fixed confidence .05 and spatial NMS .50, identical to Block47/48.
rows49=[];taken49=[]
for r in raw49[raw49.score>=.05].sort_values("score",ascending=False).itertuples():
    g=geom49(r)
    if any(iou49(g,h)>.5 for h in taken49):continue
    taken49.append(g)
    rows49.append(dict(cx=r.cx,cy=r.cy,width_m=r.width_m,height_m=r.height_m,score=r.score))
cand49=pd.DataFrame(rows49)
assert len(cand49)==260,f"Expected same 260 candidates as Block48, got {len(cand49)}"
with rasterio.open(RASTER) as ds:
    roi49=roi_v2.to_crs(ds.crs).geometry.iloc[4]
    ref49=reference_v2.to_crs(ds.crs)
    ref49=ref49[ref49.geometry.apply(roi49.covers)].reset_index(drop=True)
assert len(ref49)==198
refboxes49=[box(*g.bounds) for g in ref49.geometry]
predboxes49=[geom49(r) for r in cand49.itertuples()]
overlap49=np.zeros((len(refboxes49),len(predboxes49)),dtype=np.float32)
for i,a in enumerate(refboxes49):
    for j,b in enumerate(predboxes49):
        overlap49[i,j]=iou49(a,b)
# Assign globally to a unique reference; do not mistake spurious boxes for localization errors.
ri49,pi49=linear_sum_assignment(overlap49,maximize=True)
pairs49=[]
for i,j in zip(ri49,pi49):
    iou=float(overlap49[i,j])
    if iou<.3:continue
    true=refboxes49[i];pred=predboxes49[j]
    tx0,ty0,tx1,ty1=true.bounds
    px0,py0,px1,py1=pred.bounds
    tc=true.centroid;pc=pred.centroid
    pairs49.append(dict(ref_id=int(i),pred_id=int(j),box_iou=iou,score=float(cand49.iloc[j].score),
        dx_m=float(pc.x-tc.x),dy_m=float(pc.y-tc.y),
        centre_error_m=float(pc.distance(tc)),
        true_width_m=tx1-tx0,true_height_m=ty1-ty0,
        pred_width_m=px1-px0,pred_height_m=py1-py0,
        width_ratio=(px1-px0)/(tx1-tx0),height_ratio=(py1-py0)/(ty1-ty0),
        true_area_m2=float(ref49.geometry.iloc[i].area)))
pairs49=pd.DataFrame(pairs49)
pairs49.to_csv(D49/"matched_box_geometry.csv",index=False)
print("ROI4 matches with box IoU >= .3:",len(pairs49))
for col in ["box_iou","centre_error_m","dx_m","dy_m","width_ratio","height_ratio"]:
    print(col,"median",round(pairs49[col].median(),3),"p90",round(pairs49[col].quantile(.9),3))
pairs49["crown_area_quartile"]=pd.qcut(pairs49.true_area_m2,4,duplicates="drop")
display(pairs49.groupby("crown_area_quartile",observed=True)[
    ["box_iou","centre_error_m","width_ratio","height_ratio"]].median().round(3))
# Prespecified adjustments use ONLY predicted centre + box dimensions.
# These are exploratory scenarios; never use the reference to adjust individual predictions.
scales49=[.8,.9,1.,1.1,1.2]
summary49=[]
for scale in scales49:
    proposed=[box(r.cx-r.width_m*scale/2,r.cy-r.height_m*scale/2,
                  r.cx+r.width_m*scale/2,r.cy+r.height_m*scale/2)
              for r in cand49.itertuples()]
    mat=np.zeros((len(refboxes49),len(proposed)),np.float32)
    for i,a in enumerate(refboxes49):
        for j,b in enumerate(proposed):mat[i,j]=iou49(a,b)
    rr,cc=linear_sum_assignment(mat,maximize=True)
    vals=mat[rr,cc]
    for threshold in [.3,.5,.7]:
        tp=int((vals>=threshold).sum())
        summary49.append(dict(scale=scale,iou_threshold=threshold,predicted=len(proposed),matched=tp,
             precision=tp/len(proposed),recall=tp/len(refboxes49),
             f1=2*tp/(len(proposed)+len(refboxes49))))
scales_table49=pd.DataFrame(summary49)
scales_table49.to_csv(D49/"prespecified_box_size_sensitivity.csv",index=False)
print("Fixed 260 boxes; geometry-only scale sensitivity, IoU>=.5:")
display(scales_table49[scales_table49.iou_threshold==.5][
    ["scale","matched","precision","recall","f1"]].round(3))
fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].hist(pairs49.centre_error_m,bins=25)
axes[0].set(xlabel="Crown-centre displacement (m)",ylabel="Matched crowns",title="YOLO centre errors")
axes[1].scatter(pairs49.true_area_m2,pairs49.width_ratio,s=18,alpha=.6,label="Width")
axes[1].scatter(pairs49.true_area_m2,pairs49.height_ratio,s=18,alpha=.6,label="Height")
axes[1].axhline(1,color="grey",linestyle="--");axes[1].legend()
axes[1].set(xlabel="Reference crown polygon area (m²)",ylabel="Predicted / reference box size",
            title="Box-size bias")
sel=scales_table49[scales_table49.iou_threshold==.5]
axes[2].plot(sel.scale,sel.f1,marker="o")
axes[2].set(xlabel="Box width/height multiplier",ylabel="ROI4 box F1, IoU≥.5",title="Size correction sensitivity")
fig.tight_layout();fig.savefig(D49/"box_geometry_diagnostics.png",dpi=160);plt.show()
print("IMPORTANT: localisation diagnostics conditioned on box IoU>=.3; unmatched and duplicate candidates excluded.")
print("These corrections have not been tested as SAM3 prompts. ROI4 is development-only; do not select final hyperparameters from ROI4.")
print("BLOCK 49 PASS:",D49,"| ROI5 untouched.")


## 50 — Zero-shot SAM3 sensitivity to crown-box displacement

Using the same 169 oracle crowns and **exact crown box sizes**, shift box centres by ±0.5, ±1, ±2 m in four cardinal directions. Reuse existing exact-prompt masks from Block 41 and run only new shifts (resumable). Report paired median IoU, change from exact, fraction of crowns achieving IoU ≥ 0.5, directional effects, and displacement relative to crown diameter. This is a controlled **manual-oracle prompt sensitivity analysis**, not autonomous detection. ROI5 untouched.


In [ ]:
# BLOCK 50 — Controlled oracle crown-box DISPLACEMENT (zero-shot SAM3; ROI4 only)
# 169 existing paired oracle crowns. Exact widths/heights; shift x/y by 0, ±0.5, ±1, ±2m.
# This is a privileged-prompt sensitivity experiment, NOT independent crown detection.
from pathlib import Path
import subprocess
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box,mapping
from IPython.display import display
import matplotlib.pyplot as plt
D50=OUT/"experiments_v50_oracle_displacement";D50.mkdir(parents=True,exist_ok=True)
D40=OUT/"experiments_v40_oracle_boxes"
D41=OUT/"experiments_v41_oracle_padding"
assert Path(SAMPY).exists() and (D40/"oracle_manifest.csv").exists()
assert (D41/"oracle_padding_manifest.csv").exists(),"Run Block 41 first"
original50=pd.read_csv(D40/"oracle_manifest.csv")
pad50=pd.read_csv(D41/"oracle_padding_manifest.csv")
zero50=pad50[pad50.pad_m==0].copy()
assert len(zero50)==169,"Expected 169 zero-padding prompts from Block41"
assert zero50.crown_id.nunique()==169
with rasterio.open(RASTER) as ds:
    ref50=reference_v2.to_crs(ds.crs)
    roi50=roi_v2.to_crs(ds.crs).geometry.iloc[4]
    ref50=ref50[ref50.geometry.apply(roi50.covers)].reset_index(drop=True)
    assert len(ref50)==198
    shifts50=[("exact",0.,0.),("E05",.5,0.),("W05",-.5,0.),("N05",0.,.5),("S05",0.,-.5),
              ("E1",1.,0.),("W1",-1.,0.),("N1",0.,1.),("S1",0.,-1.),
              ("E2",2.,0.),("W2",-2.,0.),("N2",0.,2.),("S2",0.,-2.)]
    records50=[]
    for t in zero50.itertuples():
        win=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
        xmin,ymin,xmax,ymax=rasterio.windows.bounds(win,ds.transform)
        crown=ref50.geometry.iloc[int(t.crown_id)]
        left,bottom,right,top=crown.bounds
        for label,dx,dy in shifts50:
            test=box(left+dx,bottom+dy,right+dx,top+dy)
            if not box(xmin,ymin,xmax,ymax).covers(test):continue
            records50.append(dict(prompt_id=len(records50),crown_id=int(t.crown_id),
              shift=label,dx_m=dx,dy_m=dy,shift_m=float(np.hypot(dx,dy)),
              crown_area_m2=float(crown.area),equiv_diameter_m=float(2*np.sqrt(crown.area/np.pi)),
              tile_id=int(t.tile_id),path=t.path,col_off=int(t.col_off),row_off=int(t.row_off),
              width=int(t.width),height=int(t.height),
              cx=((left+right)/2+dx-xmin)/(xmax-xmin),
              cy=(ymax-((top+bottom)/2+dy))/(ymax-ymin),
              bw=(right-left)/(xmax-xmin),bh=(top-bottom)/(ymax-ymin)))
pm50=pd.DataFrame(records50)
pm50.to_csv(D50/"displacement_manifest.csv",index=False)
print("Eligible shifts:",pm50.groupby("shift").size().to_dict())
# Exact unshifted masks already exist in Block41: reuse these to reduce inference cost.
reuse50=dict(zip(zero50.crown_id.astype(int),zero50.prompt_id.astype(int)))
raw50=D50/"raw";raw50.mkdir(exist_ok=True)
for t in pm50[pm50["shift"]=="exact"].itertuples():
    dest=raw50/f"prompt_{int(t.prompt_id):04d}.npz"
    source=D41/"raw"/f"prompt_{reuse50[int(t.crown_id)]:04d}.npz"
    if not dest.exists():
        assert source.exists(),source
        import shutil
        shutil.copyfile(source,dest)
SCRIPT50=Path("/content/musanga50_displacement.py")
SCRIPT50.write_text(r'''
import sys,torch,numpy as np,pandas as pd
from pathlib import Path
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]);dest=Path(sys.argv[2]);dest.mkdir(parents=True,exist_ok=True)
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz",
                                 device="cuda",eval_mode=True)
    proc=Sam3Processor(model)
    for tid,group in df.groupby("tile_id",sort=True):
        todo=[r for r in group.itertuples() if not (dest/f"prompt_{int(r.prompt_id):04d}.npz").exists()]
        if not todo:continue
        image=Image.open(todo[0].path).convert("RGB")
        for r in todo:
            state=proc.set_image(image)
            result=proc.add_geometric_prompt(
              box=np.array([[r.cx,r.cy,r.bw,r.bh]],dtype=np.float32),
              label=np.array([1],dtype=np.int32),state=state)
            masks=result["masks"].detach().cpu().numpy()
            scores=result["scores"].detach().float().cpu().numpy().reshape(-1)
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(dest/f"prompt_{int(r.prompt_id):04d}.npz",
                               masks=(masks>.5).astype(np.uint8),scores=scores)
        print("tile",tid,":",len(todo),"shifts done",flush=True)
''')
pending50=sum(not (raw50/f"prompt_{int(i):04d}.npz").exists() for i in pm50.prompt_id)
print("SAM3 shift prompts remaining:",pending50,"(resumable; original zero-shift reused)")
if pending50:
    with (D50/"inference.log").open("w") as log:
        done=subprocess.run([str(SAMPY),"-u",str(SCRIPT50),str(D50/"displacement_manifest.csv"),
                             str(raw50)],cwd="/content/sam3",
                             stdout=log,stderr=subprocess.STDOUT,check=False)
    print("\n".join((D50/"inference.log").read_text(errors="replace").splitlines()[-15:]))
    assert done.returncode==0,"SAM3 displacement run failed; inspect inference.log"
details50=[]
with rasterio.open(RASTER) as ds:
    for t in pm50.itertuples():
        win=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
        truth=rasterize([(mapping(ref50.geometry.iloc[int(t.crown_id)]),1)],
                         out_shape=(int(t.height),int(t.width)),
                         transform=ds.window_transform(win)).astype(bool)
        with np.load(raw50/f"prompt_{int(t.prompt_id):04d}.npz") as data:
            masks=data["masks"];scores=data["scores"].reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        if len(scores):
            m=np.asarray(masks[int(np.argmax(scores))],bool)
            intersection=np.count_nonzero(m & truth)
            union=np.count_nonzero(m | truth)
            iou=intersection/union if union else 0.
            area_ratio=np.count_nonzero(m)/max(1,np.count_nonzero(truth))
        else:iou=0.;area_ratio=0.
        details50.append(dict(crown_id=int(t.crown_id),shift=t.shift,
              shift_m=t.shift_m,relative_shift=t.shift_m/t.equiv_diameter_m,
              crown_area_m2=t.crown_area_m2,iou=iou,area_ratio=area_ratio))
detail50=pd.DataFrame(details50);detail50.to_csv(D50/"per_crown_displacement.csv",index=False)
# Report only crowns with ALL 13 directions/shift combinations for apples-to-apples comparisons.
valid50=detail50.groupby("crown_id")["shift"].nunique()
paired_ids50=set(valid50[valid50==len(shifts50)].index)
paired50=detail50[detail50.crown_id.isin(paired_ids50)].copy()
print("Crowns with all displacement scenarios:",len(paired_ids50))
assert len(paired_ids50)>=100,"Too few fully paired crowns; inspect edge exclusions"
baseline50=paired50[paired50["shift"]=="exact"].set_index("crown_id").iou
paired50["delta_from_exact"]=paired50.apply(lambda r:r.iou-baseline50.loc[r.crown_id],axis=1)
summary50=paired50.groupby("shift_m").agg(
    examples=("iou","size"),crowns=("crown_id","nunique"),
    median_iou=("iou","median"),mean_iou=("iou","mean"),
    fraction_ge05=("iou",lambda z:(z>=.5).mean()),
    median_delta_iou=("delta_from_exact","median")).reset_index()
summary50.to_csv(D50/"paired_shift_summary.csv",index=False)
display(summary50.round(3))
orientation50=paired50.groupby("shift").agg(n=("iou","size"),median_iou=("iou","median"),
    median_delta=("delta_from_exact","median")).reset_index()
orientation50.to_csv(D50/"direction_summary.csv",index=False)
display(orientation50.round(3))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot(summary50.shift_m,summary50.median_iou,marker="o")
axes[0].set(xlabel="Box-centre displacement (m)",ylabel="Median crown mask IoU",
            title="Zero-shot SAM3 displacement sensitivity")
axes[1].scatter(paired50.relative_shift,paired50.delta_from_exact,s=7,alpha=.15)
axes[1].axhline(0,ls="--",color="grey")
axes[1].set(xlabel="Shift / equivalent crown diameter",ylabel="IoU change from exact box",
            title="Relative localization error")
fig.tight_layout();fig.savefig(D50/"shift_robustness.png",dpi=160);plt.show()
print("BLOCK 50 PASS:",D50,"| ROI5 untouched; privileged oracle-derived shifts, NOT autonomous detection.")


## 51 — Prompt-less baseline: original SAM automatic mask generator

Run Meta's original SAM1 ViT-B **Automatic Mask Generator** (AMG), which creates its own spatial point prompts without YOLO, manually supplied crown boxes or annotation-derived seeds. Fixed settings: 16×16 internal point grid per existing ROI4 tile, `pred_iou_thresh=0.88`, `stability_score_thresh=0.92`, no crop layers. Reuse stored ROI4 RGB tiles, cache masks resumably, polygonize and geospatially suppress overlaps; evaluate against **198 fully contained `Crown_Kristof` crowns** using one-to-one Hungarian polygon matching. This is a first computationally modest baseline, **not an exact reproduction of the Luki study**; study-specific prompts, thresholds and cleaning may differ. ROI5 untouched.


In [ ]:
# BLOCK 51 — True prompt-less SAM1 automatic-mask baseline, Baego ROI4
# Distinct from SAM3 grid-box prompts: SAM1 AMG samples INTERNAL point grids.
# ROI4 all-species crown F1 (198 fully contained Kristof crowns). ROI5 untouched.
import sys,subprocess,json
from pathlib import Path
import numpy as np,pandas as pd,rasterio,geopandas as gpd
from PIL import Image
from shapely.geometry import shape,box,Point
from shapely.validation import make_valid
from rasterio.windows import Window
from rasterio.features import shapes
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D51=OUT/"experiments_v51_promptless_sam1";D51.mkdir(parents=True,exist_ok=True)
assert "reference_v2" in globals() and "roi_v2" in globals()
try:
    from segment_anything import sam_model_registry,SamAutomaticMaskGenerator
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install",
                    "git+https://github.com/facebookresearch/segment-anything.git"],check=True)
    from segment_anything import sam_model_registry,SamAutomaticMaskGenerator
assert __import__("torch").cuda.is_available(),"GPU runtime required"
# Official original SAM ViT-B checkpoint; keep locally, avoid Drive overhead.
ck51=Path("/content/sam_vit_b_01ec64.pth")
if not ck51.exists() or ck51.stat().st_size<300_000_000:
    import urllib.request
    url51="https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    print("Downloading original SAM ViT-B checkpoint once (~375 MB)")
    urllib.request.urlretrieve(url51,str(ck51))
assert ck51.stat().st_size>300_000_000
import torch
sam51=sam_model_registry["vit_b"](checkpoint=str(ck51)).cuda()
sam51.eval()
# Fixed prompt-less baseline parameters (not fitted on ROI4).
generator51=SamAutomaticMaskGenerator(
    model=sam51,points_per_side=16,points_per_batch=64,
    pred_iou_thresh=.88,stability_score_thresh=.92,
    box_nms_thresh=.7,crop_n_layers=0,min_mask_region_area=100)
source51=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"/"valid"
tilefile51=OUT/"experiments_v22_full_roi_8m_4"/"full"/"prompt_manifest.csv"
assert tilefile51.exists(),"Run Block 22 / 28 for ROI4 tile manifest"
tiles51=pd.read_csv(tilefile51).drop_duplicates("tile_id").sort_values("tile_id")
with rasterio.open(RASTER) as src:
    crs51=src.crs;roi51=roi_v2.to_crs(crs51).geometry.iloc[4]
    ref51=reference_v2.to_crs(crs51)
    ref51=ref51[ref51.geometry.apply(roi51.covers)].reset_index(drop=True)
assert len(ref51)==198
raw51=D51/"tile_masks";raw51.mkdir(exist_ok=True)
tile_features51=[]
for t in tiles51.itertuples():
    tid=int(t.tile_id)
    filename=source51/f"roi4_tile{tid:04d}.png"
    assert filename.exists(),f"Block30 tile missing: {filename}"
    cache=raw51/f"tile_{tid:04d}.npz"
    if cache.exists():
        with np.load(cache) as z:
            masks=z["masks"];scores=z["scores"]
    else:
        rgb=np.asarray(Image.open(filename).convert("RGB"))
        out=generator51.generate(rgb)
        masks=np.asarray([q["segmentation"].astype("uint8") for q in out],dtype="uint8") if out else np.empty((0,*rgb.shape[:2]),dtype="uint8")
        scores=np.asarray([q["predicted_iou"]*q["stability_score"] for q in out],dtype="float32")
        np.savez_compressed(cache,masks=masks,scores=scores)
    win=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
    with rasterio.open(RASTER) as ds:tx=ds.window_transform(win)
    for j,(m,score) in enumerate(zip(masks,scores)):
        if not m.any():continue
        polygons=[shape(g) for g,v in shapes(m,mask=m.astype(bool),transform=tx) if v==1]
        if not polygons:continue
        geom=make_valid(max(polygons,key=lambda p:p.area))
        if geom.is_empty or not geom.is_valid:continue
        if not 4<=geom.area<=500:continue
        geom=geom.intersection(roi51)
        if geom.is_empty or geom.area<4:continue
        if not roi51.covers(geom.representative_point()):continue
        tile_features51.append(dict(tile=tid,mask_id=j,score=float(score),geometry=geom))
    print(f"ROI4 tile {tid}: {len(scores)} AMG proposals (cached)",flush=True)
candidates51=gpd.GeoDataFrame(tile_features51,geometry="geometry",crs=crs51)
print("Raw polygons after area/ROI filtering:",len(candidates51))
assert len(candidates51)>0,"No SAM automatic masks; inspect tile images or thresholds"
# Confidence-prioritized geographic NMS handles repeated crowns across overlapping tiles.
candidates51=candidates51.sort_values("score",ascending=False).reset_index(drop=True)
kept51=[];selected51=[]
for i,geom in enumerate(candidates51.geometry):
    if any(geom.intersection(p).area/max(geom.area+p.area-geom.intersection(p).area,1e-9)>.6
           for p in selected51 if geom.intersects(p)):continue
    kept51.append(i);selected51.append(geom)
crowns51=candidates51.iloc[kept51].reset_index(drop=True)
crowns51.to_file(D51/"sam1_amg_roi4_crowns.gpkg",driver="GPKG")
print("SAM1 automatic masks after NMS:",len(crowns51))
# Exact same all-species one-to-one Hungarian polygon IoU protocol.
matrix51=np.zeros((len(ref51),len(crowns51)),dtype=np.float32)
for i,a in enumerate(ref51.geometry):
    for j,b in enumerate(crowns51.geometry):
        if not a.intersects(b):continue
        intersection=a.intersection(b).area
        matrix51[i,j]=intersection/max(a.area+b.area-intersection,1e-9)
rr,cc=linear_sum_assignment(matrix51,maximize=True)
paired51=matrix51[rr,cc]
results51=[]
for thr in [.3,.5,.7]:
    tp=int((paired51>=thr).sum())
    results51.append(dict(iou=thr,reference=len(ref51),predicted=len(crowns51),matched=tp,
                          precision=tp/max(1,len(crowns51)),recall=tp/len(ref51),
                          f1=2*tp/max(1,len(ref51)+len(crowns51))))
table51=pd.DataFrame(results51)
table51.to_csv(D51/"roi4_sam1_amg_metrics.csv",index=False)
display(table51.round(3))
print("Compare at IoU >= 0.5: Block36 GIS F1=0.391, Block46 YOLO→SAM3 F1=0.373")
print("SAM1 internal point-grid automatic masks; not no prompts internally.")
print("BLOCK 51 PASS:",D51,"| ROI5 untouched.")


## 52 — Prompt-less SAM in western versus youngest stand

Repeat **identical original SAM1 Automatic Mask Generator settings** from Block 51 (16×16 point-grid per tile, ViT-B, same threshold, area filtering and NMS) on the **westernmost ROI among training ROIs 0–3** as a provisional more-mature stand. Compare its all-species crown F1 with cached **youngest ROI4** results; print six-ROI geography and crown-size/density profiles. Geographic west does not by itself prove maturity—confirm against stand-age records. Unlike ROI4, this ROI was used for earlier YOLO/SAM3 training, so interpret as a **within-landscape structural diagnostic**, not an independent model-validation score. SAM1 itself is zero-shot. ROI5 untouched. This is **not** an exact Luki workflow reproduction.


In [ ]:
# BLOCK 52 — Original SAM1 AMG on a mature/western ROI compared with young ROI4
# Distinct from SAM3 grid-box prompts: SAM1 AMG samples INTERNAL point grids.
# Same fixed generator as Block51; diagnostics in a TRAINING ROI, not independent generalization.
import sys,subprocess,json
from pathlib import Path
import numpy as np,pandas as pd,rasterio,geopandas as gpd
from PIL import Image
from shapely.geometry import shape,box,Point
from shapely.validation import make_valid
from rasterio.windows import Window
from rasterio.features import shapes
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D51=OUT/"experiments_v52_mature_roi_sam1";D51.mkdir(parents=True,exist_ok=True)
# Select westernmost of ROIs 0..3 as provisional mature-stand candidate.
# Inspect age metadata separately: westernmost does not establish actual stand age.
with rasterio.open(RASTER) as _src52:
    roi_order52=roi_v2.to_crs(_src52.crs)
    west52=sorted(range(4),key=lambda i:roi_order52.geometry.iloc[i].centroid.x)
    print("Training ROIs west to east:",west52)
ROI_TARGET52=west52[0]
print("Selected westernmost TRAINING ROI:",ROI_TARGET52,"(confirm maturity from field metadata)")
assert ROI_TARGET52 in (0,1,2,3)

assert "reference_v2" in globals() and "roi_v2" in globals()
try:
    from segment_anything import sam_model_registry,SamAutomaticMaskGenerator
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install",
                    "git+https://github.com/facebookresearch/segment-anything.git"],check=True)
    from segment_anything import sam_model_registry,SamAutomaticMaskGenerator
assert __import__("torch").cuda.is_available(),"GPU runtime required"
# Official original SAM ViT-B checkpoint; keep locally, avoid Drive overhead.
ck51=Path("/content/sam_vit_b_01ec64.pth")
if not ck51.exists() or ck51.stat().st_size<300_000_000:
    import urllib.request
    url51="https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    print("Downloading original SAM ViT-B checkpoint once (~375 MB)")
    urllib.request.urlretrieve(url51,str(ck51))
assert ck51.stat().st_size>300_000_000
import torch
sam51=sam_model_registry["vit_b"](checkpoint=str(ck51)).cuda()
sam51.eval()
# Fixed prompt-less baseline parameters (not fitted on ROI4).
generator51=SamAutomaticMaskGenerator(
    model=sam51,points_per_side=16,points_per_batch=64,
    pred_iou_thresh=.88,stability_score_thresh=.92,
    box_nms_thresh=.7,crop_n_layers=0,min_mask_region_area=100)
source51=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"/"train"
tilefile51=OUT/f"experiments_v22_full_roi_8m_{ROI_TARGET52}"/"full"/"prompt_manifest.csv"
assert tilefile51.exists(),f"Missing existing tile manifest {tilefile51}"
tiles51=pd.read_csv(tilefile51).drop_duplicates("tile_id").sort_values("tile_id")
with rasterio.open(RASTER) as src:
    crs51=src.crs;roi51=roi_v2.to_crs(crs51).geometry.iloc[ROI_TARGET52]
    ref51=reference_v2.to_crs(crs51)
    ref51=ref51[ref51.geometry.apply(roi51.covers)].reset_index(drop=True)
assert len(ref51)>0,"No fully contained manual crowns in selected ROI"
raw51=D51/"tile_masks";raw51.mkdir(exist_ok=True)
tile_features51=[]
for t in tiles51.itertuples():
    tid=int(t.tile_id)
    filename=source51/f"roi{ROI_TARGET52}_tile{tid:04d}.png"
    assert filename.exists(),f"Block30 tile missing: {filename}"
    cache=raw51/f"tile_{tid:04d}.npz"
    if cache.exists():
        with np.load(cache) as z:
            masks=z["masks"];scores=z["scores"]
    else:
        rgb=np.asarray(Image.open(filename).convert("RGB"))
        out=generator51.generate(rgb)
        masks=np.asarray([q["segmentation"].astype("uint8") for q in out],dtype="uint8") if out else np.empty((0,*rgb.shape[:2]),dtype="uint8")
        scores=np.asarray([q["predicted_iou"]*q["stability_score"] for q in out],dtype="float32")
        np.savez_compressed(cache,masks=masks,scores=scores)
    win=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
    with rasterio.open(RASTER) as ds:tx=ds.window_transform(win)
    for j,(m,score) in enumerate(zip(masks,scores)):
        if not m.any():continue
        polygons=[shape(g) for g,v in shapes(m,mask=m.astype(bool),transform=tx) if v==1]
        if not polygons:continue
        geom=make_valid(max(polygons,key=lambda p:p.area))
        if geom.is_empty or not geom.is_valid:continue
        if not 4<=geom.area<=500:continue
        geom=geom.intersection(roi51)
        if geom.is_empty or geom.area<4:continue
        if not roi51.covers(geom.representative_point()):continue
        tile_features51.append(dict(tile=tid,mask_id=j,score=float(score),geometry=geom))
    print(f"ROI {ROI_TARGET52} tile {tid}: {len(scores)} AMG proposals (cached)",flush=True)
candidates51=gpd.GeoDataFrame(tile_features51,geometry="geometry",crs=crs51)
print("Raw polygons after area/ROI filtering:",len(candidates51))
assert len(candidates51)>0,"No SAM automatic masks; inspect tile images or thresholds"
# Confidence-prioritized geographic NMS handles repeated crowns across overlapping tiles.
candidates51=candidates51.sort_values("score",ascending=False).reset_index(drop=True)
kept51=[];selected51=[]
for i,geom in enumerate(candidates51.geometry):
    if any(geom.intersection(p).area/max(geom.area+p.area-geom.intersection(p).area,1e-9)>.6
           for p in selected51 if geom.intersects(p)):continue
    kept51.append(i);selected51.append(geom)
crowns51=candidates51.iloc[kept51].reset_index(drop=True)
crowns51.to_file(D51/f"sam1_amg_roi{ROI_TARGET52}_crowns.gpkg",driver="GPKG")
print("SAM1 automatic masks after NMS:",len(crowns51))
# Exact same all-species one-to-one Hungarian polygon IoU protocol.
matrix51=np.zeros((len(ref51),len(crowns51)),dtype=np.float32)
for i,a in enumerate(ref51.geometry):
    for j,b in enumerate(crowns51.geometry):
        if not a.intersects(b):continue
        intersection=a.intersection(b).area
        matrix51[i,j]=intersection/max(a.area+b.area-intersection,1e-9)
rr,cc=linear_sum_assignment(matrix51,maximize=True)
paired51=matrix51[rr,cc]
results51=[]
for thr in [.3,.5,.7]:
    tp=int((paired51>=thr).sum())
    results51.append(dict(iou=thr,reference=len(ref51),predicted=len(crowns51),matched=tp,
                          precision=tp/max(1,len(crowns51)),recall=tp/len(ref51),
                          f1=2*tp/max(1,len(ref51)+len(crowns51))))
table51=pd.DataFrame(results51)
table51.to_csv(D51/f"roi{ROI_TARGET52}_sam1_amg_metrics.csv",index=False)
display(table51.round(3))
# Describe stand geometry without assigning age from spatial position.
with rasterio.open(RASTER) as _src52:
    roi_g52=roi_v2.to_crs(_src52.crs)
    crown_g52=reference_v2.to_crs(_src52.crs)
    profiles52=[]
    for idx52 in range(6):
        polygon52=roi_g52.geometry.iloc[idx52]
        crowns52=crown_g52[crown_g52.geometry.apply(polygon52.covers)]
        profiles52.append(dict(roi_id=idx52,easting_m=polygon52.centroid.x,
            northing_m=polygon52.centroid.y,roi_area_ha=polygon52.area/10000,
            complete_crowns=len(crowns52),
            median_crown_area_m2=crowns52.geometry.area.median() if len(crowns52) else np.nan,
            crown_density_per_ha=len(crowns52)/(polygon52.area/10000)))
profile_df52=pd.DataFrame(profiles52).sort_values("easting_m")
profile_df52.to_csv(D51/"six_roi_reference_profiles.csv",index=False)
print("ROI geography / reference structure, westernmost first:")
display(profile_df52.round(2))
young52=pd.read_csv(OUT/"experiments_v51_promptless_sam1"/"roi4_sam1_amg_metrics.csv")
young52["roi_id"]=4
table51["roi_id"]=ROI_TARGET52
comparison52=pd.concat([table51,young52],ignore_index=True)
comparison52.to_csv(D51/"mature_vs_young_sam1_metrics.csv",index=False)
print("Original SAM1 AMG, identical settings and IoU evaluation:")
display(comparison52.round(3))
print("The selected ROI was used for previous MODEL TRAINING. SAM1 is zero-shot here, but this is")
print("a stand-structure diagnostic, NOT an independent train/test comparison or a Luki replication.")
print("ROI5 untouched. BLOCK 52 PASS:",D51)


## 53 — Closest practical replication of Luki standalone SAM1

Use the published [Plumacker et al. (2026)](https://doi.org/10.1002/ece3.74260) paper and [authors' released code](https://github.com/APlumacker/Detectree2SAM) rather than guessing. Reproduce original **SAM ViT-H** with the automatic generator (`points_per_side=32`, other generator defaults), **40m core plus 60m buffer on all sides**, retain mask polygons centered within the core, allow the three largest polygons per mask, and apply published 2–2500m² limits. Compare young ROI4 and western training ROI1 against their Kristof crown reference. The 8cm imagery, specific polygon clipping, and metric matching are Baego adaptations, not an exact reproduction of Luki; original SAM1 is used, not SAM3. Cache per core on Drive; roughly 2.6GB initial weight download and potentially intensive GPU processing. **ROI5 untouched.**


In [ ]:
# BLOCK 53 — Closest practical Luki SAM-only replication (Plumacker et al. 2026)
# Author code: APlumacker/Detectree2SAM, SAM_vignette; native SAM ViT-H, AMG points_per_side=32.
# 40m core + 60m context each side; retain only polygon centroids INSIDE core.
# Compare existing young ROI4 to provisional mature ROI1. Neither SAM3 nor Detectree2 is used.
import os,sys,subprocess,urllib.request,json,math,gc
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
from rasterio.windows import from_bounds,Window
from rasterio.features import shapes
from shapely.geometry import shape,box,Point,mapping
from shapely.validation import make_valid
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D53=OUT/"experiments_v53_luki_sam_only_replication";D53.mkdir(parents=True,exist_ok=True)
assert "reference_v2" in globals() and "roi_v2" in globals()
assert torch.cuda.is_available(),"Please use GPU runtime"
try:
    from segment_anything import SamAutomaticMaskGenerator,sam_model_registry
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install",
                    "git+https://github.com/facebookresearch/segment-anything.git"],check=True)
    from segment_anything import SamAutomaticMaskGenerator,sam_model_registry
# Set to (1,) for small pilot, or (1,4) for both stands (default).
ROIS53=(1,4)
# Paper/code setup: 40m core tiling with 60m additional context on each side.
CORE_M53=40.;BUFFER_M53=60.
SAM_MODEL53="vit_h"
ck53=Path("/content/sam_vit_h_4b8939.pth")
if not ck53.exists() or ck53.stat().st_size<2_000_000_000:
    print("Downloading ORIGINAL SAM ViT-H checkpoint (~2.6 GB) once to Colab local disk.")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",ck53)
assert ck53.stat().st_size>2_000_000_000
sam53=sam_model_registry[SAM_MODEL53](checkpoint=str(ck53)).to("cuda").eval()
# Luki repo: points_per_side=32, ALL other params commented = official SAM AMG defaults.
generator53=SamAutomaticMaskGenerator(model=sam53,points_per_side=32,
    points_per_batch=32) # batch reduced ONLY to fit L4; predictions unchanged.
# True reference details: Luki code uses up to 3 geometries per mask ("all_poly=False"),
# centroid-in-core, area range 2–2500m2; no ROI4-derived tuning.
MIN_AREA53=2.;MAX_AREA53=2500.
with rasterio.open(RASTER) as rgb:
    crs53=rgb.crs;ref53=reference_v2.to_crs(crs53)
    roi53=roi_v2.to_crs(crs53)
    extent53=box(*rgb.bounds)
    gs53=abs(rgb.res[0])
print("SAM",SAM_MODEL53,"| GSD",gs53,"| core",CORE_M53,"m | buffer",BUFFER_M53,"m")
print("IMPORTANT: imagery input is native Baego 8cm, not Luki's 5cm.")
results53=[];profiles53=[]
for ROI_ID53 in ROIS53:
    polygon53=roi53.geometry.iloc[ROI_ID53]
    refs53=ref53[ref53.geometry.apply(polygon53.covers)].reset_index(drop=True)
    x0,y0,x1,y1=polygon53.bounds
    # Core grid anchored to origin of this ROI, stride 40m, overlap/context 60m.
    centers53=[]
    for gx in np.arange(x0,x1,CORE_M53):
        for gy in np.arange(y0,y1,CORE_M53):
            core=box(gx,gy,gx+CORE_M53,gy+CORE_M53)
            if polygon53.intersects(core):
                centers53.append((len(centers53),core))
    print(f"ROI {ROI_ID53}: {len(refs53)} references | {len(centers53)} 40m cores")
    cache53=D53/f"roi{ROI_ID53}_tiles";cache53.mkdir(exist_ok=True)
    polys53=[]
    for tid,core in centers53:
        tile_cache=cache53/f"core_{tid:03d}.geojson"
        if tile_cache.exists():
            feats=json.loads(tile_cache.read_text())["features"]
            recs=[dict(core_id=tid,geometry=shape(x["geometry"]),score=x["properties"].get("score",0.))
                  for x in feats]
            polys53.extend(recs)
            print(f"ROI {ROI_ID53} core {tid}: {len(recs)} proposals (cached)",flush=True)
            continue
        bx0,by0,bx1,by1=core.bounds
        context=box(bx0-BUFFER_M53,by0-BUFFER_M53,bx1+BUFFER_M53,by1+BUFFER_M53)
        with rasterio.open(RASTER) as src:
            win=from_bounds(*context.bounds,transform=src.transform).round_offsets().round_lengths()
            # Luki skips frames with >25% black or pure white input. Use boundless
            # context to reproduce mosaics at borders; no ROI polygon masking.
            im=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            tx=src.window_transform(win)
        if im.dtype!=np.uint8:
            # If source is not uint8, conservative clipping; input audit needed.
            im=np.clip(im,0,255).astype(np.uint8)
        if (np.all(im==0,axis=2).mean()>.25 or np.all(im==255,axis=2).mean()>.25):
            tile_cache.write_text(json.dumps({"type":"FeatureCollection","features":[]}))
            print(f"ROI {ROI_ID53} core {tid}: skipped (>25% black/white pixels)",flush=True)
            continue
        pred=generator53.generate(im)
        sub=[]
        for item in pred:
            mask=item["segmentation"].astype(np.uint8)
            if not mask.any():continue
            # Up to three largest connected polygons per SAM mask, matching Luki all_poly=False.
            geomlist=[make_valid(shape(g)) for g,val in shapes(mask,mask=mask.astype(bool),transform=tx) if val==1]
            geomlist=sorted((g for g in geomlist if not g.is_empty),key=lambda g:g.area,reverse=True)[:3]
            for g in geomlist:
                if not g.is_valid:g=make_valid(g)
                if not (MIN_AREA53<=g.area<=MAX_AREA53):continue
                if not core.contains(g.centroid):continue
                if not polygon53.intersects(g):continue
                clipped=g.intersection(polygon53)
                if clipped.is_empty or clipped.area<MIN_AREA53:continue
                sub.append(dict(core_id=tid,score=float(item["predicted_iou"]),geometry=clipped))
        polys53.extend(sub)
        tile_cache.write_text(json.dumps(dict(type="FeatureCollection",features=[
           dict(type="Feature",geometry=mapping(a["geometry"]),
                properties=dict(score=a["score"])) for a in sub])))
        print(f"ROI {ROI_ID53} core {tid}: {len(sub)} SAM polygons (cached)",flush=True)
        del pred,im;gc.collect()
    print(f"ROI {ROI_ID53}: total SAM predictions before geographical overlap cleaning: {len(polys53)}")
    if not polys53:
        raise RuntimeError("Zero Luki-style polygons; inspect tiling/mosaic/crown filters")
    frame=gpd.GeoDataFrame(polys53,geometry="geometry",crs=crs53)
    # Luki keeps centroid in each non-overlapping 40m core; no confidence NMS in SAM
    # path. Report this default as author-style and diagnostic NMS separately.
    frame.to_file(D53/f"roi{ROI_ID53}_luki_style_raw.gpkg",driver="GPKG")
    for method in ("author_style","nms_060"):
        sub=frame if method=="author_style" else frame.sort_values("score",ascending=False).copy()
        if method=="nms_060":
            selected=[];inds=[]
            for idx,g in zip(sub.index,sub.geometry):
                if any(g.intersects(h) and g.intersection(h).area/max(g.area+h.area-g.intersection(h).area,1e-9)>.6
                       for h in selected):continue
                selected.append(g);inds.append(idx)
            sub=sub.loc[inds]
        sub=sub.reset_index(drop=True)
        sub.to_file(D53/f"roi{ROI_ID53}_{method}.gpkg",driver="GPKG")
        mat=np.zeros((len(refs53),len(sub)),dtype=np.float32)
        for i,a in enumerate(refs53.geometry):
            for j,b in enumerate(sub.geometry):
                if a.intersects(b):
                    k=a.intersection(b).area
                    mat[i,j]=k/max(a.area+b.area-k,1e-9)
        rr,cc=linear_sum_assignment(mat,maximize=True)
        vals=mat[rr,cc]
        for threshold in (.3,.5,.7):
            tp=int((vals>=threshold).sum())
            results53.append(dict(roi=ROI_ID53,postprocess=method,iou_threshold=threshold,
              reference=len(refs53),predicted=len(sub),matched=tp,precision=tp/max(len(sub),1),
              recall=tp/max(len(refs53),1),f1=2*tp/max(len(refs53)+len(sub),1)))
metrics53=pd.DataFrame(results53)
metrics53.to_csv(D53/"luki_style_roi_comparison.csv",index=False)
print("SAM1 ViT-H, Luki-style 32 grid / 40+60m context — matched crown IoU ≥ 0.5:")
display(metrics53[metrics53.iou_threshold==.5].round(3))
print("For reference: previous simplified SAM1 ViT-B F1 ROI1=.176, ROI4=.164.")
print("Luki paper reports SAM crown F1≈.670 but its forest and imagery differ.")
print("Not exact numerical reproduction: Baego resolution/ROI limits differ; the standalone")
print("author SAM-only path and exported post-processing were reconstructed, not executed as Docker.")
print("BLOCK 53 PASS:",D53,"| ROI5 untouched.")


## 54 — High-precision evaluation: Luki-style SAM1 versus Nadine crowns

**Read-only evaluation** of existing Block 53 SAM1 ViT-H crown polygons for ROIs 1 and 4 against `Crown_Nadine.shp` (sparse, more precise polygons). Repairs invalid Nadine geometries, counts fully contained crowns and boundary-crossing exclusions, performs one-to-one polygon-IoU matching, reports annotated-crown recall at 0.3/0.5/0.7, conditional matched IoU and area ratios, exports per-crown CSVs and RGB overlays. **No precision/F1 from sparse labels**: unmatched SAM predictions may be real but unannotated crowns. No SAM rerun; ROI5 untouched.


In [ ]:
# BLOCK 54 — Luki-style SAM1 against Nadine's precise, SPARSE crown outlines
# Evaluation only. No SAM inference, no training, no change to ROI5.
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from pathlib import Path
from shapely.validation import make_valid
from shapely.geometry import Polygon,MultiPolygon,GeometryCollection,box
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D54=OUT/"experiments_v54_nadine_high_precision_eval"
D54.mkdir(parents=True,exist_ok=True)
P53=OUT/"experiments_v53_luki_sam_only_replication"
N54=AD/"Crown_Nadine.shp"
assert N54.exists(),f"Missing {N54}"
assert "roi_v2" in globals(),"Run Block 14 first"
with rasterio.open(RASTER) as src:
    crs54=src.crs
roi54=roi_v2.to_crs(crs54)
nad54=gpd.read_file(N54)
if nad54.crs is None:
    print("WARNING: Nadine shapefile lacks CRS; assuming imagery CRS",crs54)
    nad54=nad54.set_crs(crs54)
else:
    nad54=nad54.to_crs(crs54)
nad54=nad54[nad54.geometry.notna() & ~nad54.geometry.is_empty].copy()
invalid54=int((~nad54.geometry.is_valid).sum())
nad54["geometry"]=nad54.geometry.map(make_valid)
# Preserve all polygon parts; discard non-polygon components resulting from repairs.
def polygonal54(g):
    if g.is_empty:return Polygon()
    if g.geom_type in ("Polygon","MultiPolygon"):return g
    parts=[x for x in getattr(g,"geoms",[]) if x.geom_type in ("Polygon","MultiPolygon")]
    if not parts:return Polygon()
    from shapely.ops import unary_union
    return unary_union(parts)
nad54["geometry"]=nad54.geometry.map(polygonal54)
nad54=nad54[~nad54.geometry.is_empty & (nad54.geometry.area>0)].reset_index(drop=True)
nad54["nadine_id"]=np.arange(len(nad54))
print(f"Nadine crowns: {len(nad54)} usable; {invalid54} invalid repaired before filtering")
summaries54=[];details54=[];coverage54=[]
for rid54 in (1,4):
    region54=roi54.geometry.iloc[rid54]
    fully54=nad54[nad54.geometry.apply(region54.covers)].copy().reset_index(drop=True)
    intersects54=int(nad54.geometry.intersects(region54).sum())
    coverage54.append(dict(roi=rid54,nadine_fully_contained=len(fully54),
                           nadine_intersecting=intersects54,
                           excluded_boundary_crossing=intersects54-len(fully54)))
    print(f"ROI {rid54}: Nadine fully contained={len(fully54)}, intersects={intersects54}")
    if len(fully54)==0:
        print("  No suitable Nadine crowns: skipping metrics, not reporting zero recall")
        continue
    for label54 in ("author_style","nms_060"):
        path54=P53/f"roi{rid54}_{label54}.gpkg"
        assert path54.exists(),f"Please finish Block53 first: {path54}"
        pred54=gpd.read_file(path54).to_crs(crs54)
        pred54=pred54[pred54.geometry.notna()&~pred54.geometry.is_empty].copy()
        pred54["geometry"]=pred54.geometry.map(lambda g:polygonal54(make_valid(g)))
        pred54=pred54[~pred54.geometry.is_empty].reset_index(drop=True)
        # Match only the precise positive-reference crowns to predictions.
        # All other candidates are UNLABELLED, NOT automatically false positives.
        iou54=np.zeros((len(fully54),len(pred54)),dtype=np.float32)
        for i,a in enumerate(fully54.geometry):
            for j,b in enumerate(pred54.geometry):
                if not a.intersects(b):continue
                ab=a.intersection(b).area
                iou54[i,j]=ab/max(a.area+b.area-ab,1e-9)
        rows54,cols54=linear_sum_assignment(iou54,maximize=True)
        assigned54={int(i):(int(j),float(iou54[i,j])) for i,j in zip(rows54,cols54)}
        for i,refrow in fully54.iterrows():
            j,matchiou=assigned54.get(i,(-1,0.))
            if matchiou<=0:j=-1
            b=pred54.geometry.iloc[j] if j>=0 else None
            area_ratio=(b.area/refrow.geometry.area) if b is not None else np.nan
            details54.append(dict(roi=rid54,processing=label54,
              nadine_id=int(refrow.nadine_id),nadine_area_m2=refrow.geometry.area,
              matched_prediction_index=j,matched_iou=matchiou,
              pred_area_m2=(b.area if b is not None else np.nan),area_ratio=area_ratio))
        scores54=np.array([assigned54.get(i,(-1,0.))[1] for i in range(len(fully54))])
        for threshold54 in (.3,.5,.7):
            nmatch54=int((scores54>=threshold54).sum())
            positive= np.array([d["matched_iou"] for d in details54 if
                            d["roi"]==rid54 and d["processing"]==label54 and
                            d["matched_iou"]>=threshold54])
            area_ratios=np.array([d["area_ratio"] for d in details54 if
                            d["roi"]==rid54 and d["processing"]==label54 and
                            d["matched_iou"]>=threshold54],dtype=float)
            summaries54.append(dict(roi=rid54,processing=label54,iou_threshold=threshold54,
                nadine_reference=len(fully54),total_sam_predictions=len(pred54),
                annotated_crowns_recovered=nmatch54,
                annotated_crown_recall=nmatch54/len(fully54),
                median_iou_recovered=float(np.median(positive)) if len(positive) else np.nan,
                median_predicted_to_nadine_area_ratio=float(np.median(area_ratios))
                    if len(area_ratios) else np.nan))
coverage54=pd.DataFrame(coverage54);details54=pd.DataFrame(details54);summary54=pd.DataFrame(summaries54)
coverage54.to_csv(D54/"nadine_coverage_by_roi.csv",index=False)
details54.to_csv(D54/"nadine_per_crown_matches.csv",index=False)
summary54.to_csv(D54/"nadine_recall_and_boundary_quality.csv",index=False)
print("\nNadine spatial coverage (numbers may be small):")
display(coverage54)
if len(summary54):
    print("\nSparse-label recall and conditional boundary quality:")
    display(summary54[summary54.iou_threshold==.5].round(3))
    # Zero entries among all annotated crowns measure missed detections; median IoU
    # only among recovered crowns measures CONDITIONAL boundary quality.
    print("IMPORTANT: no precision/F1 calculated from partial Nadine annotations.")
    print("Area ratios are conditional on successful IoU matches, not overall area bias.")
    # Optional visual overlay for two best available matched Nadine crowns per ROI.
    from rasterio.windows import from_bounds
    for rid54 in (1,4):
        sub54=details54[(details54.roi==rid54)&(details54.processing=="author_style")]
        sub54=sub54.sort_values("matched_iou",ascending=False)
        sub54=sub54[sub54.matched_prediction_index>=0].head(2)
        if sub54.empty:continue
        preds=gpd.read_file(P53/f"roi{rid54}_author_style.gpkg").to_crs(crs54)
        fig,axes=plt.subplots(1,len(sub54),figsize=(6*len(sub54),6))
        axes=np.atleast_1d(axes)
        for ax,hit in zip(axes,sub54.itertuples()):
            reference=nad54.loc[nad54.nadine_id==hit.nadine_id].geometry.iloc[0]
            proposed=preds.geometry.iloc[int(hit.matched_prediction_index)]
            extent=box(*reference.union(proposed).buffer(3).bounds)
            with rasterio.open(RASTER) as ds:
                window=from_bounds(*extent.bounds,transform=ds.transform).round_offsets().round_lengths()
                img=ds.read([1,2,3],window=window,boundless=True,fill_value=0)
                bounds=rasterio.windows.bounds(window,ds.transform)
            rgb=img.transpose(1,2,0)
            if rgb.dtype!=np.uint8:rgb=np.clip(rgb,0,255).astype("uint8")
            ax.imshow(rgb,extent=[bounds[0],bounds[2],bounds[1],bounds[3]])
            gpd.GeoSeries([reference],crs=crs54).boundary.plot(ax=ax,color="lime",linewidth=2)
            gpd.GeoSeries([proposed],crs=crs54).boundary.plot(ax=ax,color="magenta",linewidth=2)
            ax.set_title(f"ROI{rid54} Nadine {hit.nadine_id} | IoU {hit.matched_iou:.2f}")
            ax.set_axis_off()
        plt.tight_layout()
        plt.savefig(D54/f"roi{rid54}_nadine_matched_examples.png",dpi=160,bbox_inches="tight")
        plt.show()
print("BLOCK 54 PASS:",D54)
print("Interpretation: this is validation against SPARSE Nadine crowns, not full ROI precision.")


## 55 — Nadine versus Kristof: same-crown reference diagnostic

**Read-only** comparison of Nadine and Kristof reference polygons in ROIs 1 and 4 and saved Luki-style SAM1 ViT-H masks. Spatially associate Nadine↔Kristof independently of SAM with one-to-one maximum IoU (minimum 0.10), quantify reference disagreement and area ratio, and evaluate the *same actual SAM mask* against both references. Export per-crown and aggregate CSV files plus RGB overlays (Nadine green, Kristof cyan, SAM magenta). Nadine is sparse, so **do not infer whole-ROI precision or F1**. Reference alignment is exploratory; no claim of systematic annotation bias without representative sampling. ROI5 untouched.


In [ ]:
# BLOCK 55 — Same-crown triangulation: Nadine ↔ Kristof ↔ automatic SAM1
# READ-ONLY; matched reference identities and same SAM detections.
import numpy as np,pandas as pd,geopandas as gpd,rasterio,matplotlib.pyplot as plt
from pathlib import Path
from shapely.validation import make_valid
from shapely.geometry import Polygon,box
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D55=OUT/"experiments_v55_reference_triage";D55.mkdir(parents=True,exist_ok=True)
P53=OUT/"experiments_v53_luki_sam_only_replication"
assert "reference_v2" in globals() and "roi_v2" in globals()
with rasterio.open(RASTER) as ds:crs55=ds.crs
def clean55(g):
    g=make_valid(g)
    if g.geom_type in ("Polygon","MultiPolygon"):return g
    parts=[p for p in getattr(g,"geoms",[]) if p.geom_type in ("Polygon","MultiPolygon")]
    return unary_union(parts) if parts else Polygon()
def iou55(a,b):
    if a.is_empty or b.is_empty or not a.intersects(b):return 0.
    x=a.intersection(b).area
    return float(x/max(a.area+b.area-x,1e-9))
roi55=roi_v2.to_crs(crs55)
nad55=gpd.read_file(AD/"Crown_Nadine.shp")
nad55=nad55.set_crs(crs55) if nad55.crs is None else nad55.to_crs(crs55)
nad55=nad55[nad55.geometry.notna()&~nad55.geometry.is_empty].copy()
nad55["geometry"]=nad55.geometry.map(clean55)
nad55=nad55[~nad55.geometry.is_empty].reset_index(drop=True)
nad55["nid"]=np.arange(len(nad55))
kri55=reference_v2.to_crs(crs55).copy().reset_index(drop=True)
kri55["kid"]=np.arange(len(kri55))
output55=[];counts55=[]
for rid55 in (1,4):
    poly55=roi55.geometry.iloc[rid55]
    nd=nad55[nad55.geometry.apply(poly55.covers)].reset_index(drop=True)
    kr=kri55[kri55.geometry.apply(poly55.covers)].reset_index(drop=True)
    sam=gpd.read_file(P53/f"roi{rid55}_author_style.gpkg").to_crs(crs55).reset_index(drop=True)
    assert len(nd) and len(kr) and len(sam)
    # One-to-one Nadine-Kristof associations using maximum IoU, NO matching to SAM involved.
    nk=np.array([[iou55(a,b) for b in kr.geometry] for a in nd.geometry],dtype=np.float32)
    ni,ki=linear_sum_assignment(nk,maximize=True)
    match55={int(a):(int(b),float(nk[a,b])) for a,b in zip(ni,ki) if nk[a,b]>=.10}
    # Crown-to-SAM matches are independently one-to-one within each reference set.
    def assignment55(ref):
        m=np.array([[iou55(a,b) for b in sam.geometry] for a in ref.geometry],dtype=np.float32)
        ii,jj=linear_sum_assignment(m,maximize=True)
        return {int(i):(int(j),float(m[i,j])) for i,j in zip(ii,jj) if m[i,j]>0}
    nmap=assignment55(nd);kmap=assignment55(kr)
    for i,n in nd.iterrows():
        k,nk_iou=match55.get(i,(-1,0.))
        sn,sn_iou=nmap.get(i,(-1,0.))
        sk,sk_iou=kmap.get(k,(-1,0.)) if k>=0 else (-1,0.)
        # Compare SAME SAM polygon to both references, not two independently selected winners.
        same_sam_k_iou=iou55(kr.geometry.iloc[k],sam.geometry.iloc[sn]) if k>=0 and sn>=0 else np.nan
        same_sam_n_iou=iou55(n.geometry,sam.geometry.iloc[sk]) if sk>=0 else np.nan
        output55.append(dict(roi=rid55,nadine_id=int(n.nid),
            nadine_area_m2=n.geometry.area,kristof_id=int(kr.iloc[k].kid) if k>=0 else -1,
            ref_iou=nk_iou,ref_area_ratio=(kr.geometry.iloc[k].area/n.geometry.area) if k>=0 else np.nan,
            sam_id_for_nadine=sn,sam_iou_nadine=sn_iou,sam_id_for_kristof=sk,
            sam_iou_kristof_independent=sk_iou,
            same_sam_iou_kristof=same_sam_k_iou,
            same_kristof_sam_iou_nadine=same_sam_n_iou,
            same_sam_identity=(sn>=0 and sn==sk),
            nadine_recovered_05=sn_iou>=.5,
            kristof_recovered_05=(sk_iou>=.5) if k>=0 else False))
    counts55.append(dict(roi=rid55,nadine_crowns=len(nd),kristof_crowns=len(kr),
                          sam_predictions=len(sam),nadine_kristof_associations=len(match55),
                          ref_agreement_05=sum(v[1]>=.5 for v in match55.values())))
result55=pd.DataFrame(output55);counts55=pd.DataFrame(counts55)
result55.to_csv(D55/"per_nadine_crown_same_object_comparison.csv",index=False)
counts55.to_csv(D55/"matching_coverage.csv",index=False)
print("Reference correspondence:")
display(counts55)
print("Cross-reference diagnostic (Nadine crowns are sparse and potentially selected):")
summary55=[]
for rid,grp in result55.groupby("roi"):
    paired=grp[grp.kristof_id>=0]
    strong=paired[paired.ref_iou>=.5]
    both=paired[paired.sam_id_for_nadine>=0]
    summary55.append(dict(roi=rid,nadine=len(grp),ref_correspondence=len(paired),
         ref_agreement_05=len(strong),
         nadine_sam_recall_05=grp.nadine_recovered_05.mean(),
         reference_iou_median=paired.ref_iou.median(),
         reference_area_ratio_median=paired.ref_area_ratio.median(),
         paired_same_sam_nadine_iou_median=both.sam_iou_nadine.median(),
         paired_same_sam_kristof_iou_median=both.same_sam_iou_kristof.median(),
         same_sam_id_fraction=paired.same_sam_identity.mean()))
summary55=pd.DataFrame(summary55)
summary55.to_csv(D55/"same_crown_reference_summary.csv",index=False)
display(summary55.round(3))
# Inspection: largest discrepancy where SAM fits Nadine but not Kristof.
from rasterio.windows import from_bounds
for rid in (1,4):
    x=result55[(result55.roi==rid)&(result55.kristof_id>=0)&
               (result55.sam_id_for_nadine>=0)].copy()
    if x.empty:continue
    x["iou_gap"]=x.sam_iou_nadine-x.same_sam_iou_kristof
    x=x.sort_values("iou_gap",ascending=False).head(3)
    kr=kri55;sam=gpd.read_file(P53/f"roi{rid}_author_style.gpkg").to_crs(crs55)
    fig,axes=plt.subplots(1,len(x),figsize=(5.3*len(x),5))
    for ax,row in zip(np.atleast_1d(axes),x.itertuples()):
        ng=nad55.loc[nad55.nid==row.nadine_id].geometry.iloc[0]
        kg=kr.loc[kr.kid==row.kristof_id].geometry.iloc[0]
        sg=sam.geometry.iloc[int(row.sam_id_for_nadine)]
        bounds=unary_union([ng,kg,sg]).buffer(3).bounds
        with rasterio.open(RASTER) as ds:
            win=from_bounds(*bounds,transform=ds.transform).round_offsets().round_lengths()
            img=ds.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            bb=rasterio.windows.bounds(win,ds.transform)
        if img.dtype!=np.uint8:img=np.clip(img,0,255).astype("uint8")
        ax.imshow(img,extent=[bb[0],bb[2],bb[1],bb[3]])
        for g,color in [(ng,"lime"),(kg,"cyan"),(sg,"magenta")]:
            gpd.GeoSeries([g],crs=crs55).boundary.plot(ax=ax,color=color,linewidth=2)
        ax.set_title(f"ROI{rid} | Nadine IoU {row.sam_iou_nadine:.2f}\nSame SAM vs Kristof {row.same_sam_iou_kristof:.2f}")
        ax.axis("off")
    plt.tight_layout();plt.savefig(D55/f"roi{rid}_reference_discrepancies.png",dpi=160,bbox_inches="tight");plt.show()
print("Green = Nadine; cyan = Kristof; magenta = SAME SAM prediction")
print("Selection/annotation sparseness not resolved by this analysis; ROI5 untouched.")
print("BLOCK 55 PASS:",D55)


## 56 — Automatic SAM3 pilot: real mask-generation interface

Check the official [SAM3 Hugging Face model card](https://huggingface.co/facebook/sam3), which documents `pipeline("mask-generation", model="facebook/sam3")` with internal point-grid prompting. **This differs from SAM3's text-conditioned concept detection and from SAM1's automatic generator.** Run only one *geometrically selected* central image tile in each of ROI1 and ROI4 as an environment/performance pilot, not a full Luki-equivalent benchmark. The code uses existing ROI tile manifests and genuine independent automatic proposals, caches polygons, and scores strict tile-contained Kristof and sparse Nadine crown overlaps. No SAM labels supplied as prompts, no training, ROI5 untouched. Hugging Face access to `facebook/sam3` may require accepting license terms/login. Use existing GPU L4, release earlier SAM1 model memory, and download only SAM3 weights. **A proper head-to-head comparison still requires the same imagery footprint, tiling, masks and post-processing; pilot scores are not directly comparable to whole-ROI Block53.**


In [ ]:
# BLOCK 56 — SAM3 TRUE automatic-mask mode (Hugging Face) pilot, ROI 1 and ROI 4
# Pilot only: one deterministic central RGB tile per ROI; no ground-truth-derived prompts.
# Reports tile-local all-crown F1 (Kristof) and SPARSE-label recall (Nadine).
import sys,subprocess,gc,json
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
from PIL import Image
from shapely.geometry import shape,Polygon,box
from shapely.validation import make_valid
from rasterio.features import shapes
from rasterio.windows import Window
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D56=OUT/"experiments_v56_sam3_amg_pilot";D56.mkdir(parents=True,exist_ok=True)
assert "roi_v2" in globals() and "reference_v2" in globals()
assert torch.cuda.is_available(),"Use a GPU runtime."
# Important: free the ViT-H SAM1 instance from Block53 before loading SAM3.
for _name56 in ("generator53","sam53","generator51","sam51"):
    if _name56 in globals():
        del globals()[_name56]
gc.collect();torch.cuda.empty_cache()
try:
    import transformers
    assert hasattr(transformers,"Sam3TrackerModel")
except (ImportError,AssertionError):
    subprocess.run([sys.executable,"-m","pip","-q","install","-U",
                    "transformers>=5.0.0","accelerate","huggingface_hub"],check=True)
    import importlib
    importlib.invalidate_caches()
# Official HF model card: pipeline("mask-generation",model="facebook/sam3").
try:
    from transformers import pipeline
    pipe56=pipeline("mask-generation",model="facebook/sam3",device=0)
except Exception as e:
    raise RuntimeError("Could not initialize official HF SAM3 automatic mask pipeline. "
        "This often means missing access to gated facebook/sam3 weights. "
        "Visit https://huggingface.co/facebook/sam3 , accept terms, "
        "then authenticate in Colab via huggingface_hub.login(). "
        f"Original error: {type(e).__name__}: {e}") from e
with rasterio.open(RASTER) as ds56:
    crs56=ds56.crs
nad56=gpd.read_file(AD/"Crown_Nadine.shp")
nad56=nad56.set_crs(crs56) if nad56.crs is None else nad56.to_crs(crs56)
nad56=nad56[nad56.geometry.notna()&~nad56.geometry.is_empty].copy()
nad56["geometry"]=nad56.geometry.map(make_valid)
kri56=reference_v2.to_crs(crs56)
roi56=roi_v2.to_crs(crs56)
def iou56(a,b):
    if not a.intersects(b):return 0.
    s=a.intersection(b).area
    return float(s/max(a.area+b.area-s,1e-9))
def scores56(ref,preds,limit):
    mat=np.zeros((len(ref),len(preds)),dtype="float32")
    for i,a in enumerate(ref):
        for j,b in enumerate(preds):
            if a.intersects(b):mat[i,j]=iou56(a,b)
    if mat.size:
        ii,jj=linear_sum_assignment(mat,maximize=True)
        matched=mat[ii,jj]
    else:matched=np.array([],dtype=float)
    tp=int((matched>=limit).sum())
    return tp,tp/max(len(ref),1),2*tp/max(len(ref)+len(preds),1)
rows56=[]
for rid56 in (1,4):
    manifest56=OUT/f"experiments_v22_full_roi_8m_{rid56}"/"full"/"prompt_manifest.csv"
    assert manifest56.exists(),f"Missing tile manifest {manifest56}"
    tile56=pd.read_csv(manifest56).drop_duplicates("tile_id")
    # Deterministic tile: geometric centre of ROI (no crown annotation selection).
    ctr=roi56.geometry.iloc[rid56].centroid
    with rasterio.open(RASTER) as ds:
        choice=None;mind=float("inf")
        for rr in tile56.itertuples():
            w=Window(int(rr.col_off),int(rr.row_off),int(rr.width),int(rr.height))
            b=rasterio.windows.bounds(w,ds.transform)
            d=(ctr.x-(b[0]+b[2])/2)**2+(ctr.y-(b[1]+b[3])/2)**2
            if d<mind:mind=d;choice=(rr,w,b)
        rr,w,b=choice
        image56=ds.read([1,2,3],window=w).transpose(1,2,0)
        transform56=ds.window_transform(w)
    if image56.dtype!=np.uint8:
        image56=np.clip(image56,0,255).astype("uint8")
    image_pil56=Image.fromarray(image56,"RGB")
    tile_geom56=box(*b)
    # Fully contained reference crowns only — consistent with earlier strict protocol.
    kref=kri56[kri56.geometry.apply(tile_geom56.covers)]
    nref=nad56[nad56.geometry.apply(tile_geom56.covers)]
    print(f"ROI{rid56} central tile {rr.tile_id}: {image_pil56.size}, Kristof={len(kref)}, Nadine={len(nref)}",flush=True)
    output56=D56/f"roi{rid56}_central_tile_{int(rr.tile_id)}.gpkg"
    if output56.exists():
        preds56=gpd.read_file(output56).to_crs(crs56)
    else:
        try:
            result56=pipe56(image_pil56,points_per_batch=32)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            result56=pipe56(image_pil56,points_per_batch=8)
        masks56=result56["masks"]
        qualities56=result56.get("scores",[])
        records56=[]
        for ix,m in enumerate(masks56):
            if isinstance(m,Image.Image):m=np.asarray(m)
            elif hasattr(m,"detach"):m=m.detach().cpu().numpy()
            m=np.asarray(m).squeeze().astype(bool)
            if m.shape!=image56.shape[:2]:
                raise RuntimeError(f"Mask shape {m.shape} != input image shape {image56.shape[:2]}")
            if not m.any():continue
            parts=[make_valid(shape(g)) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform56) if v==1]
            for geom in parts:
                if geom.geom_type!="Polygon" or not 2<=geom.area<=2500:continue
                if not tile_geom56.covers(geom.centroid):continue
                q=float(qualities56[ix]) if ix<len(qualities56) else 0.
                records56.append(dict(mask_id=ix,quality=q,geometry=geom))
        preds56=gpd.GeoDataFrame(records56,geometry="geometry",crs=crs56) if records56 else gpd.GeoDataFrame(
            {"mask_id":[],"quality":[]},geometry=gpd.GeoSeries([],crs=crs56),crs=crs56)
        if len(preds56):preds56.to_file(output56,driver="GPKG")
    for t in (.3,.5,.7):
        tp_k,rec_k,f1_k=scores56(list(kref.geometry),list(preds56.geometry),t)
        tp_n,rec_n,_=scores56(list(nref.geometry),list(preds56.geometry),t)
        rows56.append(dict(roi=rid56,tile_id=int(rr.tile_id),iou=t,
                           kristof_crowns=len(kref),nadine_crowns=len(nref),
                           sam3_masks=len(preds56),kristof_matched=tp_k,
                           kristof_recall=rec_k,kristof_f1=f1_k,
                           nadine_matched=tp_n,nadine_annotated_recall=rec_n if len(nref) else np.nan))
results56=pd.DataFrame(rows56)
results56.to_csv(D56/"sam3_pilot_metrics.csv",index=False)
display(results56[results56.iou==.5].round(3))
print("Caution: tile-local exploratory scores, not full ROI and not SAM1-comparable yet.")
print("No crown reference supplied as prompt; no ROI5 processing. BLOCK 56 PASS:",D56)


## 57 — SAM3 automatic-mask diagnostic before scaling up

On the **same central tiles** as Block56, inspect **raw Hugging Face SAM3 mask-generation outputs before** applying crown area/geometry filters. Re-run original pipeline settings versus a more permissive 32×32-grid configuration, log supported/unsupported kwargs, compare raw masks, retained polygons and matched reference crowns, and save RGB mask overlays and tables. This is a **tile-level diagnostic**; no whole-ROI comparison or large buffered 160 m SAM3 imagery yet. Nadine annotations are sparse; do not interpret unmatched predictions as false positives. No ROI5 access.


In [ ]:
# BLOCK 57 — Diagnose SAM3 AMG raw masks and filtering, exactly Block56 tiles
# Two modes: pipeline default and more permissive generator thresholds.
# Exploratory only; saves audit tables and visual overlays; ROI5 untouched.
import gc,json,inspect,traceback
from pathlib import Path
import numpy as np,pandas as pd,rasterio,geopandas as gpd,torch,matplotlib.pyplot as plt
from PIL import Image
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape,box
from shapely.validation import make_valid
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D57=OUT/"experiments_v57_sam3_amg_diagnostic";D57.mkdir(parents=True,exist_ok=True)
assert "roi_v2" in globals() and "reference_v2" in globals()
# Reuse loaded Hugging Face SAM3 pipeline from Block56, avoiding a second 3.4GB download.
if "pipe56" not in globals():
    from transformers import pipeline
    pipe56=pipeline("mask-generation",model="facebook/sam3",device=0)
print("Pipeline class:",type(pipe56).__name__)
print("Pipeline call signature:",inspect.signature(pipe56.__call__))
print("Pipeline preprocess signature:",inspect.signature(pipe56.preprocess))
print("Pipeline forward signature:",inspect.signature(pipe56._forward))
RIDS57=(1,4)
# Baseline is the EXACT Block56 call; permissive modifies documented HF AMG
# keyword arguments. Log unsupported kwargs rather than silently claiming a test.
SETTINGS57={
 "original56":{"points_per_batch":32},
 "permissive":{"points_per_batch":16,"points_per_side":32,
               "pred_iou_thresh":0.70,"stability_score_thresh":0.80}
}
with rasterio.open(RASTER) as src:
    crs57=src.crs
roi57=roi_v2.to_crs(crs57)
ref57=reference_v2.to_crs(crs57)
n57=gpd.read_file(AD/"Crown_Nadine.shp")
n57=n57.set_crs(crs57) if n57.crs is None else n57.to_crs(crs57)
n57=n57[n57.geometry.notna() & ~n57.geometry.is_empty].copy()
n57["geometry"]=n57.geometry.map(make_valid)
def mask_array57(m,hw):
    if isinstance(m,Image.Image):a=np.asarray(m)
    elif hasattr(m,"detach"):a=m.detach().cpu().numpy()
    else:a=np.asarray(m)
    a=np.squeeze(a)
    if a.shape!=hw:raise ValueError(f"Unexpected SAM3 mask shape {a.shape} vs image {hw}")
    return a>0
def hungarian57(refs,preds,t):
    if not len(refs):return (0,np.nan,np.nan)
    mat=np.zeros((len(refs),len(preds)),dtype=np.float32)
    for i,a in enumerate(refs):
        for j,b in enumerate(preds):
            if a.intersects(b):
                x=a.intersection(b).area;mat[i,j]=x/max(a.area+b.area-x,1e-9)
    if mat.size:
        ii,jj=linear_sum_assignment(mat,maximize=True)
        matched=int((mat[ii,jj]>=t).sum())
    else:matched=0
    return matched,matched/len(refs),2*matched/max(len(refs)+len(preds),1)
all57=[];masks57=[];tiles57=[]
for rid in RIDS57:
    man=OUT/f"experiments_v22_full_roi_8m_{rid}"/"full"/"prompt_manifest.csv"
    tiletab=pd.read_csv(man).drop_duplicates("tile_id")
    center=roi57.geometry.iloc[rid].centroid
    with rasterio.open(RASTER) as src:
        options=[]
        for rr in tiletab.itertuples():
            w=Window(int(rr.col_off),int(rr.row_off),int(rr.width),int(rr.height))
            bb=rasterio.windows.bounds(w,src.transform)
            delta=(center.x-(bb[0]+bb[2])/2)**2+(center.y-(bb[1]+bb[3])/2)**2
            options.append((delta,rr,w,bb))
        _,rr,w,bb=min(options,key=lambda z:z[0])
        rgb=src.read([1,2,3],window=w).transpose(1,2,0)
        tx=src.window_transform(w)
    if rgb.dtype!=np.uint8:rgb=np.clip(rgb,0,255).astype(np.uint8)
    pil=Image.fromarray(rgb)
    tilebox=box(*bb)
    kr=ref57[ref57.geometry.apply(tilebox.covers)]
    nd=n57[n57.geometry.apply(tilebox.covers)]
    tiles57.append(dict(roi=rid,tile=int(rr.tile_id),shape=str(rgb.shape),
                        kristof_count=len(kr),nadine_count=len(nd)))
    print(f"\nROI {rid}, tile {rr.tile_id}: {rgb.shape}; Kristof {len(kr)}, Nadine {len(nd)}")
    for label,params in SETTINGS57.items():
        try:
            result=pipe56(pil,**params)
        except Exception as ex:
            if label=="permissive":
                print("Permissive parameter call FAILED (do not assume these parameters are supported):",repr(ex))
                all57.append(dict(roi=rid,setting=label,status=f"UNSUPPORTED: {ex}",raw_masks=np.nan))
                continue
            raise
        print(label,"result keys:",list(result.keys()))
        marr=result.get("masks",[])
        qs=result.get("scores",[])
        polys=[];rawstats=[]
        canvas=np.zeros(rgb.shape[:2],dtype=np.uint16)
        for k,raw in enumerate(marr):
            arr=mask_array57(raw,rgb.shape[:2])
            if arr.any():canvas[arr]=k+1
            px=int(arr.sum())
            q=float(qs[k]) if k<len(qs) else np.nan
            area=px*abs(tx.a*tx.e)
            parts=[make_valid(shape(g)) for g,v in shapes(arr.astype(np.uint8),mask=arr,transform=tx) if v==1] if px else []
            polygonparts=[g for g in parts if g.geom_type=="Polygon"]
            kept=[g for g in polygonparts if 2<=g.area<=2500 and tilebox.covers(g.centroid)]
            polys.extend(kept)
            rawstats.append(dict(roi=rid,setting=label,mask_id=k,raw_pixels=px,
                raw_area_m2=area,score=q,connected_parts=len(polygonparts),
                retained_parts=len(kept),reason=("empty" if not px else
                    "no_polygon" if not polygonparts else "area_or_centroid" if not kept else "retained")))
        masks57.extend(rawstats)
        mk,recallk,f1k=hungarian57(list(kr.geometry),polys,.5)
        mn,recalln,_=hungarian57(list(nd.geometry),polys,.5)
        all57.append(dict(roi=rid,setting=label,status="OK",tile=int(rr.tile_id),
             raw_masks=len(marr),retained_polygons=len(polys),
             kristof_reference=len(kr),kristof_matched=mk,kristof_f1=f1k,
             nadine_reference=len(nd),nadine_matched=mn,
             nadine_annotated_recall=recalln))
        fig,axs=plt.subplots(1,2,figsize=(12,5))
        axs[0].imshow(rgb);axs[0].set_title(f"ROI{rid} source RGB")
        axs[1].imshow(rgb);axs[1].imshow(np.ma.masked_where(canvas==0,canvas),alpha=.48,cmap="nipy_spectral",
            interpolation="nearest")
        axs[1].set_title(f"{label}: {len(marr)} raw masks / {len(polys)} retained polys")
        for ax in axs:ax.axis("off")
        plt.tight_layout()
        plt.savefig(D57/f"roi{rid}_{label}_raw_mask_overlay.png",dpi=170,bbox_inches="tight")
        plt.show()
        np.savez_compressed(D57/f"roi{rid}_{label}_raw_mask_ids.npz",mask_ids=canvas)
        del result,canvas;gc.collect()
report57=pd.DataFrame(all57);masktab57=pd.DataFrame(masks57)
report57.to_csv(D57/"sam3_generator_settings_comparison.csv",index=False)
masktab57.to_csv(D57/"sam3_raw_mask_diagnostics.csv",index=False)
pd.DataFrame(tiles57).to_csv(D57/"tile_reference_coverage.csv",index=False)
print("\nPilot diagnostic:")
display(report57)
if len(masktab57):
    print("SAM3 raw mask areas and scores by setting:")
    display(masktab57.groupby(["roi","setting"]).agg(
       count=("mask_id","count"),median_area_m2=("raw_area_m2","median"),
       median_score=("score","median"),retained_mask_count=("retained_parts","sum")))
print("If permissive setting is unsupported or also returns few masks, inspect overlay before full ROI.")
print("No buffered 160m inference attempted (GPU memory and high input-resolution costs).")
print("BLOCK 57 PASS:",D57,"| ROI5 untouched.")


## 58 — Corrected five-ROI Luki-context SAM1/SAM3 benchmark

ROI5 is excluded. Restore the exact Block53 40m output cores with 60m context buffers on each side (160m imagery). Reproduce Block53 SAM1 author-style F1 on ROIs 1 and 4 with a hard gate before extending to 0,2,3. Evaluate SAM3 with the same context footprint and its previously selected exploratory thresholds. Whole-ROI F1 uses exhaustive Kristof annotations; sparse Nadine annotations provide recall/conditional IoU only. The previous reduced-context Block58 outputs remain separate. This block writes to experiments_v58b_luki_context_sam1_sam3 and caches tiles for resuming.


In [ ]:
# BLOCK 58 — Full pre-test 5-ROI SAM1 ViT-H vs SAM3 automatic benchmark
# Exact Block53 40m cores + 60m contextual buffers; ROI0-4; ROI5 sealed.
# SAM1 Luki generator: ViT-H, 32x32 internal points, SAM official defaults.
# SAM3 HF mask-generation: 32 points per crop, IoU>=.70, stability>=.80.
# Same 160m contextual imagery, native geometric filtering and matching.
# These are predeclared exploratory settings, NOT parameters tuned during this block.
import gc,json,sys,subprocess,urllib.request
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
from PIL import Image
from rasterio.windows import from_bounds
from rasterio.features import shapes
from shapely.geometry import shape,Polygon,box,mapping
from shapely.validation import make_valid
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D58=OUT/"experiments_v58b_luki_context_sam1_sam3";D58.mkdir(parents=True,exist_ok=True)
assert "roi_v2" in globals() and "reference_v2" in globals()
assert torch.cuda.is_available(),"GPU runtime required"
RIDS58=(0,1,2,3,4) # NEVER ROI5!
CORE58=40.0
BUFFER58=60.0
MIN_AREA58,MAX_AREA58=2.0,2500.0
print("ROI5 is excluded by design. Fixed configurations; no parameter selection in this block.")
with rasterio.open(RASTER) as src:
    CRS58=src.crs
    roi58=roi_v2.to_crs(CRS58)
    reference58=reference_v2.to_crs(CRS58)
nad58=gpd.read_file(AD/"Crown_Nadine.shp")
nad58=nad58.set_crs(CRS58) if nad58.crs is None else nad58.to_crs(CRS58)
nad58=nad58[nad58.geometry.notna()&~nad58.geometry.is_empty].copy()
nad58["geometry"]=nad58.geometry.map(make_valid)
def polygon58(g):
    g=make_valid(g)
    if g.geom_type in ("Polygon","MultiPolygon"):return g
    parts=[x for x in getattr(g,"geoms",[]) if x.geom_type in ("Polygon","MultiPolygon")]
    return unary_union(parts) if parts else Polygon()
def score58(ref,pred):
    n=len(ref);m=len(pred)
    mat=np.zeros((n,m),dtype=np.float32)
    for i,a in enumerate(ref):
        for j,b in enumerate(pred):
            if a.intersects(b):
                c=a.intersection(b).area
                mat[i,j]=c/max(a.area+b.area-c,1e-9)
    pairs=linear_sum_assignment(mat,maximize=True) if mat.size else ([],[])
    matched=mat[pairs] if mat.size else np.array([],dtype="float32")
    return n,m,matched
# Stable footprint: ROI-specific 40m grid, skip cores not intersecting ROI.
cores58={}
for rid in RIDS58:
    polygon=roi58.geometry.iloc[rid]
    x0,y0,x1,y1=polygon.bounds
    cores=[]
    for x in np.arange(x0,x1,CORE58):
        for y in np.arange(y0,y1,CORE58):
            square=box(x,y,x+CORE58,y+CORE58)
            if square.intersects(polygon):cores.append((len(cores),square))
    cores58[rid]=cores
    print(f"ROI{rid}: {len(cores)} cores; {sum(reference58.geometry.apply(polygon.covers))} complete Kristof crowns")
# Release previous large SAM models and switch model by stage to fit L4.
for name in ("pipe56","generator53","sam53","generator51","sam51"):
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache()
def process_model58(model_name,mask_call,rois=RIDS58):
    for rid in rois:
        polygon=roi58.geometry.iloc[rid]
        cache=D58/model_name/f"roi{rid}"
        cache.mkdir(parents=True,exist_ok=True)
        for tileid,core in cores58[rid]:
            dest=cache/f"core_{tileid:03d}.geojson"
            if dest.exists():
                print(f"{model_name} ROI{rid} {tileid+1}/{len(cores58[rid])} cached",flush=True)
                continue
            with rasterio.open(RASTER) as ds:
                cx0,cy0,cx1,cy1=core.bounds
                context=box(cx0-BUFFER58,cy0-BUFFER58,cx1+BUFFER58,cy1+BUFFER58)
                win=from_bounds(*context.bounds,transform=ds.transform).round_offsets().round_lengths()
                arr=ds.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
                transform=ds.window_transform(win)
            if arr.dtype!=np.uint8:arr=np.clip(arr,0,255).astype("uint8")
            if np.all(arr==0,axis=2).mean()>.25 or np.all(arr==255,axis=2).mean()>.25:
                dest.write_text(json.dumps({"type":"FeatureCollection","features":[]}))
                print(f"{model_name} ROI{rid} tile{tileid}: mostly blank",flush=True)
                continue
            proposed=mask_call(arr)
            features=[]
            for mask,quality in proposed:
                mask=np.asarray(mask).squeeze().astype(bool)
                if mask.shape!=arr.shape[:2]:raise ValueError(
                    f"{model_name} mask shape {mask.shape}, expected {arr.shape[:2]}")
                if not mask.any():continue
                geometries=[polygon58(shape(g)) for g,v in
                    shapes(mask.astype("uint8"),mask=mask,transform=transform) if v==1]
                # Luki-style: up to 3 largest connected polygons per mask.
                geometries=sorted((g for g in geometries if not g.is_empty),key=lambda a:a.area,reverse=True)[:3]
                for geom in geometries:
                    if geom.geom_type!="Polygon":continue
                    if not MIN_AREA58<=geom.area<=MAX_AREA58:continue
                    if not core.contains(geom.centroid):continue
                    if not polygon.intersects(geom):continue
                    clipped=polygon58(geom.intersection(polygon))
                    if clipped.is_empty or clipped.area<MIN_AREA58:continue
                    features.append(dict(type="Feature",geometry=mapping(clipped),
                        properties=dict(quality=float(quality))))
            dest.write_text(json.dumps({"type":"FeatureCollection","features":features}))
            print(f"{model_name} ROI{rid} tile{tileid+1}/{len(cores58[rid])}: {len(proposed)} raw -> {len(features)} crowns",flush=True)
            del proposed,arr
        print(f"Completed {model_name} ROI{rid}",flush=True)
# SAM1: original Meta SAM ViT-H as used by the Luki team.
from segment_anything import SamAutomaticMaskGenerator,sam_model_registry
ck58=Path("/content/sam_vit_h_4b8939.pth")
if not ck58.exists() or ck58.stat().st_size<2_000_000_000:
    print("Downloading SAM1 ViT-H weights (2.6 GB) once")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",ck58)
sam58=sam_model_registry["vit_h"](checkpoint=str(ck58)).cuda().eval()
gen58=SamAutomaticMaskGenerator(model=sam58,points_per_side=32,points_per_batch=32)
def sam1_masks58(im):
    return [(r["segmentation"],r["predicted_iou"]) for r in gen58.generate(im)]
process_model58("SAM1_ViTH",sam1_masks58,rois=(1,4))
# Hard reproduction gate: the same author-style F1 as Block 53 (before NMS).
for rid,expected in ((1,.493),(4,.394)):
    region=roi58.geometry.iloc[rid]
    refs=list(reference58[reference58.geometry.apply(region.covers)].geometry)
    polys=[]
    for tid,_ in cores58[rid]:
        path=D58/"SAM1_ViTH"/f"roi{rid}"/f"core_{tid:03d}.geojson"
        for feat in json.loads(path.read_text())["features"]:
            geom=polygon58(shape(feat["geometry"]))
            if not geom.is_empty:polys.append(geom)
    _,m,matched=score58(refs,polys)
    f1=2*int((matched>=.5).sum())/(len(refs)+m)
    print(f"BLOCK53 REPRODUCTION ROI{rid}: new F1={f1:.3f}, original={expected:.3f}")
    if abs(f1-expected)>.015:
        raise RuntimeError(f"Reproduction gate FAILED for ROI{rid}; do not extend or run SAM3. Check processing differences.")
print("Reproduction gate PASS; extending to ROI0, ROI2, ROI3")
process_model58("SAM1_ViTH",sam1_masks58,rois=(0,2,3))
del gen58,sam58;gc.collect();torch.cuda.empty_cache()
# SAM3: actual HF mask-generation; permissive thresholds derived from Block57 PILOT,
# now frozen for pre-test ROIs0-4. Not same model and not a zero-tuning validation.
from transformers import pipeline
pipe58=pipeline("mask-generation",model="facebook/sam3",device=0)
def sam3_masks58(im):
    res=pipe58(Image.fromarray(im),points_per_batch=16,points_per_crop=32,
        pred_iou_thresh=.70,stability_score_thresh=.80)
    r=[]
    for i,m in enumerate(res.get("masks",[])):
        if isinstance(m,Image.Image):m=np.asarray(m)
        elif hasattr(m,"detach"):m=m.detach().cpu().numpy()
        q=res.get("scores",[])
        score=float(q[i]) if i<len(q) else 0.
        r.append((m,score))
    return r
process_model58("SAM3_permissive",sam3_masks58)
del pipe58;gc.collect();torch.cuda.empty_cache()
# Read cached geographic predictions and compare to same COMPLETE references.
results=[];cover=[]
for rid in RIDS58:
    region=roi58.geometry.iloc[rid]
    krefs=reference58[reference58.geometry.apply(region.covers)].reset_index(drop=True)
    nrefs=nad58[nad58.geometry.apply(region.covers)].reset_index(drop=True)
    cover.append(dict(roi=rid,kristof=len(krefs),nadine_sparse=len(nrefs)))
    for method in ("SAM1_ViTH","SAM3_permissive"):
        polygons=[];scores=[]
        for tid,core in cores58[rid]:
            doc=json.loads((D58/method/f"roi{rid}"/f"core_{tid:03d}.geojson").read_text())
            for feat in doc["features"]:
                g=polygon58(shape(feat["geometry"]))
                if g.is_empty:continue
                polygons.append(g);scores.append(float(feat["properties"]["quality"]))
        frame=gpd.GeoDataFrame({"quality":scores},geometry=polygons,crs=CRS58)
        if len(frame):frame.to_file(D58/f"roi{rid}_{method}_crowns.gpkg",driver="GPKG")
        # Luki-style centroid core allocation produces non-overlapping core assignments.
        for refname,ref in (("Kristof",krefs),("Nadine_sparse",nrefs)):
            n,m,matches=score58(list(ref.geometry),polygons)
            for iou_threshold in (.3,.5,.7):
                hits=int((matches>=iou_threshold).sum())
                results.append(dict(roi=rid,model=method,reference_type=refname,
                    iou_threshold=iou_threshold,reference_count=n,predicted_count=m,
                    matches=hits,recall=hits/n if n else np.nan,
                    precision=hits/m if m and refname=="Kristof" else np.nan,
                    f1=2*hits/(n+m) if refname=="Kristof" and n+m else np.nan,
                    median_iou_recovered=float(np.median(matches[matches>=iou_threshold]))
                       if hits else np.nan))
metrics58=pd.DataFrame(results)
metrics58.to_csv(D58/"five_roi_sam1_sam3_comparison.csv",index=False)
pd.DataFrame(cover).to_csv(D58/"reference_coverage.csv",index=False)
print("\nFULL ROI0–4 pre-test — IoU >= 0.5, Kristof whole-ROI F1:")
display(metrics58[(metrics58.reference_type=="Kristof")&(metrics58.iou_threshold==.5)].round(3))
print("\nNadine sparse annotated-crown recall (no precision or F1):")
display(metrics58[(metrics58.reference_type=="Nadine_sparse")&(metrics58.iou_threshold==.5)].round(3))
print("Both models use 40m cores and identical 60m context buffers (160m images).")
print("Block53 reproduction gate validated SAM1 before extension; SAM3 uses same contextual footprint.")
print("ROI5 held out. BLOCK 58 PASS:",D58)


## 59 — Final ROI5 inference, self-contained after Block14

**Corrected:** Run the normal notebook initialization through Block14; then execute this cell. Do **not** rerun the expensive Block58B benchmark. This block defines its own tiling and model-processing functions, keeps the same 40m core / 60m context and fixed model settings, and writes only ROI5 cached GeoJSON tiles. Completed tiles are skipped. Afterward, run Block60 for ecological metrics across all six ROIs.


In [ ]:
# BLOCK 59 — stand-alone final independent ROI5; run ONLY Block14 setup beforehand
# Reproduces Block58B exact spatial settings; no need to execute Block58B itself.
import gc,json,sys,subprocess,urllib.request
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
from PIL import Image
from rasterio.windows import from_bounds
from rasterio.features import shapes
from shapely.geometry import shape,Polygon,box,mapping
from shapely.validation import make_valid
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from IPython.display import display
D58=OUT/"experiments_v58b_luki_context_sam1_sam3";D58.mkdir(parents=True,exist_ok=True)
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block14 (data initialization) first"
assert torch.cuda.is_available(),"GPU runtime required"
RIDS58=(5,) # final locked holdout
CORE58=40.0
BUFFER58=60.0
MIN_AREA58,MAX_AREA58=2.0,2500.0
print("FINAL independent ROI5; all parameters frozen before evaluating holdout.")
with rasterio.open(RASTER) as src:
    CRS58=src.crs
    roi58=roi_v2.to_crs(CRS58)
    reference58=reference_v2.to_crs(CRS58)
def polygon58(g):
    g=make_valid(g)
    if g.geom_type in ("Polygon","MultiPolygon"):return g
    parts=[x for x in getattr(g,"geoms",[]) if x.geom_type in ("Polygon","MultiPolygon")]
    return unary_union(parts) if parts else Polygon()
def score58(ref,pred):
    n=len(ref);m=len(pred)
    mat=np.zeros((n,m),dtype=np.float32)
    for i,a in enumerate(ref):
        for j,b in enumerate(pred):
            if a.intersects(b):
                c=a.intersection(b).area
                mat[i,j]=c/max(a.area+b.area-c,1e-9)
    pairs=linear_sum_assignment(mat,maximize=True) if mat.size else ([],[])
    matched=mat[pairs] if mat.size else np.array([],dtype="float32")
    return n,m,matched
# Stable footprint: ROI-specific 40m grid, skip cores not intersecting ROI.
cores58={}
for rid in RIDS58:
    polygon=roi58.geometry.iloc[rid]
    x0,y0,x1,y1=polygon.bounds
    cores=[]
    for x in np.arange(x0,x1,CORE58):
        for y in np.arange(y0,y1,CORE58):
            square=box(x,y,x+CORE58,y+CORE58)
            if square.intersects(polygon):cores.append((len(cores),square))
    cores58[rid]=cores
    print(f"ROI{rid}: {len(cores)} cores; {sum(reference58.geometry.apply(polygon.covers))} complete Kristof crowns")
# Release previous large SAM models and switch model by stage to fit L4.
for name in ("pipe56","generator53","sam53","generator51","sam51"):
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache()
def process_model58(model_name,mask_call,rois=RIDS58):
    for rid in rois:
        polygon=roi58.geometry.iloc[rid]
        cache=D58/model_name/f"roi{rid}"
        cache.mkdir(parents=True,exist_ok=True)
        for tileid,core in cores58[rid]:
            dest=cache/f"core_{tileid:03d}.geojson"
            if dest.exists():
                print(f"{model_name} ROI{rid} {tileid+1}/{len(cores58[rid])} cached",flush=True)
                continue
            with rasterio.open(RASTER) as ds:
                cx0,cy0,cx1,cy1=core.bounds
                context=box(cx0-BUFFER58,cy0-BUFFER58,cx1+BUFFER58,cy1+BUFFER58)
                win=from_bounds(*context.bounds,transform=ds.transform).round_offsets().round_lengths()
                arr=ds.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
                transform=ds.window_transform(win)
            if arr.dtype!=np.uint8:arr=np.clip(arr,0,255).astype("uint8")
            if np.all(arr==0,axis=2).mean()>.25 or np.all(arr==255,axis=2).mean()>.25:
                dest.write_text(json.dumps({"type":"FeatureCollection","features":[]}))
                print(f"{model_name} ROI{rid} tile{tileid}: mostly blank",flush=True)
                continue
            proposed=mask_call(arr)
            features=[]
            for mask,quality in proposed:
                mask=np.asarray(mask).squeeze().astype(bool)
                if mask.shape!=arr.shape[:2]:raise ValueError(
                    f"{model_name} mask shape {mask.shape}, expected {arr.shape[:2]}")
                if not mask.any():continue
                geometries=[polygon58(shape(g)) for g,v in
                    shapes(mask.astype("uint8"),mask=mask,transform=transform) if v==1]
                # Luki-style: up to 3 largest connected polygons per mask.
                geometries=sorted((g for g in geometries if not g.is_empty),key=lambda a:a.area,reverse=True)[:3]
                for geom in geometries:
                    if geom.geom_type!="Polygon":continue
                    if not MIN_AREA58<=geom.area<=MAX_AREA58:continue
                    if not core.contains(geom.centroid):continue
                    if not polygon.intersects(geom):continue
                    clipped=polygon58(geom.intersection(polygon))
                    if clipped.is_empty or clipped.area<MIN_AREA58:continue
                    features.append(dict(type="Feature",geometry=mapping(clipped),
                        properties=dict(quality=float(quality))))
            dest.write_text(json.dumps({"type":"FeatureCollection","features":features}))
            print(f"{model_name} ROI{rid} tile{tileid+1}/{len(cores58[rid])}: {len(proposed)} raw -> {len(features)} crowns",flush=True)
            del proposed,arr
        print(f"Completed {model_name} ROI{rid}",flush=True)
# Skip model initialization entirely when all tile caches already exist.
def _missing_roi5(model):
    folder=D58/model/"roi5"
    return [tileid for tileid,_ in cores58[5] if not (folder/f"core_{tileid:03d}.geojson").exists()]
if _missing_roi5("SAM1_ViTH"):
    # Original SAM1 ViT-H; exact Block53 settings
    try:
        from segment_anything import SamAutomaticMaskGenerator,sam_model_registry
    except ModuleNotFoundError as e:
        if e.name != "segment_anything":
            raise
        print("Installing original Meta segment-anything package for this fresh Colab runtime...")
        subprocess.run([sys.executable,"-m","pip","install","-q",
                        "git+https://github.com/facebookresearch/segment-anything.git"],check=True)
        from segment_anything import SamAutomaticMaskGenerator,sam_model_registry
    import urllib.request
    ck=Path("/content/sam_vit_h_4b8939.pth")
    if not ck.exists() or ck.stat().st_size<2_000_000_000:
        urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",ck)
    sam=sam_model_registry["vit_h"](checkpoint=str(ck)).cuda().eval()
    gen=SamAutomaticMaskGenerator(model=sam,points_per_side=32,points_per_batch=32)
    def _sam1_roi5(image):
        return [(m["segmentation"],m["predicted_iou"]) for m in gen.generate(image)]
    process_model58("SAM1_ViTH",_sam1_roi5,rois=(5,))
    del sam,gen;gc.collect();torch.cuda.empty_cache()
else:
    print("SAM1 ROI5 all 18 cores cached; skipping SAM1 checkpoint and inference.")
# Same 160m imagery extent, fixed SAM3 exploratory thresholds from Block57/58B
if _missing_roi5("SAM3_permissive"):
    from google.colab import userdata
    from huggingface_hub import hf_hub_download, whoami
    # Use the existing Colab secret explicitly, rather than a potentially stale HF login.
    try:
        token59=userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("Cannot read Colab secret HF_TOKEN. Enable notebook access in Colab Secrets (key icon).") from e
    if not token59:
        raise RuntimeError("Colab HF_TOKEN secret is empty or unavailable. Enable notebook access in Secrets.")
    try:
        account59=whoami(token=token59)
        hf_hub_download("facebook/sam3","config.json",token=token59)
    except Exception as e:
        raise RuntimeError("HF_TOKEN was read, but does not grant access to facebook/sam3. Check the account and gated-model approval. No token value was printed.") from e
    print("Hugging Face SAM3 access verified for:",account59["name"])
    from transformers import pipeline
    from PIL import Image
    pipe=pipeline("mask-generation",model="facebook/sam3",device=0,token=token59)
    def _sam3_roi5(image):
        out=pipe(Image.fromarray(image),points_per_batch=16,points_per_crop=32,
                 pred_iou_thresh=.70,stability_score_thresh=.80)
        scores=out.get("scores",[])
        ret=[]
        for i,m in enumerate(out.get("masks",[])):
            if isinstance(m,Image.Image):m=np.asarray(m)
            elif hasattr(m,"detach"):m=m.detach().cpu().numpy()
            ret.append((m,float(scores[i]) if i<len(scores) else 0.))
        return ret
    process_model58("SAM3_permissive",_sam3_roi5,rois=(5,))
    del pipe;gc.collect();torch.cuda.empty_cache()
else:
    print("SAM3 ROI5 already cached; skipping SAM3 checkpoint and inference.")
print("BLOCK59 ROI5 complete; cached outputs ready for Block60.")


## 60 — Ecologically meaningful crown metrics, ROIs 0–5

CPU-only reporting from cached masks. This calculates one-to-one object matching at IoU 0.3/0.5/0.7; crown counts and densities; mean/median and P10/P90 crown projected area (CPA); sum of projected crown areas; polygon-union canopy cover and overlap; centroid error; matched-crown CPA MAE/RMSE/area ratios; and Nadine annotated-crown recall and conditional IoU. **Counted polygons are visible crown objects, not all field stems.** Full-stand statistics use Kristof; Nadine is sparse and never supplies conventional precision or F1. Stand age requires separately verified plot-age metadata.

In [ ]:
# BLOCK 60 — Ecological validation for ALL SIX ROIs, cached geographic polygons only.
# Can be run in a CPU session after Block59; it does not use SAM or GPU.
import json
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from shapely.geometry import shape,box
from shapely.ops import unary_union
from shapely.validation import make_valid
from scipy.optimize import linear_sum_assignment
from IPython.display import display
assert "OUT" in globals() and "AD" in globals() and "R" not in ()
D60=OUT/"experiments_v60_ecological_validation_all6"
D60.mkdir(parents=True,exist_ok=True)
CACHED=OUT/"experiments_v58b_luki_context_sam1_sam3"
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run data initialization (Block14) first"
with rasterio.open(RASTER) as ds: crs=ds.crs
roi=roi_v2.to_crs(crs); refs_all=reference_v2.to_crs(crs)
nad=gpd.read_file(AD/"Crown_Nadine.shp")
nad=nad.set_crs(crs) if nad.crs is None else nad.to_crs(crs)
nad=nad[nad.geometry.notna() & ~nad.geometry.is_empty].copy()
def polygon(g):
    g=make_valid(g)
    if g.geom_type in ("Polygon","MultiPolygon"):return g
    items=[p for p in getattr(g,"geoms",[]) if p.geom_type in ("Polygon","MultiPolygon")]
    return unary_union(items) if items else None
nad["geometry"]=nad.geometry.map(polygon)
nad=nad[nad.geometry.notna()&~nad.geometry.is_empty].copy()
def cached(rid,model):
    folder=CACHED/model/f"roi{rid}"
    region=roi.geometry.iloc[rid]
    x0,y0,x1,y1=region.bounds
    squares=[box(x,y,x+40,y+40) for x in np.arange(x0,x1,40) for y in np.arange(y0,y1,40) if box(x,y,x+40,y+40).intersects(region)]
    files=[folder/f"core_{i:03d}.geojson" for i in range(len(squares))]
    absent=[str(p) for p in files if not p.exists()]
    if absent:raise FileNotFoundError(f"Missing {len(absent)} tile(s) for ROI{rid} {model}; finish Block59/58B first: {absent[:3]}")
    rows=[]
    for tile,p in enumerate(files):
        for f in json.loads(p.read_text())["features"]:
            geom=polygon(shape(f["geometry"]))
            if geom is not None and not geom.is_empty and geom.area>0:
                rows.append(dict(tile=tile,quality=f.get("properties",{}).get("quality",np.nan),geometry=geom))
    return gpd.GeoDataFrame(rows,geometry="geometry",crs=crs)
def match(a,b):
    matrix=np.zeros((len(a),len(b)),dtype=np.float32)
    for i,x in enumerate(a):
        for j,y in enumerate(b):
            if x.intersects(y):
                area=x.intersection(y).area
                matrix[i,j]=area/max(x.area+y.area-area,1e-9)
    if not matrix.size:return [],matrix
    ii,jj=linear_sum_assignment(matrix,maximize=True)
    return [(int(i),int(j),float(matrix[i,j])) for i,j in zip(ii,jj) if matrix[i,j]>0],matrix
def safe_mean(x):
    return float(np.mean(x)) if len(x) else np.nan
def quant(x,q):
    return float(np.quantile(x,q)) if len(x) else np.nan
def cover_stats(geoms,region):
    if not len(geoms):return dict(sum_m2=0.,union_m2=0.,overlap_m2=0.,cover_pct=0.)
    union=unary_union(list(geoms))
    area=float(sum(x.area for x in geoms))
    unique=float(union.intersection(region).area)
    return dict(sum_m2=area,union_m2=unique,overlap_m2=max(area-float(union.area),0.),
                cover_pct=100*unique/region.area)
summary=[];details=[];nadine=[];age_note="ROI age labels must be joined from independently verified metadata; index is NOT stand age."
for rid in range(6):
    region=roi.geometry.iloc[rid]
    area_ha=region.area/10000.
    kref=refs_all[refs_all.geometry.apply(region.covers)].reset_index(drop=True)
    nref=nad[nad.geometry.apply(region.covers)].reset_index(drop=True)
    a=list(kref.geometry)
    kr=cover_stats(a,region)
    for model in ("SAM1_ViTH","SAM3_permissive"):
        pred=cached(rid,model)
        b=list(pred.geometry)
        out=D60/f"roi{rid}_{model}_all_crowns.gpkg"
        if len(pred):pred.to_file(out,driver="GPKG")
        p=cover_stats(b,region)
        associations,matrix=match(a,b)
        # Thresholded one-to-one object matching, identical to Block58B.
        for threshold in (.3,.5,.7):
            chosen=[(i,j,v) for i,j,v in associations if v>=threshold]
            tp=len(chosen);nr=len(a);npred=len(b)
            ratios=np.array([b[j].area/a[i].area for i,j,_ in chosen],dtype=float)
            centroid=np.array([a[i].centroid.distance(b[j].centroid) for i,j,_ in chosen])
            relerr=np.array([100*(b[j].area-a[i].area)/a[i].area for i,j,_ in chosen])
            row=dict(roi=rid,model=model,iou_threshold=threshold,
                roi_area_ha=area_ha,reference_crowns=nr,predicted_crowns=npred,matched=tp,
                precision=tp/npred if npred else np.nan,
                recall=tp/nr if nr else np.nan,
                f1=2*tp/(nr+npred) if nr+npred else np.nan,
                count_bias_pct=100*(npred-nr)/nr if nr else np.nan,
                crown_density_ref_ha=nr/area_ha,crown_density_pred_ha=npred/area_ha,
                density_bias_ha=(npred-nr)/area_ha,
                mean_cpa_ref_m2=safe_mean([g.area for g in a]),
                mean_cpa_pred_m2=safe_mean([g.area for g in b]),
                median_cpa_ref_m2=quant([g.area for g in a],.5),
                median_cpa_pred_m2=quant([g.area for g in b],.5),
                p10_cpa_ref_m2=quant([g.area for g in a],.1),
                p10_cpa_pred_m2=quant([g.area for g in b],.1),
                p90_cpa_ref_m2=quant([g.area for g in a],.9),
                p90_cpa_pred_m2=quant([g.area for g in b],.9),
                sum_cpa_ref_m2=kr["sum_m2"],sum_cpa_pred_m2=p["sum_m2"],
                sum_cpa_bias_pct=100*(p["sum_m2"]-kr["sum_m2"])/kr["sum_m2"] if kr["sum_m2"] else np.nan,
                canopy_cover_ref_pct=kr["cover_pct"],canopy_cover_pred_pct=p["cover_pct"],
                canopy_cover_error_pp=p["cover_pct"]-kr["cover_pct"],
                canopy_cover_bias_pct=100*(p["union_m2"]-kr["union_m2"])/kr["union_m2"] if kr["union_m2"] else np.nan,
                crown_overlap_ref_m2=kr["overlap_m2"],crown_overlap_pred_m2=p["overlap_m2"],
                median_matched_iou=quant([v for _,_,v in chosen],.5),
                mean_matched_area_ratio=safe_mean(ratios),
                median_matched_area_ratio=quant(ratios,.5),
                mean_matched_area_relative_error_pct=safe_mean(relerr),
                median_matched_area_absolute_percent_error=quant(np.abs(relerr),.5),
                matched_cpa_mae_m2=safe_mean([abs(b[j].area-a[i].area) for i,j,_ in chosen]),
                matched_cpa_rmse_m2=float(np.sqrt(np.mean([(b[j].area-a[i].area)**2 for i,j,_ in chosen]))) if chosen else np.nan,
                median_matched_centroid_error_m=quant(centroid,.5))
            summary.append(row)
            for i,j,v in chosen:
                details.append(dict(roi=rid,model=model,iou_threshold=threshold,
                    kristof_index=i,prediction_index=j,iou=v,
                    reference_cpa_m2=a[i].area,predicted_cpa_m2=b[j].area,
                    area_ratio=b[j].area/a[i].area,
                    centroid_distance_m=a[i].centroid.distance(b[j].centroid)))
        # Sparse Nadine reference: positive crown recovery + conditional shape only;
        # nonmatching predictions are NOT false positives.
        assigned,_=match(list(nref.geometry),b)
        for threshold in (.3,.5,.7):
            positive=[(i,j,v) for i,j,v in assigned if v>=threshold]
            nadine.append(dict(roi=rid,model=model,iou_threshold=threshold,
                annotated_crowns=len(nref),recovered=len(positive),
                annotated_crown_recall=len(positive)/len(nref) if len(nref) else np.nan,
                median_iou_recovered=quant([v for _,_,v in positive],.5),
                median_pred_to_nadine_area_ratio=quant([b[j].area/nref.geometry.iloc[i].area for i,j,v in positive],.5)))
stats=pd.DataFrame(summary);pairs=pd.DataFrame(details);sparse=pd.DataFrame(nadine)
stats.to_csv(D60/"all6_ecological_stand_metrics.csv",index=False)
pairs.to_csv(D60/"all6_matched_crown_geometry.csv",index=False)
sparse.to_csv(D60/"all6_nadine_sparse_recall.csv",index=False)
sel=stats[stats.iou_threshold==.5].copy()
display(sel[["roi","model","reference_crowns","predicted_crowns","f1","count_bias_pct","mean_cpa_ref_m2","mean_cpa_pred_m2","sum_cpa_bias_pct","canopy_cover_error_pp","median_matched_area_ratio"]].round(3))
pooled=[]
for model,g in sel.groupby("model"):
    tp=int(g.matched.sum());nr=int(g.reference_crowns.sum());pp=int(g.predicted_crowns.sum())
    pooled.append(dict(model=model,reference_crowns=nr,predictions=pp,matches=tp,
        micro_f1=2*tp/(nr+pp),count_bias_pct=100*(pp-nr)/nr,
        summed_cpa_bias_pct=100*(g.sum_cpa_pred_m2.sum()-g.sum_cpa_ref_m2.sum())/g.sum_cpa_ref_m2.sum(),
        area_weighted_cover_error_pp=np.average(g.canopy_cover_error_pp,weights=g.roi_area_ha)))
pd.DataFrame(pooled).to_csv(D60/"all6_pooled_summary.csv",index=False)
display(pd.DataFrame(pooled).round(3))
print(age_note)
print("Crown polygons count visible delineated canopy objects, NOT all inventoried stems.")
print("Matched-crown area errors are conditional on IoU threshold; stand-level area errors include ALL predictions.")
print("BLOCK60 PASS:",D60)

## 61 — Six-ROI structural error decomposition (no GPU)

Reads the same cached geographic predictions and references as Block60. Run **Block60 first**, then this cell; no SAM inference or Hugging Face login required. Produces exact signed decomposition of *summed crown projected area* bias into (i) matched crown area differences, (ii) unmatched predicted polygon area and (iii) omitted reference polygon area. Separately reports the **non-additive** area of polygon self-overlap, spatial canopy-footprint agreement (union IoU, extra and missing footprint), duplicate candidates and potential splits/merges. Includes a **descriptive** fixed-threshold geometric NMS sensitivity table and a matched/unmatched spatial overlay per ROI. The same candidate-level summary is saved as CSV.

**Scientific safeguards:** one-to-one assignment and object IoU >=0.5; split/merge diagnostics are heuristic, not confirmed tree identities; annotations describe visible crowns rather than all stems; ROI5 is evaluated but not used to select filters or thresholds. No metrics should be presented as observed until Colab executes the cell.

In [ ]:
# BLOCK 61 — Six-ROI structural error decomposition, CPU ONLY.
# Prerequisite: execute Block60 (loads 'cached', 'roi', 'refs_all', 'crs').
# ROI5 is reporting-only; do NOT select NMS/model thresholds from ROI5.
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from shapely.strtree import STRtree
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from IPython.display import display
assert all(k in globals() for k in ("cached","roi","refs_all","crs","D60")), "Run Block60 first (CPU only)."
D61=OUT/"experiments_v61_structural_error_all6"
D61.mkdir(parents=True,exist_ok=True)
IOU61=.5                       # fixed original object criterion
NEIGHBOR_FRACTION61=.20         # diagnostic connection, NOT object matching
DUP_IOU61=.50                   # fixed overlap diagnostic, not tuning
MODELS61=("SAM1_ViTH","SAM3_permissive")
def _intersection_matrix(a,b):
    mat=np.zeros((len(a),len(b)),dtype=np.float64)
    if not a or not b:return mat
    tree=STRtree(b)
    for i,g in enumerate(a):
        for j in tree.query(g,predicate="intersects"):
            mat[i,int(j)]=g.intersection(b[int(j)]).area
    return mat
def _match61(inter,aa,bb):
    union=aa[:,None]+bb[None,:]-inter
    iou=np.divide(inter,union,out=np.zeros_like(inter),where=union>0)
    if not iou.size:return [],iou
    ii,jj=linear_sum_assignment(iou,maximize=True)
    return [(int(i),int(j),float(iou[i,j])) for i,j in zip(ii,jj) if iou[i,j]>=IOU61],iou
def _union61(geoms):
    return unary_union(geoms) if geoms else None
def _area61(g):
    return float(g.area) if g is not None else 0.
def _nms61(geoms,scores,threshold):
    # Geometric NMS diagnostic ONLY; fixed order (confidence, area, original id).
    order=sorted(range(len(geoms)),key=lambda j:(-scores[j],-geoms[j].area,j))
    selected=[]
    for j in order:
        if all(geoms[j].intersection(geoms[k]).area /
               max(geoms[j].area+geoms[k].area-geoms[j].intersection(geoms[k]).area,1e-9)
               <threshold for k in selected):
            selected.append(j)
    return selected
stand=[]; crowns=[]; sensitivity=[]
for rid in range(6):
    region=roi.geometry.iloc[rid]
    ref=list(refs_all[refs_all.geometry.apply(region.covers)].geometry)
    ar=np.array([g.area for g in ref],dtype=float)
    ref_union=_union61(ref)
    for model in MODELS61:
        pred=cached(rid,model)
        cand=list(pred.geometry)
        ap=np.array([g.area for g in cand],dtype=float)
        conf=pd.to_numeric(pred["quality"],errors="coerce").to_numpy(dtype=float) if len(pred) else np.array([])
        conf=np.nan_to_num(conf,nan=-1.,posinf=-1.,neginf=-1.)
        intr=_intersection_matrix(ref,cand)
        paired,iou=_match61(intr,ar,ap)
        ri={i for i,_,_ in paired};pj={j for _,j,_ in paired}
        unmatched_ref=[i for i in range(len(ref)) if i not in ri]
        unmatched_pred=[j for j in range(len(cand)) if j not in pj]
        # EXACT identity (additive, signed). Sum of predicted area - sum of reference area:
        matched_delta=sum(ap[j]-ar[i] for i,j,_ in paired)
        extra_area=float(sum(ap[j] for j in unmatched_pred))
        omitted_area=float(sum(ar[i] for i in unmatched_ref))
        sum_delta=float(ap.sum()-ar.sum())
        assert np.isclose(sum_delta,matched_delta+extra_area-omitted_area,atol=1e-6)
        p_union=_union61(cand)
        intersect_area=_area61(p_union.intersection(ref_union)) if p_union is not None and ref_union is not None else 0.
        union_area=_area61(p_union.union(ref_union)) if p_union is not None and ref_union is not None else (_area61(p_union)+_area61(ref_union))
        ref_union_area=_area61(ref_union);pred_union_area=_area61(p_union)
        cover_iou=intersect_area/union_area if union_area else np.nan
        # Spatial canopy disagreement is also an exact union-area decomposition.
        cover_extra=pred_union_area-intersect_area
        cover_missing=ref_union_area-intersect_area
        assert np.isclose(pred_union_area-ref_union_area,cover_extra-cover_missing,atol=1e-6)
        # Crowns intersecting multiple other crowns; overlaps >20% of EACH reference's own area
        # (candidate-side analogue uses each candidate's own area).
        links_ref=(intr/np.maximum(ar[:,None],1e-9)>=NEIGHBOR_FRACTION61) if intr.size else np.zeros_like(intr,dtype=bool)
        links_pred=(intr/np.maximum(ap[None,:],1e-9)>=NEIGHBOR_FRACTION61) if intr.size else np.zeros_like(intr,dtype=bool)
        possible_splits=int(np.sum(links_ref.sum(axis=1)>=2))
        possible_merges=int(np.sum(links_pred.sum(axis=0)>=2))
        # Exact spatial duplication of predicted polygons: summed CPA minus union CPA.
        self_overlap=float(ap.sum()-pred_union_area)
        dup_pairs=0;dup_indices=set()
        if len(cand)>1:
            self_intr=_intersection_matrix(cand,cand)
            for i in range(len(cand)):
                for j in range(i+1,len(cand)):
                    intersection=self_intr[i,j]
                    denom=ap[i]+ap[j]-intersection
                    if denom>0 and intersection/denom>=DUP_IOU61:
                        dup_pairs+=1;dup_indices.update((i,j))
        # Unmatched candidate area is NOT automatically false canopy.
        # Distinguish area overlapping annotated reference canopy from area outside it.
        unmatch_union=_union61([cand[j] for j in unmatched_pred])
        unmatch_inside=_area61(unmatch_union.intersection(ref_union)) if unmatch_union is not None and ref_union is not None else 0.
        unmatch_outside=_area61(unmatch_union)-unmatch_inside
        stand.append(dict(roi=rid,model=model,reference_n=len(ref),predicted_n=len(cand),
            matched_n=len(paired),f1=2*len(paired)/(len(ref)+len(cand)),
            sum_ref_m2=float(ar.sum()),sum_pred_m2=float(ap.sum()),
            sum_area_bias_m2=sum_delta,
            matched_boundary_area_delta_m2=matched_delta,
            unmatched_prediction_area_m2=extra_area,
            missed_reference_area_m2=omitted_area,
            signed_area_decomposition_residual_m2=sum_delta-(matched_delta+extra_area-omitted_area),
            pred_self_overlap_m2=self_overlap,
            ref_self_overlap_m2=float(ar.sum()-ref_union_area),
            pred_union_m2=pred_union_area,ref_union_m2=ref_union_area,
            canopy_intersection_m2=intersect_area,canopy_union_iou=cover_iou,
            extra_canopy_footprint_m2=cover_extra,missed_canopy_footprint_m2=cover_missing,
            net_canopy_footprint_bias_m2=cover_extra-cover_missing,
            pred_duplicate_pairs_iou_ge_05=dup_pairs,pred_polygons_in_duplicate_pairs=len(dup_indices),
            reference_potential_splits=possible_splits,prediction_potential_merges=possible_merges,
            unmatched_pred_union_inside_reference_m2=unmatch_inside,
            unmatched_pred_union_outside_reference_m2=unmatch_outside,
            matched_mean_cpa_bias_m2=float(np.mean([ap[j]-ar[i] for i,j,_ in paired])) if paired else np.nan,
            matched_area_weighted_ratio=(sum(ap[j] for i,j,_ in paired)/sum(ar[i] for i,j,_ in paired)) if paired else np.nan,
            median_pred_cpa_m2=float(np.median(ap)) if len(ap) else np.nan,
            p90_pred_cpa_m2=float(np.percentile(ap,90)) if len(ap) else np.nan))
        for j,g in enumerate(cand):
            relevant=np.flatnonzero(intr[:,j]>0) if intr.size else []
            best=float(iou[:,j].max()) if iou.size and len(ref) else 0.
            ismatched=j in pj
            crowns.append(dict(roi=rid,model=model,prediction_id=j,
                cpa_m2=ap[j],quality=conf[j],matched_iou_ge_05=ismatched,
                best_kristof_iou=best,overlapping_reference_count=len(relevant),
                count_reference_fraction_ge_20=int(links_pred[:,j].sum()) if intr.size else 0,
                part_of_duplicate_pair=j in dup_indices))
        # Descriptive sensitivity ONLY, NO ranking or threshold selection from ROI5.
        for threshold in (.3,.5,.7):
            ids=_nms61(cand,conf,threshold)
            kept=[cand[j] for j in ids]
            inter2=_intersection_matrix(ref,kept)
            matched2,_=_match61(inter2,ar,np.array([g.area for g in kept]))
            cover2=_union61(kept)
            sensitivity.append(dict(roi=rid,model=model,nms_iou=threshold,
                candidates_before=len(cand),candidates_after=len(kept),
                f1_after=2*len(matched2)/(len(ref)+len(kept)) if len(ref)+len(kept) else np.nan,
                count_bias_pct=100*(len(kept)-len(ref))/len(ref) if ref else np.nan,
                summed_area_bias_pct=100*(sum(g.area for g in kept)-sum(ar))/sum(ar) if sum(ar) else np.nan,
                canopy_cover_error_pp=100*(_area61(cover2)-ref_union_area)/region.area))
    print("ROI",rid,"complete (both models)",flush=True)
df61=pd.DataFrame(stand);crown61=pd.DataFrame(crowns);nms61=pd.DataFrame(sensitivity)
df61.to_csv(D61/"structural_error_decomposition_all6.csv",index=False)
crown61.to_csv(D61/"candidate_diagnostics_all6.csv",index=False)
nms61.to_csv(D61/"nms_sensitivity_descriptive_all6.csv",index=False)
cols=["roi","model","f1","sum_area_bias_m2","matched_boundary_area_delta_m2",
      "unmatched_prediction_area_m2","missed_reference_area_m2",
      "pred_self_overlap_m2","canopy_union_iou",
      "reference_potential_splits","prediction_potential_merges","pred_duplicate_pairs_iou_ge_05"]
display(df61[cols].round(3))
display(nms61[["roi","model","nms_iou","f1_after","count_bias_pct","summed_area_bias_pct","canopy_cover_error_pp"]].round(3))
# Publication-oriented maps: two panels per ROI, both models, same ROI and reference outlines.
# Color coding: dark lines = Kristof; green = matched prediction; orange = unmatched prediction.
for rid in range(6):
    region=roi.geometry.iloc[rid]
    ref=list(refs_all[refs_all.geometry.apply(region.covers)].geometry)
    fig,axs=plt.subplots(1,2,figsize=(13,6),sharex=True,sharey=True)
    for ax,model in zip(axs,MODELS61):
        pred=cached(rid,model);cand=list(pred.geometry)
        intr=_intersection_matrix(ref,cand)
        matched,_=_match61(intr,np.array([g.area for g in ref]),np.array([g.area for g in cand]))
        accepted={j for _,j,_ in matched}
        for g in ref:
            for p in ([g] if g.geom_type=="Polygon" else getattr(g,"geoms",[])):
                if p.geom_type=="Polygon":
                    xs,ys=p.exterior.xy
                    ax.plot(xs,ys,color="black",lw=.55,alpha=.7)
        for j,g in enumerate(cand):
            color="forestgreen" if j in accepted else "darkorange"
            for p in ([g] if g.geom_type=="Polygon" else getattr(g,"geoms",[])):
                if p.geom_type=="Polygon":
                    xs,ys=p.exterior.xy
                    ax.plot(xs,ys,color=color,lw=.75,alpha=.8)
        ax.set_title(f"ROI {rid} — {model}\nblack ref | green matched | orange unmatched")
        ax.set_aspect("equal",adjustable="box")
        ax.tick_params(labelsize=7)
    fig.tight_layout()
    fig.savefig(D61/f"roi{rid}_diagnostic_overlay.png",dpi=190)
    plt.close(fig)
print("BLOCK61 PASS:",D61)
print("Interpretation: split/merge counts are potential one-to-many overlaps, not verified tree identities.")
print("Do not add matched boundary delta, self-overlap or unmatched canopy footprint as independent additive terms.")
print("ROI5 diagnostics and NMS sensitivity must not be used for parameter tuning; select settings on ROIs 0–4 only.")

## 61B — Plot-only recovery after Block61

Run this cell **directly** in the current Colab session if Block61 calculated its tables but stopped during figure creation. It uses the already-loaded cached polygons and writes the six diagnostic overlays only. No need to rerun model inference or Block61 statistics.

In [ ]:
# BLOCK 61B — Rerun only spatial maps after a successful Block61 numerical stage.
# No inference, no recomputation of six-ROI CSV tables; execute in SAME Colab session.
assert all(k in globals() for k in ("cached","roi","refs_all","_intersection_matrix","_match61","MODELS61","D61")), "Run Block60 and the Block61 numerical stage first."
import numpy as np
import matplotlib.pyplot as plt
# Publication-oriented maps: two panels per ROI, both models, same ROI and reference outlines.
# Color coding: dark lines = Kristof; green = matched prediction; orange = unmatched prediction.
for rid in range(6):
    region=roi.geometry.iloc[rid]
    ref=list(refs_all[refs_all.geometry.apply(region.covers)].geometry)
    fig,axs=plt.subplots(1,2,figsize=(13,6),sharex=True,sharey=True)
    for ax,model in zip(axs,MODELS61):
        pred=cached(rid,model);cand=list(pred.geometry)
        intr=_intersection_matrix(ref,cand)
        matched,_=_match61(intr,np.array([g.area for g in ref]),np.array([g.area for g in cand]))
        accepted={j for _,j,_ in matched}
        for g in ref:
            for p in ([g] if g.geom_type=="Polygon" else getattr(g,"geoms",[])):
                if p.geom_type=="Polygon":
                    xs,ys=p.exterior.xy
                    ax.plot(xs,ys,color="black",lw=.55,alpha=.7)
        for j,g in enumerate(cand):
            color="forestgreen" if j in accepted else "darkorange"
            for p in ([g] if g.geom_type=="Polygon" else getattr(g,"geoms",[])):
                if p.geom_type=="Polygon":
                    xs,ys=p.exterior.xy
                    ax.plot(xs,ys,color=color,lw=.75,alpha=.8)
        ax.set_title(f"ROI {rid} — {model}\nblack ref | green matched | orange unmatched")
        ax.set_aspect("equal",adjustable="box")
        ax.tick_params(labelsize=7)
    fig.tight_layout()
    fig.savefig(D61/f"roi{rid}_diagnostic_overlay.png",dpi=190)
    plt.close(fig)
print("BLOCK61 PASS:",D61)
print("Interpretation: split/merge counts are potential one-to-many overlaps, not verified tree identities.")
print("Do not add matched boundary delta, self-overlap or unmatched canopy footprint as independent additive terms.")
print("ROI5 diagnostics and NMS sensitivity must not be used for parameter tuning; select settings on ROIs 0–4 only.")